In [ ]:
# ====================================================================================================================
# MT2KD — OFFICIAL test-clean WER OF THE DEPLOYED WEB APP (Zipformer-S + the app's own LM decoder), ONE CELL, CPU runtime, ~35 min
#   Reads: MT2KD/deploy/mt2kd_S20_main/ (the exported app folder), LibriSpeech test-clean on Drive.
#   Writes: NEW folder MT2KD/results/LM_app_decoder_test_clean/ + adds "test_clean" to deploy/mt2kd_S20_main/lm/lm_config.json.
#   STEPS 1 Drive | 2 load app | 3 test-clean posteriors + greedy | 4 LM decoding | 5 save
# ====================================================================================================================
import os, sys, json, math, time, shutil, importlib
from pathlib import Path
import numpy as np, pandas as pd

# ####################################################################################################################
# ====  PART 1: the verified modules (plain Python source, written to disk and imported)  ====
# ####################################################################################################################
_MOD_DIR = "/content/mt2kd_modules"
_MODULES = {}

# --------------------------------------------------------------------------------------------------------------
# module mt2kd_mod_asr  (verified earlier cell, unchanged; written to {_MOD_DIR}/mt2kd_mod_asr.py and imported)
# --------------------------------------------------------------------------------------------------------------
_MODULES["mt2kd_mod_asr"] = r'''# ====================================================================================================================
# MT2KD ENHANCEMENT, STEP 1 (ASR half) — WavLM-Large -> Zipformer-M feature KD + CTC teacher comparison — SINGLE CELL, end to end, resumable
#
# WHAT THIS IS: the same pipeline as your verified ASR v4 cell (sanity_2h: 2 h of train-clean-100, ScaledAdam + Eden, same student, same
#   checks), with ONE change: the teacher is WavLM-Large instead of Whisper-large-v3. WavLM-Large is the teacher planned to replace BOTH
#   Whisper (ASR) and ECAPA-TDNN (speaker verification) in our enhancement; this cell tests the ASR half.
#   Then a CTC PROBE answers the real question: freeze each student, train the same small BiLSTM-CTC on top (10 h with transcripts),
#   and measure WER / CER on dev-clean for:  WavLM-taught student | Whisper-taught student (your v4 best.pt) | random-init student (control).
#
#   teacher : microsoft/wavlm-large (315M, frozen, fp16), layer 24 (configurable), 1024-D @ 50 Hz -> pairs (2j, 2j+1) -> 2048-D @ 25 Hz
#             raw waveform input, whole utterance (no 30 s window, no chunking, no padding)
#   student : Zipformer-M (icefall), 80-D fbank, 512-D @ 25 Hz, Linear 512 -> 2048 (as in v4, only the projection size changes)
#   loss    : masked L1 on time-aligned frames (alignment MEASURED from the WavLM and Zipformer code, as in v4)
#   GPU     : T4 works (fp16 + GradScaler, ~3-4 h end to end, resumable); A100 ~1.5 h. Peak GPU memory stays well below 16 GB.
# STORAGE: Drive gets ONLY 5 files per folder (best.pt, latest.pt, history.csv, final_report.json, config.json):
#            /content/drive/MyDrive/MT2KD/results/WavLM_ASR_Stage1_sanity_2h/     (KD training)
#            /content/drive/MyDrive/MT2KD/results/ASR_teacher_comparison_ctc/     (probe: final_report.json, history.csv, config.json)
#          Your Whisper run folder ASR_Stage1_v4_sanity_2h is only READ. Everything else lives in /content (rebuilt automatically).
# RESUME:  re-run this cell after ANY interruption (finished parts are skipped / resumed from the last checkpoint).
# ====================================================================================================================
import os
os.environ.setdefault("HF_HOME", "/content/hf_cache")      # local disk (must be set before transformers is imported)

# ####################################################################################################
# ---- asr1_common ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 (WavLM-Large -> Zipformer-M) | shared configuration, paths, helpers, audit of existing work
# Every step of this pipeline is resumable: re-running a cell after a Colab disconnect continues where it stopped.
# ===========================================================================================
import os, sys, json, math, time, shutil, hashlib, subprocess, importlib.util
from pathlib import Path

CODE_VERSION = "asr1-wavlm-v1"              # bumped whenever a change affects features, alignment or training semantics

CFG = dict(
    # ---- READ-ONLY inputs on Drive (nothing is ever written below /content/drive except by class DriveStore)
    project_root="/content/drive/MyDrive/MT2KD",                       # only read (old v1 checkpoint, read-only, for an architecture comparison)
    librispeech_root="/content/drive/MyDrive/MT2KD/datasets/LibriSpeech",  # only read: LibriSpeech .tar.gz or extracted folders (searched first)
    librispeech_search=["/content/drive/MyDrive/MT2KD/datasets", "/content/drive/MyDrive/MT2KD", "/content/drive/MyDrive/LibriSpeech",
                        "/content/drive/MyDrive/datasets", "/content/drive/MyDrive"],   # also searched (read-only, 3 folder levels deep)
    librispeech_download=True,                       # not found on Drive -> download the official openslr archive to LOCAL disk (never to Drive)
    download_dir="/content/librispeech_dl",
    whisper_student_ckpt="/content/drive/MyDrive/MT2KD/results/ASR_Stage1_v4_sanity_2h/best.pt",   # your Whisper-taught student (read-only)
    # ---- Drive output folders (5 files each, written by DriveStore only)
    drive_out_root="/content/drive/MyDrive/MT2KD/results",   # KD run: <root>/WavLM_ASR_Stage1_<run>/ ; probe: <root>/ASR_teacher_comparison_ctc/
    drive_api=True,                                  # Google sign-in once per runtime: confirms uploads (md5) + deletes this folder's replaced checkpoints from the trash
    # ---- everything else goes to the Colab VM's local disk (lost when the runtime is recycled; regenerated automatically)
    write_root="/content/MT2KD_wavlm_work",
    local_audio="/content/ls_local",                 # session-local audio copy (rebuilt automatically after a restart)
    local_features="/content/wavlm_local",           # session-local copy of the teacher cache (fast random access)
    zipformer_dir="/content/MT2KD_zipformer_asr1",   # private copy of icefall's Zipformer (same files as the v4 cell)
    seed=42,
    subsets={"2h": 2.0, "10h": 10.0},                # hours (nested; the 2 h subset is the SAME as in the Whisper v4 run): 2 h = KD, 10 h = CTC probe
    # ---- teacher: WavLM-Large, 1024-D @ 50 Hz; two successive frames concatenated -> 2048-D @ 25 Hz
    teacher_model="microsoft/wavlm-large",
    teacher_layer=24,                                # 24 = final layer (= last_hidden_state), the analogue of Whisper's encoder output
    teacher_normalize=True,                          # WavLM-Large expects per-utterance normalised waveforms
    teacher_workers=2, shard_utts=256,
    # ---- student / KD
    align="shift",                                   # frame pairing from MEASURED frame centres; "truncate" = no offset (comparison only)
    target_norm="none",                              # paper: plain L1 on the raw teacher features
    init_bias="train_mean",                          # projection bias starts at the mean aligned teacher frame (engineering choice)
    # ---- optimisation: paper / icefall Zipformer recipe = ScaledAdam + Eden (identical to the Whisper v4 run)
    optimizer="scaled_adam",
    base_lr=0.045, lr_batches=7500, lr_epochs=3.5, eden_warmup_batches=500, eden_warmup_start=0.1, clipping_scale=2.0,
    lr=1.5e-4, weight_decay=1e-2, warmup_updates=500, min_lr_frac=0.05, grad_clip=1.0,      # used only when optimizer == "adamw"
    max_batch_seconds=1000.0, max_batch_utts=128, accum_steps=1, ref_duration=600.0,
    mem_fraction=0.85,                               # memory probe: largest batch whose measured peak stays below this share of the GPU
    max_consecutive_skips=25,
    max_session_hours=None,                          # e.g. 2.0: stop this cell run after ~2 h of training (state saved, resumable)
    # ---- validation / early stopping / checkpoints
    val_split="dev-clean", patience=5, min_delta=1e-4, overfit_gap_ratio=1.15,
    local_ckpt_every_updates=100, drive_ckpt_every_updates=500, ckpt_every_minutes=15.0,   # + both at every epoch end
    log_every=50, num_workers=min(8, os.cpu_count() or 2),
    # ---- CTC probe (identical for every student; fixed epochs, no model selection on dev-clean)
    probe_train_subset="10h", probe_epochs=30, probe_batch_seconds=200.0, probe_hidden=384, probe_layers=2, probe_dropout=0.1,
    probe_lr=1e-3, probe_seed=1234,
)
PRESETS = {
    # identical to the Whisper v4 "sanity_2h" run (same subset, epochs, batches, patience), so the two students are directly comparable
    "sanity_2h": dict(train_subset="2h", max_epochs=30, max_batch_seconds=120.0, patience=8, gate_ratio=0.90),
}
RUN_PRESETS = ["sanity_2h"]
def drive_dir(preset): return f"{CFG['drive_out_root']}/WavLM_ASR_Stage1_{preset}"
def probe_drive_dir(): return f"{CFG['drive_out_root']}/ASR_teacher_comparison_ctc"
OFFICIAL_UTTS = {"train-clean-100": 28539, "dev-clean": 2703, "test-clean": 2620}
SR = 16000

def log(*a): print(*a, flush=True)

DRIVE = "/content/drive"
def assert_local(p):
    """Every LOCAL write goes through here: refuses anything on Google Drive (Drive writes exist only in class DriveStore)."""
    rp = os.path.realpath(str(p))
    if rp.startswith(DRIVE) or str(p).startswith(DRIVE): raise PermissionError(f"refusing to write to Google Drive: {p}")
    return p

def paths(preset=None):
    R = Path(CFG["project_root"]); W = Path(assert_local(CFG["write_root"]))
    for k in ("local_audio", "local_features", "zipformer_dir", "download_dir"): assert_local(CFG[k])
    P = dict(
        root=R, ls=Path(CFG["librispeech_root"]),                         # read-only (Drive)
        meta=W / "datasets" / "LibriSpeech",                              # inventories, manifests, selection record (local)
        features=W / "features" / "wavlm",
        ckpt=W / "checkpoints" / "ASR_Stage1", logs=W / "logs" / "ASR_Stage1", results=W / "results" / "ASR_Stage1")
    if preset:
        run = f"{preset}_{PRESETS[preset]['train_subset']}"
        for k in ["ckpt", "logs", "results"]: P[k] = P[k] / run
        P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest")
    return P

def ensure_dirs(P):
    for k, v in P.items():
        if k not in ("root", "ls") and isinstance(v, Path): assert_local(v); v.mkdir(parents=True, exist_ok=True)

def pip_if_missing(*mods):
    for m in mods:
        mod, pkg = (m, m) if isinstance(m, str) else m
        if importlib.util.find_spec(mod) is None: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def atomic_json(obj, p):
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp")
    json.dump(obj, open(tmp, "w"), indent=1, default=str); os.replace(tmp, p)

def atomic_torch_save(obj, p):
    import torch
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, p)

def sha_of(items): return hashlib.sha1("\n".join(map(str, items)).encode()).hexdigest()[:12]

def file_sha1(p, chunk=1 << 22):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def code_fingerprint():
    """Single-cell version: the smoke-test report is tied to CODE_VERSION (plus the cache folder and the measured alignment, checked separately)."""
    return CODE_VERSION

def set_seed(s):
    import random, numpy as np, torch
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def oom_error():
    """CUDA out-of-memory exception class (torch.OutOfMemoryError exists only in newer PyTorch)."""
    import torch
    return getattr(torch, "OutOfMemoryError", None) or torch.cuda.OutOfMemoryError

def gpu_mem():
    import torch
    if not torch.cuda.is_available(): return {}
    return dict(gpu_alloc_gib=torch.cuda.memory_allocated() / 2**30, gpu_peak_gib=torch.cuda.max_memory_allocated() / 2**30,
                gpu_reserved_gib=torch.cuda.memory_reserved() / 2**30, gpu_total_gib=torch.cuda.get_device_properties(0).total_memory / 2**30)

def disk_free_gib(p):
    p = Path(p)
    while not p.exists(): p = p.parent
    return shutil.disk_usage(p).free / 2**30

def require_space(path, need_gib, what, margin_gib=1.0):
    free = disk_free_gib(path)
    if need_gib > free - margin_gib: raise RuntimeError(f"not enough space for {what}: need {need_gib:.1f} GiB (+{margin_gib} margin), {free:.1f} GiB free at {path}")
    return free

def env_report():
    """Environment check (nothing is installed or downgraded)."""
    import torch, platform
    r = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda, gpu=None, code_version=CODE_VERSION)
    try:
        import torchaudio; r["torchaudio"] = torchaudio.__version__
    except Exception as e: r["torchaudio"] = f"missing ({e})"
    try:
        import transformers; r["transformers"] = transformers.__version__
    except Exception: r["transformers"] = "missing (installed on first teacher use)"
    try:
        import k2; r["k2"] = getattr(k2, "__version__", "installed")
    except Exception: r["k2"] = "not installed (pure-PyTorch Swoosh port used; verified in the smoke test)"
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0); r.update(gpu=p.name, gpu_gib=round(p.total_memory / 2**30, 1), compute_capability=f"{p.major}.{p.minor}",
                                                           amp="bfloat16" if p.major >= 8 else "float16 + GradScaler")
    r["disk_free_local_gib"] = round(disk_free_gib("/content" if Path("/content").exists() else "/"), 1)
    r["disk_free_drive_gib"] = round(disk_free_gib(CFG["project_root"]), 1) if Path("/content/drive/MyDrive").exists() else None
    for k, v in r.items(): log(f"  {k:22s}: {v}")
    if r["gpu"] is None: log("  WARNING: no GPU. Runtime > Change runtime type > T4 GPU")
    return r

# ####################################################################################################
# ---- asr1_data ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 | LibriSpeech: official-split inventory, strict integrity checks, reproducible nested 2 h / 10 h subsets,
#                    manifests, local audio
#   * the inventory reads only FLAC headers (streamed through the .tar.gz, nothing extracted) -> exact durations
#   * every selected file is then fully decoded once (corrupt / wrong-rate / silent files are excluded and the selection redone)
#   * no duration filter: utterances longer than 30 s are kept (WavLM encodes every utterance whole)
# ===========================================================================================
import os, json, time, shutil, tarfile, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd

SELECTION_VERSION = 2          # v1 dropped utterances > 30 s; v2 keeps them
MANIFEST_COLS = ["uid", "speaker", "chapter", "split", "n_samples", "duration_s", "relpath", "text"]
DTYPES = {"uid": str, "speaker": str, "chapter": str, "text": str, "relpath": str}

def flac_streaminfo(head):
    """First bytes of a .flac -> (sample_rate, channels, bits, total_samples) from the STREAMINFO block, or None."""
    if len(head) < 42 or head[:4] != b"fLaC" or (head[4] & 0x7F) != 0: return None
    x = int.from_bytes(head[18:26], "big")
    return x >> 44, ((x >> 41) & 0x7) + 1, ((x >> 36) & 0x1F) + 1, x & ((1 << 36) - 1)

LS_SHA256 = {   # official archives (same values as torchaudio.datasets.LIBRISPEECH)
    "train-clean-100": "d4ddd1d5a6ab303066f14971d768ee43278a5f2a0aa43dc716b0e64ecbbbf6e2",
    "dev-clean": "76f87d090650617fca0cac8f88b9416e0ebf80350acb97b343a85fa903728ab3"}
LS_MIRRORS = ["https://www.openslr.org/resources/12", "https://openslr.elda.org/resources/12", "https://openslr.trmal.net/resources/12",
              "https://us.openslr.org/resources/12"]
_SRC = {}

def _search(root, split):
    root = Path(root)
    if not root.is_dir(): return None, None
    for t in [root / f"{split}.tar.gz", root / "tars" / f"{split}.tar.gz"]:
        if t.exists(): return "tar", t
    for base in [root, root / "LibriSpeech", root / "LibriSpeech" / "LibriSpeech"]:
        if (base / split).is_dir() and any((base / split).iterdir()): return "dir", base
    for dp, dn, fn in os.walk(root):
        if f"{split}.tar.gz" in fn: return "tar", Path(dp) / f"{split}.tar.gz"
        if split in dn and any((Path(dp) / split).iterdir()): return "dir", Path(dp)
        if len(Path(dp).relative_to(root).parts) >= 3: dn[:] = []
        dn[:] = [d for d in dn if not d.startswith(".") and d not in ("teacher_reps", "results", "checkpoints", "features")]
    return None, None

def download_split(split):
    """Official openslr archive -> LOCAL disk (resumable wget, sha256 checked). Never written to Drive."""
    d = Path(assert_local(CFG["download_dir"])); d.mkdir(parents=True, exist_ok=True); dst = d / f"{split}.tar.gz"; okf = d / f"{split}.sha256.ok"
    if dst.exists() and okf.exists() and okf.read_text() == LS_SHA256[split]: return dst
    require_space(d, 7.0 if "train" in split else 0.5, f"the {split} download", 3.0); part = d / f"{split}.tar.gz.part"
    for m in LS_MIRRORS:
        url = f"{m}/{split}.tar.gz"; log(f"[{split}] downloading {url} -> {part} (local disk; resumable) ..."); t0 = time.time()
        r = subprocess.run(["wget", "-c", "--tries=5", "--timeout=60", "--progress=dot:giga", "-O", str(part), url])
        if r.returncode: log(f"  download from {m} failed (wget exit {r.returncode}); trying the next mirror"); continue
        log(f"  downloaded {part.stat().st_size / 2**30:.2f} GiB in {(time.time() - t0) / 60:.1f} min; verifying sha256 ...")
        h = hashlib.sha256()
        with open(part, "rb") as f:
            for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
        if h.hexdigest() == LS_SHA256[split]: os.replace(part, dst); okf.write_text(LS_SHA256[split]); log(f"  sha256 OK -> {dst}"); return dst
        log(f"  sha256 MISMATCH ({h.hexdigest()[:16]}...): file removed, trying the next mirror"); part.unlink()
    raise RuntimeError(f"could not download a verified {split}.tar.gz from {LS_MIRRORS}")

def find_source(split, download=False):
    """('tar', archive) or ('dir', folder containing <split>). Searches the configured Drive locations (read-only), then the local download;
    with download=True and nothing found, downloads the official archive to local disk."""
    if split in _SRC: return _SRC[split]
    for root in [CFG["librispeech_root"]] + list(CFG.get("librispeech_search", [])) + [CFG["download_dir"]]:
        if split not in LS_SHA256 and root == CFG["download_dir"]: continue
        kind, src = _search(root, split)
        if kind == "tar" and str(src).startswith(str(CFG["download_dir"])):
            okf = Path(CFG["download_dir"]) / f"{split}.sha256.ok"
            if not (okf.exists() and okf.read_text() == LS_SHA256.get(split)): kind = None
        if kind: log(f"[{split}] source: {kind} {src}"); _SRC[split] = (kind, src); return kind, src
    if download and CFG.get("librispeech_download") and split in LS_SHA256:
        log(f"[{split}] not found on Drive under {[CFG['librispeech_root']] + CFG['librispeech_search']}: downloading it to local disk")
        _SRC[split] = ("tar", download_split(split)); return _SRC[split]
    return None, None

def _parse_member(name):
    """'LibriSpeech/train-clean-100/19/198/19-198-0000.flac' -> (uid, speaker, chapter) or None (official naming only)."""
    p = name.replace("\\", "/").split("/")
    if len(p) < 3 or not p[-1].endswith(".flac"): return None
    uid = p[-1][:-5]; sp = uid.split("-")
    if len(sp) != 3 or sp[0] != p[-3] or sp[1] != p[-2] or not all(s.isdigit() for s in sp): return None
    return uid, sp[0], sp[1]

def inventory(split, rebuild=False):
    """Valid utterances of an official split (exact sample counts, transcripts). Invalid ones -> inventory_v2_<split>_dropped.csv."""
    P = paths(); P["meta"].mkdir(parents=True, exist_ok=True); f = P["meta"] / f"inventory_v2_{split}.csv"; fd = P["meta"] / f"inventory_v2_{split}_dropped.csv"
    if f.exists() and not rebuild:
        inv = pd.read_csv(f, dtype=DTYPES, keep_default_na=False); nd = len(pd.read_csv(fd)) if fd.exists() and fd.stat().st_size > 30 else 0
        log(f"[{split}] inventory (cached): {len(inv):,} valid utterances, {inv.duration_s.sum() / 3600:.2f} h, {nd} invalid (see {fd.name})"); return inv
    kind, src = find_source(split, download=True)
    if kind is None: raise FileNotFoundError(f"LibriSpeech {split} not found under {[CFG['librispeech_root']] + CFG['librispeech_search']} "
                                              f"(expected {split}.tar.gz or an extracted folder) and automatic download is off")
    rows, texts, t0 = [], {}, time.time()
    def add_texts(raw):
        for line in raw.splitlines():
            if line.strip():
                k, _, t = line.partition(" ")
                if k in texts: raise RuntimeError(f"[{split}] transcript id {k} appears twice")
                texts[k] = t.strip()
    if kind == "tar":
        log(f"[{split}] reading FLAC headers and transcripts from {src.name} (streamed, nothing extracted) ...")
        with tarfile.open(src, "r|gz") as tf:
            for m in tf:
                if not m.isfile() or f"/{split}/" not in f"/{m.name}": continue
                if m.name.endswith(".trans.txt"): add_texts(tf.extractfile(m).read().decode("utf-8")); continue
                pm = _parse_member(m.name)
                if pm: rows.append(dict(uid=pm[0], speaker=pm[1], chapter=pm[2], relpath=m.name, header=flac_streaminfo(tf.extractfile(m).read(64))))
                elif m.name.endswith(".flac"): rows.append(dict(uid=os.path.basename(m.name)[:-5], speaker="", chapter="", relpath=m.name, header="badname"))
    else:
        base = src / split; log(f"[{split}] reading FLAC headers and transcripts under {base} ...")
        files = []
        for dp, _, fns in os.walk(base):
            for fn in fns:
                full = os.path.join(dp, fn)
                if fn.endswith(".trans.txt"): add_texts(open(full, encoding="utf-8").read())
                elif fn.endswith(".flac"): files.append(full)
        def head(p):
            try:
                with open(p, "rb") as fh: return flac_streaminfo(fh.read(64))
            except OSError: return None
        with ThreadPoolExecutor(32) as ex: infos = list(ex.map(head, files))
        for full, si in zip(files, infos):
            rel = os.path.relpath(full, src); pm = _parse_member(rel)
            rows.append(dict(uid=pm[0], speaker=pm[1], chapter=pm[2], relpath=rel, header=si) if pm else
                        dict(uid=os.path.basename(full)[:-5], speaker="", chapter="", relpath=rel, header="badname"))
    good, bad = [], []
    for r in rows:
        h = r.pop("header"); reason = ""
        if h == "badname": reason = "file name/folder does not follow the official speaker-chapter-utterance layout"
        elif h is None: reason = "FLAC header unreadable (corrupt or not FLAC)"
        elif h[0] != SR: reason = f"sample rate {h[0]} Hz (expected {SR})"
        elif h[1] != 1: reason = f"{h[1]} channels (expected mono)"
        elif h[3] <= 0: reason = "zero / unknown length in header"
        elif r["uid"] not in texts: reason = "missing transcript"
        if reason: bad.append(dict(r, reason=reason))
        else: good.append(dict(r, n_samples=int(h[3]), text=texts[r["uid"]]))
    inv = pd.DataFrame(good, columns=["uid", "speaker", "chapter", "relpath", "n_samples", "text"]).sort_values("uid").reset_index(drop=True)
    if inv.uid.duplicated().any(): raise RuntimeError(f"[{split}] duplicate utterance ids: {inv.uid[inv.uid.duplicated()].tolist()[:5]}")
    orphan = sorted(set(texts) - set(inv.uid) - {b["uid"] for b in bad})
    for u in orphan: bad.append(dict(uid=u, speaker=u.split("-")[0], chapter=u.split("-")[1] if "-" in u else "", relpath="", reason="transcript without audio"))
    inv["split"] = split; inv["duration_s"] = inv.n_samples / SR
    n_files = len(rows); exp = OFFICIAL_UTTS.get(split)
    if exp and n_files != exp: raise RuntimeError(f"[{split}] {n_files:,} audio files found, the official split has {exp:,}: the copy is incomplete")
    pd.DataFrame(bad, columns=["uid", "speaker", "chapter", "relpath", "reason"]).to_csv(fd, index=False)
    inv[MANIFEST_COLS].to_csv(f, index=False)
    log(f"[{split}] {n_files:,} files: {len(inv):,} valid ({inv.speaker.nunique()} speakers, {inv.duration_s.sum() / 3600:.2f} h), {len(bad)} invalid -> {fd.name} | "
        f"longest {inv.duration_s.max():.1f} s, {int((inv.duration_s > 30).sum())} longer than 30 s (kept) | {time.time() - t0:.0f} s")
    return inv

def select_subsets(inv, seed, hours, exclude=()):
    """Reproducible nested subsets of COMPLETE utterances: valid utterances (minus `exclude`), sorted by id, permuted with a fixed seed;
    each subset is the shortest prefix of that order whose total duration reaches its target (so 2 h is a prefix of 10 h).
    No duration filter (WavLM encodes long utterances whole)."""
    pool = inv[~inv.uid.isin(set(exclude))].sort_values("uid").reset_index(drop=True)
    order = np.random.default_rng(seed).permutation(len(pool)); cum = pool.duration_s.values[order].cumsum(); out = {}
    for name, h in sorted(hours.items(), key=lambda kv: kv[1]):
        k = len(pool) if not math.isfinite(h) else int(np.searchsorted(cum, h * 3600.0)) + 1
        if k > len(pool): raise ValueError(f"subset {name}: only {cum[-1] / 3600:.1f} h available")
        out[name] = pool.iloc[order[:k]].assign(sel_rank=np.arange(k)).reset_index(drop=True)
    return out

def local_path(relpath): return str(Path(CFG["local_audio"]) / relpath)

def materialize(df, split):
    """Puts exactly the manifest's files on local disk (only missing ones; safe to re-run after a restart)."""
    need = [r for r in df.relpath if not os.path.exists(local_path(r))]
    if not need: log(f"[{split}] local audio ready ({len(df):,} files)"); return
    est = df[df.relpath.isin(set(need))].n_samples.sum() * 2 * 0.7 / 2**30           # FLAC ~70 % of 16-bit PCM for speech (upper estimate)
    require_space(CFG["local_audio"], est, f"{split} audio on local disk", 2.0)
    kind, src = find_source(split); t0 = time.time(); Path(CFG["local_audio"]).mkdir(parents=True, exist_ok=True)
    log(f"[{split}] copying {len(need):,} selected files to local disk ({kind}: {src})")
    if kind == "tar":
        lst = Path(CFG["local_audio"]) / f".members_{split}.txt"; lst.write_text("\n".join(need) + "\n")
        r = subprocess.run(["tar", "-xzf", str(src), "-C", CFG["local_audio"], "-T", str(lst)], capture_output=True, text=True)
        if r.returncode: raise RuntimeError(f"tar extraction failed: {r.stderr[-300:]}")
    else:
        def cp(rel):
            dst = Path(local_path(rel)); dst.parent.mkdir(parents=True, exist_ok=True); tmp = str(dst) + ".part"
            shutil.copyfile(src / rel, tmp); os.replace(tmp, dst)
        with ThreadPoolExecutor(32) as ex: list(ex.map(cp, need))
    miss = [r for r in df.relpath if not os.path.exists(local_path(r))]
    if miss: raise RuntimeError(f"[{split}] {len(miss)} files still missing after the copy, e.g. {miss[:3]}")
    log(f"[{split}] {len(need):,} files ready in {time.time() - t0:.0f} s")

def decode_check(df, split):
    """Full decode of every file once: rate, channels, exact length, finite, not digital silence. Verified ids are remembered (local disk)."""
    import soundfile as sf
    fv = paths()["meta"] / f"decode_verified_v2_{split}.csv"
    ok = set(pd.read_csv(fv, dtype={"uid": str}).uid) if fv.exists() else set()
    todo = df[~df.uid.isin(ok)]
    def chk(r):
        try:
            w, sr = sf.read(local_path(r.relpath), dtype="float32", always_2d=True)
            if sr != SR: return r.uid, f"decoded rate {sr}"
            if w.shape[1] != 1: return r.uid, f"{w.shape[1]} channels"
            if len(w) != r.n_samples: return r.uid, f"decoded {len(w)} samples, header {r.n_samples}"
            if not np.isfinite(w).all(): return r.uid, "non-finite samples"
            if np.abs(w).max() == 0: return r.uid, "digital silence"
            return r.uid, ""
        except Exception as e: return r.uid, f"decode error: {type(e).__name__}: {e}"[:200]
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(chk, todo.itertuples()))
    bad = {u: m for u, m in res if m}; good = sorted(ok | {u for u, m in res if not m})
    tmp = str(fv) + ".tmp"; pd.DataFrame(dict(uid=good)).to_csv(tmp, index=False); os.replace(tmp, fv)
    log(f"[{split}] full decode check: {len(todo):,} files decoded in {time.time() - t0:.0f} s ({len(df) - len(todo):,} verified earlier) | failures: {len(bad)}")
    return bad

def _record_failures(split, bad):
    f = paths()["meta"] / f"decode_failures_v2_{split}.csv"
    old = pd.read_csv(f, dtype={"uid": str}) if f.exists() else pd.DataFrame(columns=["uid", "reason"])
    new = pd.concat([old, pd.DataFrame([dict(uid=u, reason=m) for u, m in bad.items()])]).drop_duplicates("uid")
    new.to_csv(f, index=False); return set(new.uid)

def _failures(split):
    f = paths()["meta"] / f"decode_failures_v2_{split}.csv"
    return set(pd.read_csv(f, dtype={"uid": str}).uid) if f.exists() else set()

def prepare_manifests():
    """train_2h / train_10h / dev_clean manifests + subset_selection.json. Idempotent; never silently overwrites a different selection."""
    P = paths(); P["meta"].mkdir(parents=True, exist_ok=True); tr = inventory("train-clean-100"); dv = inventory(CFG["val_split"])
    assert not set(tr.uid) & set(dv.uid) and not set(tr.speaker) & set(dv.speaker), "train/dev overlap"
    kind, _ = find_source("test-clean"); te = inventory("test-clean") if kind else None
    if te is not None: assert not set(tr.uid) & set(te.uid) and not set(tr.speaker) & set(te.speaker), "train/test overlap"
    sel_f = P["meta"] / "subset_selection.json"
    if sel_f.exists() and json.load(open(sel_f)).get("selection_version") != SELECTION_VERSION:      # manifests from the previous code version
        arc = P["meta"] / "archive_manifests" / time.strftime("%Y%m%d_%H%M%S"); arc.mkdir(parents=True)
        for n in ["subset_selection.json", "train_2h_manifest.csv", "train_10h_manifest.csv", "dev_clean_manifest.csv"]:
            if (P["meta"] / n).exists(): shutil.move(str(P["meta"] / n), arc / n)
        log(f"manifests from an older selection rule moved to {arc} (nothing deleted)")
    # ---- training subsets: select, extract, decode every selected file; corrupt ones are excluded and the selection redone (deterministic)
    for it in range(20):
        excl = _failures("train-clean-100"); subs = select_subsets(tr, CFG["seed"], CFG["subsets"], excl)
        big = subs[max(CFG["subsets"], key=CFG["subsets"].get)]; materialize(big, "train-clean-100"); bad = decode_check(big, "train-clean-100")
        if not bad: break
        _record_failures("train-clean-100", bad); log(f"  excluded {len(bad)} corrupt files {sorted(bad)[:5]} and re-selected")
    else: raise RuntimeError("too many corrupt files in train-clean-100")
    sel = dict(selection_version=SELECTION_VERSION, seed=CFG["seed"], source_split="train-clean-100", inventory_sha=sha_of(tr.uid),
               rule="valid utterances of train-clean-100 (official naming, 16 kHz mono FLAC, transcript present, fully decodable), sorted by id, "
                    "permuted by numpy.random.default_rng(seed).permutation; each subset = shortest prefix reaching its target hours (nested). "
                    "No duration filter.", excluded_corrupt=sorted(_failures("train-clean-100")), subsets={})
    for name, df in subs.items():
        f = P["meta"] / f"train_{name}_manifest.csv"; uid_sha = sha_of(sorted(df.uid))
        if f.exists():
            old = pd.read_csv(f, dtype=DTYPES, keep_default_na=False)
            if sha_of(sorted(old.uid)) != uid_sha: raise RuntimeError(f"{f} exists with a DIFFERENT selection; not overwritten. Move it away if you mean to replace it.")
            log(f"train_{name}: existing manifest matches the selection (reused)")
        else: df[MANIFEST_COLS + ["sel_rank"]].to_csv(f, index=False)
        sel["subsets"][name] = dict(manifest=str(f), utterances=len(df), speakers=int(df.speaker.nunique()), chapters=int(df.chapter.nunique()),
                                    hours=round(df.duration_s.sum() / 3600, 4), min_s=round(df.duration_s.min(), 2), max_s=round(df.duration_s.max(), 2),
                                    longer_than_30s=int((df.duration_s > 30).sum()), uid_sha=uid_sha)
        log(f"train_{name}: {len(df):,} utterances | {df.speaker.nunique()} speakers | {df.duration_s.sum():.1f} s = {df.duration_s.sum() / 3600:.3f} h "
            f"(target {CFG['subsets'][name] * 3600:.0f} s) | "
            f"{int((df.duration_s > 30).sum())} longer than 30 s -> {f.name}")
    names = sorted(subs, key=lambda n: CFG["subsets"][n])
    for a, b in zip(names, names[1:]): assert set(subs[a].uid) <= set(subs[b].uid), f"{a} not nested in {b}"
    # ---- validation: the whole official dev-clean (only invalid/corrupt files excluded, and reported)
    for it in range(5):
        dvv = dv[~dv.uid.isin(_failures(CFG["val_split"]))]; materialize(dvv, CFG["val_split"]); bad = decode_check(dvv, CFG["val_split"])
        if not bad: break
        _record_failures(CFG["val_split"], bad)
    fdev = P["meta"] / "dev_clean_manifest.csv"; dvv[MANIFEST_COLS].to_csv(fdev, index=False)
    fdrop = P["meta"] / f"inventory_v2_{CFG['val_split']}_dropped.csv"
    n_drop = len(pd.read_csv(fdrop)) if fdrop.exists() and fdrop.stat().st_size > 30 else 0
    n_off = OFFICIAL_UTTS.get(CFG["val_split"], len(dv) + n_drop)
    sel["validation"] = dict(split=CFG["val_split"], manifest=str(fdev), utterances=len(dvv), official_utterances=n_off,
                             coverage=f"{len(dvv)}/{n_off}" + ("" if len(dvv) == n_off else " (FILTERED dev-clean: invalid/corrupt files excluded, see dropped lists)"),
                             hours=round(dvv.duration_s.sum() / 3600, 4), longer_than_30s=int((dvv.duration_s > 30).sum()), speakers=int(dvv.speaker.nunique()))
    sel["test"] = "test-clean is reserved for downstream (Stage 2) WER; Stage 1 reads only its headers for the overlap check" if te is not None else \
                  "test-clean not present on Drive (not needed for Stage 1)"
    atomic_json(sel, P["meta"] / "subset_selection.json")
    log(f"validation (official {CFG['val_split']}): {len(dvv):,}/{n_off} utterances, {dvv.duration_s.sum() / 3600:.2f} h, "
        f"{sel['validation']['longer_than_30s']} longer than 30 s (kept, encoded whole) | coverage {sel['validation']['coverage']}")
    log(f"no train/dev{'/test' if te is not None else ''} utterance or speaker overlap | selection record: {P['meta'] / 'subset_selection.json'}")
    return sel

def load_manifest(name):
    P = paths(); f = P["meta"] / ("dev_clean_manifest.csv" if name == "dev" else f"train_{name}_manifest.csv")
    if not f.exists(): raise FileNotFoundError(f"{f} missing: run the manifest cell first")
    sel = json.load(open(P["meta"] / "subset_selection.json"))
    if sel.get("selection_version") != SELECTION_VERSION: raise RuntimeError("manifests come from an older selection rule: re-run the manifest cell")
    df = pd.read_csv(f, dtype=DTYPES, keep_default_na=False)
    assert df.uid.is_unique and (df.text.str.len() > 0).all() and (df.n_samples > 0).all(), f"{f}: duplicate ids / empty transcripts / bad lengths"
    return df

def load_audio(relpath, n_expected):
    """16 kHz mono float32 in [-1, 1]; raises on wrong rate/length or non-finite samples."""
    import soundfile as sf
    wav, sr = sf.read(local_path(relpath), dtype="float32", always_2d=True)
    if sr != SR: raise ValueError(f"{relpath}: {sr} Hz (expected {SR})")
    if wav.shape[1] != 1: raise ValueError(f"{relpath}: {wav.shape[1]} channels (expected mono)")
    wav = wav[:, 0]
    if len(wav) != n_expected: raise ValueError(f"{relpath}: {len(wav)} samples, manifest says {n_expected}")
    if not np.isfinite(wav).all(): raise ValueError(f"{relpath}: non-finite samples")
    return np.ascontiguousarray(wav)

# ####################################################################################################
# ---- asr1_wavlm ----
# ####################################################################################################
# ===========================================================================================
# WavLM ASR Stage 1 | WavLM-Large teacher: loading, preprocessing, layer features, verification, timing measurement,
#                     resumable integrity-checked cache
#   WavLM-Large (microsoft/wavlm-large, 315M parameters, 24 Transformer layers, 1024-D @ 50 Hz, input = raw 16 kHz waveform).
#   Same representation recipe as the Whisper run: two successive 50 Hz frames concatenated -> 2048-D @ 25 Hz,
#   target frame j = [h_{2j} ; h_{2j+1}] (an odd last 50 Hz frame is dropped).
#   No 30 s window and no chunking: every utterance is encoded alone, at its exact length (no padding at all).
# ===========================================================================================
import os, json, math, time, shutil
from pathlib import Path
import numpy as np, pandas as pd

HOP_T = 640                                   # samples per 25 Hz target frame
CACHE_FORMAT = 3
WAVLM_DIM = 1024
WAVLM_CONV = [(10, 5), (3, 2), (3, 2), (3, 2), (3, 2), (2, 2), (2, 2)]     # (kernel, stride) of WavLM's convolutional feature encoder
def wavlm_frames50(n):
    """Exact frame count of the conv feature encoder (no padding): L -> (L - k) // s + 1 per layer (= (n - 400) // 320 + 1)."""
    L = int(n)
    for k, s in WAVLM_CONV: L = (L - k) // s + 1
    return max(L, 0)
def teacher_frames(n): return wavlm_frames50(n) // 2                       # 25 Hz target frames (pairs of 50 Hz frames)
def wavlm_centre50(m):
    """Analytic centre (in samples) of 50 Hz frame m: the middle of its receptive field (= 320 m + 199.5)."""
    x = float(m)
    for k, s in reversed(WAVLM_CONV): x = x * s + (k - 1) / 2
    return x

def wavlm_prep(wav):
    """Per-utterance zero-mean / unit-variance waveform (WavLM-Large was pre-trained on normalised input, cfg.normalize = True;
    same formula as transformers' Wav2Vec2FeatureExtractor.zero_mean_unit_var_norm)."""
    w = np.asarray(wav, np.float32)
    if not CFG["teacher_normalize"]: return np.ascontiguousarray(w)
    return np.ascontiguousarray(((w - w.mean()) / np.sqrt(w.var() + 1e-7)).astype(np.float32))

def _verification_file(): return paths()["logs"] / "wavlm_verification.json"

class WavLMTeacher:
    """Frozen WavLM-Large. One utterance per forward pass (no padding, so no attention mask is involved)."""
    def __init__(self, device="cuda", dtype=None):
        pip_if_missing("transformers")
        import torch, transformers
        from transformers import WavLMModel
        name = CFG["teacher_model"]; t0 = time.time()
        if dtype is None:          # the precision chosen by the verification of this runtime (fp32 only if fp16 was found inaccurate)
            vf = _verification_file()
            dtype = json.load(open(vf)).get("dtype_used", "float16") if vf.exists() else "float16"
        self.dtype_name = dtype; tdt = torch.float16 if dtype == "float16" else torch.float32
        major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
        kw = dict(low_cpu_mem_usage=True); kw["dtype" if (major, minor) >= (4, 56) else "torch_dtype"] = tdt
        model = WavLMModel.from_pretrained(name, **kw)
        cfg = model.config; self.commit = getattr(cfg, "_commit_hash", None) or "unknown"
        self.enc = model.to(device=device, dtype=tdt).eval()
        for p in self.enc.parameters(): p.requires_grad_(False)
        self.device = device; self.d_model = int(cfg.hidden_size); self.n_layers = int(cfg.num_hidden_layers); self.layer = int(CFG["teacher_layer"])
        self.hf_do_normalize = None
        try:
            from transformers import Wav2Vec2FeatureExtractor
            self.hf_do_normalize = bool(Wav2Vec2FeatureExtractor.from_pretrained(name).do_normalize)
        except Exception as e: log(f"  (feature-extractor config not read: {type(e).__name__}; the waveform normalisation is done by this code anyway)")
        conv = [(int(k), int(s)) for k, s in zip(cfg.conv_kernel, cfg.conv_stride)]
        self.info = dict(model=name, commit=self.commit, transformers=transformers.__version__, torch=torch.__version__,
                         sampling_rate=SR, d_model=self.d_model, encoder_layers=self.n_layers, conv=conv,
                         params_M=round(sum(p.numel() for p in self.enc.parameters()) / 1e6, 1), dtype=dtype,
                         layer=f"hidden_states[{self.layer}] of {self.n_layers}" + (" (= last_hidden_state, after the final LayerNorm)" if self.layer == self.n_layers else ""),
                         pairing="concatenate 50 Hz frames (2j, 2j+1) -> 25 Hz, 2 x d_model", frames_rule="T = ((n - 400) // 320 + 1) // 2",
                         normalize="per-utterance zero mean / unit variance" if CFG["teacher_normalize"] else "none", hf_feature_extractor_do_normalize=self.hf_do_normalize,
                         chunking="none (whole utterance, no padding)", load_s=round(time.time() - t0, 1))
        problems = []
        if conv != WAVLM_CONV: problems.append(f"conv feature encoder {conv}, expected {WAVLM_CONV}")
        if self.d_model != WAVLM_DIM or self.n_layers != 24: problems.append(f"hidden size {self.d_model}, layers {self.n_layers} (expected 1024 / 24: WavLM-Large)")
        if not 1 <= self.layer <= self.n_layers: problems.append(f"teacher_layer {self.layer} outside 1..{self.n_layers}")
        if any(p.requires_grad for p in self.enc.parameters()) or self.enc.training: problems.append("teacher not frozen / not in eval mode")
        if problems: raise RuntimeError("WavLM check failed: " + "; ".join(problems))
        log(f"WavLM loaded: {name} (commit {self.commit[:10]}) | {self.info['params_M']}M params, {self.n_layers} layers, {dtype}, frozen, eval | "
            f"layer {self.layer} | d_model {self.d_model} | {self.info['load_s']} s | " + ", ".join(f"{k} {v:.2f}" for k, v in gpu_mem().items()))

    def set_dtype(self, dtype):
        import torch
        self.enc.to(torch.float16 if dtype == "float16" else torch.float32); self.dtype_name = dtype; self.info["dtype"] = dtype

    def hidden(self, x):
        """x: normalised waveform (numpy [n]) -> the selected layer [T50, 1024] on the GPU (fp32 retry if fp16 overflows)."""
        import torch
        dt = next(self.enc.parameters()).dtype
        def run(dtype):
            inp = torch.as_tensor(x)[None].to(self.device, dtype=dtype)
            with torch.inference_mode():
                if self.layer == self.n_layers: return self.enc(input_values=inp).last_hidden_state[0]
                return self.enc(input_values=inp, output_hidden_states=True).hidden_states[self.layer][0]
        h = run(dt)
        if not torch.isfinite(h).all():                       # rare fp16 overflow: recompute this utterance in fp32
            self.enc.float(); h = run(torch.float32); self.enc.to(dt)
            if not torch.isfinite(h).all(): raise FloatingPointError("WavLM output non-finite even in fp32")
        return h

    def encode_wav(self, wav):
        """16 kHz mono float32 waveform -> float16 array [teacher_frames(n), 2048] (pairs of 50 Hz frames)."""
        import torch
        n = len(wav); h = self.hidden(wavlm_prep(wav)); T50 = wavlm_frames50(n)
        if h.shape[0] != T50 or h.shape[1] != self.d_model: raise RuntimeError(f"WavLM output {tuple(h.shape)} for {n} samples, expected ({T50}, {self.d_model})")
        T = T50 // 2
        return h[:2 * T].reshape(T, 2 * self.d_model).to(torch.float16).cpu().numpy()

class _WavDS:       # CPU workers only read audio (they never touch the GPU model)
    def __init__(self, rows): self.r = rows
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        r = self.r[i]; n = int(r["n_samples"]); return r["uid"], n, load_audio(r["relpath"], n)
def _collate(b): return b

# ------------------------------------------------------------------ verification on real audio
def _cos(a, b):
    a = np.asarray(a, np.float32).ravel(); b = np.asarray(b, np.float32).ravel(); return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def pick_test_utts(train_df, dev_df):
    """short, medium, near-30 s (<= 30 s) and > 30 s (dev-clean, or two real training utterances joined if none) for the checks."""
    pool = pd.concat([train_df, dev_df], ignore_index=True)
    short = pool.loc[pool.duration_s.idxmin()]; med = pool.iloc[(pool.duration_s - pool.duration_s.median()).abs().argsort().iloc[0]]
    near = pool[pool.duration_s <= 30].iloc[(pool[pool.duration_s <= 30].duration_s - 29.5).abs().argsort().iloc[0]]
    longs = pool[pool.duration_s > 30]
    return dict(short=short, medium=med, near30=near, long=longs.iloc[0] if len(longs) else None)

def verify_wavlm(teacher, train_df, dev_df):
    """Preprocessing, dimensions, frame-count rule, layer identity, frame pairing, long-utterance handling, repeatability and precision.
    If fp16 is not accurate enough (cosine to fp32 < 0.999), the teacher switches to fp32 for the cache (recorded)."""
    import torch
    torch.cuda.reset_peak_memory_stats(); R = dict(info=dict(teacher.info)); U = pick_test_utts(train_df, dev_df)
    W = {k: (r.uid, load_audio(r.relpath, int(r.n_samples))) for k, r in U.items() if r is not None}
    if U["long"] is None:
        a, b = W["near30"][1], W["medium"][1]; W["long"] = ("joined:" + W["near30"][0] + "+" + W["medium"][0], np.concatenate([a, b]))
        log("  no utterance > 30 s in the manifests: a > 30 s test signal is built by joining two real utterances")
    for k, (u, w) in W.items():
        assert w.ndim == 1 and w.dtype == np.float32 and np.isfinite(w).all() and np.abs(w).max() <= 1.0
        log(f"  {k:7s} {u}: {len(w)} samples = {len(w) / SR:.2f} s, mono float32 @ {SR} Hz")
    # frame-count rule: the formula used for the cache == transformers' own output-length rule == closed form, for many lengths
    ns = list(range(400, 60000, 37)) + [480000, 523456]
    assert [wavlm_frames50(n) for n in ns] == [(n - 400) // 320 + 1 for n in ns], "WavLM frame-count rule mismatch"
    try:
        hf = [int(teacher.enc._get_feat_extract_output_lengths(torch.tensor(n)).item()) for n in ns]; hf_ok = hf == [wavlm_frames50(n) for n in ns]
    except Exception as e: hf_ok = f"not checked ({type(e).__name__})"
    assert hf_ok is not False, "transformers' WavLM output-length rule differs from (n - 400) // 320 + 1"
    log(f"  frame rule: 50 Hz frames = (n - 400) // 320 + 1 (transformers' own rule agrees: {hf_ok}); 25 Hz targets = that // 2; "
        f"checked against the real model output below")
    # preprocessing
    w0 = W["medium"][1]; x0 = wavlm_prep(w0)
    R["prep"] = dict(mean=float(x0.mean()), std=float(x0.std()), hf_do_normalize=teacher.hf_do_normalize)
    if CFG["teacher_normalize"]: assert abs(R["prep"]["mean"]) < 1e-3 and abs(R["prep"]["std"] - 1) < 1e-3, R["prep"]
    log(f"  preprocessing: raw 16 kHz waveform, {teacher.info['normalize']} (mean {R['prep']['mean']:+.1e}, std {R['prep']['std']:.4f}); "
        f"transformers' feature-extractor config for this model says do_normalize = {teacher.hf_do_normalize}")
    # layer identity + raw output geometry
    inp = torch.as_tensor(x0)[None].to(teacher.device, dtype=next(teacher.enc.parameters()).dtype)
    with torch.inference_mode(): o = teacher.enc(input_values=inp, output_hidden_states=True)
    hs = o.hidden_states; assert len(hs) == teacher.n_layers + 1, f"{len(hs)} hidden states for {teacher.n_layers} layers"
    lhs = o.last_hidden_state; d_last = float((hs[-1] - lhs).abs().max()); scale = max(1.0, float(lhs.abs().max()))
    if d_last <= 1e-3 * scale:
        how = f"hidden_states[{teacher.n_layers}] == last_hidden_state (|diff| {d_last:.1e}): stored after the final LayerNorm"
    else:                     # some transformers versions store the last layer's output BEFORE the encoder's final LayerNorm
        ln = getattr(getattr(teacher.enc, "encoder", None), "layer_norm", None)
        assert ln is not None, f"hidden_states[{teacher.n_layers}] differs from last_hidden_state by {d_last} and the encoder has no final LayerNorm"
        with torch.inference_mode(): d_ln = float((ln(hs[-1]) - lhs).abs().max())
        assert d_ln <= 1e-2 * scale, f"last_hidden_state is neither hidden_states[-1] (|diff| {d_last}) nor its LayerNorm (|diff| {d_ln})"
        how = (f"last_hidden_state == LayerNorm(hidden_states[{teacher.n_layers}]) (|diff| {d_ln:.1e}); this transformers version stores hidden_states[-1] "
               f"before the final LayerNorm")
    sel = lhs if teacher.layer == teacher.n_layers else hs[teacher.layer]          # exactly what WavLMTeacher.hidden() returns
    T50 = hs[0].shape[1]; assert T50 == wavlm_frames50(len(w0)) and sel.shape[-1] == teacher.d_model
    R["layer_check"] = dict(hidden_states=len(hs), used=teacher.info["layer"], detail=how)
    log(f"  layer: {how}; using {teacher.info['layer']} | {len(w0)} samples -> {T50} frames @ 50 Hz, {hs[0].shape[-1]}-D")
    # pairing / concatenation order (same forward path as the cache)
    out = teacher.encode_wav(w0); T = teacher_frames(len(w0)); hb = sel[0].float().cpu().numpy(); j = T // 2
    assert out.shape == (T, 2 * teacher.d_model) and np.isfinite(out).all()
    c_a, c_b = _cos(out[j, :teacher.d_model], hb[2 * j]), _cos(out[j, teacher.d_model:], hb[2 * j + 1])
    assert c_a > 0.9999 and c_b > 0.9999, (c_a, c_b)
    R["pairing"] = dict(frames50=T50, targets=T, dropped_odd_frame=bool(T50 % 2), cos=[c_a, c_b])
    log(f"  pairing: target[j] = [h(2j) ; h(2j+1)] verified (cos {c_a:.6f} / {c_b:.6f}); {T50} frames @ 50 Hz -> {out.shape} @ 25 Hz"
        + (" (odd last 50 Hz frame dropped)" if T50 % 2 else ""))
    del o, hs, hb, lhs, sel
    # every test length incl. > 30 s: shape rule, finite, no chunking
    for k, (u, w) in W.items():
        t = teacher.encode_wav(w); assert t.shape == (teacher_frames(len(w)), 2 * teacher.d_model) and np.isfinite(t).all(), (k, t.shape)
        R[f"frames_{k}"] = [len(w), len(t)]
    log(f"  all test lengths (incl. {len(W['long'][1]) / SR:.1f} s, encoded whole, no 30 s window): " + ", ".join(f"{k} {v[0]} -> {v[1]}" for k, v in R.items() if k.startswith("frames_")))
    # repeatability and precision
    again = teacher.encode_wav(w0); R["repeat_maxdiff"] = float(np.abs(again.astype(np.float32) - out).max())
    if teacher.dtype_name == "float16":
        teacher.set_dtype("float32"); t32 = teacher.encode_wav(w0); teacher.set_dtype("float16")
        R["fp16_vs_fp32_cos"] = _cos(out, t32); R["fp16_vs_fp32_rel_l1"] = float(np.abs(out.astype(np.float32) - t32).mean() / (np.abs(t32).mean() + 1e-12))
        if R["fp16_vs_fp32_cos"] < 0.999:
            log(f"  fp16 vs fp32 cosine only {R['fp16_vs_fp32_cos']:.5f}: the cache will be computed in fp32 (slower, exact)"); teacher.set_dtype("float32")
    else: R["fp16_vs_fp32_cos"] = None; R["fp16_vs_fp32_rel_l1"] = None
    R["dtype_used"] = teacher.dtype_name
    st = out.astype(np.float32); R.update(target_mean=float(st.mean()), target_std=float(st.std()), target_absmax=float(np.abs(st).max()), **gpu_mem())
    log(f"  repeat max |diff| {R['repeat_maxdiff']:.4f} | fp16 vs fp32 cosine {R['fp16_vs_fp32_cos']} (mean rel. L1 {R['fp16_vs_fp32_rel_l1']}) | cache dtype: {R['dtype_used']} | all finite")
    log(f"  target stats: mean {R['target_mean']:.3f}, std {R['target_std']:.3f}, |max| {R['target_absmax']:.1f} | GPU peak {R['gpu_peak_gib']:.2f} GiB")
    atomic_json(R, _verification_file()); log(f"  saved {_verification_file()}")
    return R

def measure_teacher_timing(teacher):
    """Measures, with the imported WavLM code, where each target frame sits in time: every input sample of a probe signal is perturbed
    (one at a time) and the 50 Hz frames of the REAL conv feature encoder that change are recorded (fp32, TF32 off). The centre of each frame's
    receptive field is fitted linearly; the pairing (2j, 2j+1) is then applied. Returns a linear map j -> centre (samples)."""
    import torch, copy
    fe = copy.deepcopy(teacher.enc.feature_extractor).float().eval(); dev = teacher.device
    tf = None
    try: tf = (torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32); torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
    except Exception: pass
    try:
        L, K, B = 4000, 3200, 128; g = torch.Generator().manual_seed(0); x = (torch.randn(1, L, generator=g) * 0.1).to(dev)
        infl = None
        with torch.inference_mode():
            for s in range(0, K, B):
                ks = list(range(s, min(K, s + B))); xb = x.repeat(len(ks) + 1, 1)
                for i, k in enumerate(ks): xb[i + 1, k] += 0.5
                y = fe(xb)                                                                  # [b, C, M]; item 0 = unperturbed (same batch -> same kernels)
                if infl is None: infl = [[] for _ in range(y.shape[-1])]
                ch = (y[1:] - y[:1]).abs().amax(1) > 1e-5                                  # [b, M]
                for i, k in enumerate(ks):
                    for m in torch.nonzero(ch[i]).flatten().tolist(): infl[m].append(k)
    finally:
        try:
            if tf is not None: torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32 = tf
        except Exception: pass
        del fe; torch.cuda.empty_cache()
    ms = [m for m in range(len(infl)) if infl[m] and min(infl[m]) > 0 and max(infl[m]) < K - 1]
    if len(ms) < 4:          # e.g. a GroupNorm feature encoder (every frame depends on every sample): use the layer-geometry map
        a_w, b_w = 320.0, wavlm_centre50(0)
        log(f"  teacher timing: the perturbation probe found only {len(ms)} local frames (feature encoder normalises over time?) -> "
            f"using the layer-geometry map 50 Hz frame m centre = {a_w:.0f} m + {b_w:.1f} samples")
        return dict(frame50_slope=a_w, frame50_intercept=b_w, fit_residual=None, measured=False,
                    centre_samples=dict(slope=2 * a_w, intercept=b_w + 0.5 * a_w))
    cm = np.array([(min(infl[m]) + max(infl[m])) / 2 for m in ms]); width = sorted({max(infl[m]) - min(infl[m]) + 1 for m in ms})
    a_w, b_w = np.polyfit(ms, cm, 1); res_w = float(np.abs(np.polyval([a_w, b_w], ms) - cm).max())
    slope = 2 * a_w; icpt = b_w + 0.5 * a_w                                                 # target j = 50 Hz frames (2j, 2j+1): centre index 2j + 0.5
    R = dict(frame50_slope=float(a_w), frame50_intercept=float(b_w), fit_residual=res_w, receptive_field_samples=width,
             influence_example={int(m): [min(infl[m]), max(infl[m])] for m in ms[:3]}, centre_samples=dict(slope=float(slope), intercept=float(icpt)))
    log(f"  teacher timing (measured): 50 Hz frame m <- samples centred at {a_w:.3f} m + {b_w:.2f} (receptive field {width} samples, fit residual {res_w:.3f}) "
        f"| target j centre = {slope:.1f} j + {icpt:.1f} samples = {slope / 16:.1f} j + {icpt / 16:.2f} ms")
    return R

# ------------------------------------------------------------------ cache
FEATURE_KEYS = ["model", "commit", "layer", "pairing", "dim", "frames_rule", "normalize", "sampling_rate", "cache_format", "dtype_stored"]

def cache_dir(split, teacher_commit):
    return paths()["features"] / f"{CFG['teacher_model'].split('/')[-1]}_{teacher_commit[:8]}_L{CFG['teacher_layer']}_cat2_v{CACHE_FORMAT}" / split

def _committed(d, full_check=False):
    """{uid -> (shard, offset, length)} for committed shards. A shard counts only if its .npy, .json and index .csv all exist and agree;
    inconsistent shards are moved to <cache>/corrupt/ (never deleted) and their utterances are recomputed."""
    parts = []
    for f in sorted(d.glob("index_*.csv")):
        k = f.stem.split("_")[1]; s, j = d / f"shard_{k}.npy", d / f"shard_{k}.json"; why = ""
        try:
            ix = pd.read_csv(f, dtype={"uid": str, "speaker": str}, keep_default_na=False); meta = json.load(open(j))
            if s.stat().st_size != meta["nbytes"]: why = f"size {s.stat().st_size} != {meta['nbytes']}"
            else:
                m = np.load(s, mmap_mode="r")
                if m.shape != (int(ix.length.sum()), meta["dim"]) or m.dtype != np.float16: why = f"shape {m.shape} / {m.dtype}"
                elif full_check and file_sha1(s) != meta["sha1"]: why = "sha1 mismatch"
        except Exception as e: why = f"{type(e).__name__}: {e}"
        if why:
            bad = d / "corrupt" / time.strftime("%Y%m%d_%H%M%S"); bad.mkdir(parents=True, exist_ok=True)
            for x in (f, s, j):
                if x.exists(): shutil.move(str(x), bad / x.name)
            log(f"  WARNING shard {k} failed the integrity check ({why}); moved to {bad}; its utterances will be recomputed"); continue
        parts.append(ix)
    idx = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["uid", "speaker", "n_samples", "shard", "offset", "length"])
    if idx.uid.duplicated().any(): raise RuntimeError(f"cache {d}: utterances in more than one shard: {idx.uid[idx.uid.duplicated()].tolist()[:5]}")
    return idx

def check_meta(d, meta):
    if (d / "meta.json").exists():
        old = json.load(open(d / "meta.json"))
        diff = {k: (old.get(k), meta.get(k)) for k in FEATURE_KEYS if old.get(k) != meta.get(k)}
        if diff: raise RuntimeError(f"existing cache in {d} was made differently: {diff}. Move that folder away (nothing is reused silently).")

def build_cache(teacher, df, split, encode_rows=None):
    """Resumable: committed utterances are skipped; an interrupted shard is redone (at most shard_utts utterances).
    Commit order per shard: .npy (atomic) -> .json (sha1, size, shape) -> index .csv (the commit marker)."""
    d = cache_dir(split, teacher.commit); d.mkdir(parents=True, exist_ok=True)
    meta = dict(teacher.info, split=split, dim=2 * teacher.d_model, rate_hz=25, sample_rate=SR, cache_format=CACHE_FORMAT, dtype_stored="float16",
                preprocessing="16 kHz mono float32 -> per-utterance zero-mean/unit-variance -> WavLM-Large (whole utterance, no padding) -> "
                              f"hidden_states[{teacher.layer}] -> frames (2j, 2j+1) concatenated (odd last frame dropped)")
    meta["sampling_rate"] = SR; check_meta(d, meta); atomic_json(meta, d / "meta.json")
    for f in list(d.glob("shard_*.npy")) + list(d.glob("shard_*.json")):          # orphans from an interrupted commit
        if not (d / f"index_{f.stem.split('_')[1]}.csv").exists(): f.unlink()
    for f in list(d.glob("*.tmp*")): f.unlink()
    done = _committed(d); todo = df[~df.uid.isin(set(done.uid))].sort_values("uid")
    need_gib = sum(teacher_frames(n) for n in todo.n_samples) * 2 * teacher.d_model * 2 / 2**30
    log(f"[{split}] teacher cache {d}\n  committed {len(done):,} | to compute {len(todo):,} utterances (~{need_gib:.2f} GiB)")
    if len(todo): require_space(d, need_gib, "the teacher cache on local disk", 3.0)
    used = [int(f.stem.split("_")[1]) for f in list(d.glob("index_*.csv")) + list(d.glob("corrupt/*/*.npy")) + list(d.glob("corrupt/*/*.csv"))]
    k = max(used, default=-1) + 1                                                  # never reuse a shard number
    t0 = time.time(); n_done = 0; s_done = 0.0; rows = todo.to_dict("records")
    for s in range(0, len(rows), CFG["shard_utts"]):
        chunk = rows[s:s + CFG["shard_utts"]]; arrs, recs, off = [], [], 0
        if encode_rows is not None: res = encode_rows(chunk)              # test hook
        else:
            import torch
            dl = torch.utils.data.DataLoader(_WavDS(chunk), batch_size=4, num_workers=CFG["teacher_workers"], collate_fn=_collate,
                                             prefetch_factor=2 if CFG["teacher_workers"] else None)
            res = [(u, n, teacher.encode_wav(w)) for b in dl for u, n, w in b]
        by = {u: (n, a) for u, n, a in res}
        for r in chunk:
            n, a = by[r["uid"]]
            if a.shape != (teacher_frames(n), 2 * teacher.d_model) or not np.isfinite(a).all(): raise RuntimeError(f"{r['uid']}: bad teacher output {a.shape}")
            arrs.append(a); recs.append(dict(uid=r["uid"], speaker=r["speaker"], n_samples=n, shard=k, offset=off, length=len(a))); off += len(a)
        A = np.concatenate(arrs).astype(np.float16); tmp = d / f"shard_{k:05d}.tmp.npy"; np.save(tmp, A); os.replace(tmp, d / f"shard_{k:05d}.npy")
        atomic_json(dict(sha1=file_sha1(d / f"shard_{k:05d}.npy"), nbytes=(d / f"shard_{k:05d}.npy").stat().st_size, rows=int(A.shape[0]), dim=int(A.shape[1]),
                         utterances=len(chunk)), d / f"shard_{k:05d}.json")
        tmpc = d / f"index_{k:05d}.tmp.csv"; pd.DataFrame(recs).to_csv(tmpc, index=False); os.replace(tmpc, d / f"index_{k:05d}.csv")
        n_done += len(chunk); s_done += sum(r["n_samples"] for r in chunk) / SR; el = time.time() - t0
        log(f"  shard {k:05d} committed | {n_done:,}/{len(rows):,} | {n_done / el:.1f} utt/s ({s_done / el:.0f} s of audio per s) | "
            f"ETA {(len(rows) - n_done) / max(n_done / el, 1e-9) / 60:.0f} min" + (f" | GPU peak {gpu_mem().get('gpu_peak_gib', 0):.1f} GiB" if encode_rows is None else ""))
        k += 1
    allidx = _committed(d); miss = set(df.uid) - set(allidx.uid)
    if miss: raise RuntimeError(f"{len(miss)} utterances still missing from the cache")
    allidx.to_csv(d / "index.csv", index=False)
    log(f"[{split}] cache complete: {len(allidx):,} utterances, {allidx.length.sum():,} frames x {2 * teacher.d_model} "
        f"({allidx.length.sum() * 2 * teacher.d_model * 2 / 2**30:.2f} GiB)")
    return d

def find_cache(split):
    """The (single) compatible teacher cache folder for a split."""
    c = sorted({Path(p).parent for p in paths()["features"].glob(f"*/{split}/meta.json")})
    ok = []
    for d in c:
        m = json.load(open(d / "meta.json"))
        if (m.get("model") == CFG["teacher_model"] and m.get("dim") == 2 * WAVLM_DIM and m.get("cache_format") == CACHE_FORMAT
                and m.get("layer", "").startswith(f"hidden_states[{CFG['teacher_layer']}] ")): ok.append(d)
        else: log(f"  (ignoring incompatible cache {d}: model {m.get('model')}, format {m.get('cache_format')}, dim {m.get('dim')}, layer {m.get('layer')})")
    if not ok: raise FileNotFoundError(f"no compatible WavLM {2 * WAVLM_DIM}-D cache for {split} under {paths()['features']}: run the teacher-cache step")
    if len(ok) > 1: raise RuntimeError(f"several compatible caches for {split}: {ok}. Keep one.")
    return ok[0]

class TeacherCache:
    """Read-only access by utterance id. Shards are memory-mapped (never loaded whole into RAM); copied to local disk and sha1-checked first."""
    def __init__(self, d, uids, local=True):
        d = Path(d); idx = _committed(d).set_index("uid"); uids = list(uids); miss = set(uids) - set(idx.index)
        if miss: raise RuntimeError(f"{len(miss)} utterances are not in the teacher cache {d} (e.g. {sorted(miss)[:3]}): run the cache step")
        self.meta = json.load(open(d / "meta.json")); self.idx = idx.loc[uids]; self.dim = int(self.meta["dim"]); self.src = d
        bad = self.idx[self.idx.length != [teacher_frames(n) for n in self.idx.n_samples]]
        if len(bad): raise RuntimeError(f"{len(bad)} cache entries have lengths that do not match ((n - 400) // 320 + 1) // 2, e.g. {bad.index[:3].tolist()}")
        shards = sorted(set(int(s) for s in self.idx.shard))
        if local and not str(d).startswith("/content/drive"):      # cache already on local disk: verify every shard's sha1 in place (once per shard)
            for s in shards:
                a, sha = d / f"shard_{s:05d}.npy", json.load(open(d / f"shard_{s:05d}.json"))["sha1"]; okf = d / f"shard_{s:05d}.ok"
                if okf.exists() and okf.read_text() == sha and okf.stat().st_mtime >= a.stat().st_mtime: continue
                if file_sha1(a) != sha: raise RuntimeError(f"shard {a} fails its sha1 check: move it to corrupt/ (or re-run after deleting it)")
                okf.write_text(sha)
            self.d = d
        elif local:
            loc = Path(CFG["local_features"]) / d.parent.name / d.name; loc.mkdir(parents=True, exist_ok=True)
            need = sum((d / f"shard_{s:05d}.npy").stat().st_size for s in shards if not (loc / f"shard_{s:05d}.npy").exists()) / 2**30
            if need: require_space(loc, need, "the local teacher-cache copy", 2.0)
            for s in shards:
                a, b, sha = d / f"shard_{s:05d}.npy", loc / f"shard_{s:05d}.npy", json.load(open(d / f"shard_{s:05d}.json"))["sha1"]
                ok_flag = loc / f"shard_{s:05d}.ok"
                if b.exists() and ok_flag.exists() and ok_flag.read_text() == sha: continue
                for attempt in range(2):
                    shutil.copyfile(a, str(b) + ".part"); os.replace(str(b) + ".part", b)
                    if file_sha1(b) == sha: ok_flag.write_text(sha); break
                else: raise RuntimeError(f"shard {a} fails its sha1 check after copying twice: move it to corrupt/ and rebuild")
            self.d = loc
        else: self.d = d
        self._m = {}
    def get(self, uid):
        r = self.idx.loc[uid]; s = int(r.shard)
        if s not in self._m: self._m[s] = np.load(self.d / f"shard_{s:05d}.npy", mmap_mode="r")
        return np.asarray(self._m[s][int(r.offset): int(r.offset) + int(r.length)])
    def baselines(self, ranges, cache_file=None, median_frames=20000, seed=0):
        """Constant-prediction baselines over the ALIGNED teacher frames only: ranges = {uid: (first, count)}.
        mean: per-dimension mean (exact, streamed); median: per-dimension median of a seeded random sample of frames
        (the median is the L1-optimal constant; the mean is what the projection bias is initialised to)."""
        if cache_file and Path(cache_file).exists():
            z = np.load(cache_file); return z["mean"], z["median"]
        s = np.zeros(self.dim, np.float64); n = 0; rng = np.random.default_rng(seed); tot = sum(c for _, c in ranges.values())
        p = min(1.0, median_frames / max(tot, 1)); sample = []
        for u, (a, c) in ranges.items():
            x = self.get(u)[a:a + c].astype(np.float32); s += x.astype(np.float64).sum(0); n += len(x)
            m = rng.random(len(x)) < p
            if m.any(): sample.append(x[m])
        mean = (s / n).astype(np.float32); med = np.median(np.concatenate(sample), 0).astype(np.float32)
        if cache_file: np.savez(cache_file, mean=mean, median=med)
        return mean, med

def verify_cache_vs_fresh(teacher, cache, df, n=8, seed=0):
    """Re-extracts n utterances (always including the longest) and compares them with the cache."""
    rng = np.random.default_rng(seed); rows = df.sort_values("duration_s").iloc[[-1] + list(rng.choice(len(df) - 1, min(n, len(df)) - 1, replace=False))]
    out = []
    for r in rows.itertuples():
        f = teacher.encode_wav(load_audio(r.relpath, int(r.n_samples))); c = cache.get(r.uid)
        assert f.shape == c.shape, (r.uid, f.shape, c.shape)
        out.append(dict(uid=r.uid, seconds=round(r.duration_s, 2), frames=len(c), cos=_cos(f, c), maxdiff=float(np.abs(f.astype(np.float32) - c).max())))
    worst = min(o["cos"] for o in out)
    log(f"  cache vs fresh extraction on {len(out)} utterances (longest {rows.duration_s.max():.1f} s): min cosine {worst:.6f}, "
        f"max |diff| {max(o['maxdiff'] for o in out):.4f}")
    return dict(items=out, min_cos=worst, passed=worst > 0.9995)

# ####################################################################################################
# ---- asr1_student ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 | Zipformer-M student (your existing architecture, unchanged), source inspection, Swoosh port + verification,
#                    fbank frontend, projection, MEASURED teacher/student frame timing, alignment, masked L1, data loading
# ===========================================================================================
import os, re, sys, json, math, shutil, subprocess, importlib, hashlib
from pathlib import Path
import numpy as np

D_S = 512                                    # Zipformer-M final output (max of encoder_dim)
ZIP_ARGS = dict(output_downsampling_factor=2, downsampling_factor=(1, 2, 4, 8, 4, 2), num_encoder_layers=(2, 2, 3, 4, 3, 2),
                encoder_dim=(192, 256, 384, 512, 384, 256), encoder_unmasked_dim=(192, 192, 256, 256, 256, 192), query_head_dim=(32,) * 6,
                pos_head_dim=(4,) * 6, value_head_dim=(12,) * 6, num_heads=(4, 4, 4, 8, 4, 4), feedforward_dim=(512, 768, 1024, 1536, 1024, 768),
                cnn_module_kernel=(31, 31, 15, 15, 15, 31), pos_dim=48, causal=False)        # identical to your v1 code

# ------------------------------------------------------------------ frame counts (pure Python, unit-tested)
def fbank_frames(n): return (n + 80) // 160                         # kaldi fbank, 10 ms shift, snip_edges=False
def embed_frames(n): return (fbank_frames(n) - 7) // 2              # Conv2dSubsampling: T' = (T - 7) // 2  (~50 Hz)
def student_frames(n): return (embed_frames(n) + 1) // 2            # final Zipformer output (~25 Hz), = your v1 formula

# ------------------------------------------------------------------ alignment from measured frame centres (pure Python)
def centre(A, who, j): return A[who]["slope"] * j + A[who]["intercept"]          # samples
def offset_from(A):
    """Teacher frame index offset that puts frame centres closest: student j <-> teacher j + offset."""
    for k in ("student", "teacher"):
        if abs(A[k]["slope"] - 640.0) > 1.0: raise RuntimeError(f"{k} frame stride measured as {A[k]['slope']:.2f} samples, expected 640")
    return int(round((A["student"]["intercept"] - A["teacher"]["intercept"]) / 640.0))
def valid_pairs(n, Ts, Tt, A, off):
    """Number of usable pairs (student j, teacher j+off), j = 0..v-1: both frames exist and both frame centres lie inside the audio.
    Unmatched frames (teacher frames before off / after the last pair, frames centred beyond the end of the audio) are excluded."""
    if off < 0: raise RuntimeError("negative offsets are not supported (teacher frames would have to be dropped at the start of the student)")
    v = max(0, min(Ts, Tt - off))
    while v > 0 and (centre(A, "student", v - 1) > n or centre(A, "teacher", v - 1 + off) > n): v -= 1
    return v
def analytic_alignment():
    """Geometric cross-check from the layer definitions (kernel, stride, padding); the MEASURED values are what training uses."""
    def c(i, chain):
        x = float(i)
        for k, s, p in chain: x = x * s - p + (k - 1) / 2
        return x
    s0 = 160 * c(0, [(2, 2, 0), (3, 1, 0), (3, 2, 0), (3, 1, 0)]) + 80; s1 = 160 * c(1, [(2, 2, 0), (3, 1, 0), (3, 2, 0), (3, 1, 0)]) + 80
    t0 = wavlm_centre50(0.5); t1 = wavlm_centre50(2.5)            # WavLM target j = 50 Hz frames (2j, 2j+1): centre index 2j + 0.5
    return dict(student=dict(slope=s1 - s0, intercept=s0), teacher=dict(slope=t1 - t0, intercept=t0), source="analytic")

# ------------------------------------------------------------------ Swoosh: pure-PyTorch port of k2's implementation (k2/python/csrc/torch/v2/autograd/swoosh.cu)
K2_SWOOSH_SRC = r"""# Pure-PyTorch port of k2's Swoosh functions (k2/python/csrc/torch/v2/autograd/swoosh.cu), written by the MT2KD ASR Stage 1 pipeline.
# Same formulas, same fp32 compute, same 8-bit stochastically-rounded derivative in the autograd path. Differences from k2:
# the random numbers for the stochastic rounding come from torch's RNG (k2: its own torch::rand call) -> statistically identical, not bitwise.
import torch
_P = {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}

def _forward(x, shift, coeff, offset):                    # k2 SwooshForward
    x = x.to(torch.float32); xo = x - shift
    l = torch.log1p(torch.exp(xo)); l = torch.where(torch.isinf(l), xo, l)
    return l - coeff * x - offset

def _forward_and_deriv(x, shift, coeff, offset):          # k2 SwooshForwardAndDeriv
    x = x.to(torch.float32); xo = x - shift
    denom = 1.0 + torch.exp(xo); d = (1.0 - coeff) - 1.0 / denom
    ld = torch.log(denom); ld = torch.where(torch.isinf(ld), xo, ld)
    return ld - coeff * x - offset, d

class _SwooshFunction(torch.autograd.Function):           # k2 SwooshFunction (autograd path)
    @staticmethod
    def forward(ctx, x, shift, coeff, offset, dropout_prob):
        x = x.to(torch.float32).contiguous(); xo = x - shift; e = torch.exp(xo)
        g = 1.0 - 1.0 / (1.0 + e)
        l = torch.log1p(e); l = torch.where(torch.isinf(l), xo, l); y = l - coeff * x - offset
        if dropout_prob != 0.0:
            drop = torch.rand_like(x) < dropout_prob
            y = torch.where(drop, torch.zeros_like(y), y * (1.0 / (1.0 - dropout_prob))); g = torch.where(drop, torch.full_like(g, coeff), g)
        g_int = (g * (255.0 / 1.005) + torch.rand_like(x)).to(torch.uint8)
        ctx.save_for_backward(g_int); ctx.coeff = coeff; ctx.dropout_prob = dropout_prob
        return y
    @staticmethod
    def backward(ctx, y_grad):
        (g,) = ctx.saved_tensors
        f = (g.to(torch.float32) * (1.005 / 255.0) - ctx.coeff) / (1.0 - ctx.dropout_prob)
        return y_grad.to(torch.float32) * f, None, None, None, None

def _swoosh(x, which, dropout_prob):
    shift, coeff, offset = _P[which]
    if not x.requires_grad:                               # k2: logaddexp path when no gradient is needed
        x = x.to(torch.float32); return torch.logaddexp(torch.zeros(1, device=x.device), x - shift) - coeff * x - offset
    return _SwooshFunction.apply(x, shift, coeff, offset, float(dropout_prob))

def swoosh_l(x, dropout_prob=0.0): return _swoosh(x, "l", dropout_prob)
def swoosh_r(x, dropout_prob=0.0): return _swoosh(x, "r", dropout_prob)
def swoosh_l_forward(x): return _forward(x, *_P["l"])
def swoosh_r_forward(x): return _forward(x, *_P["r"])
def swoosh_l_forward_and_deriv(x): return _forward_and_deriv(x, *_P["l"])
def swoosh_r_forward_and_deriv(x): return _forward_and_deriv(x, *_P["r"])
"""
STUB_FUNCS = {"swoosh_l", "swoosh_r", "swoosh_l_forward", "swoosh_r_forward", "swoosh_l_forward_and_deriv", "swoosh_r_forward_and_deriv"}

def setup_zipformer():
    """Private copy of icefall's Zipformer source (unchanged except `import k2` -> the Swoosh port). Returns source information."""
    ZDIR = CFG["zipformer_dir"]; SRC_Z = "/content/icefall/egs/librispeech/ASR/zipformer"
    if not os.path.isdir(SRC_Z):
        subprocess.run("rm -rf /content/icefall; git clone --depth 1 https://github.com/k2-fsa/icefall.git /content/icefall", shell=True, check=True)
    os.makedirs(ZDIR + "/icefall", exist_ok=True)
    for f in ["zipformer.py", "scaling.py", "subsampling.py", "encoder_interface.py", "optim.py"]:
        if not os.path.exists(f"{ZDIR}/{f}"): shutil.copy(f"{SRC_Z}/{f}", f"{ZDIR}/{f}")
    o_ = open(f"{ZDIR}/optim.py").read()             # ScaledAdam / Eden, unchanged; lhotse is used only by its self-tests (fix_random_seed)
    if "from lhotse.utils import fix_random_seed" in o_:
        o_ = o_.replace("from lhotse.utils import fix_random_seed",
                        "def fix_random_seed(seed):  # MT2KD: replaces lhotse.utils.fix_random_seed (used only by optim.py's self-tests)\n"
                        "    import random as _r, numpy as _np; _r.seed(seed); _np.random.seed(seed); torch.manual_seed(seed)", 1)
        open(f"{ZDIR}/optim.py", "w").write(o_)
    if re.search(r"^\s*(from|import) lhotse", o_, re.M): raise RuntimeError("icefall optim.py imports lhotse in an unexpected place")
    open(ZDIR + "/icefall/__init__.py", "a").close()
    open(ZDIR + "/icefall/utils.py", "w").write(
        'import torch\n\ndef torch_autocast(device_type="cuda", **kwargs):\n    try:\n        return torch.amp.autocast(device_type, **kwargs)\n'
        '    except (AttributeError, TypeError):\n        return torch.cuda.amp.autocast(**kwargs)\n')
    if not os.path.exists(f"{ZDIR}/k2_swoosh.py") or open(f"{ZDIR}/k2_swoosh.py").read() != K2_SWOOSH_SRC: open(f"{ZDIR}/k2_swoosh.py", "w").write(K2_SWOOSH_SRC)
    t_ = open(f"{ZDIR}/scaling.py").read()
    if "\nimport k2\n" in t_: t_ = t_.replace("\nimport k2\n", "\nimport k2_swoosh as k2\n", 1); open(f"{ZDIR}/scaling.py", "w").write(t_)
    used = set(re.findall(r"\bk2\.(\w+)", t_ + open(f"{ZDIR}/zipformer.py").read() + open(f"{ZDIR}/subsampling.py").read()))
    if used - STUB_FUNCS: raise RuntimeError(f"the Zipformer source uses k2 functions the port does not provide: {used - STUB_FUNCS}")
    sys.path = [p for p in sys.path if p != ZDIR]; sys.path.insert(0, ZDIR); importlib.invalidate_caches()
    for m_ in [k for k in list(sys.modules) if k in ("zipformer", "scaling", "subsampling", "encoder_interface", "k2_swoosh", "optim") or k == "icefall" or k.startswith("icefall.")]:
        if not (getattr(sys.modules[m_], "__file__", "") or "").startswith(ZDIR): del sys.modules[m_]          # e.g. imported earlier from another folder
    try: commit = subprocess.run(["git", "-C", "/content/icefall", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: commit = "unknown"
    return dict(dir=ZDIR, icefall_commit=commit, k2_functions_used=sorted(used),
                files={f: file_sha1(f"{ZDIR}/{f}")[:12] for f in ["zipformer.py", "scaling.py", "subsampling.py", "k2_swoosh.py", "optim.py"]})

def build_models(teacher_dim):
    """Returns (student, proj) on CPU. Classes are created here because they need torch + icefall."""
    import torch, torch.nn as nn
    info = setup_zipformer()
    from subsampling import Conv2dSubsampling
    from zipformer import Zipformer2
    import zipformer as zmod, subsampling as smod
    assert zmod.__file__.startswith(CFG["zipformer_dir"]) and smod.__file__.startswith(CFG["zipformer_dir"]), "Zipformer imported from the wrong folder"
    class ZipformerStudent(nn.Module):
        """fbank (B, T, 80) -> Conv2dSubsampling (80 -> 192, ~50 Hz) -> Zipformer2 -> (B, ~T/4, 512) at ~25 Hz. Same keys as v1."""
        def __init__(self):
            super().__init__(); self.encoder_embed = Conv2dSubsampling(in_channels=80, out_channels=192); self.encoder = Zipformer2(**ZIP_ARGS)
        def forward(self, feats, feat_lens, return_embed=False):
            x, xl = self.encoder_embed(feats, feat_lens)
            enc, el = self.encoder(x.permute(1, 0, 2), xl, ~lengths_to_mask(xl, x.size(1)))
            return (enc.permute(1, 0, 2), el, x, xl) if return_embed else (enc.permute(1, 0, 2), el)
    class ASRProj(nn.Module):
        """Paper: a linear layer maps the student's ASR representation to the teacher dimension (512 -> 2048 for WavLM)."""
        def __init__(self): super().__init__(); self.lin = nn.Linear(D_S, teacher_dim)
        def forward(self, x): return self.lin(x)
    s = ZipformerStudent(); s.source_info = info
    return s, ASRProj()

def inspect_student(student, v1_ckpt=None):
    """Architecture actually instantiated from the imported source: stacks, layers, dims, parameters; optional key/shape match with v1."""
    enc = student.encoder; stacks = []
    for i, m in enumerate(enc.encoders):
        inner = getattr(m, "encoder", m); stacks.append(dict(type=type(m).__name__, downsample=int(enc.downsampling_factor[i]), layers=len(inner.layers),
                                                             dim=int(enc.encoder_dim[i]), params=sum(p.numel() for p in m.parameters())))
    R = dict(source=student.source_info, embed=type(student.encoder_embed).__name__, embed_in=80, embed_out=int(student.encoder_embed.out_channels)
             if hasattr(student.encoder_embed, "out_channels") else None, stacks=stacks, output_downsampling=int(enc.output_downsampling_factor),
             output_dim=int(max(enc.encoder_dim)), total_layers=sum(s["layers"] for s in stacks), params=sum(p.numel() for p in student.parameters()))
    exp = ZIP_ARGS
    assert [s["layers"] for s in stacks] == list(exp["num_encoder_layers"]) and [s["dim"] for s in stacks] == list(exp["encoder_dim"])
    assert [s["downsample"] for s in stacks] == list(exp["downsampling_factor"]) and R["output_dim"] == D_S and R["output_downsampling"] == 2
    if v1_ckpt and Path(v1_ckpt).exists():
        import torch
        st = torch.load(v1_ckpt, map_location="cpu", weights_only=False).get("student", {}); mine = student.state_dict()
        R["v1_match"] = dict(path=str(v1_ckpt), same_keys=set(st) == set(mine), same_shapes=all(tuple(st[k].shape) == tuple(mine[k].shape) for k in set(st) & set(mine)),
                             tensors=len(mine))
        if not (R["v1_match"]["same_keys"] and R["v1_match"]["same_shapes"]): raise RuntimeError(f"architecture differs from your v1 student: {R['v1_match']}")
    log(f"  Zipformer-M from {student.source_info['dir']} (icefall {student.source_info['icefall_commit'][:10]}): stacks "
        + ", ".join(f"{s['layers']}x{s['dim']}/{s['downsample']}" for s in stacks) + f" | {R['total_layers']} layers | output {R['output_dim']}-D, /2 -> 25 Hz | "
        f"{R['params'] / 1e6:.2f}M params" + (f" | identical keys+shapes to {Path(v1_ckpt).parent.name}/{Path(v1_ckpt).name}" if R.get("v1_match") else ""))
    return R

def verify_swoosh(device="cuda"):
    """Swoosh port vs reference: forward exactness (fp32/fp16 inputs, extremes), derivative, 8-bit stochastic backward (bounded, unbiased),
    the icefall modules that call it, and the real k2 if it happens to be installed."""
    import torch, torch.nn.functional as F
    setup_zipformer(); import k2_swoosh as P, scaling as S
    dev = torch.device(device if torch.cuda.is_available() else "cpu"); R = dict(source="k2/python/csrc/torch/v2/autograd/swoosh.cu (ported)")
    xs = torch.cat([torch.linspace(-60, 60, 200001), torch.tensor([-65504., -1e4, -100., 100., 1e4, 65504.])]).to(dev)
    for w, (sh, co, of) in {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}.items():
        x64 = xs.double(); ref = torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x64 - sh) - co * x64 - of; dref = torch.sigmoid(x64 - sh) - co
        tol = 2e-6 * (1 + ref.abs())
        y = getattr(P, f"swoosh_{w}_forward")(xs); yd, d = getattr(P, f"swoosh_{w}_forward_and_deriv")(xs)
        e_fwd = float(((y.double() - ref).abs() / tol).max()); e_fd = float(((yd.double() - ref).abs() / tol).max()); e_d = float((d.double() - dref).abs().max())
        y16 = getattr(P, f"swoosh_{w}_forward")(xs.half()); y16_ok = bool(torch.isfinite(y16).all()) and y16.dtype == torch.float32
        xg = xs.clone().requires_grad_(True); yg = getattr(P, f"swoosh_{w}")(xg); yg.backward(torch.ones_like(yg)); g = xg.grad.double()
        q = 1.005 / 255.0; e_q = float((g - dref).abs().max()); bias = float((g - dref).mean())
        xr = torch.randn(2_000_000, device=dev) * 6; xr.requires_grad_(True); yr = getattr(P, f"swoosh_{w}")(xr); yr.backward(torch.ones_like(yr))
        bias_r = float((xr.grad.double() - (torch.sigmoid(xr.detach().double() - sh) - co)).mean())
        e_nog = float(((getattr(P, f"swoosh_{w}")(xs).double() - ref).abs() / tol).max())
        mod = (S.SwooshL if w == "l" else S.SwooshR)(); e_mod = float(((mod(xs).double() - ref).abs() / tol).max())
        own = (S.SwooshLForward if w == "l" else S.SwooshRForward)(xs); e_own = float(((own.double() - ref).abs() / tol)[torch.isfinite(own)].max())
        R[w] = dict(forward_err_rel=e_fwd, forward_and_deriv_err_rel=e_fd, deriv_maxerr=e_d, fp16_input_finite_fp32_output=y16_ok, no_grad_path_err_rel=e_nog,
                    backward_maxerr=e_q, backward_quant_step=q, backward_bias_grid=bias, backward_bias_random=bias_r, icefall_module_err_rel=e_mod,
                    icefall_reference_SwooshForward_err_rel=e_own)
        ok = e_fwd <= 1 and e_fd <= 1 and e_d < 1e-6 and y16_ok and e_nog <= 1 and e_q <= q + 1e-6 and abs(bias_r) < 2e-4 and e_mod <= 1 and e_own <= 1
        R[w]["passed"] = ok
        log(f"  swoosh_{w}: forward rel.err {e_fwd:.2f} (<=1 = within 2e-6 relative) | deriv {e_d:.1e} | fp16 inputs finite {y16_ok} | backward |err| {e_q:.4f} "
            f"<= 8-bit step {q:.4f}, mean bias {bias_r:+.1e} | icefall SwooshL/R module {e_mod:.2f}, icefall reference {e_own:.2f} -> {'OK' if ok else 'FAIL'}")
    # the fused ActivationDropoutAndLinear path (uses forward_and_deriv in backward) vs autograd through the reference, dropout 0
    torch.manual_seed(0); lin = S.ActivationDropoutAndLinear(64, 32, bias=True, activation="SwooshL", dropout_p=0.0).to(dev)
    x = (torch.randn(50, 64, device=dev) * 4).requires_grad_(True); y = lin(x); y.square().sum().backward(); gx = x.grad.clone(); gw = lin.weight.grad.clone()
    x2 = x.detach().double().requires_grad_(True); W = lin.weight.detach().double().requires_grad_(True); b = lin.bias.detach().double()
    y2 = F.linear(torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x2 - 4) - 0.08 * x2 - 0.035, W, b); y2.square().sum().backward()
    R["fused_linear"] = dict(out_rel=float((y.double() - y2).abs().max() / y2.abs().max()), dx_rel=float((gx.double() - x2.grad).abs().max() / x2.grad.abs().max()),
                             dw_rel=float((gw.double() - W.grad).abs().max() / W.grad.abs().max()))
    R["fused_linear"]["passed"] = max(R["fused_linear"].values()) < 1e-4
    log(f"  ActivationDropoutAndLinear(SwooshL) vs float64 autograd: output {R['fused_linear']['out_rel']:.1e}, dx {R['fused_linear']['dx_rel']:.1e}, "
        f"dW {R['fused_linear']['dw_rel']:.1e} (relative)")
    try:
        import k2 as real
        if not hasattr(real, "swoosh_l_forward"): raise ImportError
        xr = torch.randn(100000, device=dev) * 8
        R["real_k2"] = dict(forward_l_maxdiff=float((real.swoosh_l_forward(xr) - P.swoosh_l_forward(xr)).abs().max()),
                            forward_r_maxdiff=float((real.swoosh_r_forward(xr) - P.swoosh_r_forward(xr)).abs().max()),
                            deriv_l_maxdiff=float((real.swoosh_l_forward_and_deriv(xr)[1] - P.swoosh_l_forward_and_deriv(xr)[1]).abs().max()))
        log(f"  compared with the installed k2: {R['real_k2']}")
    except Exception: R["real_k2"] = "k2 not installed for this PyTorch (not downgraded); compared against k2's published formulas and algorithm instead"
    R["passed"] = R["l"]["passed"] and R["r"]["passed"] and R["fused_linear"]["passed"]
    R["claim"] = ("forward and derivative: same formulas as k2 in fp32 (verified to <= 2e-6 relative against float64); autograd backward: same 8-bit "
                  "stochastic quantisation of the derivative as k2 (bounded by one step, unbiased); random numbers differ, so results are not bit-identical to a k2 run")
    return R

def measure_student_timing(student):
    """Measures, with the imported code, where each student output frame sits in time:
    (1) kaldi fbank frame centre from impulse probes through the real fbank() below, (2) Conv2dSubsampling index map from perturbation probes,
    (3) the output downsampler (SimpleDownsample) index map. Returns a linear map j -> centre (samples)."""
    import torch
    S = np.arange(8000, 8000 + 1280, 4); arg = []
    for s in S:
        w = np.zeros(16000, np.float32); w[s] = 0.5; arg.append(int(fbank(w)[:100].sum(1).argmax()))
    arg = np.array(arg); cen = {int(k): float(S[arg == k].mean()) for k in np.unique(arg) if (arg == k).sum() >= 8}
    c_fb = float(np.median([c - 160 * k for k, c in cen.items()]))
    dev = next(student.parameters()).device; was = student.training; student.eval(); L = 80
    g = torch.Generator().manual_seed(0); x = torch.randn(1, L, 80, generator=g).to(dev); xl = torch.tensor([L], device=dev)
    with torch.no_grad():
        y0, _ = student.encoder_embed(x, xl); infl = [[] for _ in range(y0.shape[1])]
        for k in range(L):
            x2 = x.clone(); x2[:, k] += 5.0; ch = (student.encoder_embed(x2, xl)[0] - y0).abs().amax(-1)[0] > 1e-5
            for i in torch.nonzero(ch).flatten().tolist(): infl[i].append(k)
        ii = [i for i in range(6, y0.shape[1] - 6) if infl[i]]; ci = np.array([(min(infl[i]) + max(infl[i])) / 2 for i in ii])
        a_e, b_e = np.polyfit(ii, ci, 1); res_e = float(np.abs(np.polyval([a_e, b_e], ii) - ci).max())
        ds = student.encoder.downsample_output; C = max(ZIP_ARGS["encoder_dim"]); T2 = 20
        z = torch.randn(T2, 1, C, generator=g).to(dev); o0 = ds(z); inf2 = [[] for _ in range(o0.shape[0])]
        for t in range(T2):
            z2 = z.clone(); z2[t] += 3.0; ch = (ds(z2) - o0).abs().amax(-1)[:, 0] > 1e-6
            for j in torch.nonzero(ch).flatten().tolist(): inf2[j].append(t)
        jj = [j for j in range(o0.shape[0] - 1) if inf2[j]]; cj = np.array([(min(inf2[j]) + max(inf2[j])) / 2 for j in jj])
        a_d, b_d = np.polyfit(jj, cj, 1); res_d = float(np.abs(np.polyval([a_d, b_d], jj) - cj).max())
    student.train(was)
    slope = 160 * a_e * a_d; icpt = c_fb + 160 * (a_e * b_d + b_e)
    R = dict(fbank_centre_offset_samples=c_fb, embed_slope=float(a_e), embed_intercept=float(b_e), embed_fit_residual=res_e, embed_influence_example={int(i): infl[i] for i in ii[:2]},
             down_slope=float(a_d), down_intercept=float(b_d), down_fit_residual=res_d, down_influence_example={int(j): inf2[j] for j in jj[:3]},
             centre_samples=dict(slope=float(slope), intercept=float(icpt)))
    log(f"  student timing (measured): fbank frame k centre = 160 k + {c_fb:.1f} samples | Conv2dSubsampling: embed frame i <- fbank frames centred at "
        f"{a_e:.3f} i + {b_e:.3f} (residual {res_e:.3f}) | output downsample: j <- embed {a_d:.3f} j + {b_d:.3f} | student j centre = {slope / 16:.1f} j + {icpt / 16:.2f} ms")
    return R

def lengths_to_mask(lens, T):
    import torch
    return torch.arange(T, device=lens.device)[None, :] < lens[:, None]

def set_batch_count(model, bc):
    for name, mod in model.named_modules():
        if hasattr(mod, "batch_count"): mod.batch_count = bc
        if hasattr(mod, "name"): mod.name = name

def fbank(wav):
    """Student frontend = your v1 ASR fbank (80 bins, 25 ms / 10 ms, povey window, snip_edges=False)."""
    import torch, torchaudio.compliance.kaldi as kaldi
    return kaldi.fbank(torch.from_numpy(wav)[None], num_mel_bins=80, frame_length=25.0, frame_shift=10.0, dither=0.0, energy_floor=1e-10,
                       sample_frequency=float(SR), snip_edges=False, low_freq=20.0, high_freq=-400.0, remove_dc_offset=True,
                       preemphasis_coefficient=0.97, window_type="povey", use_energy=False)

# ------------------------------------------------------------------ alignment and loss
def align_targets(t, t_lens, s_lens, n_samples, S_T, A, off):
    """t [B, Tt, D] padded teacher -> (targets [B, S_T, D] fp32, valid pairs [B]): student frame j <-> teacher frame j + off for j < v,
    v = valid_pairs(...). Frames without a valid partner are excluded from the loss by the mask (no interpolation, no padding as target)."""
    import torch
    B, _, Dm = t.shape; out = torch.zeros(B, S_T, Dm, device=t.device, dtype=torch.float32); v = torch.zeros(B, dtype=torch.long, device=t.device)
    for b in range(B):
        n_ = valid_pairs(int(n_samples[b]), int(s_lens[b]), int(t_lens[b]), A, off)
        out[b, :n_] = t[b, off:off + n_].float(); v[b] = n_
    return out, v

def masked_l1(pred, tgt, valid):
    """Paper's ASR KD loss: L1 between projected student and teacher features, mean over the teacher dims and over VALID frames only."""
    mask = lengths_to_mask(valid, pred.size(1)).float(); nf = mask.sum().clamp(min=1)
    return ((pred.float() - tgt).abs().mean(-1) * mask).sum() / nf, nf

# ------------------------------------------------------------------ data
def make_batches(lens, epoch, seed, shuffle, max_sec, max_utts, idx=None):
    """Duration batches over `idx` (default: all): padded seconds <= max_sec and <= max_utts items; sorted inside shuffled pools of 1000;
    deterministic for (seed, epoch, idx)."""
    lens = np.asarray(lens); idx = np.arange(len(lens)) if idx is None else np.asarray(sorted(idx))
    rng = np.random.default_rng(seed + 1000 * epoch + (0 if len(idx) == len(lens) else 7919 * len(idx)))
    order = idx[rng.permutation(len(idx))] if shuffle else idx; out = []
    for s in range(0, len(order), 1000):
        ids = order[s:s + 1000]; ids = ids[np.argsort(lens[ids], kind="stable")]; cur, mx = [], 0
        for i in ids:
            nm = max(mx, lens[i])
            if cur and (nm * (len(cur) + 1) / SR > max_sec or len(cur) >= max_utts): out.append(cur); cur, nm = [], lens[i]
            cur.append(int(i)); mx = nm
        if cur: out.append(cur)
    if shuffle: out = [out[j] for j in rng.permutation(len(out))]
    return out

class KDData:
    def __init__(self, df, cache): self.r = df.to_dict("records"); self.c = cache
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        import torch
        r = self.r[i]; w = load_audio(r["relpath"], int(r["n_samples"])); t = self.c.get(r["uid"])
        return dict(uid=r["uid"], n=len(w), fb=fbank(w), t=torch.from_numpy(t))

def collate(b):
    import torch, math
    from torch.nn.utils.rnn import pad_sequence
    return dict(uid=[x["uid"] for x in b], n=torch.tensor([x["n"] for x in b]),
                fb=pad_sequence([x["fb"] for x in b], batch_first=True, padding_value=math.log(1e-10)), fb_lens=torch.tensor([x["fb"].size(0) for x in b]),
                t=pad_sequence([x["t"] for x in b], batch_first=True), t_lens=torch.tensor([x["t"].size(0) for x in b]))

def loader(df, cache, batches, num_workers, seed=0):
    import torch
    def wi(_): torch.set_num_threads(1)
    return torch.utils.data.DataLoader(KDData(df, cache), batch_sampler=batches, num_workers=num_workers, collate_fn=collate, pin_memory=True,
                                       worker_init_fn=wi, prefetch_factor=4 if num_workers else None,
                                       generator=torch.Generator().manual_seed(int(seed)))      # own generator: the global RNG streams are not consumed

# ####################################################################################################
# ---- asr1_train ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 | alignment measurement, smoke test (critical checks, machine-readable report), resumable KD training,
#                    validation, early stopping, memory probe, plots, report, single-file export to MT2KD/ASR_outputs/
#   Resume semantics (stated honestly): training resumes from the last saved UPDATE BOUNDARY (local / Drive checkpoints),
#   with model, projection, optimiser, GradScaler, schedule position, epoch, batch index, seen utterances, RNG states and history restored.
#   Updates made after that save are recomputed; replays use the same batches and RNG state but GPU kernels are not bit-deterministic.
# ===========================================================================================
import os, json, math, time, shutil
from pathlib import Path
import numpy as np, pandas as pd

def ALIGN_F(): return paths()["logs"] / "alignment.json"
def SMOKE_F(): return paths()["logs"] / "smoke" / "smoke_report.json"
V1_CKPT = lambda: paths()["root"] / "stage1_asr_kd" / "best.pt"

# ===========================================================================================
# alignment: measured with the imported WavLM and Zipformer code
# ===========================================================================================
def measure_alignment(teacher, train_df, dev_df):
    """Measures both frame-centre maps, checks them against the layer-geometry analysis, derives the offset, and tests the pairing on
    short / medium / near-30 s / > 30 s utterances. Writes logs/ASR_Stage1/alignment.json (training reads it)."""
    import torch
    T = measure_teacher_timing(teacher); student, _ = build_models(2 * WAVLM_DIM); S = measure_student_timing(student)
    M = dict(student=S["centre_samples"], teacher=T["centre_samples"]); An = analytic_alignment()
    dev_s = {k: abs(M[k]["intercept"] - An[k]["intercept"]) for k in ("student", "teacher")}
    off = offset_from(M); res_ms = (centre(M, "student", 0) - centre(M, "teacher", off)) / 16
    log(f"  measured vs analytic frame centres: student {M['student']['intercept'] / 16:.2f} vs {An['student']['intercept'] / 16:.2f} ms, "
        f"teacher {M['teacher']['intercept'] / 16:.2f} vs {An['teacher']['intercept'] / 16:.2f} ms | both strides {M['student']['slope'] / 16:.1f} / "
        f"{M['teacher']['slope'] / 16:.1f} ms")
    log(f"  => student frame j <-> teacher frame j + {off}; remaining centre difference {res_ms:+.2f} ms (for offset 0 it would be "
        f"{(centre(M, 'student', 0) - centre(M, 'teacher', 0)) / 16:+.1f} ms)")
    table = []
    for k, r in pick_test_utts(train_df, dev_df).items():
        if r is None: continue
        n = int(r.n_samples); Ts, Tt = student_frames(n), teacher_frames(n); v = valid_pairs(n, Ts, Tt, M, off)
        table.append(dict(kind=k, uid=r.uid, seconds=round(n / SR, 3), student_frames=Ts, teacher_frames=Tt, pairs=v, unmatched_teacher=Tt - v,
                          unmatched_student=Ts - v, first_pair_ms=[centre(M, "student", 0) / 16, centre(M, "teacher", off) / 16],
                          last_pair_ms=[centre(M, "student", v - 1) / 16, centre(M, "teacher", v - 1 + off) / 16], audio_end_ms=1000 * n / SR))
        log(f"  {k:7s} {r.uid}: {n / SR:6.2f} s | student {Ts} | teacher {Tt} | pairs {v} (unmatched: teacher {Tt - v}, student {Ts - v}) | "
            f"last pair centres {centre(M, 'student', v - 1) / 16:.0f} / {centre(M, 'teacher', v - 1 + off) / 16:.0f} ms, audio ends {1000 * n / SR:.0f} ms")
    ok = (max(dev_s.values()) <= 16 and abs(res_ms) <= 20 and all(t["pairs"] > 0 and t["unmatched_student"] <= 1 for t in table))
    A = dict(measured=M, analytic=An, measured_minus_analytic_samples=dev_s, offset=off, residual_ms=res_ms, teacher_detail=T, student_detail=S,
             table=table, passed=ok, note="training uses the MEASURED maps; the analytic values are an independent cross-check")
    atomic_json(A, ALIGN_F()); log(f"  alignment {'PASSED' if ok else 'FAILED'} -> {ALIGN_F()}")
    del student; torch.cuda.empty_cache()
    return A

def load_alignment():
    if not ALIGN_F().exists(): raise FileNotFoundError(f"{ALIGN_F()} missing: run the alignment cell")
    A = json.load(open(ALIGN_F()))
    if not A.get("passed"): raise RuntimeError("the alignment measurement did not pass; see alignment.json")
    return A

# ===========================================================================================
# shared context
# ===========================================================================================
def context(preset, local=True, with_alignment=True):
    """Manifests, local audio, teacher caches, and (with_alignment) the measured alignment and aligned-frame baselines."""
    sub = PRESETS[preset]["train_subset"]; P = paths(preset); ensure_dirs(P)
    tr = load_manifest(sub); dv = load_manifest("dev")
    materialize(tr, "train-clean-100"); materialize(dv, CFG["val_split"])
    dtr, ddv = find_cache("train-clean-100"), find_cache(CFG["val_split"])
    TCtr, TCdv = TeacherCache(dtr, tr.uid, local), TeacherCache(ddv, dv.uid, local)
    base = dict(preset=preset, sub=sub, P=P, tr=tr, dv=dv, TCtr=TCtr, TCdv=TCdv, teacher_meta=TCtr.meta, manifest_sha=sha_of(sorted(tr.uid)), dev_sha=sha_of(sorted(dv.uid)))
    if not with_alignment: return base
    A = load_alignment(); M = A["measured"]; off = A["offset"] if CFG["align"] == "shift" else 0
    rng = {}
    for name, df, TC in [("train", tr, TCtr), ("dev", dv, TCdv)]:
        Tt = TC.idx.loc[df.uid, "length"].values; Ts = np.array([student_frames(n) for n in df.n_samples])
        v = np.array([valid_pairs(int(n), int(s), int(t), M, off) for n, s, t in zip(df.n_samples, Ts, Tt)])
        if (v <= 0).any(): raise RuntimeError(f"{name}: utterances without a valid frame pair: {df.uid[v <= 0].tolist()[:3]}")
        log(f"{name}: {len(df):,} utts | pairs {v.sum():,} of {Ts.sum():,} student / {Tt.sum():,} teacher frames | unmatched per utterance: "
            f"student max {int((Ts - v).max())}, teacher {int((Tt - v).min())}-{int((Tt - v).max())}")
        if name == "train": rng = {u: (off, int(x)) for u, x in zip(df.uid, v)}
    mean, med = TCtr.baselines(rng, cache_file=P["logs"] / f"baselines_{sha_of(sorted(tr.uid))}_{off}.npz")
    return dict(base, A=M, offset=off, align=A, mean=mean, median=med)

def amp_dtype():
    import torch
    return torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16

def forward_loss(student, proj, b, C, dev, amp):
    import torch
    fb, fl = b["fb"].to(dev, non_blocking=True), b["fb_lens"].to(dev); t = b["t"].to(dev, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=amp):
        enc, el = student(fb, fl); pred = proj(enc)
    tgt, v = align_targets(t, b["t_lens"], el, b["n"], enc.size(1), C["A"], C["offset"])
    l1, nf = masked_l1(pred, tgt, v)
    with torch.no_grad():
        mask = lengths_to_mask(v, enc.size(1)).float()
        mu, md = torch.as_tensor(C["mean"], device=dev), torch.as_tensor(C["median"], device=dev)
        b_mean = ((tgt - mu).abs().mean(-1) * mask).sum() / nf; b_med = ((tgt - md).abs().mean(-1) * mask).sum() / nf
        cos = (torch.nn.functional.cosine_similarity(pred.float(), tgt, dim=-1) * mask).sum() / nf
    return l1, dict(base_mean=float(b_mean), base_median=float(b_med), cos=float(cos), frames=int(nf), enc=enc, el=el, tgt=tgt, v=v, pred=pred)

def evaluate(student, proj, C, dev, amp):
    """Frame-weighted validation metrics over the whole dev manifest (no parameter updates)."""
    import torch
    was = student.training; student.eval(); proj.eval(); s = bm = bd = c_ = 0.0; n = 0; nu = 0
    bt = make_batches(C["dv"].n_samples.values, 0, CFG["seed"], False, 300.0, CFG["max_batch_utts"])
    with torch.no_grad():
        for b in loader(C["dv"], C["TCdv"], bt, CFG["num_workers"]):
            l1, i = forward_loss(student, proj, b, C, dev, amp)
            if not torch.isfinite(l1): raise FloatingPointError("non-finite validation loss")
            f = i["frames"]; s += float(l1) * f; bm += i["base_mean"] * f; bd += i["base_median"] * f; c_ += i["cos"] * f; n += f; nu += len(b["uid"])
    student.train(was); proj.train(was)
    assert nu == len(C["dv"]), f"validation covered {nu} of {len(C['dv'])} utterances"
    return dict(val_l1=s / n, val_base_mean_l1=bm / n, val_base_median_l1=bd / n, val_ratio_mean=s / bm, val_ratio_median=s / bd, val_cos=c_ / n,
                val_frames=n, val_utts=nu)

def new_models(C, dev):
    import torch
    set_seed(CFG["seed"]); student, proj = build_models(int(C["teacher_meta"]["dim"]))
    if CFG["init_bias"] == "train_mean":
        with torch.no_grad(): proj.lin.bias.copy_(torch.as_tensor(C["mean"]))
    return student.to(dev), proj.to(dev)

def make_optimizer(student, proj, cfg):
    """(optimizer, scheduler). Paper / icefall: ScaledAdam(named params, clipping_scale=2.0) + Eden(lr_batches, lr_epochs, warmup_start=0.1).
    icefall groups parameters by per-module 'lr_scale'; no Zipformer module defines one, so a single group is identical to the recipe.
    Eden's lr is a pure function of (global step, epoch), both stored in the training state, so a resume reproduces it exactly."""
    import torch
    if cfg["optimizer"] == "scaled_adam":
        setup_zipformer(); from optim import ScaledAdam, Eden
        named = [("student." + n, p) for n, p in student.named_parameters()] + [("proj." + n, p) for n, p in proj.named_parameters()]
        opt = ScaledAdam([dict(named_params=named, lr=cfg["base_lr"])], lr=cfg["base_lr"], clipping_scale=cfg["clipping_scale"])
        sch = Eden(opt, cfg["lr_batches"], cfg["lr_epochs"], warmup_batches=cfg["eden_warmup_batches"], warmup_start=cfg["eden_warmup_start"])
        sch.step_epoch(0); sch.step_batch(0)
        return opt, sch
    if cfg["optimizer"] == "adamw":
        params = list(student.parameters()) + list(proj.parameters())
        return torch.optim.AdamW([dict(params=[p for p in params if p.ndim >= 2], weight_decay=cfg["weight_decay"]),
                                  dict(params=[p for p in params if p.ndim < 2], weight_decay=0.0)], lr=cfg["lr"]), None
    raise ValueError(f"unknown optimizer {cfg['optimizer']}")

def memory_probe(student, proj, C, dev, amp, max_sec):
    """Measured (not assumed) peak GPU memory of the worst-case training batch: the longest training utterances, padded, forward + backward.
    The batch limit is lowered by 20 % steps until the peak is below mem_fraction of the GPU."""
    import torch
    L = int(C["tr"].n_samples.max()); total = torch.cuda.get_device_properties(0).total_memory; scaler = torch.amp.GradScaler("cuda", enabled=amp == torch.float16)
    while max_sec >= 20:
        nb = max(1, min(CFG["max_batch_utts"], int(max_sec * SR // L))); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); peak = float("nan")
        try:
            b = dict(fb=torch.randn(nb, fbank_frames(L), 80), fb_lens=torch.full((nb,), fbank_frames(L)), t=torch.randn(nb, teacher_frames(L), int(C["teacher_meta"]["dim"])).half(),
                     t_lens=torch.full((nb,), teacher_frames(L)), n=torch.full((nb,), L))
            student.train(); proj.train(); l1, _ = forward_loss(student, proj, b, C, dev, amp); scaler.scale(l1).backward()
            peak = torch.cuda.max_memory_allocated(); ok = peak < CFG["mem_fraction"] * total
        except oom_error(): ok = False
        for p in list(student.parameters()) + list(proj.parameters()): p.grad = None
        b = l1 = None; torch.cuda.empty_cache()
        log(f"  memory probe: {nb} x {L / SR:.1f} s (longest training utterance) = {nb * L / SR:.0f} s padded -> peak {peak / 2**30:.2f} GiB of "
            f"{total / 2**30:.1f} GiB {'OK' if ok else '-> too much, reducing'}")
        if ok: return max_sec, peak / 2**30
        max_sec *= 0.8
    raise RuntimeError("even a 20 s batch does not fit in GPU memory")

# ===========================================================================================
# smoke test
# ===========================================================================================
import re
UNUSED_ICEFALL = re.compile(r"(^|\.)layers\.\d+\.bypass_scale$")     # Zipformer2EncoderLayer.bypass_scale: defined, never used by icefall

def smoke_test(preset="pilot", n_utts=4):
    """All checks; critical failures block training. Writes logs/ASR_Stage1/smoke/smoke_report.json."""
    import torch, gc
    dev = torch.device("cuda"); amp = amp_dtype(); R = []; st = {}; out = SMOKE_F().parent; out.mkdir(parents=True, exist_ok=True)
    sc = 1024.0 if amp == torch.float16 else 1.0
    def _sbw(loss):          # fp16 (T4): scaled backward + unscaled gradients (what GradScaler does); bf16 (A100): plain backward
        (loss * sc).backward()
        if sc != 1.0:
            for p in list(st["S"].parameters()) + list(st["Pj"].parameters()):
                if p.grad is not None: p.grad.div_(sc)
    def chk(no, name, fn, critical=True):
        t0 = time.time()
        try: info = fn(); R.append(dict(no=no, check=name, critical=critical, status="PASS", info=info, seconds=round(time.time() - t0, 1))); log(f"[{no}] PASS  {name}: {info}")
        except Exception as e:
            R.append(dict(no=no, check=name, critical=critical, status="FAIL", info=f"{type(e).__name__}: {e}", seconds=round(time.time() - t0, 1)))
            log(f"[{no}] {'FAIL' if critical else 'WARN'}  {name}: {type(e).__name__}: {e}")
    def d1():
        P = paths(); sel = json.load(open(P["meta"] / "subset_selection.json")); s2, dv = load_manifest("2h"), load_manifest("dev"); s10 = s2
        tgt = 3600 * CFG["subsets"]["2h"]; assert tgt <= s2.duration_s.sum() < tgt + s2.duration_s.max() + 1e-6, f"2 h subset is {s2.duration_s.sum():.1f} s"
        assert not set(s10.uid) & set(dv.uid) and not set(s10.speaker) & set(dv.speaker), "train/dev overlap"
        kind, _ = find_source("test-clean")
        if kind: te = inventory("test-clean"); assert not set(s10.uid) & set(te.uid) and not set(s10.speaker) & set(te.speaker), "train/test overlap"
        for split, df in [("train-clean-100", s10), (CFG["val_split"], dv)]:
            ver = set(pd.read_csv(P["meta"] / f"decode_verified_v2_{split}.csv", dtype={"uid": str}).uid); miss = set(df.uid) - ver
            assert not miss, f"{len(miss)} {split} manifest files were never fully decoded"
        assert all(df.split.eq(sp).all() for df, sp in [(s10, "train-clean-100"), (dv, CFG["val_split"])])
        st["dfs"] = (s2, dv)
        return (f"2h {len(s2)} utts / {s2.duration_s.sum():.1f} s | dev {sel['validation']['coverage']} | "
                f"ids unique, transcripts present, all files fully decoded, no train/dev{'/test' if kind else ''} overlap")
    def d2():
        C = context(preset, with_alignment=False); st["C"] = C
        rows = C["tr"].sort_values("duration_s").iloc[np.linspace(0, len(C["tr"]) - 1, n_utts).astype(int)].reset_index(drop=True); st["rows"] = rows
        ws = [load_audio(r.relpath, int(r.n_samples)) for r in rows.itertuples()]; st["w"] = ws
        return f"{n_utts} utterances {[len(w) for w in ws]} samples, mono float32 {SR} Hz, finite, exact lengths"
    def w1():
        st["T"] = WavLMTeacher(); T = st["T"]; return f"{T.info['model']} commit {T.commit[:10]}, {T.n_layers} layers, using layer {T.layer}, d_model {T.d_model}, {T.dtype_name}, eval, frozen"
    def w2():
        s2, dv = st["dfs"]; Rw = verify_wavlm(st["T"], st["C"]["tr"], dv)
        return f"frame rule OK, layer identity OK, pairing cos {min(Rw['pairing']['cos']):.6f}, fp16 vs fp32 cos {Rw['fp16_vs_fp32_cos']}, long utterances whole, cache dtype {Rw['dtype_used']}"
    def w3():
        C = st["C"]; nt, nd = len(C["TCtr"].idx), len(C["TCdv"].idx)
        assert C["TCtr"].idx.index.is_unique and C["TCdv"].idx.index.is_unique and nt == len(C["tr"]) and nd == len(C["dv"])
        for TC in (C["TCtr"], C["TCdv"]):
            for k in ["model", "layer", "pairing", "dim", "frames_rule", "normalize", "cache_format"]: assert TC.meta[k] == st["T"].info.get(k, TC.meta[k]), k
            assert TC.meta["commit"] == st["T"].commit, "cache made with a different WavLM revision"
        return f"one entry per manifest utterance (train {nt}, dev {nd}); shards sha1-verified; metadata matches the loaded WavLM (commit, layer, pairing, normalisation)"
    def w4():
        C = st["C"]; a = verify_cache_vs_fresh(st["T"], C["TCtr"], C["tr"], 6); b = verify_cache_vs_fresh(st["T"], C["TCdv"], C["dv"], 4, seed=1)
        assert a["passed"] and b["passed"]; return f"min cosine train {a['min_cos']:.6f}, dev {b['min_cos']:.6f} (includes the longest utterances)"
    def a1():
        s2, dv = st["dfs"]; A = measure_alignment(st["T"], st["C"]["tr"], dv); assert A["passed"]; st["T"].enc.cpu(); del st["T"]; gc.collect(); torch.cuda.empty_cache()
        st["C"] = context(preset)                 # re-read: uses the alignment just measured
        return f"student j <-> teacher j+{A['offset']}, centre difference {A['residual_ms']:+.2f} ms, measured = analytic within {max(A['measured_minus_analytic_samples'].values()):.1f} samples"
    def s1():
        Rs = verify_swoosh(); st["swoosh"] = Rs; assert Rs["passed"]; return "forward exact to 2e-6 rel., derivative exact, backward = k2's 8-bit stochastic scheme (bounded, unbiased)"
    def z1():
        C = st["C"]; st["S"], st["Pj"] = new_models(C, dev); Ri = inspect_student(st["S"], V1_CKPT()); st["arch"] = Ri
        return f"{Ri['total_layers']} layers, stacks {[s['layers'] for s in Ri['stacks']]}, dims {[s['dim'] for s in Ri['stacks']]}, {Ri['params'] / 1e6:.2f}M params" + \
               (", same keys/shapes as v1" if Ri.get("v1_match") else "")
    def z2():
        b = collate([KDData(st["rows"], st["C"]["TCtr"])[i] for i in range(n_utts)]); st["b"] = b
        assert b["fb"].shape[-1] == 80 and b["fb_lens"].tolist() == [fbank_frames(n) for n in b["n"].tolist()] and torch.isfinite(b["fb"]).all()
        return f"fbank {tuple(b['fb'].shape)} (80-D, 10 ms), lens {b['fb_lens'].tolist()}"
    def z3():
        S = st["S"]; b = st["b"]; S.eval()
        with torch.no_grad():
            enc, el, emb, xl = S(b["fb"].to(dev), b["fb_lens"].to(dev), return_embed=True); diffs = []
            for k in range(n_utts):                                         # padding mask: padded batch == each utterance alone
                L = int(b["fb_lens"][k]); e1, l1_ = S(b["fb"][k:k + 1, :L].to(dev), b["fb_lens"][k:k + 1].to(dev)); n_ = int(l1_[0]) - 4
                # last 4 frames excluded: icefall's stack downsampling pairs the last valid frame with a padded one (inherent, also in icefall training)
                diffs.append(float((enc[k, :n_] - e1[0, :n_]).abs().max() / e1[0, :n_].abs().max()))
        S.train()
        assert emb.shape[-1] == 192 and xl.tolist() == [embed_frames(n) for n in b["n"].tolist()]
        assert enc.shape[-1] == D_S and el.tolist() == [student_frames(n) for n in b["n"].tolist()] and torch.isfinite(enc).all() and max(diffs) < 2e-3, diffs
        return f"embed {tuple(emb.shape)} -> encoder {tuple(enc.shape)}, lens {el.tolist()}; padded batch == single utterances except the last 4 frames (max rel. diff {max(diffs):.1e})"
    def l1():
        C = st["C"]; b = st["b"]; S, Pj = st["S"], st["Pj"]; set_batch_count(S, 100000)   # regularisers at end values: every layer in the graph
        loss, i = forward_loss(S, Pj, b, C, dev, amp); st["loss"] = loss; st["i"] = i
        pred = i["pred"].detach().float().cpu().numpy(); tl = b["t"].float().numpy(); vals = []
        for k in range(n_utts):                                             # independent numpy re-implementation of pairing + mask + L1
            n = int(b["n"][k]); v = valid_pairs(n, student_frames(n), teacher_frames(n), C["A"], C["offset"])
            vals.append(np.abs(pred[k, :v] - tl[k, C["offset"]:C["offset"] + v]).mean(-1))
        man = float(np.concatenate(vals).mean()); assert torch.isfinite(loss) and abs(float(loss) - man) < 1e-3 * max(1.0, man), (float(loss), man)
        b2 = dict(b); t2 = b["t"].clone()
        for k, L in enumerate(b["t_lens"].tolist()): t2[k, L:] = 1e4; t2[k, :C["offset"]] = -1e4      # padding and unmatched leading frames
        b2["t"] = t2; S.eval(); Pj.eval()
        with torch.no_grad(): la, _ = forward_loss(S, Pj, b, C, dev, amp); lb, _ = forward_loss(S, Pj, b2, C, dev, amp)
        S.train(); Pj.train(); assert abs(float(la) - float(lb)) < 1e-6, (float(la), float(lb))
        return (f"L1 {float(loss):.4f} == numpy {man:.4f} | pred {tuple(i['pred'].shape)} vs target {tuple(i['tgt'].shape)} | pairs {i['v'].tolist()} | "
                f"garbage in padding/unmatched teacher frames: no effect | baselines mean {i['base_mean']:.4f}, median {i['base_median']:.4f}")
    def g1(): _sbw(st["loss"]); return "backward OK" + (" (fp16: loss-scaled)" if sc != 1.0 else "")
    def g2():
        S, Pj = st["S"], st["Pj"]; out_ = []
        for g, m in {"encoder_embed": S.encoder_embed, **{f"block{k + 1}": S.encoder.encoders[k] for k in range(6)}, "downsample_output": S.encoder.downsample_output, "projection": Pj}.items():
            gs = [p.grad for p in m.parameters() if p.grad is not None]; assert gs, f"{g}: no gradient"
            nrm = float(torch.sqrt(sum((x.float() ** 2).sum() for x in gs))); assert all(torch.isfinite(x).all() for x in gs) and nrm > 0, g; out_.append(f"{g} {nrm:.1e}")
        return " | ".join(out_)
    def g3():
        S, Pj = st["S"], st["Pj"]; none = [n for n, p in list(S.named_parameters()) + list(Pj.named_parameters()) if p.requires_grad and p.grad is None]
        known = [n for n in none if UNUSED_ICEFALL.search(n)]; other = sorted(set(none) - set(known)); st["unused"] = set(known)
        assert not other, f"{len(other)} trainable tensors received no gradient (not a known unused icefall parameter): {other[:8]}"
        return ("every trainable student/projection tensor received a gradient" if not known else
                f"every USED tensor received a gradient; {len(known)} layer-level 'bypass_scale' tensors get none because icefall's Zipformer2EncoderLayer "
                f"defines but never uses them ('# TODO: remove it'; the bypass uses bypass.bypass_scale / bypass_mid.bypass_scale)")
    def g4():
        S, Pj = st["S"], st["Pj"]; params = list(S.parameters()) + list(Pj.parameters())
        assert all(p.requires_grad for p in params), "some student/projection parameters are frozen"
        return "no teacher in the graph (cached features); WavLM was frozen (eval, requires_grad False) and is released"
    def o1():
        S, Pj = st["S"], st["Pj"]; params = list(S.parameters()) + list(Pj.parameters())
        opt, _ = make_optimizer(S, Pj, dict(CFG, **PRESETS[preset])); st["opt"] = opt
        named = dict(list(S.named_parameters()) + [("proj." + k, v) for k, v in Pj.named_parameters()]); before = {k: v.detach().clone() for k, v in named.items()}
        unused = {k for k, v in named.items() if v.grad is None}
        assert all(UNUSED_ICEFALL.search(k) for k in unused), f"tensors without gradient: {sorted(unused)[:8]}"
        if CFG["optimizer"] == "adamw": torch.nn.utils.clip_grad_norm_(params, CFG["grad_clip"])
        opt.step(); opt.zero_grad(set_to_none=True)
        unchanged = [k for k, v in named.items() if torch.equal(v.detach(), before[k]) and k not in unused]
        must = ["proj.lin.weight", "proj.lin.bias"] + [k for k in named if k.startswith("encoder_embed.")][:1] + \
               [next(k for k in named if k.startswith(f"encoder.encoders.{i}.") and k not in unused) for i in range(6)]
        assert not set(must) & set(unchanged), f"key tensors not updated: {sorted(set(must) & set(unchanged))}"
        assert len(unchanged) <= 0.01 * len(named), f"{len(unchanged)} of {len(named)} tensors not updated: {unchanged[:8]}"
        return (f"{type(opt).__name__}: {len(named) - len(unused) - len(unchanged)}/{len(named) - len(unused)} used student/projection tensors updated"
                + (f" (unchanged: {unchanged[:5]})" if unchanged else "") + f"; {len(unused)} unused icefall 'bypass_scale' leftovers have no gradient")
    def c1():
        f = out / "smoke_ckpt.pt"; atomic_torch_save(dict(student=st["S"].state_dict(), asr_proj=st["Pj"].state_dict()), f)
        ck = torch.load(f, map_location=dev, weights_only=False); S2, P2 = new_models(st["C"], dev)
        S2.load_state_dict(ck["student"], strict=True); P2.load_state_dict(ck["asr_proj"], strict=True); b = st["b"]; outs = []
        for S_, P_ in [(st["S"], st["Pj"]), (S2, P2)]:
            S_.eval(); P_.eval()
            with torch.no_grad(): e, _ = S_(b["fb"].to(dev), b["fb_lens"].to(dev)); outs.append(P_(e))
            S_.train(); P_.train()
        d = float((outs[0] - outs[1]).abs().max()); os.remove(f); assert d < 1e-5, d; return f"atomic save + strict reload, identical outputs (max diff {d:.1e})"
    def c2():
        """Recovery: state saved after one update; the next update is done, the state restored, the same update redone -> same result."""
        S, Pj, opt = st["S"], st["Pj"], st["opt"]; b = st["b"]; C = st["C"]; f = out / "smoke_resume.pt"
        atomic_torch_save(dict(s=S.state_dict(), p=Pj.state_dict(), o=opt.state_dict(), rng=torch.get_rng_state(), rngc=torch.cuda.get_rng_state()), f)
        def step():
            set_batch_count(S, 100000); l, _ = forward_loss(S, Pj, b, C, dev, amp); _sbw(l); opt.step(); opt.zero_grad(set_to_none=True)
            return float(l), float(Pj.lin.weight.detach().double().sum())
        a = step(); ck = torch.load(f, map_location=dev, weights_only=False)
        S.load_state_dict(ck["s"]); Pj.load_state_dict(ck["p"]); opt.load_state_dict(ck["o"]); torch.set_rng_state(ck["rng"].cpu()); torch.cuda.set_rng_state(ck["rngc"].cpu())
        b_ = step(); os.remove(f)
        assert abs(a[0] - b_[0]) < 1e-3 * abs(a[0]) and abs(a[1] - b_[1]) < 1e-3 * max(1, abs(a[1])), (a, b_)
        return f"replayed update after restore: loss {a[0]:.5f} vs {b_[0]:.5f}, projection checksum {a[1]:.4f} vs {b_[1]:.4f}"
    def m1():
        mb, peak = memory_probe(st["S"], st["Pj"], st["C"], dev, amp, CFG["max_batch_seconds"]); st["mem"] = (mb, peak)
        return f"worst-case batch fits: max_batch_seconds {mb:.0f} -> measured peak {peak:.2f} GiB"
    def t2():
        set_batch_count(st["S"], 100000); l, _ = forward_loss(st["S"], st["Pj"], st["b"], st["C"], dev, amp); _sbw(l); st["opt"].step(); st["opt"].zero_grad(set_to_none=True)
        assert torch.isfinite(l); return f"another training step OK, L1 {float(l):.4f} | " + ", ".join(f"{k} {v:.2f}" for k, v in gpu_mem().items())
    plan = [("D1", "dataset integrity (splits, nesting, ids, transcripts, decoded files, overlap)", d1), ("D2", "LibriSpeech audio loads", d2),
            ("W1", "WavLM-Large loads (frozen, eval, 24 layers, 1024-D)", w1), ("W2", "WavLM preprocessing / frame rule / layer / pairing / long utterances / precision", w2),
            ("W3", "teacher cache integrity and metadata", w3), ("W4", "cached features == freshly extracted", w4),
            ("A1", "teacher/student temporal alignment measured from the imported code", a1), ("S1", "Swoosh port == k2 algorithm", s1),
            ("Z1", "Zipformer-M source and configuration", z1), ("Z2", "80-D fbank frontend", z2), ("Z3", "student shapes, lengths, padding masks", z3),
            ("L1", "masked L1 == independent calculation; padding/unmatched frames ignored", l1), ("G1", "backpropagation", g1),
            ("G2", "non-zero finite gradients in every block and the projection", g2), ("G3", "every trainable tensor gets a gradient", g3),
            ("G4", "teacher frozen / not in the graph", g4), ("O1", "optimizer updates student and projection", o1),
            ("C1", "checkpoint save and strict reload", c1), ("C2", "checkpoint recovery (restore + replay)", c2), ("M1", "measured GPU memory of the worst-case batch", m1),
            ("T2", "another training step", t2)]
    for no, name, fn in plan:
        chk(no, name, fn, critical=(no != "G3"))
        if no in ("D2", "W1") and R[-1]["status"] == "FAIL": log("cannot continue without this"); break
    if "T" in st: del st["T"]
    gc.collect(); torch.cuda.empty_cache()
    crit_fail = [r["no"] for r in R if r["critical"] and r["status"] == "FAIL"]; ran = {r["no"] for r in R}; missing = [n for n, _, _ in plan if n not in ran]
    rep = dict(preset=preset, code_version=CODE_VERSION, code_fingerprint=code_fingerprint(), time=time.strftime("%Y-%m-%d %H:%M:%S"), checks=R,
               critical_failed=crit_fail + missing, all_critical_passed=not crit_fail and not missing,
               teacher_cache=str(st["C"]["TCtr"].src) if "C" in st else None, alignment_sha=sha_of([json.dumps(load_alignment()["measured"], sort_keys=True)]) if ALIGN_F().exists() else None,
               max_batch_seconds_probe=st.get("mem", (None, None))[0], swoosh=st.get("swoosh"), architecture=st.get("arch"))
    atomic_json(rep, SMOKE_F())
    log(f"\nSMOKE TEST: {sum(r['status'] == 'PASS' for r in R)}/{len(plan)} passed | critical failures: {rep['critical_failed'] or 'none'} -> {SMOKE_F()}"
        + ("" if rep["all_critical_passed"] else "\nTRAINING IS BLOCKED until every critical check passes"))
    return rep

def _gate(C):
    if not SMOKE_F().exists(): raise RuntimeError("smoke test has not been run: run the smoke-test cell first")
    r = json.load(open(SMOKE_F()))
    if not r.get("all_critical_passed"): raise RuntimeError(f"smoke test has critical failures {r.get('critical_failed')}: training blocked")
    if r.get("code_fingerprint") != code_fingerprint(): raise RuntimeError("the code changed since the smoke test: run the smoke-test cell again")
    if r.get("teacher_cache") != str(C["TCtr"].src): raise RuntimeError("the teacher cache differs from the one the smoke test checked: run the smoke test again")
    if r.get("alignment_sha") != sha_of([json.dumps(C["align"]["measured"], sort_keys=True)]): raise RuntimeError("alignment changed since the smoke test: run it again")

# ===========================================================================================
# training
# ===========================================================================================

# ####################################################################################################
# ---- Google Drive output store: the ONLY code in this cell that writes to Google Drive
# ####################################################################################################
import threading, random, hashlib, itertools
DRIVE_FILES = ("best.pt", "latest.pt", "history.csv", "final_report.json", "config.json")
RESUME_KEYS = {"sig", "student", "asr_proj", "opt", "scaler", "scheduler", "state", "rng_cpu", "rng_cuda", "rng_np", "rng_py"}

class SimulatedDisconnect(Exception):
    """Raised by the resume test to imitate a runtime that dies: nothing is saved at that moment."""

class DriveStore:
    """Writes ONLY into one folder (drive_dir(<run>)) and ONLY the five files in DRIVE_FILES (plus '<name>.tmp' while a write is in progress).
    * small files (config.json, history.csv, final_report.json): written synchronously, atomically (.tmp then rename)
    * checkpoints (latest.pt, best.pt): copied in a background thread from the local checkpoint (training does not wait),
      written to '<name>.tmp' first and renamed only when the copy is complete -> an interrupted upload never replaces a good file
    * with the Drive API (CFG['drive_api']): each upload is confirmed by comparing Drive's md5 with the local md5, and the copies that
      Drive moves to the trash when a checkpoint is replaced are deleted permanently (only files of THIS folder with THESE names),
      because trashed files keep using Drive quota for 30 days.
    fake=True: a local stand-in folder (used by the resume test; nothing goes to Drive)."""
    def __init__(self, folder, use_api=True, fake=False):
        self.dir, self.fake = Path(folder), fake
        if fake: assert_local(folder)
        else:
            if not os.path.isdir(DRIVE + "/MyDrive"): raise RuntimeError("Google Drive is not mounted")
            # the path as written must be inside MyDrive; its resolved location may be elsewhere on the Drive mount, because a folder
            # such as MT2KD can be a Drive SHORTCUT (resolved to /content/drive/.shortcut-targets-by-id/...)
            if not os.path.normpath(os.path.abspath(str(folder))).startswith(DRIVE + "/MyDrive/"): raise PermissionError(f"Drive output folder must be inside MyDrive: {folder}")
            if not os.path.realpath(str(folder)).startswith(os.path.realpath(DRIVE) + "/"): raise PermissionError(f"{folder} resolves outside Google Drive")
        self.dir.mkdir(parents=True, exist_ok=True)
        self.api = self.fid = None; self.api_note = "not used (stand-in folder)" if fake else "disabled (CFG['drive_api'] = False)"
        if use_api and not fake: self._init_api()
        self.cv = threading.Condition(); self.pending = {}; self.busy = None; self.errors = []; self.done = {}; self._ids = itertools.count()
        threading.Thread(target=self._worker, daemon=True).start()

    # ---------------------------------------------------------------- guards / API
    def _p(self, name):
        base = name[:-4] if name.endswith(".tmp") else name
        if base not in DRIVE_FILES: raise PermissionError(f"'{name}' is not one of the files allowed on Drive: {DRIVE_FILES}")
        return self.dir / name
    def _init_api(self):
        try:
            from google.colab import auth; auth.authenticate_user()
            from googleapiclient.discovery import build
            self.api = build("drive", "v3", cache_discovery=False)
            self.api_note = "ON: uploads confirmed by md5; trashed copies of this folder's checkpoints are deleted permanently"
        except Exception as e:
            self.api = None; self.api_note = (f"UNAVAILABLE ({type(e).__name__}: {e}): uploads are not confirmed and replaced checkpoints stay in the Drive "
                                              f"trash (empty it to recover the space)")
    def _folder_id(self):
        if self.fid or self.api is None: return self.fid
        rel = os.path.relpath(os.path.normpath(os.path.abspath(str(self.dir))), DRIVE + "/MyDrive").split(os.sep); parent = "root"
        try:
            for part in rel:                                  # walk the path as written; follow Drive shortcuts to their target folder
                q = (f"name = '{part}' and '{parent}' in parents and trashed = false and "
                     f"(mimeType = 'application/vnd.google-apps.folder' or mimeType = 'application/vnd.google-apps.shortcut')")
                r = self.api.files().list(q=q, fields="files(id,mimeType,shortcutDetails)", pageSize=10, supportsAllDrives=True,
                                          includeItemsFromAllDrives=True).execute().get("files", [])
                if not r: return None                         # folder not synced to Drive yet: retried later
                x = r[0]; parent = x["shortcutDetails"]["targetId"] if x["mimeType"].endswith("shortcut") else x["id"]
            self.fid = parent
        except Exception: return None
        return self.fid

    # ---------------------------------------------------------------- storage information
    def quota(self):
        if self.api:
            try:
                q = self.api.about().get(fields="storageQuota").execute()["storageQuota"]; lim = int(q.get("limit", 0)); use = int(q.get("usage", 0))
                return dict(limit=lim or None, usage=use, trash=int(q.get("usageInDriveTrash", 0)), free=(lim - use) if lim else None, source="Drive API")
            except Exception: pass
        du = shutil.disk_usage(self.dir if self.fake else DRIVE + "/MyDrive")
        return dict(limit=du.total, usage=du.used, trash=None, free=du.free, source="mounted file system (approximate)")
    def files(self): return {f.name: f.stat().st_size for f in sorted(self.dir.iterdir()) if f.is_file()}
    def status(self, expected_bytes=None):
        q = self.quota(); fl = self.files(); g = lambda b: "?" if b is None else f"{b / 2**30:.2f} GiB"
        log(f"  Drive output folder : {self.dir}")
        log(f"  folder size now     : {g(sum(fl.values()))} | files: " + (", ".join(f"{k} ({v / 2**20:.1f} MiB)" for k, v in fl.items()) or "none"))
        log(f"  Drive storage       : free {g(q['free'])} of {g(q['limit'])} (used {g(q['usage'])}" + (f", in trash {g(q['trash'])}" if q["trash"] is not None else "")
            + f") [{q['source']}]")
        log(f"  Drive API           : {self.api_note}")
        if expected_bytes: log(f"  this run will keep  : about {g(expected_bytes)} in this folder (best.pt + latest.pt + small files)")
        unexpected = [k for k in fl if (k[:-4] if k.endswith(".tmp") else k) not in DRIVE_FILES]
        if unexpected: log(f"  NOTE: files not written by this code are in the folder (left untouched): {unexpected}")
        return dict(quota=q, files=fl)
    def check_space(self, need_bytes, what):
        """Refuses to start writing large files when the free Drive space is below what the next writes need."""
        q = self.quota()
        if q["free"] is not None and q["free"] < need_bytes:
            raise RuntimeError(f"Google Drive has {q['free'] / 2**30:.2f} GiB free; {what} needs about {need_bytes / 2**30:.2f} GiB. Nothing large was written. "
                               f"Free at least {(need_bytes - q['free']) / 2**30:.2f} GiB more on Drive (deleted files only free space once the Drive trash is "
                               f"emptied) and re-run the cell.")
        free = "?" if q["free"] is None else f"{q['free'] / 2**30:.2f}"
        log(f"  Drive space check OK for {what}: need ~{need_bytes / 2**30:.2f} GiB, free {free} GiB")

    # ---------------------------------------------------------------- writes
    def write_text(self, name, text):
        tmp, dst = self._p(name + ".tmp"), self._p(name)
        with open(tmp, "w") as fh: fh.write(text); fh.flush(); os.fsync(fh.fileno())
        os.replace(tmp, dst)
    def write_json(self, name, obj): self.write_text(name, json.dumps(obj, indent=1, default=str))
    def push(self, local_file, name):
        """Queues an upload of a local checkpoint file. The file is snapshotted NOW (hard link), so later local saves cannot change what is
        uploaded; a newer request for the same name replaces a queued (not yet started) one."""
        self._p(name); snap = f"{local_file}.upload{next(self._ids)}"
        try: os.link(local_file, snap)
        except OSError: shutil.copyfile(local_file, snap)
        with self.cv:
            if self.errors: os.remove(snap); raise RuntimeError(f"a Google Drive upload failed: {self.errors[-1]} (training stopped; local state is saved)")
            old = self.pending.get(name); self.pending[name] = snap; self.cv.notify_all()
        if old and os.path.exists(old): os.remove(old)
    def _worker(self):
        while True:
            with self.cv:
                while not self.pending: self.cv.wait()
                name = next(iter(self.pending)); src = self.pending.pop(name); self.busy = name
            try: self._upload(src, name)
            except Exception as e: self.errors.append(f"{name}: {type(e).__name__}: {e}"); log(f"  GOOGLE DRIVE UPLOAD FAILED ({name}): {type(e).__name__}: {e}")
            finally:
                try: os.remove(src)
                except OSError: pass
                with self.cv: self.busy = None; self.cv.notify_all()
    def _upload(self, src, name):
        t0 = time.time(); tmp, dst = self._p(name + ".tmp"), self._p(name); h = hashlib.md5(); n = 0
        with open(src, "rb") as fi, open(tmp, "wb") as fo:          # the open handle keeps a consistent snapshot even if the local file is replaced meanwhile
            for b in iter(lambda: fi.read(1 << 24), b""): fo.write(b); h.update(b); n += len(b)
            fo.flush(); os.fsync(fo.fileno())
        os.replace(tmp, dst); md5 = h.hexdigest()
        self.done[name] = dict(bytes=n, md5=md5, written=time.strftime("%Y-%m-%d %H:%M:%S"), confirmed_by_drive_api=None)
        log(f"  Drive: {name} written ({n / 2**20:.0f} MiB in {time.time() - t0:.0f} s)" + ("; waiting for Drive to confirm the upload" if self.api else ""))
        if self.api:
            self.done[name]["confirmed_by_drive_api"] = self._confirm(name, md5); self.purge_trash()
    def _confirm(self, name, md5, timeout=1800):
        t0 = time.time()
        while time.time() - t0 < timeout:
            with self.cv:
                if name in self.pending: return "superseded by a newer upload"
            fid = self._folder_id()
            if fid:
                try:
                    r = self.api.files().list(q=f"name = '{name}' and '{fid}' in parents and trashed = false", fields="files(id,md5Checksum)").execute().get("files", [])
                    if any(x.get("md5Checksum") == md5 for x in r): log(f"  Drive: {name} upload CONFIRMED (md5 {md5[:10]} matches)"); return True
                except Exception: pass
            time.sleep(15)
        log(f"  Drive: {name} upload not confirmed within {timeout // 60} min (it may still be syncing)"); return False
    def purge_trash(self):
        """Deletes permanently the TRASHED copies of this folder's own files (replaced checkpoints); nothing else is touched."""
        fid = self._folder_id()
        if not fid: return 0
        freed = 0
        try:
            for x in self.api.files().list(q=f"'{fid}' in parents and trashed = true", fields="files(id,name,size)", pageSize=200).execute().get("files", []):
                base = x["name"][:-4] if x["name"].endswith(".tmp") else x["name"]
                if base in DRIVE_FILES: self.api.files().delete(fileId=x["id"]).execute(); freed += int(x.get("size", 0))
        except Exception as e: log(f"  (could not clean this folder's trashed copies: {type(e).__name__}: {e})")
        if freed: log(f"  Drive: {freed / 2**20:.0f} MiB of replaced copies removed from the trash")
        return freed
    def wait(self, timeout=None):
        """Blocks until every queued upload is written (and confirmed, with the API)."""
        t0 = time.time()
        with self.cv:
            while self.pending or self.busy:
                if timeout and time.time() - t0 > timeout: return False
                self.cv.wait(5)
        if self.api: self.purge_trash()
        if self.errors: raise RuntimeError(f"Google Drive upload failed: {self.errors[-1]}")
        return True

    # ---------------------------------------------------------------- reading back
    def candidates(self, name): return [f for f in (self._p(name), self._p(name + ".tmp")) if f.exists()]
    def promote_tmp(self, name):
        """A complete '<name>.tmp' whose rename was interrupted becomes '<name>'."""
        os.replace(self._p(name + ".tmp"), self._p(name)); return self._p(name)
    def clean_stale_tmp(self):
        for f in self.dir.glob("*.tmp"):
            if f.name[:-4] in DRIVE_FILES: f.unlink(); log(f"  removed an incomplete upload left by an interrupted run: {f.name}")
    def best_info(self): return best_meta(self._p("best.pt")) if self._p("best.pt").exists() else None

def drive_need_bytes(n_par, drive):
    """Drive space the run needs beyond what its own files already use: best.pt (fp32 weights) + latest.pt (weights + 2 AdamW moments),
    each written as a temporary copy before the old file is replaced (peak = 2 x each)."""
    best_b, latest_b = 4.05 * n_par, 20.0 * n_par          # latest.pt: weights + optimizer state (ScaledAdam keeps more per weight than AdamW)
    have = sum(v for k, v in drive.files().items() if k in ("best.pt", "latest.pt"))
    return max(0.0, 2 * (best_b + latest_b) - have) + 50e6, best_b + latest_b

def best_meta(f):
    import torch
    try: ck = torch.load(f, map_location="cpu", weights_only=False, mmap=True)
    except Exception:
        try: ck = torch.load(f, map_location="cpu", weights_only=False)
        except Exception as e: return dict(error=f"{type(e).__name__}: {e}")
    r = dict(val_l1=float(ck["val"]["val_l1"]), epoch=int(ck["epoch"]), update=int(ck["update"]), sig=ck.get("sig")); del ck
    return r

def discard_summary():
    """Every utterance excluded from the data, with its reason (inventory checks + full decode)."""
    m = paths()["meta"]; out = {}
    for split in ["train-clean-100", CFG["val_split"]]:
        d = {}; f = m / f"inventory_v2_{split}_dropped.csv"
        if f.exists() and f.stat().st_size > 30:
            x = pd.read_csv(f, dtype=str); d["invalid_files"] = {u: r for u, r in zip(x.uid, x.reason)}
        f = m / f"decode_failures_v2_{split}.csv"
        if f.exists():
            x = pd.read_csv(f, dtype=str); d["failed_full_decode"] = {u: r for u, r in zip(x.uid, x.reason)}
        out[split] = d or "none excluded"
    return out

def _load_resumable(f):
    import torch
    try: ck = torch.load(f, map_location="cpu", weights_only=False)
    except Exception as e: return None, f"unreadable ({type(e).__name__}: {str(e)[:120]})"
    if not isinstance(ck, dict) or not RESUME_KEYS <= set(ck): return None, "not a complete resumable checkpoint"
    return ck, ""

def _position(S): return (S["update"], S["epoch"], len(S["seen"]))

# ####################################################################################################
# ---- training (resumable from local disk or Google Drive)
# ####################################################################################################
def train(preset="pilot", C=None, P=None, drive=None, overrides=None, interrupt_after_update=None, probe=True):
    """KD training. Resumes automatically from the most advanced VALID checkpoint among: local latest, Drive latest.pt, Drive latest.pt.tmp."""
    import torch
    C = C or context(preset); _gate(C); P = P or paths(preset); ensure_dirs(P); dev = torch.device("cuda"); amp = amp_dtype()
    cfg = dict(CFG, **PRESETS[preset]); cfg.update(overrides or {})
    LATEST, BEST = P["latest"] / "checkpoint.pt", P["best"] / "best.pt"; N = len(C["tr"]); lens = C["tr"].n_samples.values
    student, proj = new_models(C, dev)
    params = list(student.parameters()) + list(proj.parameters())
    opt, sch = make_optimizer(student, proj, cfg)
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 12, enabled=amp == torch.float16)
    sig_keys = ["seed", "optimizer", "base_lr", "lr_batches", "lr_epochs", "eden_warmup_batches", "eden_warmup_start", "clipping_scale",
                "lr", "weight_decay", "warmup_updates", "min_lr_frac", "grad_clip", "accum_steps", "align", "target_norm", "init_bias", "max_batch_utts"]
    # (max_epochs is NOT in the signature: raising it later continues the same run instead of refusing its checkpoints)
    SIG = sha_of([json.dumps({k: cfg[k] for k in sig_keys}, sort_keys=True), C["manifest_sha"], C["dev_sha"], str(C["TCtr"].src), C["offset"], CODE_VERSION])
    S = dict(epoch=0, batch=0, update=0, best=None, es_counter=0, train_hist=[], val_hist=[], elapsed=0.0, skips=0, finished=None, seen=[], plan_remaining=None,
             ep_sum=0.0, ep_fr=0.0, max_batch_seconds=None, sessions=0, resumes=[])

    # ---- 1) look for the most advanced valid checkpoint (local disk first, then Drive, including an upload whose rename was interrupted)
    for f in list(P["ckpt"].rglob("*.upload*")): f.unlink()                 # snapshots left by an interrupted upload
    cands = [("local disk", LATEST)] + ([("Google Drive", f) for f in drive.candidates("latest.pt")] if drive else []); chosen = None
    for src, f in cands:
        if not Path(f).exists(): continue
        ck, why = _load_resumable(f)
        if ck is None: log(f"  ignored {src} file {f}: {why} (e.g. an interrupted save or upload)"); continue
        if ck["sig"] != SIG and src == "local disk":
            dst = P["ckpt"].parent / f"{P['ckpt'].name}_other_config_{time.strftime('%Y%m%d_%H%M%S')}"; del ck; os.replace(P["ckpt"], dst); ensure_dirs(P)
            log(f"  local checkpoints of a DIFFERENT configuration moved aside (local disk): {dst}"); continue
        if ck["sig"] != SIG:
            raise RuntimeError(f"{f} ({src}) is a checkpoint of a DIFFERENT configuration / data / code version (sig {ck['sig']}, this run {SIG}). "
                               f"It is left untouched. Move it out of the folder yourself to start a new run, or restore the settings it was made with.")
        k = _position(ck["state"]); log(f"  valid checkpoint on {src}: {Path(f).name}, epoch {ck['state']['epoch'] + 1}, batch {ck['state']['batch']}, global step {k[0]}")
        if chosen is None or k > chosen[2]: chosen = (src, f, k, ck)
        else: del ck
    if chosen:
        src, f, k, ck = chosen
        if src == "Google Drive" and str(f).endswith(".tmp"): f = drive.promote_tmp("latest.pt"); log("  (the newest Drive checkpoint was a complete upload whose rename was interrupted: renamed)")
        student.load_state_dict(ck["student"], strict=True); proj.load_state_dict(ck["asr_proj"], strict=True); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        S = ck["state"]; torch.set_rng_state(ck["rng_cpu"].cpu()); torch.cuda.set_rng_state(ck["rng_cuda"].cpu()); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
        log("\n" + "#" * 70 + "\n#  RESUMING FROM CHECKPOINT\n" + "#" * 70)
        log(f"  source {src}: {f}\n  epoch {S['epoch'] + 1}/{cfg['max_epochs']} | batch position {S['batch']} ({len(S['seen'])}/{N} utterances of this epoch done) | "
            f"global optimizer step {S['update']} | lr schedule at step {ck['scheduler']['update']} (lr {ck['scheduler']['lr']:.3e}) | best val L1 "
            f"{S['best']['val_l1'] if S['best'] else None} | early-stop counter {S['es_counter']}/{cfg['patience']}")
        S.setdefault("resumes", []).append(dict(source=src, update=S["update"], epoch=S["epoch"], batch=S["batch"], seen=len(S["seen"]), time=time.strftime("%Y-%m-%d %H:%M:%S")))
        if src == "Google Drive":
            shutil.copyfile(f, str(LATEST) + ".part"); os.replace(str(LATEST) + ".part", LATEST)       # local copy (later local saves start from here)
        del ck
        if cfg["max_batch_seconds"] < S["max_batch_seconds"]:
            log(f"  max_batch_seconds lowered {S['max_batch_seconds']:.0f} -> {cfg['max_batch_seconds']:.0f}: the REMAINING utterances of this epoch are re-batched")
            S["max_batch_seconds"] = cfg["max_batch_seconds"]; S["plan_remaining"] = sorted(set(range(N)) - set(S["seen"])); S["batch"] = 0
    else:
        log("\n" + "#" * 70 + "\n#  STARTING NEW TRAINING RUN\n" + "#" * 70 + "\n  (no valid checkpoint on local disk" + (" or in " + str(drive.dir) if drive else "") + ")")
    if drive: drive.clean_stale_tmp()

    # ---- 2) the best checkpoint is never replaced by a worse model. A best.pt (Drive or local) better than the resumed state's best
    #         (saved during work lost in the interruption) becomes a floor: best.pt is only replaced by a model that beats it.
    lb = best_meta(BEST) if BEST.exists() else None
    if lb and lb.get("sig") != SIG:
        dst = BEST.with_name(f"best_other_config_{time.strftime('%Y%m%d_%H%M%S')}.pt"); os.replace(BEST, dst); lb = None
        log(f"  local best.pt of a different configuration moved aside: {dst}")
    db = drive.best_info() if drive else None
    if db and "error" in db: log(f"  Drive best.pt is unreadable ({db['error']}): it will be replaced by the next best model"); db = None
    if db and db.get("sig") != SIG:
        raise RuntimeError(f"{drive.dir / 'best.pt'} is not a best checkpoint of this configuration ({db}); left untouched. Move it away to start a new run.")
    floor = None
    for src_, m in (("Google Drive", db), ("local disk", lb)):
        if m and (S["best"] is None or m["val_l1"] < S["best"]["val_l1"]) and (floor is None or m["val_l1"] < floor["val_l1"]): floor = dict(m, source=src_)
    if floor: log(f"  existing best.pt on {floor['source']} (val L1 {floor['val_l1']:.5f}, epoch {floor['epoch']}) is newer than the resumed state: "
                  f"it is only replaced by a model with a lower validation loss")
    if drive:
        if db and (lb is None or lb["val_l1"] > db["val_l1"]): shutil.copyfile(drive.dir / "best.pt", str(BEST) + ".part"); os.replace(str(BEST) + ".part", BEST)
        elif lb and (db is None or lb["val_l1"] < db["val_l1"]): drive.push(BEST, "best.pt"); log("  local best.pt is newer than the Drive copy (upload was interrupted): re-uploading")
    S["sessions"] += 1

    # ---- 3) Drive space (estimated from the parameter count: fp32 weights; AdamW keeps two more fp32 tensors per weight)
    if drive and not drive.fake: drive.check_space(drive_need_bytes(sum(p.numel() for p in params), drive)[0], "best.pt + latest.pt (each replaced via a temporary copy)")
    if S["max_batch_seconds"] is None:
        S["max_batch_seconds"] = memory_probe(student, proj, C, dev, amp, cfg["max_batch_seconds"])[0] if probe else cfg["max_batch_seconds"]
    MBS = S["max_batch_seconds"]

    def lr_at(update, prog):
        """linear warm-up over the first warmup_updates, then cosine over training PROGRESS (fraction of max_epochs x utterances)."""
        w = cfg["lr"] * min(1.0, (update + 1) / cfg["warmup_updates"])
        cosv = cfg["min_lr_frac"] + (1 - cfg["min_lr_frac"]) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
        return min(w, cfg["lr"] * cosv) if update < cfg["warmup_updates"] else cfg["lr"] * cosv
    def lr_now(update):
        """learning rate for the next optimizer step (global step `update` + 1), also written into the optimizer.
        ScaledAdam/Eden: as icefall (scheduler.step_epoch(epoch - 1) at each epoch start, step_batch(batch_idx_train) before each step)."""
        if sch is not None:
            sch.step_epoch(S["epoch"]); sch.step_batch(update + 1); return sch.get_last_lr()[0]
        lr = lr_at(update, progress())
        for g in opt.param_groups: g["lr"] = lr
        return lr
    progress = lambda: (S["epoch"] + len(S["seen"]) / N) / cfg["max_epochs"]
    tr = C["tr"]; dvf = C["dv"]
    run_cfg = dict(code_version=CODE_VERSION, sig=SIG, preset=preset, seed=cfg["seed"], amp=str(amp), updated=time.strftime("%Y-%m-%d %H:%M:%S"),
                   data=dict(source="LibriSpeech train-clean-100 (official)", train_subset=PRESETS[preset]["train_subset"], target_seconds=3600 * CFG["subsets"].get(PRESETS[preset]["train_subset"], 0),
                             train_utterances=N, train_seconds=round(float(tr.duration_s.sum()), 2), train_hours=round(float(tr.duration_s.sum() / 3600), 4),
                             train_speakers=int(tr.speaker.nunique()), train_manifest_sha=C["manifest_sha"],
                             selection_rule="valid utterances sorted by id, permuted with numpy default_rng(seed); shortest prefix reaching the target duration",
                             validation=dict(split=CFG["val_split"], utterances=len(dvf), hours=round(float(dvf.duration_s.sum() / 3600), 4), manifest_sha=C["dev_sha"]),
                             test="test-clean is not used in Stage 1", discarded=discard_summary(), train_uids=sorted(tr.uid.tolist())),
                   teacher=dict(model=C["teacher_meta"].get("model"), commit=C["teacher_meta"].get("commit"), layer=C["teacher_meta"].get("layer"),
                                representation=f"{C['teacher_meta'].get('layer')}, 1024-D @ 50 Hz; frames (2j, 2j+1) concatenated -> 2048-D @ 25 Hz", frozen=True,
                                preprocessing=C["teacher_meta"].get("normalize"), dtype=C["teacher_meta"].get("dtype"),
                                long_utterances="encoded whole (no window, no chunking, no padding)", cache="local disk only (/content), regenerated after a runtime recycle"),
                   student=dict(model="Zipformer-M (icefall zipformer.py)", input="80-D log-mel fbank, 10 ms", frontend="Conv2dSubsampling (icefall)", zip_args=ZIP_ARGS,
                                output="512-D @ 25 Hz", projection=f"Linear 512 -> {C['teacher_meta']['dim']}", params_student=sum(p.numel() for p in student.parameters()),
                                params_projection=sum(p.numel() for p in proj.parameters()), swoosh="pure-PyTorch port of k2's Swoosh (k2 not required), verified numerically"),
                   alignment=dict(rule=f"student frame j <-> teacher frame j+{C['offset']} (measured frame centres)", residual_ms=C["align"]["residual_ms"], offset=C["offset"]),
                   loss=f"masked L1 between projected student and teacher features (mean over {C['teacher_meta']['dim']} dims and valid aligned frames)",
                   training={k: cfg[k] for k in ["lr", "weight_decay", "warmup_updates", "min_lr_frac", "grad_clip", "accum_steps", "max_epochs", "patience", "min_delta",
                                                  "max_batch_utts", "local_ckpt_every_updates", "drive_ckpt_every_updates", "ckpt_every_minutes", "init_bias", "target_norm"]}
                            | ({k: cfg[k] for k in ["optimizer", "base_lr", "lr_batches", "lr_epochs", "eden_warmup_batches", "eden_warmup_start", "clipping_scale"]}
                               if cfg["optimizer"] == "scaled_adam" else dict(optimizer="AdamW", schedule="linear warm-up, then cosine over progress"))
                            | dict(max_batch_seconds_effective=MBS),
                   drive_files=dict(folder=str(drive.dir) if drive else None, files=list(DRIVE_FILES)))
    atomic_json(run_cfg, P["logs"] / "run_config.json")
    if drive: drive.write_json("config.json", run_cfg)
    log(f"  train {N:,} utts / {tr.duration_s.sum():.0f} s / {run_cfg['data']['train_speakers']} speakers | val {len(dvf):,} utts / {dvf.duration_s.sum() / 3600:.2f} h | "
        f"max {cfg['max_epochs']} epochs | " + (f"ScaledAdam + Eden (base lr {cfg['base_lr']}, lr_batches {cfg['lr_batches']}, lr_epochs {cfg['lr_epochs']})"
        if cfg["optimizer"] == "scaled_adam" else f"AdamW lr {cfg['lr']}") + f" | {str(amp).split('.')[-1]}{' + GradScaler' if scaler.is_enabled() else ''} | batches <= {MBS:.0f} s "
        f"| checkpoints: local every {cfg['local_ckpt_every_updates']} steps / {cfg['ckpt_every_minutes']:.0f} min, Drive every {cfg['drive_ckpt_every_updates']} steps, both every epoch")

    t_ref = [time.time()]; last_local = [time.time()]; t_session = time.time(); budget = cfg.get("max_session_hours")
    def sched(): return dict(kind=("Eden: a pure function of (global step, epoch), both restored from 'state'" if sch is not None else
                                   "warm-up + cosine on progress: a pure function of (global step, progress), both restored from 'state'"),
                             update=S["update"], progress=progress(), lr=lr_now(S["update"]), eden=sch.state_dict() if sch is not None else None)
    def save_latest(to_drive=False):
        S["elapsed"] += time.time() - t_ref[0]; t_ref[0] = time.time()
        atomic_torch_save(dict(sig=SIG, student=student.state_dict(), asr_proj=proj.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), scheduler=sched(),
                               state=S, rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state(), rng_np=np.random.get_state(), rng_py=random.getstate(),
                               zip_args=ZIP_ARGS, code_version=CODE_VERSION), LATEST)
        pd.DataFrame(S["train_hist"]).to_csv(P["logs"] / "train_history.csv", index=False); last_local[0] = time.time()
        if to_drive and drive: drive.push(LATEST, "latest.pt")
    def write_history():
        h = pd.DataFrame([dict(epoch=v["epoch"], global_step=v["update"], train_loss=v["train_l1"], val_loss=v["val_l1"], lr=v["lr"], best=v["best"]) for v in S["val_hist"]],
                         columns=["epoch", "global_step", "train_loss", "val_loss", "lr", "best"])
        h.to_csv(P["logs"] / "history.csv", index=False)
        if drive: drive.write_text("history.csv", h.to_csv(index=False))

    student.train(); proj.train(); win = []; pending = []; stop_reason = None
    try:
        while S["epoch"] < cfg["max_epochs"] and not S["finished"]:
            ep = S["epoch"]; batches = make_batches(lens, ep, cfg["seed"], True, MBS, cfg["max_batch_utts"], idx=S["plan_remaining"])
            k0 = S["batch"]; t_ep = time.time(); torch.cuda.reset_peak_memory_stats()
            nb = [len(x) for x in batches]
            log(f"epoch {ep + 1}: {len(batches)} batches, {min(nb)}-{max(nb)} utterances per batch (mean {np.mean(nb):.1f})"
                + (f" | continuing at batch {k0}/{len(batches)} ({len(S['seen'])}/{N} utterances already seen)" if k0 or S["plan_remaining"] is not None else ""))
            for k, b in enumerate(loader(C["tr"], C["TCtr"], batches[k0:], cfg["num_workers"], seed=cfg["seed"] + ep), start=k0):
                set_batch_count(student, S["update"] * MBS * cfg["accum_steps"] / cfg["ref_duration"])
                l1, i = forward_loss(student, proj, b, C, dev, amp)
                if not torch.isfinite(l1):
                    S["skips"] += 1; log(f"  non-finite loss at step {S['update']} (batch skipped, {S['skips']} in a row)"); opt.zero_grad(set_to_none=True)
                    for kk, _, _ in pending: S["seen"] += batches[kk]
                    pending = []
                    if S["skips"] > cfg["max_consecutive_skips"]: save_latest(True); raise FloatingPointError("too many non-finite losses; state saved")
                    S["seen"] += batches[k]; S["batch"] = k + 1; continue
                scaler.scale(l1 / cfg["accum_steps"]).backward(); win.append((float(l1), i["base_mean"], i["cos"], i["frames"])); pending.append((k, float(l1), i["frames"]))
                if (k + 1) % cfg["accum_steps"] and k + 1 < len(batches): continue
                lr = lr_now(S["update"])
                # ScaledAdam clips internally (clipping_scale x median recent norm): here the norm is only measured (max_norm = inf)
                scaler.unscale_(opt); gn = float(torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"] if sch is None else float("inf")))
                if math.isfinite(gn): scaler.step(opt); S["skips"] = 0
                else:
                    S["skips"] += 1; log(f"  non-finite gradient at step {S['update']} (update skipped, {S['skips']} in a row)")
                    if scaler.is_enabled(): scaler.step(opt)
                    if S["skips"] > cfg["max_consecutive_skips"]: save_latest(True); raise FloatingPointError("too many non-finite gradients; state saved")
                scaler.update(); opt.zero_grad(set_to_none=True); S["update"] += 1
                for kk, lv, fr in pending: S["seen"] += batches[kk]; S["ep_sum"] += lv * fr; S["ep_fr"] += fr
                pending = []; S["batch"] = k + 1; u = S["update"]
                if u % cfg["log_every"] == 0:
                    w = np.array(win); fr = w[:, 3]; win = []
                    rec = dict(update=u, epoch=ep + 1, batch=k + 1, progress=round(progress(), 5), l1=float(np.average(w[:, 0], weights=fr)),
                               base_mean=float(np.average(w[:, 1], weights=fr)), cos=float(np.average(w[:, 2], weights=fr)), grad_norm=gn, lr=lr,
                               loss_scale=scaler.get_scale() if scaler.is_enabled() else 1.0, **{a: round(v, 3) for a, v in gpu_mem().items()})
                    S["train_hist"].append(rec); el = time.time() - t_ep
                    log(f"[ep {ep + 1}/{cfg['max_epochs']} | b {k + 1}/{len(batches)} | step {u}] L1 {rec['l1']:.4f} | mean-baseline {rec['base_mean']:.4f} "
                        f"(ratio {rec['l1'] / rec['base_mean']:.3f}) | cos {rec['cos']:.3f} | |g| {gn:.2f} | lr {lr:.2e} | GPU peak {rec.get('gpu_peak_gib', 0):.1f} GiB | "
                        f"epoch ETA {el / (k + 1 - k0) * (len(batches) - k - 1) / 60:.1f} min")
                if drive and u % cfg["drive_ckpt_every_updates"] == 0: save_latest(True); log(f"  checkpoint saved at step {u}: local disk + Google Drive (upload in background)")
                elif u % cfg["local_ckpt_every_updates"] == 0 or time.time() - last_local[0] > 60 * cfg["ckpt_every_minutes"]: save_latest(); log(f"  checkpoint saved at step {u} (local disk)")
                if interrupt_after_update is not None and u == interrupt_after_update: raise SimulatedDisconnect(f"simulated runtime loss after step {u}")
                if budget and time.time() - t_session > 3600 * budget:
                    save_latest(True); write_history(); stop_reason = f"session time budget of {budget} h reached"; break
            if stop_reason: break
            assert len(set(S["seen"])) == N == len(S["seen"]), f"epoch {ep + 1} saw {len(set(S['seen']))} distinct of {N} utterances ({len(S['seen'])} total)"
            v = evaluate(student, proj, C, dev, amp); tr_l1 = S["ep_sum"] / max(S["ep_fr"], 1)
            v.update(epoch=ep + 1, update=S["update"], train_l1=tr_l1, lr=lr_now(S["update"]), epoch_minutes=(time.time() - t_ep) / 60,
                     **{a: round(x, 3) for a, x in gpu_mem().items()})
            improved = S["best"] is None or v["val_l1"] < S["best"]["val_l1"] - cfg["min_delta"]
            if improved and floor and v["val_l1"] >= floor["val_l1"]:
                S["best"] = {k: floor[k] for k in ("val_l1", "epoch", "update")}; S["es_counter"] = 0; improved = False
                log(f"  (val L1 {v['val_l1']:.5f} is not lower than the existing best.pt ({floor['val_l1']:.5f}): best.pt kept)")
                floor = None
            elif improved:
                S["best"] = dict(val_l1=v["val_l1"], epoch=ep + 1, update=S["update"], val_cos=v["val_cos"], val_ratio_mean=v["val_ratio_mean"]); S["es_counter"] = 0; floor = None
                atomic_torch_save(dict(student=student.state_dict(), asr_proj=proj.state_dict(), zip_args=ZIP_ARGS, cfg=cfg, val=v, epoch=ep + 1, update=S["update"],
                                       teacher=C["teacher_meta"], manifest_sha=C["manifest_sha"], alignment=run_cfg["alignment"], seed=cfg["seed"], sig=SIG,
                                       code_version=CODE_VERSION), BEST)
                if drive: drive.push(BEST, "best.pt")
            else: S["es_counter"] += 1
            vh = S["val_hist"]; flags = []
            if len(vh) >= 2 and v["val_l1"] > vh[-1]["val_l1"] > vh[-2]["val_l1"] and tr_l1 < vh[-1]["train_l1"] < vh[-2]["train_l1"]:
                flags.append("validation rose for 2 checks while training fell")
            if v["val_l1"] / tr_l1 > cfg["overfit_gap_ratio"]: flags.append(f"val/train L1 = {v['val_l1'] / tr_l1:.2f} > {cfg['overfit_gap_ratio']}")
            v["overfit_flags"] = "; ".join(flags); v["best"] = improved; v["es_counter"] = S["es_counter"]; vh.append(v)
            log(f"=== epoch {ep + 1}: train L1 {tr_l1:.4f} | VAL L1 {v['val_l1']:.4f} (mean-baseline {v['val_base_mean_l1']:.4f}, median-baseline "
                f"{v['val_base_median_l1']:.4f}) | cos {v['val_cos']:.3f} | step {S['update']} | {v['epoch_minutes']:.1f} min"
                + (" | NEW BEST (saved)" if improved else f" | no improvement ({S['es_counter']}/{cfg['patience']})")
                + (f" | POSSIBLE OVERFITTING: {v['overfit_flags']}" if flags else "") + " ===")
            S["epoch"] = ep + 1; S["batch"] = 0; S["seen"] = []; S["plan_remaining"] = None; S["ep_sum"] = S["ep_fr"] = 0.0
            if S["es_counter"] >= cfg["patience"]: S["finished"] = f"early stopping after epoch {ep + 1} (best epoch {S['best']['epoch']})"
            elif S["epoch"] >= cfg["max_epochs"]: S["finished"] = f"max_epochs {cfg['max_epochs']} reached"
            save_latest(True); write_history(); plots(P)
    except oom_error():
        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); save_latest(True); write_history()
        if drive: drive.wait()
        raise RuntimeError(f"CUDA out of memory. State saved at step {S['update']}. Set CFG['max_batch_seconds'] lower (effective {MBS:.0f}) and re-run: "
                           f"the rest of the epoch is re-batched.")
    except KeyboardInterrupt:
        save_latest(True); write_history(); log("interrupted: state saved locally; waiting for the Google Drive upload ...")
        if drive: drive.wait()
        make_report(preset, C, P, drive, verify_best=False); raise
    if S["finished"]:
        ck = torch.load(BEST, map_location=dev, weights_only=False); student.load_state_dict(ck["student"]); proj.load_state_dict(ck["asr_proj"])
        log(f"finished: {S['finished']}. Best checkpoint (epoch {ck['epoch']}) restored; kept at {BEST}" + (f" and {drive.dir / 'best.pt'}" if drive else "")); del ck
    elif stop_reason: log(f"{stop_reason}: state saved at step {S['update']}. Re-run the cell to continue.")
    del student, proj, opt, params; torch.cuda.empty_cache()
    if drive: log("waiting for the Google Drive uploads to finish ..."); drive.wait()
    return make_report(preset, C, P, drive, verify_best=bool(S["finished"]))

# ####################################################################################################
# ---- resume test: deliberately interrupted short runs must continue from the correct epoch, batch and optimizer step
# ####################################################################################################
def resume_test(C, n_train=24, n_dev=8):
    """Short runs on 24 training / 8 validation utterances, with a LOCAL stand-in folder for Drive (nothing is written to Google Drive):
       reference : uninterrupted run
       run A     : killed after step c1 without saving (as a dying runtime would) ...
       recycle   : local checkpoints deleted (= runtime recycled) and an incomplete 'latest.pt.tmp' planted in the Drive folder
       run B     : must resume from the Drive copy (the corrupt upload ignored), and is killed again after step c2
       run C     : must resume from the local checkpoint (more advanced than the Drive one) and finish.
    Checks: resume source / epoch / batch / global step; every step done exactly once with the reference's (epoch, batch); losses equal the
    reference's (same data order and RNG; GPU kernels are not bit-exact); same final state; no utterance repeated or skipped (asserted per epoch)."""
    import torch, gc
    root = Path(CFG["write_root"]) / "resume_test"
    if root.exists(): shutil.rmtree(root)
    trs = C["tr"].sort_values(["duration_s", "uid"]); tr = trs.iloc[np.linspace(0, len(trs) // 2, n_train).astype(int)].drop_duplicates("uid").reset_index(drop=True)
    dvs = C["dv"].sort_values(["duration_s", "uid"]); dv = dvs.iloc[np.linspace(0, len(dvs) // 2, n_dev).astype(int)].drop_duplicates("uid").reset_index(drop=True)
    Ct = dict(C, tr=tr, dv=dv, manifest_sha=sha_of(sorted(tr.uid)), dev_sha=sha_of(sorted(dv.uid)))
    LE, DE = 3, 7
    ov = dict(max_epochs=4, warmup_updates=4, local_ckpt_every_updates=LE, drive_ckpt_every_updates=DE, ckpt_every_minutes=1e9, log_every=1, max_batch_seconds=40.0,
              patience=99, max_session_hours=None)
    def mkP(name):
        P = dict(ckpt=root / name / "ckpt", logs=root / name / "logs", results=root / name / "results"); P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest")
        return P
    def go(P, ddir, interrupt=None):
        drv = DriveStore(ddir, fake=True); out = "finished"
        try: train(RUN_PRESETS[0], C=Ct, P=P, drive=drv, overrides=ov, interrupt_after_update=interrupt, probe=False)   # the optimizer actually used (ScaledAdam + Eden)
        except SimulatedDisconnect as e: out = str(e)
        drv.wait(); gc.collect(); torch.cuda.empty_cache(); return out
    def state(P): ck = torch.load(P["latest"] / "checkpoint.pt", map_location="cpu", weights_only=False); s = ck["state"]; del ck; return s
    log("---- reference run (uninterrupted)"); Pr = mkP("reference"); go(Pr, root / "drive_reference"); R = state(Pr)
    ref = {h["update"]: h for h in R["train_hist"]}; total = R["update"]; ends = {v["update"] for v in R["val_hist"]}
    lpts = {u for u in ref if u % LE == 0} | ends; dpts = {u for u in ref if u % DE == 0} | ends
    lastle = lambda pts, u: max([p for p in pts if p <= u], default=0)
    c1 = next((u for u in range(DE + 2, total) if u not in lpts | dpts), None)
    c2 = next((u for u in range(c1 + 2, total) if u not in lpts | dpts and lastle(lpts, u) > lastle(dpts, u) > lastle(dpts, c1)), None) if c1 else None
    if c2 is None: raise RuntimeError(f"resume test: the reference run has only {total} steps, too few to place the two interruptions")
    def expect(u):                       # saved position after step u
        return (ref[u]["epoch"], 0) if u in ends else (ref[u]["epoch"] - 1, ref[u]["batch"])
    Pb, Db = mkP("interrupted"), root / "drive_interrupted"; checks = []
    log(f"---- run A: killed after step {c1} (local checkpoint at step {lastle(lpts, c1)}, Drive at step {lastle(dpts, c1)})"); go(Pb, Db, c1)
    shutil.rmtree(Pb["ckpt"]); (Db / "latest.pt.tmp").write_bytes(os.urandom(1 << 20))
    log(f"---- runtime recycle simulated (local checkpoints deleted; incomplete upload planted) -> run B must resume from Drive at step {lastle(dpts, c1)}; killed after step {c2}")
    go(Pb, Db, c2)
    log(f"---- run C: must resume from the local checkpoint at step {lastle(lpts, c2)} (Drive has step {lastle(dpts, c2)}) and finish"); go(Pb, Db)
    F = state(Pb); res = F["resumes"]
    exp = [("Google Drive", lastle(dpts, c1)), ("local disk", lastle(lpts, c2))]
    checks.append(("resume sources and steps", [(r["source"], r["update"]) for r in res] == exp, f"{[(r['source'], r['update']) for r in res]} expected {exp}"))
    pos = [(r["epoch"], r["batch"]) for r in res]; epos = [expect(u) for _, u in exp]
    checks.append(("resumed epoch and batch position", pos == epos, f"{pos} expected {epos} (epoch index, batch index)"))
    upd = [h["update"] for h in F["train_hist"]]
    checks.append(("every optimizer step done exactly once", upd == list(range(1, total + 1)), f"{len(upd)} steps, reference {total}"))
    same_pos = all((h["epoch"], h["batch"]) == (ref[h["update"]]["epoch"], ref[h["update"]]["batch"]) for h in F["train_hist"])
    checks.append(("each step used the reference's epoch/batch", same_pos, ""))
    rel = {h["update"]: abs(h["l1"] - ref[h["update"]]["l1"]) / ref[h["update"]]["l1"] for h in F["train_hist"]}
    first = [rel[u + 1] for _, u in exp]
    checks.append(("first loss after each resume == reference", max(first) < 1e-2, f"relative differences {[f'{x:.1e}' for x in first]}"))
    checks.append(("all losses == reference", max(rel.values()) < 5e-2, f"max relative difference {max(rel.values()):.1e}"))
    checks.append(("final state == reference", F["epoch"] == R["epoch"] and F["update"] == R["update"] and bool(F["finished"])
                   and abs(F["val_hist"][-1]["val_l1"] - R["val_hist"][-1]["val_l1"]) / R["val_hist"][-1]["val_l1"] < 5e-2,
                   f"epochs {F['epoch']}/{R['epoch']}, steps {F['update']}/{R['update']}, final val L1 {F['val_hist'][-1]['val_l1']:.4f}/{R['val_hist'][-1]['val_l1']:.4f}"))
    checks.append(("incomplete upload ignored and removed", not (Db / "latest.pt.tmp").exists(), ""))
    bm = best_meta(Db / "best.pt"); vbest = min(v["val_l1"] for v in F["val_hist"])
    checks.append(("best.pt in the Drive folder holds the lowest validation loss", bm["val_l1"] <= vbest + ov.get("min_delta", CFG["min_delta"]) + 1e-9,
                   f"best.pt {bm['val_l1']:.5f}, lowest epoch val {vbest:.5f}"))
    for n, okk, info in checks: log(f"  [{'PASS' if okk else 'FAIL'}] {n}: {info}")
    rep = dict(passed=all(c[1] for c in checks), code_version=CODE_VERSION, time=time.strftime("%Y-%m-%d %H:%M:%S"), interrupts=[c1, c2], total_steps=total,
               checks=[dict(check=n, passed=bool(o), info=i) for n, o, i in checks])
    atomic_json(rep, paths()["logs"] / "resume_test.json")
    if rep["passed"]: shutil.rmtree(root); log("RESUME TEST PASSED (test folders removed)")
    else: log(f"RESUME TEST FAILED: training is blocked (test folders kept in {root})")
    return rep

# ####################################################################################################
# ---- plots + final report
# ####################################################################################################
def plots(P):
    """Training curves (local disk only). Columns are read with [...] (T.update would be the DataFrame.update METHOD).
    A plotting problem is reported but never stops training."""
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        ft, fv = P["logs"] / "train_history.csv", P["logs"] / "history.csv"
        if not fv.exists() or fv.stat().st_size < 40: return
        V = pd.read_csv(fv); T = pd.read_csv(ft) if ft.exists() and ft.stat().st_size > 10 else None
        if T is not None and not len(T): T = None
        best = V["best"].astype(str).str.lower().eq("true")
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
        if T is not None: ax[0].plot(T["update"], T["l1"], alpha=0.5, label="train L1 (logged steps)")
        ax[0].plot(V["global_step"], V["train_loss"], "o-", label="train L1 (epoch)"); ax[0].plot(V["global_step"], V["val_loss"], "o-", label="validation L1 (dev-clean)")
        if best.any():
            b = V[best].iloc[-1]; ax[0].axvline(b["global_step"], c="green", ls=":", label=f"best (epoch {int(b['epoch'])})")
        ax[0].set(title="ASR KD loss", xlabel="global step", ylabel="L1"); ax[0].legend()
        if T is not None:
            ax[1].semilogy(T["update"], T["grad_norm"].clip(lower=1e-4)); ax[1].set(title="gradient norm (before clipping)", xlabel="global step")
            ax[2].plot(T["update"], T["lr"]); ax[2].set(title="learning rate", xlabel="global step")
        fig.tight_layout(); fig.savefig(P["results"] / "training_curves.png", dpi=110); plt.close(fig)
    except Exception as e:
        log(f"  (plot not updated: {type(e).__name__}: {e})")

def make_report(preset, C, P, drive=None, verify_best=True):
    import torch
    LATEST, BEST = P["latest"] / "checkpoint.pt", P["best"] / "best.pt"
    if not LATEST.exists(): log("no training state yet"); return None
    ck = torch.load(LATEST, map_location="cpu", weights_only=False); S = ck["state"]; del ck
    rc = json.load(open(P["logs"] / "run_config.json")); V = pd.DataFrame(S["val_hist"]); d = rc["data"]
    rep = dict(
        status=("COMPLETED: " + S["finished"]) if S["finished"] else
               f"INTERRUPTED / NOT FINISHED at epoch {S['epoch'] + 1}, batch {S['batch']}, global step {S['update']} (re-run the cell to resume)",
        training_completed=bool(S["finished"]),
        dataset=dict(train=f"LibriSpeech train-clean-100, reproducible '{d['train_subset']}' subset (seed {rc['seed']}, target {d['target_seconds']:.0f} s)",
                     train_utterances=d["train_utterances"], train_seconds=d["train_seconds"], train_speakers=d["train_speakers"],
                     validation=f"{d['validation']['split']}: {d['validation']['utterances']} utterances, {d['validation']['hours']} h",
                     test="test-clean not used in Stage 1", discarded=d["discarded"]),
        teacher=rc["teacher"], student=rc["student"],
        feature_dimensions=dict(student_input="80-D fbank @ 100 Hz", student_output="512-D @ 25 Hz", projection="512 -> 2048", teacher="WavLM-Large 1024-D @ 50 Hz -> 2048-D @ 25 Hz"),
        distillation_loss=rc["loss"], alignment=rc["alignment"],
        epochs_completed=S["epoch"], total_optimizer_steps=S["update"], best_validation=S["best"],
        last_epoch=dict(train_l1=float(V.train_l1.iloc[-1]), val_l1=float(V.val_l1.iloc[-1]), val_mean_baseline_l1=float(V.val_base_mean_l1.iloc[-1]),
                        val_median_baseline_l1=float(V.val_base_median_l1.iloc[-1]), val_cos=float(V.val_cos.iloc[-1])) if len(V) else None,
        overfitting_flags=[f"epoch {int(r.epoch)}: {r.overfit_flags}" for r in V.itertuples() if isinstance(r.overfit_flags, str) and r.overfit_flags] if len(V) else [],
        early_stopping=dict(counter=S["es_counter"], patience=rc["training"]["patience"]),
        training_time_hours=round(S["elapsed"] / 3600, 3), sessions=S["sessions"], resumes=S.get("resumes", []),
        checkpoints=dict(drive_best=str(drive.dir / "best.pt") if drive else None, drive_latest=str(drive.dir / "latest.pt") if drive else None,
                         local_best=str(BEST), local_latest=str(LATEST)),
        drive_uploads=dict(drive.done) if drive else None,
        not_paper_exact=["ENHANCEMENT, not the paper: teacher WavLM-Large instead of Whisper-large-v3",
                         "reduced-data pilot: 2 h subset of train-clean-100, not the paper's training data",
                         "optimizer/schedule: ScaledAdam + Eden as in the paper / icefall (identical to the Whisper v4 run)",
                         "k2 not used: Swoosh activations are a pure-PyTorch port of k2's implementation, verified numerically",
                         "frame pairing (student j <-> teacher j+offset) derived from measured frame centres; not verified against the authors' code",
                         "projection bias initialised to the mean teacher frame (engineering choice)",
                         "WavLM: every utterance encoded whole (no chunking); odd last 50 Hz frame dropped by the pairing"],
        note="Stage 1 = feature-level distillation; the CTC probe of this cell gives a frozen-encoder WER for the teacher comparison (not the final Stage 2 WER).")
    if verify_best and BEST.exists():
        dev = torch.device("cuda"); student, proj = new_models(C, dev); b = torch.load(BEST, map_location=dev, weights_only=False)
        student.load_state_dict(b["student"], strict=True); proj.load_state_dict(b["asr_proj"], strict=True)
        v = evaluate(student, proj, C, dev, amp_dtype()); del student, proj; torch.cuda.empty_cache()
        rep["final_evaluation_best_checkpoint"] = dict(split=CFG["val_split"], epoch=b["epoch"], global_step=b["update"], val_l1=v["val_l1"], stored_val_l1=b["val"]["val_l1"],
                                                       reload_matches=abs(v["val_l1"] - b["val"]["val_l1"]) < 2e-3, val_cos=v["val_cos"],
                                                       ratio_to_mean_baseline=v["val_ratio_mean"], ratio_to_median_baseline=v["val_ratio_median"])
    atomic_json(rep, P["results"] / "final_report.json"); plots(P)
    if drive: drive.write_json("final_report.json", rep)
    log("\n" + "=" * 90 + f"\nWavLM ASR STAGE 1 FINAL REPORT\n" + "=" * 90)
    for k in ["status", "dataset", "epochs_completed", "total_optimizer_steps", "best_validation", "last_epoch", "final_evaluation_best_checkpoint", "training_time_hours",
              "sessions", "resumes", "overfitting_flags", "checkpoints"]:
        if k in rep: log(f"{k:34s}: {rep[k]}")
    log(f"(full report: {P['results'] / 'final_report.json'}" + (f" and {drive.dir / 'final_report.json'}" if drive else "") + ")")
    return rep


# ####################################################################################################
# ---- CTC PROBE: does the WavLM-taught student carry more SPEECH-RECOGNITION information than the Whisper-taught one?
# ####################################################################################################
# Each student's encoder is FROZEN; only a small probe is trained on top of it, identically for every student:
#   student output (512-D @ 25 Hz) -> per-dimension standardisation (train statistics) -> each frame repeated twice (50 Hz, so that
#   CTC has room for repeated letters) -> 2-layer BiLSTM -> Linear -> 29 symbols (CTC blank, space, apostrophe, A-Z) -> CTC loss.
#   Data: the 10 h subset of train-clean-100 (it contains the 2 h KD subset) with its transcripts; evaluation: the whole dev-clean, greedy
#   decoding, WER and CER computed by this code. Fixed seed, fixed number of epochs (no model selection on dev-clean), same settings for all.
#   Students compared: WavLM-taught (this cell), Whisper-taught (your ASR v4 sanity_2h best.pt, read-only), random-init (control = no teacher).
VOCAB = ["<blank>", " ", "'"] + [chr(c) for c in range(ord("A"), ord("Z") + 1)]
CHAR2ID = {c: i for i, c in enumerate(VOCAB)}

def text_to_ids(t):
    t = " ".join(str(t).upper().split()); bad = sorted(set(t) - set(CHAR2ID))
    if bad: raise ValueError(f"unexpected characters {bad} in transcript {t[:60]!r}")
    return [CHAR2ID[c] for c in t]

def ctc_min_frames(ids):
    """Fewest frames a CTC path needs: one per label plus a blank between two equal neighbours."""
    return len(ids) + sum(1 for a, b in zip(ids, ids[1:]) if a == b)

def greedy_decode(best):
    """best: list of argmax symbol ids per frame -> text (repeats collapsed, blanks removed)."""
    out = []; prev = -1
    for k in best:
        if k != prev and k != 0: out.append(VOCAB[k])
        prev = k
    return " ".join("".join(out).split())

def edit_distance(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, y in enumerate(b, 1): cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y))
        prev = cur
    return prev[-1]

def error_rates(refs, hyps, with_cer=True):
    refs = [" ".join(r.upper().split()) for r in refs]; hyps = [" ".join(h.split()) for h in hyps]
    we = sum(edit_distance(r.split(), h.split()) for r, h in zip(refs, hyps)); wn = sum(len(r.split()) for r in refs)
    R = dict(wer=we / max(wn, 1), word_errors=we, words=wn, utterances=len(refs))
    if with_cer:
        ce = sum(edit_distance(list(r), list(h)) for r, h in zip(refs, hyps)); cn = sum(len(r) for r in refs); R.update(cer=ce / max(cn, 1), char_errors=ce, chars=cn)
    return R

class _FbankDS:
    def __init__(self, df): self.r = df.to_dict("records")
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        r = self.r[i]; w = load_audio(r["relpath"], int(r["n_samples"])); return dict(uid=r["uid"], n=len(w), fb=fbank(w))
def _fb_collate(b):
    import torch, math
    from torch.nn.utils.rnn import pad_sequence
    return dict(uid=[x["uid"] for x in b], n=torch.tensor([x["n"] for x in b]),
                fb=pad_sequence([x["fb"] for x in b], batch_first=True, padding_value=math.log(1e-10)), fb_lens=torch.tensor([x["fb"].size(0) for x in b]))

def student_features(student, df, dev, amp):
    """Frozen student encoder outputs for every utterance of df -> {uid: float16 CPU tensor [T, 512]} (25 Hz)."""
    import torch
    student.eval(); out = {}
    bt = make_batches(df.n_samples.values, 0, CFG["seed"], False, 300.0, CFG["max_batch_utts"])
    dl = torch.utils.data.DataLoader(_FbankDS(df), batch_sampler=bt, num_workers=CFG["num_workers"], collate_fn=_fb_collate,
                                     prefetch_factor=4 if CFG["num_workers"] else None)
    with torch.no_grad():
        for b in dl:
            with torch.autocast(device_type="cuda", dtype=amp): enc, el = student(b["fb"].to(dev), b["fb_lens"].to(dev))
            enc = enc.float()
            if not torch.isfinite(enc).all(): raise FloatingPointError("non-finite student output")
            for k, u in enumerate(b["uid"]):
                L = int(el[k]); assert L == student_frames(int(b["n"][k])), (u, L); out[u] = enc[k, :L].half().cpu()
    assert len(out) == len(df), f"features for {len(out)} of {len(df)} utterances"
    return out

def build_probe(d_in):
    import torch, torch.nn as nn
    from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
    class CTCProbe(nn.Module):
        def __init__(s):
            super().__init__(); s.register_buffer("mu", torch.zeros(d_in)); s.register_buffer("sd", torch.ones(d_in))
            s.lstm = nn.LSTM(d_in, CFG["probe_hidden"], num_layers=CFG["probe_layers"], bidirectional=True, batch_first=True, dropout=CFG["probe_dropout"])
            s.drop = nn.Dropout(CFG["probe_dropout"]); s.out = nn.Linear(2 * CFG["probe_hidden"], len(VOCAB))
        def forward(s, x, lens):                         # x [B, T, 512] @ 25 Hz -> log-probs [B, 2T, 29] @ 50 Hz, lengths 2 * lens
            x = ((x.float() - s.mu) / s.sd).repeat_interleave(2, dim=1); L = lens * 2
            p = pack_padded_sequence(x, L.cpu(), batch_first=True, enforce_sorted=False)
            y, _ = s.lstm(p); y, _ = pad_packed_sequence(y, batch_first=True, total_length=x.size(1))
            return s.out(s.drop(y)).float().log_softmax(-1), L
    return CTCProbe()

def _probe_batches(uids, F, epoch, shuffle, max_sec):
    lens = np.array([len(F[u]) * HOP_T for u in uids])              # duration-equivalent lengths (samples) of the 25 Hz features
    return make_batches(lens, epoch, CFG["probe_seed"], shuffle, max_sec, 256)

def _probe_batch(uids, F, labels, idx, dev):
    import torch
    from torch.nn.utils.rnn import pad_sequence
    us = [uids[i] for i in idx]; x = pad_sequence([F[u].float() for u in us], batch_first=True).to(dev)
    lens = torch.tensor([len(F[u]) for u in us], device=dev)
    if labels is None: return us, x, lens, None, None
    tg = [labels[u] for u in us]
    return us, x, lens, torch.tensor([c for t in tg for c in t], device=dev, dtype=torch.long), torch.tensor([len(t) for t in tg], device=dev, dtype=torch.long)

def probe_decode(probe, uids, F, dev):
    import torch
    hyps = {}; probe.eval()
    with torch.no_grad():
        for idx in _probe_batches(uids, F, 0, False, CFG["probe_batch_seconds"]):
            us, x, lens, _, _ = _probe_batch(uids, F, None, idx, dev); lp, L = probe(x, lens); best = lp.argmax(-1).cpu()
            for k, u in enumerate(us): hyps[u] = greedy_decode(best[k, :int(L[k])].tolist())
    probe.train(); return hyps

def run_probe(name, Ftr, Fdv, tr_df, dv_df, dev):
    """Trains one CTC probe (identical settings and seed for every student) and evaluates it on dev-clean."""
    import torch, torch.nn.functional as Fn
    t0 = time.time(); set_seed(CFG["probe_seed"])
    labels = {u: text_to_ids(t) for u, t in zip(tr_df.uid, tr_df.text)}
    feasible = [u for u in tr_df.uid if 2 * len(Ftr[u]) >= ctc_min_frames(labels[u])]; n_inf = len(tr_df) - len(feasible)
    s = torch.zeros(D_S, dtype=torch.float64); s2 = torch.zeros(D_S, dtype=torch.float64); nfr = 0
    for u in feasible: x = Ftr[u].double(); s += x.sum(0); s2 += (x * x).sum(0); nfr += len(x)
    mu = s / nfr; sd = (s2 / nfr - mu * mu).clamp(min=1e-8).sqrt()
    probe = build_probe(D_S).to(dev); probe.mu.copy_(mu.float()); probe.sd.copy_(sd.float())
    opt = torch.optim.Adam(probe.parameters(), lr=CFG["probe_lr"])
    nb = len(_probe_batches(feasible, Ftr, 0, True, CFG["probe_batch_seconds"])); total = nb * CFG["probe_epochs"]; warm = max(1, min(300, total // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda st: min(1.0, (st + 1) / warm) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, st / total)))))
    hist = []; step = 0
    log(f"  [{name}] probe: {len(feasible):,} training utterances ({n_inf} too short for their transcript at 50 Hz, excluded), {nb} batches/epoch, "
        f"{CFG['probe_epochs']} epochs, BiLSTM {CFG['probe_layers']} x {CFG['probe_hidden']}, Adam {CFG['probe_lr']}")
    for ep in range(CFG["probe_epochs"]):
        tl = tn = 0.0
        for idx in _probe_batches(feasible, Ftr, ep, True, CFG["probe_batch_seconds"]):
            us, x, lens, tg, tl_ = _probe_batch(feasible, Ftr, labels, idx, dev)
            lp, L = probe(x, lens)
            loss = Fn.ctc_loss(lp.transpose(0, 1), tg, L, tl_, blank=0, reduction="sum", zero_infinity=True) / tl_.sum()
            if not torch.isfinite(loss): opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(probe.parameters(), 5.0); opt.step(); sched.step(); step += 1
            tl += float(loss) * float(tl_.sum()); tn += float(tl_.sum())
        rec = dict(probe=name, epoch=ep + 1, step=step, train_ctc_per_char=tl / max(tn, 1), lr=sched.get_last_lr()[0])
        if (ep + 1) % 5 == 0 or ep + 1 == CFG["probe_epochs"]:
            hy = probe_decode(probe, list(dv_df.uid), Fdv, dev); rec["dev_wer_greedy"] = error_rates(list(dv_df.text), [hy[u] for u in dv_df.uid], with_cer=False)["wer"]
        hist.append(rec)
        log(f"  [{name}] epoch {ep + 1}/{CFG['probe_epochs']}: train CTC {rec['train_ctc_per_char']:.4f} per character"
            + (f" | dev-clean WER {rec['dev_wer_greedy'] * 100:.2f}%" if "dev_wer_greedy" in rec else "") + f" | {(time.time() - t0) / 60:.1f} min")
    hy = probe_decode(probe, list(dv_df.uid), Fdv, dev); er = error_rates(list(dv_df.text), [hy[u] for u in dv_df.uid])
    ex = [dict(uid=u, ref=r, hyp=hy[u]) for u, r in list(zip(dv_df.uid, dv_df.text))[:3]]
    del probe, opt; torch.cuda.empty_cache()
    return dict(name=name, dev_clean_wer=er["wer"], dev_clean_cer=er["cer"], errors=er, train_utterances=len(feasible), excluded_too_short=n_inf,
                epochs=CFG["probe_epochs"], steps=step, minutes=round((time.time() - t0) / 60, 2), examples=ex), hist

def load_student_ckpt(path):
    """A trained student from a best.pt (read-only): returns (student on CPU, info)."""
    import torch
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict) or "student" not in ck: raise RuntimeError(f"{path} has no 'student' weights")
    if ck.get("zip_args") is not None and dict(ck["zip_args"]) != dict(ZIP_ARGS): raise RuntimeError(f"{path}: Zipformer arguments differ from this cell's")
    set_seed(CFG["seed"]); s, _ = build_models(1); s.load_state_dict(ck["student"], strict=True)
    v = ck.get("val") or {}
    info = dict(path=str(path), code_version=ck.get("code_version"), teacher=(ck.get("teacher") or {}).get("model"), epoch=ck.get("epoch"), update=ck.get("update"),
                kd_val_l1=v.get("val_l1"), kd_ratio_to_mean_baseline=v.get("val_ratio_mean"), kd_val_cos=v.get("val_cos"), manifest_sha=ck.get("manifest_sha"),
                file_sha1=file_sha1(path)[:16])
    del ck; return s, info

def probe_comparison(wavlm_best, drive=None):
    """Runs the CTC probe for: WavLM-taught student, Whisper-taught student (if its best.pt exists), random-init student. Resumable per student."""
    import torch, gc
    dev = torch.device("cuda"); amp = amp_dtype(); Pp = paths()["logs"] / "ctc_probe"; Pp.mkdir(parents=True, exist_ok=True)
    tr = load_manifest(CFG["probe_train_subset"]); dv = load_manifest("dev"); s2 = load_manifest(PRESETS[RUN_PRESETS[0]]["train_subset"])
    materialize(tr, "train-clean-100"); materialize(dv, CFG["val_split"])
    pcfg = {k: CFG[k] for k in CFG if k.startswith("probe_")}
    log(f"  probe data: {CFG['probe_train_subset']} = {len(tr):,} utterances / {tr.duration_s.sum() / 3600:.2f} h (contains the {len(s2):,}-utterance KD subset: "
        f"{set(s2.uid) <= set(tr.uid)}) | evaluation: dev-clean {len(dv):,} utterances | settings {pcfg}")
    students = [("WavLM-taught", wavlm_best)]
    wb = Path(CFG["whisper_student_ckpt"])
    if wb.exists(): students.append(("Whisper-taught", wb))
    else: log(f"  Whisper-taught student not found at {wb}: it is skipped (set CFG['whisper_student_ckpt'] to its best.pt)")
    students.append(("random-init (no teacher)", None))
    old = {}
    if drive is not None and (drive.dir / "final_report.json").exists():
        try: old = {r["name"]: r for r in json.load(open(drive.dir / "final_report.json")).get("results", [])}
        except Exception: old = {}
    results, hist_all = [], []
    if (Pp / "history.csv").exists():
        try: hist_all = pd.read_csv(Pp / "history.csv").to_dict("records")
        except Exception: hist_all = []
    for name, ck in students:
        if ck is None: set_seed(CFG["seed"]); st, _ = build_models(1); info = dict(path=None, teacher=None, note="Zipformer-M with random weights (seed %d), never trained" % CFG["seed"])
        else: st, info = load_student_ckpt(ck)
        if info.get("manifest_sha") and info["manifest_sha"] != sha_of(sorted(s2.uid)): log(f"  NOTE [{name}]: its KD training subset differs from this cell's 2 h subset")
        sig = sha_of([json.dumps(pcfg, sort_keys=True), info.get("file_sha1"), name, sha_of(sorted(tr.uid)), sha_of(sorted(dv.uid)), CODE_VERSION])
        if name in old and old[name].get("sig") == sig:
            log(f"  [{name}] already done (same checkpoint and probe settings): WER {old[name]['dev_clean_wer'] * 100:.2f}% -> reused"); results.append(old[name]); continue
        log(f"\n  ---- [{name}] {info.get('path') or 'random weights'} (KD teacher: {info.get('teacher')}, KD epoch {info.get('epoch')}, "
            f"KD ratio to mean baseline {info.get('kd_ratio_to_mean_baseline')})")
        st = st.to(dev); t0 = time.time(); Ftr = student_features(st, tr, dev, amp); Fdv = student_features(st, dv, dev, amp)
        st.cpu(); del st; torch.cuda.empty_cache(); log(f"  [{name}] frozen encoder features extracted in {(time.time() - t0) / 60:.1f} min")
        r, h = run_probe(name, Ftr, Fdv, tr, dv, dev); r.update(student=info, sig=sig); results.append(r)
        hist_all = [x for x in hist_all if x.get("probe") != name] + h
        del Ftr, Fdv; gc.collect()
        atomic_json(r, Pp / f"{sha_of([name])}.json"); pd.DataFrame(hist_all).to_csv(Pp / "history.csv", index=False)
        rep = dict(code_version=CODE_VERSION, updated=time.strftime("%Y-%m-%d %H:%M:%S"), probe_settings=pcfg,
                   data=dict(train=f"LibriSpeech train-clean-100 {CFG['probe_train_subset']} subset ({len(tr)} utterances)", evaluation=f"dev-clean ({len(dv)} utterances)"),
                   results=list(results), status="in progress (re-run the cell to continue with the remaining students)")
        if drive is not None: drive.write_json("final_report.json", rep); drive.write_text("history.csv", pd.DataFrame(hist_all).to_csv(index=False))
        log(f"  [{name}] dev-clean WER {r['dev_clean_wer'] * 100:.2f}% | CER {r['dev_clean_cer'] * 100:.2f}% | probe {r['minutes']:.1f} min | example: "
            f"REF {r['examples'][0]['ref'][:70]!r} / HYP {r['examples'][0]['hyp'][:70]!r}")
    rep = dict(code_version=CODE_VERSION, updated=time.strftime("%Y-%m-%d %H:%M:%S"), probe_settings=pcfg,
               data=dict(train=f"LibriSpeech train-clean-100 {CFG['probe_train_subset']} subset ({len(tr)} utterances)", evaluation=f"dev-clean ({len(dv)} utterances)"),
               results=results,
               reading=("lower WER = the frozen student carries more speech-recognition information. Same student architecture, same 2 h KD audio, same KD "
                        "recipe; only the teacher differs (the random-init row shows what the probe gets from an untrained encoder). This is a probe on a frozen "
                        "Stage-1 encoder, not the final MT2KD ASR WER (Stage 2 fine-tunes the encoder with a transducer)."))
    atomic_json(rep, Pp / "final_report.json")
    if drive is not None:
        drive.write_json("final_report.json", rep); drive.write_text("history.csv", pd.DataFrame(hist_all).to_csv(index=False))
        drive.write_json("config.json", dict(probe_settings=pcfg, students=[dict(name=n, checkpoint=str(c) if c else None) for n, c in students], code_version=CODE_VERSION))
    return rep

'''

# --------------------------------------------------------------------------------------------------------------
# module mt2kd_mod_at  (verified earlier cell, unchanged; written to {_MOD_DIR}/mt2kd_mod_at.py and imported)
# --------------------------------------------------------------------------------------------------------------
_MODULES["mt2kd_mod_at"] = r'''# ====================================================================================================================
# MT2KD ENHANCEMENT — AUDIO-TAGGING TEACHER TEST: BEATs (current) vs CED-Base / CED-Small / CED-Mini vs SSLAM on FSD50K — SINGLE CELL, NO training
#
# WHY: before replacing BEATs (as we did for Whisper -> WavLM), measure on OUR data whether a newer AudioSet teacher is really better.
#   Every teacher scores the same 16 kHz, 10 s FSD50K crops (the AT2/AT3/AT v2 audio cache: MT2KD/cache16k) of the WHOLE official
#   validation split (4,170 clips; decision) and evaluation split (10,231 clips; confirmation). Metric: mAP over the FSD50K classes whose
#   AudioSet ID is among the 527 outputs: the same mapping and mAP code as the AT v2 cell (BEATs there: 50.29 % on val).
#   Paired bootstrap (100 resamples) gives a 95 % interval for each candidate's difference to BEATs.
# DECISION RULE: CED-Small if within ~1 mAP of the best candidate; otherwise the best candidate; keep BEATs if no candidate is
#   significantly better (95 % interval of the difference above 0).
# TEACHERS: BEATs = your fine-tuned AS2M cpt2 checkpoint on Drive (as in AT v2). CED (Apache-2.0) and SSLAM (MIT) are downloaded from
#   Hugging Face to local disk. A candidate that fails to load is skipped with a message (never fatal); its label order is checked.
# DRIVE: writes only MT2KD/results/AT_teacher_comparison/ (final_report.json, history.csv, config.json; a few KB). Everything else is read-only:
#   the audio crops are restored from MT2KD/cache16k when present, missing ones are built from the WAVs onto LOCAL disk only.
# TIME: ~45-75 min on a T4 (mostly restoring/building the eval-split audio the first time), ~25-40 min on an A100. Resumable.
# ====================================================================================================================
import os, sys, json, math, time, shutil, hashlib, subprocess, importlib, importlib.util, re, random, threading, itertools
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

CODE_VERSION = "at-teacher-test-v2"
os.environ.setdefault("HF_HOME", "/content/hf_home")   # model downloads stay on local disk

CFG = dict(
    # ---- read-only inputs on Drive
    project_root="/content/drive/MyDrive/MT2KD",
    fsd_root="/content/drive/MyDrive/MT2KD/datasets/FSD50K",                 # FSD50K.dev_audio/, FSD50K.ground_truth/, processed/
    friend_at_root="/content/drive/MyDrive/MT2KD/outputs_AT_KD",               # AT1/AT2 outputs (class_mapping.csv, teacher_*.npz): reference only
    beats_search=["/content/drive/MyDrive/MT2KD/datasets/FSD50K", "/content/drive/MyDrive/MT2KD/datasets/FSD50K/checkpoint",
                  "/content/drive/MyDrive/MT2KD/checkpoints/BEATs", "/content/drive/MyDrive/MT2KD/checkpoints", "/content/drive/MyDrive/MT2KD"],
    unilm_repo="https://github.com/microsoft/unilm.git",                       # official BEATs model code (only the beats/ folder is fetched)
    # ---- outputs
    drive_out_root="/content/drive/MyDrive/MT2KD/results",                     # each run: <root>/AT_Stage1_v2_<run>/ (5 files)
    drive_api=True,
    drive_audio_cache="/content/drive/MyDrive/MT2KD/cache16k",                 # <split>_16k_c10.tar (same format as AT3): restored, and updated when clips are added
    save_audio_cache_to_drive=True,
    write_root="/content/MT2KD_at_teacher_test", local_audio="/content/fsd_cache16k", local_beats="/content/beats_local",
    unilm_dir="/content/unilm_beats", zipformer_dir="/content/MT2KD_zipformer_mt2kd",
    # ---- data (identical crop to AT2/AT3, so the cached reference teacher logits refer to the same audio)
    seed=42, max_seconds=10.0, cache_threads=16,
    # ---- teacher
    teacher_batch=32, teacher_match_tol=1e-3, require_teacher_match=True,      # new BEATs probabilities must equal AT2's cached ones within 1e-3
    # ---- student / KD
    n_classes=527, init_bias="teacher_prior",                                  # head bias starts at the mean teacher probability (label-free)
    # ---- optimisation: paper / icefall = ScaledAdam + Eden
    base_lr=0.045, lr_batches=7500, lr_epochs=3.5, eden_warmup_batches=500, eden_warmup_start=0.1, clipping_scale=2.0,
    max_batch_seconds=600.0, max_batch_utts=128, ref_duration=600.0, mem_fraction=0.85, max_consecutive_skips=25,
    max_session_hours=None,
    # ---- validation / early stopping / checkpoints
    patience=5, min_rel_delta=1e-3, eval_batch_seconds=600.0,
    local_ckpt_every_updates=100, drive_ckpt_every_updates=500, ckpt_every_minutes=15.0,
    log_every=50, num_workers=min(8, os.cpu_count() or 2),
)
PRESETS = {
    # quick gate (~1 h on an A100): the 2,000 training clips of AT3's smoke run (their audio and BEATs targets are already cached),
    # small batches -> ~100 steps/epoch, ~3,000 steps. Validation = the whole official FSD50K validation split (4,170 clips).
    "at_sanity": dict(n_train=2000, max_epochs=30, max_batch_seconds=120.0, patience=8, gate_ratio=0.85),
    # all 36,796 FSD50K training clips (~3-4 h on an A100 after ~1 h of one-time audio/teacher preparation)
    "at_full": dict(n_train=None, max_epochs=30, max_batch_seconds=600.0, patience=5),
}
RUN_PRESETS = ["at_sanity"]                          # add "at_full" to continue with all training clips (it starts only if at_sanity passes)
CFG.update(test_splits=["val"], n_bootstrap=100,                  # validation only (decision split); add "eval" for confirmation
           save_audio_cache_to_drive=False)   # test only: audio built here stays on local disk (no large tar written to Drive)   # official FSD50K validation (decision) + evaluation (confirmation) splits
OFFICIAL = {"train": 36796, "val": 4170, "eval": 10231}
SR = 16000
PREV_AT3 = dict(val_mAP_mapped=0.0540, eval_mAP_mapped=0.0433, teacher_val_mAP_mapped=0.5337, note="AT3 smoke run (AdamW, 250 steps, 186-class KD, 500 val clips)")
def drive_dir(preset): return f"{CFG['drive_out_root']}/AT_Stage1_v2_{preset}"

def paths(preset=None):
    W = Path(CFG["write_root"]); P = dict(meta=W / "meta", teacher=W / "teacher", logs=W / "logs", ckpt=W / "checkpoints", results=W / "results")
    if preset:
        for k in ("ckpt", "logs", "results"): P[k] = P[k] / preset
        P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest")
    return P
def ensure_dirs(P):
    for v in P.values():
        if isinstance(v, Path): assert_local(v); v.mkdir(parents=True, exist_ok=True)
def oom_error():
    import torch
    return getattr(torch, "OutOfMemoryError", None) or torch.cuda.OutOfMemoryError
def code_fingerprint(): return CODE_VERSION

# ####################################################################################################
# ---- reused UNCHANGED from the verified ASR v4 cell: helpers, Zipformer-M + verified Swoosh port, fbank, batching, Drive store
# ####################################################################################################
def log(*a): print(*a, flush=True)

DRIVE = "/content/drive"

def assert_local(p):
    """Every LOCAL write goes through here: refuses anything on Google Drive (Drive writes exist only in class DriveStore)."""
    rp = os.path.realpath(str(p))
    if rp.startswith(DRIVE) or str(p).startswith(DRIVE): raise PermissionError(f"refusing to write to Google Drive: {p}")
    return p

def pip_if_missing(*mods):
    for m in mods:
        mod, pkg = (m, m) if isinstance(m, str) else m
        if importlib.util.find_spec(mod) is None: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def atomic_json(obj, p):
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp")
    json.dump(obj, open(tmp, "w"), indent=1, default=str); os.replace(tmp, p)

def atomic_torch_save(obj, p):
    import torch
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, p)

def sha_of(items): return hashlib.sha1("\n".join(map(str, items)).encode()).hexdigest()[:12]

def file_sha1(p, chunk=1 << 22):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def set_seed(s):
    import random, numpy as np, torch
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def gpu_mem():
    import torch
    if not torch.cuda.is_available(): return {}
    return dict(gpu_alloc_gib=torch.cuda.memory_allocated() / 2**30, gpu_peak_gib=torch.cuda.max_memory_allocated() / 2**30,
                gpu_reserved_gib=torch.cuda.memory_reserved() / 2**30, gpu_total_gib=torch.cuda.get_device_properties(0).total_memory / 2**30)

def disk_free_gib(p):
    p = Path(p)
    while not p.exists(): p = p.parent
    return shutil.disk_usage(p).free / 2**30

def require_space(path, need_gib, what, margin_gib=1.0):
    free = disk_free_gib(path)
    if need_gib > free - margin_gib: raise RuntimeError(f"not enough space for {what}: need {need_gib:.1f} GiB (+{margin_gib} margin), {free:.1f} GiB free at {path}")
    return free

D_S = 512

ZIP_ARGS = dict(output_downsampling_factor=2, downsampling_factor=(1, 2, 4, 8, 4, 2), num_encoder_layers=(2, 2, 3, 4, 3, 2),
                encoder_dim=(192, 256, 384, 512, 384, 256), encoder_unmasked_dim=(192, 192, 256, 256, 256, 192), query_head_dim=(32,) * 6,
                pos_head_dim=(4,) * 6, value_head_dim=(12,) * 6, num_heads=(4, 4, 4, 8, 4, 4), feedforward_dim=(512, 768, 1024, 1536, 1024, 768),
                cnn_module_kernel=(31, 31, 15, 15, 15, 31), pos_dim=48, causal=False)

def fbank_frames(n): return (n + 80) // 160

def embed_frames(n): return (fbank_frames(n) - 7) // 2

def student_frames(n): return (embed_frames(n) + 1) // 2

K2_SWOOSH_SRC = r"""# Pure-PyTorch port of k2's Swoosh functions (k2/python/csrc/torch/v2/autograd/swoosh.cu), written by the MT2KD ASR Stage 1 pipeline.
# Same formulas, same fp32 compute, same 8-bit stochastically-rounded derivative in the autograd path. Differences from k2:
# the random numbers for the stochastic rounding come from torch's RNG (k2: its own torch::rand call) -> statistically identical, not bitwise.
import torch
_P = {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}

def _forward(x, shift, coeff, offset):                    # k2 SwooshForward
    x = x.to(torch.float32); xo = x - shift
    l = torch.log1p(torch.exp(xo)); l = torch.where(torch.isinf(l), xo, l)
    return l - coeff * x - offset

def _forward_and_deriv(x, shift, coeff, offset):          # k2 SwooshForwardAndDeriv
    x = x.to(torch.float32); xo = x - shift
    denom = 1.0 + torch.exp(xo); d = (1.0 - coeff) - 1.0 / denom
    ld = torch.log(denom); ld = torch.where(torch.isinf(ld), xo, ld)
    return ld - coeff * x - offset, d

class _SwooshFunction(torch.autograd.Function):           # k2 SwooshFunction (autograd path)
    @staticmethod
    def forward(ctx, x, shift, coeff, offset, dropout_prob):
        x = x.to(torch.float32).contiguous(); xo = x - shift; e = torch.exp(xo)
        g = 1.0 - 1.0 / (1.0 + e)
        l = torch.log1p(e); l = torch.where(torch.isinf(l), xo, l); y = l - coeff * x - offset
        if dropout_prob != 0.0:
            drop = torch.rand_like(x) < dropout_prob
            y = torch.where(drop, torch.zeros_like(y), y * (1.0 / (1.0 - dropout_prob))); g = torch.where(drop, torch.full_like(g, coeff), g)
        g_int = (g * (255.0 / 1.005) + torch.rand_like(x)).to(torch.uint8)
        ctx.save_for_backward(g_int); ctx.coeff = coeff; ctx.dropout_prob = dropout_prob
        return y
    @staticmethod
    def backward(ctx, y_grad):
        (g,) = ctx.saved_tensors
        f = (g.to(torch.float32) * (1.005 / 255.0) - ctx.coeff) / (1.0 - ctx.dropout_prob)
        return y_grad.to(torch.float32) * f, None, None, None, None

def _swoosh(x, which, dropout_prob):
    shift, coeff, offset = _P[which]
    if not x.requires_grad:                               # k2: logaddexp path when no gradient is needed
        x = x.to(torch.float32); return torch.logaddexp(torch.zeros(1, device=x.device), x - shift) - coeff * x - offset
    return _SwooshFunction.apply(x, shift, coeff, offset, float(dropout_prob))

def swoosh_l(x, dropout_prob=0.0): return _swoosh(x, "l", dropout_prob)
def swoosh_r(x, dropout_prob=0.0): return _swoosh(x, "r", dropout_prob)
def swoosh_l_forward(x): return _forward(x, *_P["l"])
def swoosh_r_forward(x): return _forward(x, *_P["r"])
def swoosh_l_forward_and_deriv(x): return _forward_and_deriv(x, *_P["l"])
def swoosh_r_forward_and_deriv(x): return _forward_and_deriv(x, *_P["r"])
"""

STUB_FUNCS = {"swoosh_l", "swoosh_r", "swoosh_l_forward", "swoosh_r_forward", "swoosh_l_forward_and_deriv", "swoosh_r_forward_and_deriv"}

def setup_zipformer():
    """Private copy of icefall's Zipformer source (unchanged except `import k2` -> the Swoosh port). Returns source information."""
    ZDIR = CFG["zipformer_dir"]; SRC_Z = "/content/icefall/egs/librispeech/ASR/zipformer"
    if not os.path.isdir(SRC_Z):
        subprocess.run("rm -rf /content/icefall; git clone --depth 1 https://github.com/k2-fsa/icefall.git /content/icefall", shell=True, check=True)
    os.makedirs(ZDIR + "/icefall", exist_ok=True)
    for f in ["zipformer.py", "scaling.py", "subsampling.py", "encoder_interface.py", "optim.py"]:
        if not os.path.exists(f"{ZDIR}/{f}"): shutil.copy(f"{SRC_Z}/{f}", f"{ZDIR}/{f}")
    o_ = open(f"{ZDIR}/optim.py").read()             # ScaledAdam / Eden, unchanged; lhotse is used only by its self-tests (fix_random_seed)
    if "from lhotse.utils import fix_random_seed" in o_:
        o_ = o_.replace("from lhotse.utils import fix_random_seed",
                        "def fix_random_seed(seed):  # MT2KD: replaces lhotse.utils.fix_random_seed (used only by optim.py's self-tests)\n"
                        "    import random as _r, numpy as _np; _r.seed(seed); _np.random.seed(seed); torch.manual_seed(seed)", 1)
        open(f"{ZDIR}/optim.py", "w").write(o_)
    if re.search(r"^\s*(from|import) lhotse", o_, re.M): raise RuntimeError("icefall optim.py imports lhotse in an unexpected place")
    open(ZDIR + "/icefall/__init__.py", "a").close()
    open(ZDIR + "/icefall/utils.py", "w").write(
        'import torch\n\ndef torch_autocast(device_type="cuda", **kwargs):\n    try:\n        return torch.amp.autocast(device_type, **kwargs)\n'
        '    except (AttributeError, TypeError):\n        return torch.cuda.amp.autocast(**kwargs)\n')
    if not os.path.exists(f"{ZDIR}/k2_swoosh.py") or open(f"{ZDIR}/k2_swoosh.py").read() != K2_SWOOSH_SRC: open(f"{ZDIR}/k2_swoosh.py", "w").write(K2_SWOOSH_SRC)
    t_ = open(f"{ZDIR}/scaling.py").read()
    if "\nimport k2\n" in t_: t_ = t_.replace("\nimport k2\n", "\nimport k2_swoosh as k2\n", 1); open(f"{ZDIR}/scaling.py", "w").write(t_)
    used = set(re.findall(r"\bk2\.(\w+)", t_ + open(f"{ZDIR}/zipformer.py").read() + open(f"{ZDIR}/subsampling.py").read()))
    if used - STUB_FUNCS: raise RuntimeError(f"the Zipformer source uses k2 functions the port does not provide: {used - STUB_FUNCS}")
    sys.path = [p for p in sys.path if p != ZDIR]; sys.path.insert(0, ZDIR); importlib.invalidate_caches()
    for m_ in [k for k in list(sys.modules) if k in ("zipformer", "scaling", "subsampling", "encoder_interface", "k2_swoosh", "optim") or k == "icefall" or k.startswith("icefall.")]:
        if not (getattr(sys.modules[m_], "__file__", "") or "").startswith(ZDIR): del sys.modules[m_]          # e.g. imported earlier from another folder
    try: commit = subprocess.run(["git", "-C", "/content/icefall", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: commit = "unknown"
    return dict(dir=ZDIR, icefall_commit=commit, k2_functions_used=sorted(used),
                files={f: file_sha1(f"{ZDIR}/{f}")[:12] for f in ["zipformer.py", "scaling.py", "subsampling.py", "k2_swoosh.py", "optim.py"]})

def verify_swoosh(device="cuda"):
    """Swoosh port vs reference: forward exactness (fp32/fp16 inputs, extremes), derivative, 8-bit stochastic backward (bounded, unbiased),
    the icefall modules that call it, and the real k2 if it happens to be installed."""
    import torch, torch.nn.functional as F
    setup_zipformer(); import k2_swoosh as P, scaling as S
    dev = torch.device(device if torch.cuda.is_available() else "cpu"); R = dict(source="k2/python/csrc/torch/v2/autograd/swoosh.cu (ported)")
    xs = torch.cat([torch.linspace(-60, 60, 200001), torch.tensor([-65504., -1e4, -100., 100., 1e4, 65504.])]).to(dev)
    for w, (sh, co, of) in {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}.items():
        x64 = xs.double(); ref = torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x64 - sh) - co * x64 - of; dref = torch.sigmoid(x64 - sh) - co
        tol = 2e-6 * (1 + ref.abs())
        y = getattr(P, f"swoosh_{w}_forward")(xs); yd, d = getattr(P, f"swoosh_{w}_forward_and_deriv")(xs)
        e_fwd = float(((y.double() - ref).abs() / tol).max()); e_fd = float(((yd.double() - ref).abs() / tol).max()); e_d = float((d.double() - dref).abs().max())
        y16 = getattr(P, f"swoosh_{w}_forward")(xs.half()); y16_ok = bool(torch.isfinite(y16).all()) and y16.dtype == torch.float32
        xg = xs.clone().requires_grad_(True); yg = getattr(P, f"swoosh_{w}")(xg); yg.backward(torch.ones_like(yg)); g = xg.grad.double()
        q = 1.005 / 255.0; e_q = float((g - dref).abs().max()); bias = float((g - dref).mean())
        xr = torch.randn(2_000_000, device=dev) * 6; xr.requires_grad_(True); yr = getattr(P, f"swoosh_{w}")(xr); yr.backward(torch.ones_like(yr))
        bias_r = float((xr.grad.double() - (torch.sigmoid(xr.detach().double() - sh) - co)).mean())
        e_nog = float(((getattr(P, f"swoosh_{w}")(xs).double() - ref).abs() / tol).max())
        mod = (S.SwooshL if w == "l" else S.SwooshR)(); e_mod = float(((mod(xs).double() - ref).abs() / tol).max())
        own = (S.SwooshLForward if w == "l" else S.SwooshRForward)(xs); e_own = float(((own.double() - ref).abs() / tol)[torch.isfinite(own)].max())
        R[w] = dict(forward_err_rel=e_fwd, forward_and_deriv_err_rel=e_fd, deriv_maxerr=e_d, fp16_input_finite_fp32_output=y16_ok, no_grad_path_err_rel=e_nog,
                    backward_maxerr=e_q, backward_quant_step=q, backward_bias_grid=bias, backward_bias_random=bias_r, icefall_module_err_rel=e_mod,
                    icefall_reference_SwooshForward_err_rel=e_own)
        ok = e_fwd <= 1 and e_fd <= 1 and e_d < 1e-6 and y16_ok and e_nog <= 1 and e_q <= q + 1e-6 and abs(bias_r) < 2e-4 and e_mod <= 1 and e_own <= 1
        R[w]["passed"] = ok
        log(f"  swoosh_{w}: forward rel.err {e_fwd:.2f} (<=1 = within 2e-6 relative) | deriv {e_d:.1e} | fp16 inputs finite {y16_ok} | backward |err| {e_q:.4f} "
            f"<= 8-bit step {q:.4f}, mean bias {bias_r:+.1e} | icefall SwooshL/R module {e_mod:.2f}, icefall reference {e_own:.2f} -> {'OK' if ok else 'FAIL'}")
    # the fused ActivationDropoutAndLinear path (uses forward_and_deriv in backward) vs autograd through the reference, dropout 0
    torch.manual_seed(0); lin = S.ActivationDropoutAndLinear(64, 32, bias=True, activation="SwooshL", dropout_p=0.0).to(dev)
    x = (torch.randn(50, 64, device=dev) * 4).requires_grad_(True); y = lin(x); y.square().sum().backward(); gx = x.grad.clone(); gw = lin.weight.grad.clone()
    x2 = x.detach().double().requires_grad_(True); W = lin.weight.detach().double().requires_grad_(True); b = lin.bias.detach().double()
    y2 = F.linear(torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x2 - 4) - 0.08 * x2 - 0.035, W, b); y2.square().sum().backward()
    R["fused_linear"] = dict(out_rel=float((y.double() - y2).abs().max() / y2.abs().max()), dx_rel=float((gx.double() - x2.grad).abs().max() / x2.grad.abs().max()),
                             dw_rel=float((gw.double() - W.grad).abs().max() / W.grad.abs().max()))
    R["fused_linear"]["passed"] = max(R["fused_linear"].values()) < 1e-4
    log(f"  ActivationDropoutAndLinear(SwooshL) vs float64 autograd: output {R['fused_linear']['out_rel']:.1e}, dx {R['fused_linear']['dx_rel']:.1e}, "
        f"dW {R['fused_linear']['dw_rel']:.1e} (relative)")
    try:
        import k2 as real
        if not hasattr(real, "swoosh_l_forward"): raise ImportError
        xr = torch.randn(100000, device=dev) * 8
        R["real_k2"] = dict(forward_l_maxdiff=float((real.swoosh_l_forward(xr) - P.swoosh_l_forward(xr)).abs().max()),
                            forward_r_maxdiff=float((real.swoosh_r_forward(xr) - P.swoosh_r_forward(xr)).abs().max()),
                            deriv_l_maxdiff=float((real.swoosh_l_forward_and_deriv(xr)[1] - P.swoosh_l_forward_and_deriv(xr)[1]).abs().max()))
        log(f"  compared with the installed k2: {R['real_k2']}")
    except Exception: R["real_k2"] = "k2 not installed for this PyTorch (not downgraded); compared against k2's published formulas and algorithm instead"
    R["passed"] = R["l"]["passed"] and R["r"]["passed"] and R["fused_linear"]["passed"]
    R["claim"] = ("forward and derivative: same formulas as k2 in fp32 (verified to <= 2e-6 relative against float64); autograd backward: same 8-bit "
                  "stochastic quantisation of the derivative as k2 (bounded by one step, unbiased); random numbers differ, so results are not bit-identical to a k2 run")
    return R

def lengths_to_mask(lens, T):
    import torch
    return torch.arange(T, device=lens.device)[None, :] < lens[:, None]

def set_batch_count(model, bc):
    for name, mod in model.named_modules():
        if hasattr(mod, "batch_count"): mod.batch_count = bc
        if hasattr(mod, "name"): mod.name = name

def fbank(wav):
    """Student frontend = your v1 ASR fbank (80 bins, 25 ms / 10 ms, povey window, snip_edges=False)."""
    import torch, torchaudio.compliance.kaldi as kaldi
    return kaldi.fbank(torch.from_numpy(wav)[None], num_mel_bins=80, frame_length=25.0, frame_shift=10.0, dither=0.0, energy_floor=1e-10,
                       sample_frequency=float(SR), snip_edges=False, low_freq=20.0, high_freq=-400.0, remove_dc_offset=True,
                       preemphasis_coefficient=0.97, window_type="povey", use_energy=False)

def make_batches(lens, epoch, seed, shuffle, max_sec, max_utts, idx=None):
    """Duration batches over `idx` (default: all): padded seconds <= max_sec and <= max_utts items; sorted inside shuffled pools of 1000;
    deterministic for (seed, epoch, idx)."""
    lens = np.asarray(lens); idx = np.arange(len(lens)) if idx is None else np.asarray(sorted(idx))
    rng = np.random.default_rng(seed + 1000 * epoch + (0 if len(idx) == len(lens) else 7919 * len(idx)))
    order = idx[rng.permutation(len(idx))] if shuffle else idx; out = []
    for s in range(0, len(order), 1000):
        ids = order[s:s + 1000]; ids = ids[np.argsort(lens[ids], kind="stable")]; cur, mx = [], 0
        for i in ids:
            nm = max(mx, lens[i])
            if cur and (nm * (len(cur) + 1) / SR > max_sec or len(cur) >= max_utts): out.append(cur); cur, nm = [], lens[i]
            cur.append(int(i)); mx = nm
        if cur: out.append(cur)
    if shuffle: out = [out[j] for j in rng.permutation(len(out))]
    return out

def amp_dtype():
    import torch
    return torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16

DRIVE_FILES = ("best.pt", "latest.pt", "history.csv", "final_report.json", "config.json")

class SimulatedDisconnect(Exception):
    """Raised by the resume test to imitate a runtime that dies: nothing is saved at that moment."""

class DriveStore:
    """Writes ONLY into one folder (drive_dir(<run>)) and ONLY the five files in DRIVE_FILES (plus '<name>.tmp' while a write is in progress).
    * small files (config.json, history.csv, final_report.json): written synchronously, atomically (.tmp then rename)
    * checkpoints (latest.pt, best.pt): copied in a background thread from the local checkpoint (training does not wait),
      written to '<name>.tmp' first and renamed only when the copy is complete -> an interrupted upload never replaces a good file
    * with the Drive API (CFG['drive_api']): each upload is confirmed by comparing Drive's md5 with the local md5, and the copies that
      Drive moves to the trash when a checkpoint is replaced are deleted permanently (only files of THIS folder with THESE names),
      because trashed files keep using Drive quota for 30 days.
    fake=True: a local stand-in folder (used by the resume test; nothing goes to Drive)."""
    def __init__(self, folder, use_api=True, fake=False):
        self.dir, self.fake = Path(folder), fake
        if fake: assert_local(folder)
        else:
            if not os.path.isdir(DRIVE + "/MyDrive"): raise RuntimeError("Google Drive is not mounted")
            # the path as written must be inside MyDrive; its resolved location may be elsewhere on the Drive mount, because a folder
            # such as MT2KD can be a Drive SHORTCUT (resolved to /content/drive/.shortcut-targets-by-id/...)
            if not os.path.normpath(os.path.abspath(str(folder))).startswith(DRIVE + "/MyDrive/"): raise PermissionError(f"Drive output folder must be inside MyDrive: {folder}")
            if not os.path.realpath(str(folder)).startswith(os.path.realpath(DRIVE) + "/"): raise PermissionError(f"{folder} resolves outside Google Drive")
        self.dir.mkdir(parents=True, exist_ok=True)
        self.api = self.fid = None; self.api_note = "not used (stand-in folder)" if fake else "disabled (CFG['drive_api'] = False)"
        if use_api and not fake: self._init_api()
        self.cv = threading.Condition(); self.pending = {}; self.busy = None; self.errors = []; self.done = {}; self._ids = itertools.count()
        threading.Thread(target=self._worker, daemon=True).start()

    # ---------------------------------------------------------------- guards / API
    def _p(self, name):
        base = name[:-4] if name.endswith(".tmp") else name
        if base not in DRIVE_FILES: raise PermissionError(f"'{name}' is not one of the files allowed on Drive: {DRIVE_FILES}")
        return self.dir / name
    def _init_api(self):
        try:
            from google.colab import auth; auth.authenticate_user()
            from googleapiclient.discovery import build
            self.api = build("drive", "v3", cache_discovery=False)
            self.api_note = "ON: uploads confirmed by md5; trashed copies of this folder's checkpoints are deleted permanently"
        except Exception as e:
            self.api = None; self.api_note = (f"UNAVAILABLE ({type(e).__name__}: {e}): uploads are not confirmed and replaced checkpoints stay in the Drive "
                                              f"trash (empty it to recover the space)")
    def _folder_id(self):
        if self.fid or self.api is None: return self.fid
        rel = os.path.relpath(os.path.normpath(os.path.abspath(str(self.dir))), DRIVE + "/MyDrive").split(os.sep); parent = "root"
        try:
            for part in rel:                                  # walk the path as written; follow Drive shortcuts to their target folder
                q = (f"name = '{part}' and '{parent}' in parents and trashed = false and "
                     f"(mimeType = 'application/vnd.google-apps.folder' or mimeType = 'application/vnd.google-apps.shortcut')")
                r = self.api.files().list(q=q, fields="files(id,mimeType,shortcutDetails)", pageSize=10, supportsAllDrives=True,
                                          includeItemsFromAllDrives=True).execute().get("files", [])
                if not r: return None                         # folder not synced to Drive yet: retried later
                x = r[0]; parent = x["shortcutDetails"]["targetId"] if x["mimeType"].endswith("shortcut") else x["id"]
            self.fid = parent
        except Exception: return None
        return self.fid

    # ---------------------------------------------------------------- storage information
    def quota(self):
        if self.api:
            try:
                q = self.api.about().get(fields="storageQuota").execute()["storageQuota"]; lim = int(q.get("limit", 0)); use = int(q.get("usage", 0))
                return dict(limit=lim or None, usage=use, trash=int(q.get("usageInDriveTrash", 0)), free=(lim - use) if lim else None, source="Drive API")
            except Exception: pass
        du = shutil.disk_usage(self.dir if self.fake else DRIVE + "/MyDrive")
        return dict(limit=du.total, usage=du.used, trash=None, free=du.free, source="mounted file system (approximate)")
    def files(self): return {f.name: f.stat().st_size for f in sorted(self.dir.iterdir()) if f.is_file()}
    def status(self, expected_bytes=None):
        q = self.quota(); fl = self.files(); g = lambda b: "?" if b is None else f"{b / 2**30:.2f} GiB"
        log(f"  Drive output folder : {self.dir}")
        log(f"  folder size now     : {g(sum(fl.values()))} | files: " + (", ".join(f"{k} ({v / 2**20:.1f} MiB)" for k, v in fl.items()) or "none"))
        log(f"  Drive storage       : free {g(q['free'])} of {g(q['limit'])} (used {g(q['usage'])}" + (f", in trash {g(q['trash'])}" if q["trash"] is not None else "")
            + f") [{q['source']}]")
        log(f"  Drive API           : {self.api_note}")
        if expected_bytes: log(f"  this run will keep  : about {g(expected_bytes)} in this folder (best.pt + latest.pt + small files)")
        unexpected = [k for k in fl if (k[:-4] if k.endswith(".tmp") else k) not in DRIVE_FILES]
        if unexpected: log(f"  NOTE: files not written by this code are in the folder (left untouched): {unexpected}")
        return dict(quota=q, files=fl)
    def check_space(self, need_bytes, what):
        """Refuses to start writing large files when the free Drive space is below what the next writes need."""
        q = self.quota()
        if q["free"] is not None and q["free"] < need_bytes:
            raise RuntimeError(f"Google Drive has {q['free'] / 2**30:.2f} GiB free; {what} needs about {need_bytes / 2**30:.2f} GiB. Nothing large was written. "
                               f"Free at least {(need_bytes - q['free']) / 2**30:.2f} GiB more on Drive (deleted files only free space once the Drive trash is "
                               f"emptied) and re-run the cell.")
        free = "?" if q["free"] is None else f"{q['free'] / 2**30:.2f}"
        log(f"  Drive space check OK for {what}: need ~{need_bytes / 2**30:.2f} GiB, free {free} GiB")

    # ---------------------------------------------------------------- writes
    def write_text(self, name, text):
        tmp, dst = self._p(name + ".tmp"), self._p(name)
        with open(tmp, "w") as fh: fh.write(text); fh.flush(); os.fsync(fh.fileno())
        os.replace(tmp, dst)
    def write_json(self, name, obj): self.write_text(name, json.dumps(obj, indent=1, default=str))
    def push(self, local_file, name):
        """Queues an upload of a local checkpoint file. The file is snapshotted NOW (hard link), so later local saves cannot change what is
        uploaded; a newer request for the same name replaces a queued (not yet started) one."""
        self._p(name); snap = f"{local_file}.upload{next(self._ids)}"
        try: os.link(local_file, snap)
        except OSError: shutil.copyfile(local_file, snap)
        with self.cv:
            if self.errors: os.remove(snap); raise RuntimeError(f"a Google Drive upload failed: {self.errors[-1]} (training stopped; local state is saved)")
            old = self.pending.get(name); self.pending[name] = snap; self.cv.notify_all()
        if old and os.path.exists(old): os.remove(old)
    def _worker(self):
        while True:
            with self.cv:
                while not self.pending: self.cv.wait()
                name = next(iter(self.pending)); src = self.pending.pop(name); self.busy = name
            try: self._upload(src, name)
            except Exception as e: self.errors.append(f"{name}: {type(e).__name__}: {e}"); log(f"  GOOGLE DRIVE UPLOAD FAILED ({name}): {type(e).__name__}: {e}")
            finally:
                try: os.remove(src)
                except OSError: pass
                with self.cv: self.busy = None; self.cv.notify_all()
    def _upload(self, src, name):
        t0 = time.time(); tmp, dst = self._p(name + ".tmp"), self._p(name); h = hashlib.md5(); n = 0
        with open(src, "rb") as fi, open(tmp, "wb") as fo:          # the open handle keeps a consistent snapshot even if the local file is replaced meanwhile
            for b in iter(lambda: fi.read(1 << 24), b""): fo.write(b); h.update(b); n += len(b)
            fo.flush(); os.fsync(fo.fileno())
        os.replace(tmp, dst); md5 = h.hexdigest()
        self.done[name] = dict(bytes=n, md5=md5, written=time.strftime("%Y-%m-%d %H:%M:%S"), confirmed_by_drive_api=None)
        log(f"  Drive: {name} written ({n / 2**20:.0f} MiB in {time.time() - t0:.0f} s)" + ("; waiting for Drive to confirm the upload" if self.api else ""))
        if self.api:
            self.done[name]["confirmed_by_drive_api"] = self._confirm(name, md5); self.purge_trash()
    def _confirm(self, name, md5, timeout=1800):
        t0 = time.time()
        while time.time() - t0 < timeout:
            with self.cv:
                if name in self.pending: return "superseded by a newer upload"
            fid = self._folder_id()
            if fid:
                try:
                    r = self.api.files().list(q=f"name = '{name}' and '{fid}' in parents and trashed = false", fields="files(id,md5Checksum)").execute().get("files", [])
                    if any(x.get("md5Checksum") == md5 for x in r): log(f"  Drive: {name} upload CONFIRMED (md5 {md5[:10]} matches)"); return True
                except Exception: pass
            time.sleep(15)
        log(f"  Drive: {name} upload not confirmed within {timeout // 60} min (it may still be syncing)"); return False
    def purge_trash(self):
        """Deletes permanently the TRASHED copies of this folder's own files (replaced checkpoints); nothing else is touched."""
        fid = self._folder_id()
        if not fid: return 0
        freed = 0
        try:
            for x in self.api.files().list(q=f"'{fid}' in parents and trashed = true", fields="files(id,name,size)", pageSize=200).execute().get("files", []):
                base = x["name"][:-4] if x["name"].endswith(".tmp") else x["name"]
                if base in DRIVE_FILES: self.api.files().delete(fileId=x["id"]).execute(); freed += int(x.get("size", 0))
        except Exception as e: log(f"  (could not clean this folder's trashed copies: {type(e).__name__}: {e})")
        if freed: log(f"  Drive: {freed / 2**20:.0f} MiB of replaced copies removed from the trash")
        return freed
    def wait(self, timeout=None):
        """Blocks until every queued upload is written (and confirmed, with the API)."""
        t0 = time.time()
        with self.cv:
            while self.pending or self.busy:
                if timeout and time.time() - t0 > timeout: return False
                self.cv.wait(5)
        if self.api: self.purge_trash()
        if self.errors: raise RuntimeError(f"Google Drive upload failed: {self.errors[-1]}")
        return True

    # ---------------------------------------------------------------- reading back
    def candidates(self, name): return [f for f in (self._p(name), self._p(name + ".tmp")) if f.exists()]
    def promote_tmp(self, name):
        """A complete '<name>.tmp' whose rename was interrupted becomes '<name>'."""
        os.replace(self._p(name + ".tmp"), self._p(name)); return self._p(name)
    def clean_stale_tmp(self):
        for f in self.dir.glob("*.tmp"):
            if f.name[:-4] in DRIVE_FILES: f.unlink(); log(f"  removed an incomplete upload left by an interrupted run: {f.name}")
    def best_info(self): return best_meta(self._p("best.pt")) if self._p("best.pt").exists() else None

def drive_need_bytes(n_par, drive):
    """Drive space the run needs beyond what its own files already use: best.pt (fp32 weights) + latest.pt (weights + 2 AdamW moments),
    each written as a temporary copy before the old file is replaced (peak = 2 x each)."""
    best_b, latest_b = 4.05 * n_par, 20.0 * n_par          # latest.pt: weights + optimizer state (ScaledAdam keeps more per weight than AdamW)
    have = sum(v for k, v in drive.files().items() if k in ("best.pt", "latest.pt"))
    return max(0.0, 2 * (best_b + latest_b) - have) + 50e6, best_b + latest_b

def _position(S): return (S["update"], S["epoch"], len(S["seen"]))

# ####################################################################################################
# ---- environment
# ####################################################################################################
def env_report():
    import torch, platform
    r = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda, gpu=None, code_version=CODE_VERSION)
    try:
        import torchaudio; r["torchaudio"] = torchaudio.__version__
    except Exception as e: r["torchaudio"] = f"missing ({e})"
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0); r.update(gpu=p.name, gpu_gib=round(p.total_memory / 2**30, 1), amp="bfloat16" if p.major >= 8 else "float16 + GradScaler")
    r["disk_free_local_gib"] = round(disk_free_gib("/content" if Path("/content").exists() else "/"), 1)
    for k, v in r.items(): log(f"  {k:22s}: {v}")
    if r["gpu"] is None: raise RuntimeError("no GPU: Runtime > Change runtime type > A100 (or T4) GPU")
    return r

# ####################################################################################################
# ---- FSD50K: official splits, labels, vocabulary, 16 kHz 10 s centre-crop audio cache (same as AT2/AT3)
# ####################################################################################################
import numpy as np, pandas as pd

def load_fsd():
    """train / val (from FSD50K.dev) and eval: file ids, labels [N, 200], and the FSD50K vocabulary. Checked against the official sizes."""
    root = Path(CFG["fsd_root"]); proc = root / "processed"; gt = root / "FSD50K.ground_truth"
    for f in [proc / "train_metadata.csv", proc / "val_metadata.csv", proc / "y_train.npy", proc / "y_val.npy", gt / "vocabulary.csv"]:
        if not f.exists(): raise FileNotFoundError(f"{f} not found (expected the FSD50K folder used by AT2/AT3)")
    voc = pd.read_csv(gt / "vocabulary.csv", header=None, names=["idx", "label", "mid"])
    if list(voc.idx) != list(range(200)): raise RuntimeError("vocabulary.csv must list classes 0..199 in order")
    F = dict(vocab=voc, splits={})
    for name, sub in [("train", "FSD50K.dev_audio"), ("val", "FSD50K.dev_audio"), ("eval", "FSD50K.eval_audio")]:
        mf = proc / f"{name}_metadata.csv"
        if not mf.exists(): continue
        df = pd.read_csv(mf, dtype={"fname": str}); y = np.load(proc / f"y_{name}.npy") > 0.5
        if y.shape != (len(df), 200): raise RuntimeError(f"y_{name}.npy has shape {y.shape}, expected ({len(df)}, 200)")
        if len(df) != OFFICIAL[name]: raise RuntimeError(f"{name}: {len(df):,} clips, the official split has {OFFICIAL[name]:,}")
        F["splits"][name] = dict(fnames=df.fname.tolist(), y=y, wav=[str(root / sub / f"{f}.wav") for f in df.fname])
    a, b = set(F["splits"]["train"]["fnames"]), set(F["splits"]["val"]["fnames"])
    if a & b: raise RuntimeError(f"train and val share {len(a & b)} clips")
    return F

def subset_idx(n_total, n, offset):
    """Same rule as AT2/AT3: numpy default_rng(seed + offset).permutation, first n (None = all). offset 0 train, 100 val, 200 eval."""
    perm = np.random.default_rng(CFG["seed"] + offset).permutation(n_total); return [int(i) for i in (perm if n is None else perm[:n])]

def audio_dir(split): return Path(CFG["local_audio"]) / f"{split}_16k_c{int(CFG['max_seconds'])}"
def audio_tar(split): return Path(CFG["drive_audio_cache"]) / f"{split}_16k_c{int(CFG['max_seconds'])}.tar"

def wav_to_npy(src, dst):
    """Identical to AT2/AT3: mono mean -> torchaudio resample to 16 kHz -> centre crop to 10 s -> int16."""
    import soundfile as sf, torch, torchaudio
    x, sr0 = sf.read(src, dtype="float32", always_2d=True); x = x.mean(axis=1); w = torch.from_numpy(np.ascontiguousarray(x))
    if sr0 != SR: w = torchaudio.functional.resample(w, sr0, SR)
    MAXN = int(CFG["max_seconds"] * SR)
    if w.numel() > MAXN: s = (w.numel() - MAXN) // 2; w = w[s:s + MAXN]
    if w.numel() < 1600 or not bool(torch.isfinite(w).all()): raise ValueError(f"bad audio {src}")
    a = (w.clamp(-1.0, 1.0) * 32767.0).round().to(torch.int16).numpy(); tmp = str(dst) + ".part"
    with open(tmp, "wb") as f: np.save(f, a)
    os.replace(tmp, dst)

def ensure_audio(F, split, idx):
    """Local 10 s crops for exactly these clips: restored from the Drive tar when possible, the rest built from the WAVs (resumable)."""
    S = F["splits"][split]; d = audio_dir(split); assert_local(d); d.mkdir(parents=True, exist_ok=True); fn = S["fnames"]
    for p in d.glob("*.part"): p.unlink()
    def missing():
        have = set(os.listdir(d)); return [i for i in sorted(set(idx)) if f"{fn[i]}.npy" not in have]
    need = missing()
    if need and audio_tar(split).exists():
        t0 = time.time(); log(f"  [{split}] restoring the audio cache from {audio_tar(split)} ...")
        subprocess.run(["tar", "-xf", str(audio_tar(split)), "-C", str(d)], capture_output=True); need = missing()
        log(f"  [{split}] restored in {time.time() - t0:.0f} s | still missing {len(need):,}/{len(set(idx)):,}")
    built, errs = 0, {}
    if need:
        require_space(d, len(need) * 0.33 / 1024, f"{split} audio cache")
        t0 = time.time(); log(f"  [{split}] building {len(need):,} clips from the WAVs on Drive ({CFG['cache_threads']} threads; slow the first time, resumable) ...")
        def work(i):
            for _ in range(3):
                try: wav_to_npy(S["wav"][i], d / f"{fn[i]}.npy"); return True
                except Exception as e: errs[i] = f"{type(e).__name__}: {e}"; time.sleep(1.0)
            return False
        with ThreadPoolExecutor(CFG["cache_threads"]) as ex:
            for b0 in range(0, len(need), 2000):
                built += sum(ex.map(work, need[b0:b0 + 2000])); log(f"  [{split}] {min(b0 + 2000, len(need)):,}/{len(need):,} | {time.time() - t0:.0f} s")
    bad = missing()
    if bad: raise RuntimeError(f"[{split}] {len(bad)} clips could not be read, e.g. {[(fn[i], errs.get(i)) for i in bad[:3]]}; re-run (it resumes)")
    if built and CFG["save_audio_cache_to_drive"]:
        tar = audio_tar(split); tar.parent.mkdir(parents=True, exist_ok=True); tmp = str(tar) + ".part"
        r = subprocess.run(["tar", "-cf", tmp, "-C", str(d), "."], capture_output=True, text=True)
        if r.returncode == 0: os.replace(tmp, tar); log(f"  [{split}] audio cache tar updated on Drive: {tar.name} ({len(os.listdir(d)):,} clips)")
        else: log(f"  [{split}] (could not write the cache tar: {r.stderr[-200:]})")
    log(f"  [{split}] local audio ready: {len(set(idx)):,} clips")

def clip_lengths(F, split, idx):
    """Sample counts of the cached clips (read from the .npy headers; remembered in a small local file)."""
    P = paths(); P["meta"].mkdir(parents=True, exist_ok=True); f = P["meta"] / f"lens_{split}.json"
    known = json.load(open(f)) if f.exists() else {}; fn = F["splits"][split]["fnames"]; d = audio_dir(split); new = 0
    for i in idx:
        if fn[i] not in known: known[fn[i]] = int(np.load(d / f"{fn[i]}.npy", mmap_mode="r").shape[0]); new += 1
    if new: atomic_json(known, f)
    return np.array([known[fn[i]] for i in idx], dtype=np.int64)

def load_clip(split, fname):
    a = np.load(audio_dir(split) / f"{fname}.npy"); return a.astype(np.float32) / 32768.0

# ####################################################################################################
# ---- BEATs teacher (official microsoft/unilm code, fine-tuned AS2M cpt2 checkpoint), frozen
# ####################################################################################################
def setup_beats_code():
    d = Path(CFG["unilm_dir"]); assert_local(d); bd = d / "beats"
    if not (bd / "BEATs.py").exists():
        shutil.rmtree(d, ignore_errors=True)
        cmd = (f"git clone -q --depth 1 --filter=blob:none --sparse {CFG['unilm_repo']} {d} && git -C {d} sparse-checkout set beats")
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode or not (bd / "BEATs.py").exists(): raise RuntimeError(f"could not fetch the BEATs code from {CFG['unilm_repo']}: {r.stderr[-300:]}")
    if str(bd) not in sys.path: sys.path.append(str(bd))                      # appended: the Zipformer folder keeps priority
    importlib.invalidate_caches(); import BEATs as beats_mod
    try: commit = subprocess.run(["git", "-C", str(d), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: commit = "unknown"
    return beats_mod, commit

def _ckpt_kind(path):
    """'finetuned' (has the 527-class predictor and label_dict), 'pretrained' (no predictor), or an error string."""
    import torch
    try: ck = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e: return f"unreadable ({type(e).__name__}: {str(e)[:120]})", None
    if not isinstance(ck, dict) or "model" not in ck or "cfg" not in ck: return "not a BEATs checkpoint (no 'model'/'cfg')", None
    pw = ck["model"].get("predictor.weight")
    if pw is None or not ck["cfg"].get("finetuned_model", False): return "pretrained", None
    if tuple(pw.shape) != (527, 768) or len(ck.get("label_dict", {})) != 527: return f"unexpected head {tuple(pw.shape)} / {len(ck.get('label_dict', {}))} labels", None
    return "finetuned", ck

def find_beats_checkpoint():
    """Searches the Drive folders in CFG['beats_search'] (one level deep) for BEATs*.pt / *.pt.zip / *.zip and returns the fine-tuned one."""
    cands = []
    for root in CFG["beats_search"]:
        r = Path(root)
        if not r.is_dir(): continue
        top = list(r.glob("*"))                                                # one level deeper only inside folders named like checkpoints/BEATs
        deeper = [q for d_ in top if d_.is_dir() and any(w in d_.name.lower() for w in ("checkpoint", "beats", "ckpt")) for q in d_.glob("*")]
        for p in top + deeper:
            n = p.name.lower()
            if p.is_file() and "beats" in n and (n.endswith(".pt") or n.endswith(".zip")) and "tokenizer" not in n: cands.append(p)
    cands = sorted(set(cands), key=lambda p: (0 if "finetuned" in p.name.lower() else 1, 0 if "cpt2" in p.name.lower() else 1, p.name))
    if not cands: raise FileNotFoundError(f"no BEATs checkpoint file found in {CFG['beats_search']}. Put BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt "
                                          f"(the 'Fine-tuned BEATs_iter3+ (AS2M) (cpt2)' link of github.com/microsoft/unilm/tree/master/beats) in {CFG['fsd_root']}")
    seen = []
    for p in cands:
        loc = Path(CFG["local_beats"]) / (p.name[:-4] if p.name.lower().endswith(".pt.zip") else p.name if p.name.endswith(".pt") else p.name + ".pt")
        loc.parent.mkdir(parents=True, exist_ok=True)
        if not loc.exists() or loc.stat().st_size != p.stat().st_size: shutil.copyfile(p, str(loc) + ".part"); os.replace(str(loc) + ".part", loc)
        kind, ck = _ckpt_kind(loc); seen.append(f"{p.name}: {kind}")
        if kind == "finetuned": return p, loc, ck
    raise RuntimeError("no FINE-TUNED BEATs checkpoint found. Files checked:\n  " + "\n  ".join(seen) +
                       "\n'pretrained' = BEATs_iter3_plus_AS2M.pt (no 527-class head). Download 'Fine-tuned BEATs_iter3+ (AS2M) (cpt2)' "
                       "(last column, last row of the table at github.com/microsoft/unilm/tree/master/beats) and upload it as ONE .pt file.")

class BEATsTeacher:
    """Frozen fine-tuned BEATs. logits(waves) = mean over patches of BEATs' own 527-class predictor (= the logit of BEATs' output probability)."""
    def __init__(self, device="cuda"):
        import torch
        self.mod, self.code_commit = setup_beats_code(); self.src, self.local, ck = find_beats_checkpoint()
        cfg = self.mod.BEATsConfig(ck["cfg"]); self.model = self.mod.BEATs(cfg); self.model.load_state_dict(ck["model"], strict=True)
        self.model = self.model.to(device).eval(); self.device = device
        for p in self.model.parameters(): p.requires_grad_(False)
        self.labels = [ck["label_dict"][i] for i in range(527)]; self.sha1 = file_sha1(self.local)
        self.info = dict(source=str(self.src), sha1=self.sha1, unilm_commit=self.code_commit, labels=527, params_M=round(sum(p.numel() for p in self.model.parameters()) / 1e6, 1),
                         precision="float32", clip_logits="mean over patches of the 527-class predictor (BEATs' own head, eval mode)",
                         preprocessing="16 kHz waveform x 2^15 -> kaldi fbank 128 mel (25/10 ms) -> (x - 15.41663) / (2 x 6.55582)  [BEATs.preprocess]")
        del ck
        log(f"BEATs teacher loaded: {self.src.name} | {self.info['params_M']}M params | 527 classes | fp32, frozen, eval | unilm {self.code_commit[:10]} | sha1 {self.sha1[:10]}")
    def logits(self, waves):
        """waves: list of float32 arrays of EQUAL length (no padding, so a batch equals single-clip extraction) -> [B, 527] numpy."""
        import torch
        assert len({len(w) for w in waves}) == 1, "teacher batches must hold clips of equal length"
        x = torch.from_numpy(np.stack(waves)).to(self.device)
        with torch.inference_mode():
            pred = self.model.predictor; self.model.predictor = None
            try: feats, _ = self.model.extract_features(x, padding_mask=None)
            finally: self.model.predictor = pred
            out = pred(feats).mean(dim=1)
        if not torch.isfinite(out).all(): raise FloatingPointError("BEATs produced non-finite logits")
        return out.float().cpu().numpy()
    def logits_many(self, split, fnames):
        """Clips grouped by identical length (most FSD50K crops are exactly 10 s) and processed in batches."""
        lens = {}; out = {}
        for f in fnames: lens.setdefault(int(np.load(audio_dir(split) / f"{f}.npy", mmap_mode="r").shape[0]), []).append(f)
        for n, group in sorted(lens.items()):
            for s in range(0, len(group), CFG["teacher_batch"]):
                g = group[s:s + CFG["teacher_batch"]]; lg = self.logits([load_clip(split, f) for f in g])
                for f, row in zip(g, lg): out[f] = row
        return out

def teacher_store(split): return paths()["teacher"] / f"beats_{split}.npz"
def build_teacher_logits(T, F, split, idx):
    """Resumable local store of BEATs clip logits for the selected clips (float32 [N, 527]); saved every 2,000 clips."""
    f = teacher_store(split); f.parent.mkdir(parents=True, exist_ok=True); fn = F["splits"][split]["fnames"]
    have = {}
    if f.exists():
        z = np.load(f); meta = json.loads(str(z["meta"]))
        if meta.get("teacher_sha1") == T.sha1: have = dict(zip(z["uids"].tolist(), z["logits"]))
        else: log(f"  [{split}] existing teacher store was made with another checkpoint: recomputed")
    todo = [fn[i] for i in idx if fn[i] not in have]; t0 = time.time()
    if todo: log(f"  [{split}] BEATs logits: {len(have):,} stored, computing {len(todo):,} ...")
    def save():
        u = sorted(have); tmp = str(f) + ".tmp.npz"
        np.savez(tmp, uids=np.array(u), logits=np.stack([have[k] for k in u]).astype(np.float32), meta=json.dumps(dict(teacher_sha1=T.sha1, split=split)))
        os.replace(tmp, f)
    for s in range(0, len(todo), 2000):
        have.update(T.logits_many(split, todo[s:s + 2000])); save()
        log(f"  [{split}] {min(s + 2000, len(todo)):,}/{len(todo):,} | {(min(s + 2000, len(todo))) / max(time.time() - t0, 1e-9):.0f} clips/s")
    if not f.exists(): save()
    return f

def load_teacher_logits(split, fnames):
    z = np.load(teacher_store(split)); pos = {u: i for i, u in enumerate(z["uids"].tolist())}; miss = [u for u in fnames if u not in pos]
    if miss: raise RuntimeError(f"[{split}] {len(miss)} clips have no BEATs logits (e.g. {miss[:3]}): run the teacher step")
    return z["logits"][[pos[u] for u in fnames]].astype(np.float32), json.loads(str(z["meta"]))

def compare_with_at2(T, F, n_per_split=64):
    """New BEATs logits vs the logits cached by your friend's AT2 run on the same clips (same 10 s crop)."""
    rows = {}
    for split in ["train", "val", "eval"]:
        f = Path(CFG["friend_at_root"]) / f"teacher_{split}.npz"
        if not f.exists() or split not in F["splits"]: continue
        z = np.load(f); uids = z["uids"].tolist()[:n_per_split]; ref = z["logits"][:n_per_split].astype(np.float32)
        if not all((audio_dir(split) / f"{u}.npy").exists() for u in uids):
            ensure_audio(F, split, [F["splits"][split]["fnames"].index(u) for u in uids])
        new = T.logits_many(split, uids); mine = np.stack([new[u] for u in uids])
        sig = lambda x: 1 / (1 + np.exp(-x))
        rows[split] = dict(clips=len(uids), max_prob_diff=float(np.abs(sig(mine) - sig(ref)).max()), mean_prob_diff=float(np.abs(sig(mine) - sig(ref)).mean()),
                           max_logit_diff=float(np.abs(mine - ref).max()), top1_agreement=float((mine.argmax(1) == ref.argmax(1)).mean()))
        log(f"  [{split}] {len(uids)} clips: max |prob diff| {rows[split]['max_prob_diff']:.2e} | max |logit diff| {rows[split]['max_logit_diff']:.3f} | "
            f"top-1 class agreement {rows[split]['top1_agreement'] * 100:.0f}%")
    ok = bool(rows) and all(r["max_prob_diff"] < CFG["teacher_match_tol"] for r in rows.values())
    return dict(splits=rows, passed=ok, available=bool(rows))

# ####################################################################################################
# ---- FSD50K class mapping (for EVALUATION only): FSD50K index -> BEATs output index, exact AudioSet ID match
# ####################################################################################################
def build_mapping(F, labels):
    voc = F["vocab"]; a_of = {m: i for i, m in enumerate(labels)}
    rows = [dict(fsd_index=int(r.idx), fsd_label=r.label, fsd_mid=r.mid, beats_index=a_of.get(r.mid, -1)) for r in voc.itertuples()]
    M = pd.DataFrame(rows); M["mapped"] = M.beats_index >= 0
    ref = Path(CFG["friend_at_root"]) / "mapping" / "class_mapping.csv"; cmp = None
    if ref.exists():
        R = pd.read_csv(ref); Ru = R[R.used_for_kd.astype(bool)]
        mine = set(zip(M[M.mapped].fsd_index, M[M.mapped].beats_index)); theirs = set(zip(Ru.fsd_index.astype(int), Ru.audioset_index.astype(int)))
        cmp = dict(identical=mine == theirs, only_new=sorted(mine - theirs), only_at1=sorted(theirs - mine))
    log(f"  mapping: {int(M.mapped.sum())} of 200 FSD50K classes have their AudioSet ID among BEATs' 527 outputs; unmapped: "
        + ", ".join(M[~M.mapped].fsd_label) + (f" | identical to AT1's class_mapping.csv: {cmp['identical']}" if cmp else " | (AT1 mapping file not found)"))
    return M, cmp


# ####################################################################################################
# ---- AT TEACHER COMPARISON: BEATs (current) vs CED-Base / CED-Small / CED-Mini vs SSLAM, on FSD50K, NO training
# ####################################################################################################
# Every teacher scores the SAME 10 s, 16 kHz clips (the AT2/AT3/AT v2 audio cache). Score = mAP over the FSD50K classes whose AudioSet ID
# is one of the 527 AudioSet outputs (the same class mapping and the same mAP code as the AT v2 cell; BEATs' val result there: 50.29 %).
# Each candidate's 527 outputs are assumed to follow the standard AudioSet order (as BEATs' label_dict); this is CHECKED twice:
#   (1) its own label names vs the FSD50K names of the mapped classes, (2) per-class correlation with BEATs' outputs over all clips.
# A candidate whose order cannot be confirmed is reported but NOT ranked. A candidate that cannot be loaded is skipped (logged), never fatal.
SIG = lambda x: 1.0 / (1.0 + np.exp(-x))

def mean_ap(y, p):
    from sklearn.metrics import average_precision_score
    has = y.sum(0) > 0; ap = np.full(y.shape[1], np.nan)
    for j in np.where(has)[0]: ap[j] = average_precision_score(y[:, j], p[:, j])
    return float(np.nanmean(ap)), ap

def _fast_map(y, p):
    """Vectorised average precision per class (non-interpolated; equals sklearn's value when scores have no ties). Classes without positives skipped."""
    o = np.argsort(-p, axis=0, kind="stable"); ys = np.take_along_axis(y.astype(np.float64), o, axis=0)
    tp = np.cumsum(ys, 0); prec = tp / np.arange(1, len(y) + 1)[:, None]; npos = ys.sum(0); keep = npos > 0
    return float(((prec * ys).sum(0)[keep] / npos[keep]).mean())

def paired_bootstrap(y, p_a, p_b, n=100, seed=0):
    """mAP(a) - mAP(b) on resampled clip sets -> (mean, 2.5 %, 97.5 %). Same resamples for both models (paired)."""
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y)); yy = y[i]
        if not (yy.sum(0) > 0).any(): continue
        d.append(_fast_map(yy, p_a[i]) - _fast_map(yy, p_b[i]))
    d = np.array(d); return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def _norm_name(s):
    s = str(s).lower().replace("_", " ")
    for w in (" and ", ",", "(", ")", "'", "-", ".", "/"): s = s.replace(w, " ")
    return "".join(s.split())

def _hf_compat_shims():
    """Remote model code written for transformers 4.x imports a few helpers that 5.x may have renamed/removed: add no-op/alias versions if missing."""
    import transformers, transformers.utils as tu
    noop = lambda *a, **k: (lambda f: f)
    for n in ("add_code_sample_docstrings", "add_start_docstrings", "add_start_docstrings_to_model_forward", "replace_return_docstrings"):
        if not hasattr(tu, n): setattr(tu, n, noop)
    try:
        from transformers import PreTrainedModel
        if not hasattr(PreTrainedModel, "all_tied_weights_keys"): PreTrainedModel.all_tied_weights_keys = {}
    except Exception: pass
    if not hasattr(transformers, "PretrainedConfig") and hasattr(transformers, "PreTrainedConfig"):
        transformers.PretrainedConfig = transformers.PreTrainedConfig
        import transformers.configuration_utils as cu
        if not hasattr(cu, "PretrainedConfig"): cu.PretrainedConfig = transformers.PreTrainedConfig

BEATS_MIDS = None          # set when BEATs is loaded: the AudioSet IDs of BEATs' 527 outputs (the reference order)

def load_weights_from_file(model, repo, fname="model.safetensors"):
    """Independent of transformers' loader (old remote code + transformers 5 can silently leave weights wrong): reads the checkpoint
    file, matches tensors to the model by name (trying common prefixes), reports how many were ALREADY correct, then copies them all in."""
    import torch
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    sd = load_file(hf_hub_download(repo, fname)); msd = model.state_dict()
    pref = {""} | {k.split(".")[0] + "." for k in sd} | {k.split(".")[0] + "." for k in msd}
    best = ({}, "", "")
    for strip in pref:
        for add in pref:
            m = {}
            for k in sd:
                if strip and not k.startswith(strip): continue
                kk = add + k[len(strip):]
                if kk in msd and tuple(msd[kk].shape) == tuple(sd[k].shape): m[kk] = k
            if len(m) > len(best[0]): best = (m, strip, add)
    m = best[0]; cover = len(m) / max(len(msd), 1)
    already = sum(1 for kk, k in m.items() if torch.equal(msd[kk].detach().cpu().to(sd[k].dtype), sd[k]))
    with torch.no_grad():
        for kk, k in m.items(): msd[kk].copy_(sd[k].to(msd[kk].device, msd[kk].dtype))
    missing = sorted(set(msd) - set(m))
    log(f"    weights: {len(m)}/{len(msd)} model tensors matched in {fname} (prefix '{best[1]}' -> '{best[2]}'); {already} were already correct "
        f"after transformers' loading, {len(m) - already} were WRONG and are now fixed" + (f" | not in the file: {missing[:4]}" if missing else ""))
    if cover < 0.95: raise RuntimeError(f"only {cover:.0%} of the model's tensors found in the checkpoint file: weights cannot be trusted")
    return dict(matched=len(m), total=len(msd), already_correct=already, fixed=len(m) - already, missing=missing[:20])

AUDIOSET_CSV_URL = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"   # official AudioSet class list

def model_class_names(repo):
    """The 527 class names exactly as written in the model's own config.json (read as a file, not through transformers)."""
    from huggingface_hub import hf_hub_download
    c = json.load(open(hf_hub_download(repo, "config.json"))); i2l = c.get("id2label") or {}
    return [i2l.get(str(i), i2l.get(i, "")) for i in range(527)] if len(i2l) >= 527 else None

def audioset_names_to_mids():
    """display_name -> AudioSet ID from the official class list (None if it cannot be downloaded)."""
    try:
        A = pd.read_csv(AUDIOSET_CSV_URL); return dict(zip(A.display_name, A.mid)), list(A.mid)
    except Exception as e:
        log(f"    (official AudioSet class list not downloadable: {type(e).__name__}; the class order is then checked from the outputs only)"); return None, None

class BEATsCandidate:
    """The current teacher, exactly as in the AT v2 cell (fp32, mean over patches of the fine-tuned AS2M cpt2 head)."""
    name = "BEATs iter3+ AS2M cpt2 (current)"; key = "beats"
    def __init__(self):
        global BEATS_MIDS
        self.T = BEATsTeacher(); self.labels_mid = self.T.labels; self.names = None; self.params_M = self.T.info["params_M"]; self.outputs = "logits"
        BEATS_MIDS = list(self.T.labels)
    def scores(self, split, fnames): return self.T.logits_many(split, fnames)
    def release(self): self.T.model.cpu(); del self.T

class CEDCandidate:
    """mispeech/ced-* (Apache-2.0). Front end computed HERE with CED's own recipe (torchaudio MelSpectrogram: 16 kHz, n_fft 512, win 512,
    hop 160, 64 mels, f 0-8000 Hz, center -> AmplitudeToDB top_db 120, per clip), so it does not depend on the remote feature-extractor code.
    Weights are re-checked against model.safetensors; the 527 output columns are put in BEATs' order via AudioSet IDs (class_labels_indices.csv)."""
    def __init__(self, repo, name, key):
        import torch, torchaudio
        pip_if_missing("transformers", "safetensors"); _hf_compat_shims()
        from transformers import AutoModelForAudioClassification, AutoConfig
        self.name, self.key, self.repo = name, key, repo
        self.model = AutoModelForAudioClassification.from_pretrained(repo, trust_remote_code=True).eval()
        self.wcheck = load_weights_from_file(self.model, repo); self.model = self.model.to("cuda").eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        cfg = self.model.config; g = lambda k, d: getattr(cfg, k, d) if getattr(cfg, k, None) is not None else d
        self.params_M = round(sum(p.numel() for p in self.model.parameters()) / 1e6, 1)
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=int(g("n_fft", 512)), win_length=int(g("win_size", 512)), hop_length=int(g("hop_size", 160)),
                                                        f_min=float(g("f_min", 0)), f_max=float(g("f_max", 8000)), n_mels=int(g("n_mels", 64)), center=bool(g("center", True))).to("cuda")
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=120).to("cuda")
        tl = g("target_length", None); self.max_samples = (int(tl) - 1) * int(g("hop_size", 160)) if tl else None
        names = model_class_names(repo); self.perm = np.arange(527); self.names = names     # native order = standard AudioSet index order
        log(f"    class order: standard AudioSet index order (first names {names[:3] if names else None}); re-ordered to BEATs' classes by AudioSet ID in STEP 4")
        self.pooling = g("pooling", "?"); self.outputs = "?"; self.fe_check = None
        log(f"  {name}: {repo} | {self.params_M}M params | pooling '{self.pooling}' | target_length {tl} | class list in BEATs' order already: "
            f"{bool((self.perm == np.arange(527)).all())} | own mel front end | fp32, eval")
    def _feats(self, waves):
        import torch
        if self.max_samples and len(waves[0]) > self.max_samples: waves = [w[:self.max_samples] for w in waves]   # stay within ONE CED chunk
        x = torch.from_numpy(np.stack(waves)).to("cuda")
        return self.db(self.mel(x).unsqueeze(1)).squeeze(1)              # [B, 64, T]; dB range clipped per clip (top_db 120)
    def _run(self, waves):
        import torch
        x = self._feats(waves)
        if self.fe_check is None:                                         # one-time comparison with the model's own feature extractor (information only)
            try:
                from transformers import AutoFeatureExtractor
                fe = AutoFeatureExtractor.from_pretrained(self.repo, trust_remote_code=True)
                y = fe([waves[0]], sampling_rate=SR, return_tensors="pt")["input_values"][0].to("cuda")
                self.fe_check = f"shape {tuple(y.shape)} vs {tuple(x[0].shape)}, max |diff| {float((y[..., :x.shape[-1]] - x[0][..., :y.shape[-1]]).abs().max()):.4f}" if y.shape[0] == x.shape[1] else f"shape {tuple(y.shape)} vs {tuple(x[0].shape)}"
            except Exception as e: self.fe_check = f"not available ({type(e).__name__})"
            log(f"    front-end check vs the repository's feature extractor: {self.fe_check}")
        with torch.inference_mode(): out = self.model(input_values=x).logits.float()
        if out.dim() == 3: out = out.mean(1)
        if out.shape[-1] != 527 or not torch.isfinite(out).all(): raise RuntimeError(f"unexpected output {tuple(out.shape)}")
        return out.cpu().numpy()[:, self.perm]
    def scores(self, split, fnames):
        lens = {}; out = {}
        for f in fnames: lens.setdefault(int(np.load(audio_dir(split) / f"{f}.npy", mmap_mode="r").shape[0]), []).append(f)
        for n, group in sorted(lens.items()):
            for s in range(0, len(group), CFG["teacher_batch"]):
                g = group[s:s + CFG["teacher_batch"]]
                for f, row in zip(g, self._run([load_clip(split, f) for f in g])): out[f] = row
        v = np.stack(list(out.values()))
        self.outputs = "probabilities (sigmoid already applied)" if (v.min() >= 0 and v.max() <= 1) else "logits"
        return out
    def release(self): self.model.cpu(); del self.model

class SSLAMCandidate:
    """ta012/SSLAM_AS2M_Finetuned (MIT). Model-card preprocessing: waveform - mean -> kaldi fbank 128 mel (hanning, 10 ms) ->
    pad/cut to 1024 frames -> (x + 4.268) / (4.569 x 2) -> [B, 1, T, F]; outputs 527 logits."""
    name = "SSLAM AS2M fine-tuned"; key = "sslam"
    def __init__(self, repo="ta012/SSLAM_AS2M_Finetuned"):
        import torch
        pip_if_missing("transformers", "timm"); _hf_compat_shims()
        from transformers import AutoModel
        self.repo = repo; self.model = AutoModel.from_pretrained(repo, trust_remote_code=True).eval()
        self.wcheck = load_weights_from_file(self.model, repo); self.model = self.model.to("cuda").eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        cfg = self.model.config; i2l = getattr(cfg, "id2label", None) or {}
        self.names = [i2l.get(i, i2l.get(str(i), "")) for i in range(527)] if len(i2l) >= 527 else None
        self.labels_mid = None; self.params_M = round(sum(p.numel() for p in self.model.parameters()) / 1e6, 1); self.outputs = "logits"
        log(f"  {self.name}: {repo} | {self.params_M}M params | {len(i2l)} label names | fp32, eval")
    def _feat(self, w):
        import torch, torchaudio
        x = torch.from_numpy(w).float(); x = x - x.mean()
        m = torchaudio.compliance.kaldi.fbank(x.unsqueeze(0), htk_compat=True, sample_frequency=SR, use_energy=False, window_type="hanning",
                                              num_mel_bins=128, dither=0.0, frame_shift=10)
        T = 1024
        m = torch.nn.functional.pad(m, (0, 0, 0, T - m.shape[0])) if m.shape[0] < T else m[:T]
        return (m + 4.268) / (4.569 * 2)
    def scores(self, split, fnames):
        import torch
        out = {}
        for s in range(0, len(fnames), CFG["teacher_batch"]):
            g = fnames[s:s + CFG["teacher_batch"]]; x = torch.stack([self._feat(load_clip(split, f)) for f in g]).unsqueeze(1).to("cuda")
            with torch.inference_mode(): o = self.model(x)
            lg = getattr(o, "logits", o if torch.is_tensor(o) else o[0]).float()
            if lg.dim() == 3: lg = lg.mean(1)
            if lg.shape[-1] != 527 or not torch.isfinite(lg).all(): raise RuntimeError(f"unexpected output {tuple(lg.shape)}")
            for f, row in zip(g, lg.cpu().numpy()): out[f] = row
        return out
    def release(self): self.model.cpu(); del self.model

CANDIDATES = [  # (key, display name, constructor)
    ("beats", "BEATs iter3+ AS2M cpt2 (current)", lambda: BEATsCandidate()),
    ("ced_base", "CED-Base (86M)", lambda: CEDCandidate("mispeech/ced-base", "CED-Base (86M)", "ced_base")),
    ("ced_small", "CED-Small (22M)", lambda: CEDCandidate("mispeech/ced-small", "CED-Small (22M)", "ced_small")),
    ("ced_mini", "CED-Mini (9.6M)", lambda: CEDCandidate("mispeech/ced-mini", "CED-Mini (9.6M)", "ced_mini")),
    ("sslam", "SSLAM AS2M fine-tuned (88M)", lambda: SSLAMCandidate()),
]

def score_store(key, split): return paths()["teacher"] / (f"scores_{key}_{split}.npz" if key == "beats" else f"scores_v2_{key}_{split}.npz")

def candidate_scores(key, cand, split, fnames):
    """Resumable local store of one candidate's 527 outputs for every clip of the split."""
    import torch
    f = score_store(key, split); f.parent.mkdir(parents=True, exist_ok=True)
    if f.exists():
        z = np.load(f, allow_pickle=False); u = z["uids"].tolist()
        if u == list(fnames): return z["scores"].astype(np.float32), json.loads(str(z["meta"]))
    torch.cuda.reset_peak_memory_stats(); t0 = time.time(); S = {}
    for s in range(0, len(fnames), 2000):
        S.update(cand.scores(split, fnames[s:s + 2000])); log(f"    [{split}] {min(s + 2000, len(fnames)):,}/{len(fnames):,} clips | {time.time() - t0:.0f} s")
    sc = np.stack([S[u] for u in fnames]).astype(np.float32); sec = time.time() - t0
    meta = dict(outputs=cand.outputs, seconds=round(sec, 1), clips_per_s=round(len(fnames) / max(sec, 1e-9), 1), gpu_peak_gib=round(gpu_mem().get("gpu_peak_gib", 0), 2))
    tmp = str(f) + ".tmp.npz"; np.savez(tmp, uids=np.array(fnames), scores=sc, meta=json.dumps(meta)); os.replace(tmp, f)
    return sc, meta

def check_label_order(cand, sc, ref_beats, M):
    """(1) names of the mapped classes vs FSD50K names, (2) for each output class, is the most correlated BEATs output the SAME index?"""
    R = {}
    mm = M[M.mapped]
    if cand.names:
        ok = [_norm_name(cand.names[a]) == _norm_name(l) for a, l in zip(mm.beats_index, mm.fsd_label)]
        R["name_match_mapped"] = float(np.mean(ok))
    else: R["name_match_mapped"] = None
    if ref_beats is not None:
        a = sc - sc.mean(0); b = ref_beats - ref_beats.mean(0)
        a /= (np.linalg.norm(a, axis=0) + 1e-12); b /= (np.linalg.norm(b, axis=0) + 1e-12)
        C = a.T @ b                                                    # [527 candidate, 527 BEATs] correlations over all clips
        idx = np.asarray(mm.beats_index)                               # the classes that are scored (mapped FSD50K classes)
        R["corr_argmax_same_index"] = float((C[idx].argmax(1) == idx).mean())
        R["mean_diag_corr"] = float(C[idx, idx].mean())
        R["median_best_corr_any_class"] = float(np.median(C[idx].max(1)))   # high but diag low = shuffled class order; both low = outputs meaningless
        R["top1_agreement_with_beats"] = float((sc.argmax(1) == ref_beats.argmax(1)).mean())
    else: R["corr_argmax_same_index"] = None; R["mean_diag_corr"] = None
    R["order_confirmed"] = bool((R["mean_diag_corr"] or 0) >= 0.3 or (R["corr_argmax_same_index"] or 0) >= 0.5)   # outputs must track the same classes as BEATs
    return R


'''

# --------------------------------------------------------------------------------------------------------------
# module mt2kd_mod_sv  (verified earlier cell, unchanged; written to {_MOD_DIR}/mt2kd_mod_sv.py and imported)
# --------------------------------------------------------------------------------------------------------------
_MODULES["mt2kd_mod_sv"] = r'''# ====================================================================================================================
# MT2KD STAGE 1 -- SPEAKER-VERIFICATION KNOWLEDGE DISTILLATION: ReDimNet2-B6 (frozen teacher) -> Zipformer-M (third encoder block + M_SV)
# SINGLE COLAB CELL. Choose MODE at the top of the RUN part (end of the cell): "full" (all official VoxCeleb1-dev training utterances) or "sanity".
#
# Loss (MT2KD Eq. 8):  L_SV_KD = 1 - CosineSim(v_teacher, M_SV(R_student^(3)))      -- no speaker labels in the loss
#   v_teacher : frozen ReDimNet2-B6 (PalabraAI/redimnet2, VoxCeleb2-dev, large-margin fine-tuned; 192-D speaker embedding), code pinned to one
#               commit, weights sha256-checked. Optional label-free target normalisation (none / mean-centred / standardised with TRAINING-set
#               statistics), chosen by an explicit experiment on held-out DEV speakers (never on the official test set).
#   R_student : output of Zipformer-M's THIRD encoder stack (encoder.encoders[2]: 3 layers, 384-D, run at 12.5 Hz inside, returned at ~50 Hz)
#   M_SV      : attentive pooling with two 1-D convolutions (paper) -> attentive mean + std -> Linear to the teacher dimension (192)
#
# STORAGE: everything large and temporary in /content (extracted + copied audio, ReDimNet2 code and weights, logs, local checkpoints).
#          Google Drive: /content/drive/MyDrive/MT2KD/results/SV_Stage1_ReDimNet2/ only (manifests, reports, the small utterance-level teacher
#          embedding cache, checkpoints). The dataset folder on Drive is only READ.
# RESUME:  re-run the cell after ANY interruption (disconnect, runtime recycled, Colab limit): finished steps are skipped, the teacher cache
#          resumes block by block (restored from Drive), training resumes from the most advanced valid checkpoint (local disk or Drive) at the
#          exact epoch / batch / optimizer step. Drive checkpoints: every drive_ckpt_every_minutes, every drive_ckpt_every_updates, every epoch.
# ====================================================================================================================
import os
os.environ.setdefault("TORCH_HOME", "/content/torch_cache")    # ReDimNet2 weights on LOCAL disk
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import sys, re, json, math, time, shutil, hashlib, subprocess, importlib, importlib.util, threading, random, itertools, zipfile, struct
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd

CODE_VERSION = "sv1-redimnet2-v1"

CFG = dict(
    # ---- data (READ ONLY on Drive) and outputs
    vox_root="/content/drive/MyDrive/MT2KD/datasets/VoxCeleb1",
    drive_out="/content/drive/MyDrive/MT2KD/results/SV_Stage1_ReDimNet2",
    drive_api=True,                                   # Google sign-in once per runtime: confirms uploads (md5) + deletes this folder's replaced checkpoints from the trash
    write_root="/content/MT2KD_SV_ReDimNet2_work", local_audio="/content/vox1_local", zipformer_dir="/content/MT2KD_zipformer_sv1",
    extract_dir="/content/vox1_extracted",           # tar / tar.gz archives (also split into .part-NNN pieces) are extracted HERE (local disk)
    seed=42, sample_rate=16000,
    # ---- partitions
    n_val_speakers=40,                                # NON-OFFICIAL validation: 40 VoxCeleb1-dev speakers held out (VoxCeleb1 has no official SV validation split)
    sanity_train_utts=2000,
    sanity_dev_utts=8000,                             # sanity run: only this many dev utterances (+ the official test utterances) are taken from the archive
    n_val_speakers_subset=10,                         # held-out validation speakers when only a subset of VoxCeleb1-dev is available
    crop_seconds=20.0,                                # training / validation KD: teacher AND student see the same centred <= 20 s segment
    # ---- teacher: ReDimNet2-B6, VoxCeleb2-dev only (no speaker overlap with VoxCeleb1), large-margin fine-tuned ("lm")
    redimnet_label="ReDimNet2-B6", redimnet_repo="https://github.com/PalabraAI/redimnet2.git",
    redimnet_commit="c5bbe0b76e37df698c403f8844e41304ceab6307",   # source code pinned (2026-08-28)
    redimnet_model="b6", redimnet_dataset="vox2", redimnet_train_type="lm",
    redimnet_weights_sha256="bc45f032099f3c9f0fb6ad756bca23daeeba456d00c515056e388824ef36f41b",   # b6-vox2-lm.pt (release v1.0.0, 50.9 MB)
    redimnet_dir="/content/redimnet2_src", torch_home="/content/torch_cache",
    teacher_amp_dtype="float16",                      # used only if it matches fp32 (cos >= 0.999, checked); same choice on every GPU -> caches stay valid
    teacher_batch_seconds=240.0, teacher_min_speaker_gap=0.2,
    selection_val_utts_per_speaker=30, selection_trials=20000,   # normalisation experiment on the held-out DEV speakers (never the test set)
    # ---- student (Zipformer-M, unchanged) + SV branch
    sv_block=3, msv_attention_channels=128, cos_eps=1e-8,
    # ---- optimisation (MT2KD / icefall: ScaledAdam + Eden, base lr 0.045)
    optimizer="scaled_adam", base_lr=0.045, lr_batches=7500, lr_epochs=3.5, eden_warmup_batches=500, eden_warmup_start=0.1, clipping_scale=2.0,
    msv_lr_scale=1.0,                                 # separate parameter group for M_SV (same lr by default)
    adamw_lr=3e-4, weight_decay=1e-2, warmup_updates=500, min_lr_frac=0.05, grad_clip=1.0,    # used only with optimizer="adamw"
    batch_count_mode="progress",                      # "progress": Zipformer's step-dependent regularisation (layer skipping, dropout) follows training
                                                      # progress, reaching icefall's end values by the end of a short run; "icefall": icefall's own formula
    max_batch_seconds=600.0, max_batch_utts=128, accum_steps=1, ref_duration=600.0, mem_fraction=0.85,
    local_ckpt_every_updates=200, drive_ckpt_every_updates=1000, ckpt_every_minutes=10.0, drive_ckpt_every_minutes=20.0,
    log_every=20, num_workers=min(4, os.cpu_count() or 2), max_consecutive_skips=20, min_delta=1e-4,
    stall_minutes=15,                                 # archive extraction: stop (instead of hanging) if nothing arrives from Drive for this long
    learning_check_min_cos=0.5,                       # quick check: on 64 training utterances the student must reach this cosine with the teacher
)
MODES = {
    "sanity": dict(max_epochs=10, patience=99, eden_warmup_batches=50),    # pipeline check on ~2,000 utterances: NOT a research result
    "full":   dict(max_epochs=10, patience=3),
}
OFFICIAL = dict(dev_speakers=1211, dev_utts=148642, test_speakers=40, test_utts=4874, veri_test=37720, veri_test2=37611)
TEST_SPEAKERS_DOC = [f"id{n}" for n in range(10270, 10310)]                # VoxCeleb1 test speakers (id10270-id10309), used only if no trial list is found
SR = CFG["sample_rate"]

def log(*a): print(*a, flush=True)
DRIVE = "/content/drive"
def assert_local(p):
    """Every LOCAL write goes through here: refuses Google Drive (Drive writes exist only in class DriveStore)."""
    rp = os.path.realpath(str(p))
    if rp.startswith(DRIVE) or str(p).startswith(DRIVE): raise PermissionError(f"refusing to write to Google Drive: {p}")
    return p

def paths(mode=None):
    W = Path(assert_local(CFG["write_root"]))
    P = dict(meta=W / "meta", teacher=W / "teacher", logs=W / "logs", ckpt=W / "checkpoints", results=W / "results")
    if mode:
        for k in ["ckpt", "logs", "results"]: P[k] = P[k] / mode
        P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest")
    return P

def ensure_dirs(P):
    for k, v in P.items():
        if isinstance(v, Path): assert_local(v); v.mkdir(parents=True, exist_ok=True)

def pip_if_missing(*mods):
    for m in mods:
        mod, pkg = (m, m) if isinstance(m, str) else m
        if importlib.util.find_spec(mod) is None: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def atomic_json(obj, p):
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp")
    json.dump(obj, open(tmp, "w"), indent=1, default=str); os.replace(tmp, p)

def atomic_csv(df, p):
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp"); df.to_csv(tmp, index=False); os.replace(tmp, p)

def atomic_torch_save(obj, p):
    import torch
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, p)

def sha_of(items): return hashlib.sha1("\n".join(map(str, items)).encode()).hexdigest()[:12]

def file_sha1(p, chunk=1 << 22):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def file_sha256(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def set_seed(s):
    import torch
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def gpu_mem():
    import torch
    if not torch.cuda.is_available(): return {}
    return dict(gpu_alloc_gib=torch.cuda.memory_allocated() / 2**30, gpu_peak_gib=torch.cuda.max_memory_allocated() / 2**30,
                gpu_total_gib=torch.cuda.get_device_properties(0).total_memory / 2**30)

def disk_free_gib(p):
    p = Path(p)
    while not p.exists(): p = p.parent
    return shutil.disk_usage(p).free / 2**30

def require_space(path, need_gib, what, margin_gib=1.0):
    free = disk_free_gib(path)
    if need_gib > free - margin_gib: raise RuntimeError(f"not enough space for {what}: need {need_gib:.1f} GiB (+{margin_gib} margin), {free:.1f} GiB free at {path}")
    return free

def amp_dtype():
    import torch
    return torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16

def env_report():
    import torch, platform
    r = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda, gpu=None, code_version=CODE_VERSION)
    for m in ["torchaudio", "scipy", "soundfile"]:
        try: r[m] = importlib.import_module(m).__version__
        except Exception as e: r[m] = f"missing ({type(e).__name__})"
    try:
        import k2; r["k2"] = getattr(k2, "__version__", "installed")
    except Exception: r["k2"] = "not installed (pure-PyTorch Swoosh port used; verified)"
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0); r.update(gpu=p.name, gpu_gib=round(p.total_memory / 2**30, 1), compute_capability=f"{p.major}.{p.minor}",
                                                           amp="bfloat16" if p.major >= 8 else "float16 + GradScaler")
    r["disk_free_local_gib"] = round(disk_free_gib("/content" if Path("/content").exists() else "/"), 1)
    for k, v in r.items(): log(f"  {k:22s}: {v}")
    if r["gpu"] is None: raise RuntimeError("no GPU: Runtime > Change runtime type > GPU")
    return r
# ####################################################################################################
# ---- Google Drive output store: the ONLY code in this cell that writes to Google Drive
# ####################################################################################################
class SimulatedDisconnect(Exception):
    """Raised by the resume test to imitate a runtime that dies: nothing is saved at that moment."""

class DriveStore:
    """Writes ONLY into one folder and ONLY the file names in `allowed` (plus files under the sub-folder prefixes in `prefixes`, and '<name>.tmp'
    while a write is in progress).
    * small files (config.json, history.csv, final_report.json): written synchronously, atomically (.tmp then rename)
    * checkpoints (latest.pt, best.pt): copied in a background thread from the local checkpoint (training does not wait),
      written to '<name>.tmp' first and renamed only when the copy is complete -> an interrupted upload never replaces a good file
    * with the Drive API (CFG['drive_api']): each upload is confirmed by comparing Drive's md5 with the local md5, and the copies that
      Drive moves to the trash when a checkpoint is replaced are deleted permanently (only files of THIS folder with THESE names),
      because trashed files keep using Drive quota for 30 days.
    fake=True: a local stand-in folder (used by the resume test; nothing goes to Drive)."""
    def __init__(self, folder, allowed, prefixes=(), use_api=True, fake=False, api_from=None):
        self.dir, self.fake, self.allowed, self.prefixes = Path(folder), fake, tuple(allowed), tuple(prefixes)
        if fake: assert_local(folder)
        else:
            if not os.path.isdir(DRIVE + "/MyDrive"): raise RuntimeError("Google Drive is not mounted")
            # the path as written must be inside MyDrive; its resolved location may be elsewhere on the Drive mount, because a folder
            # such as MT2KD can be a Drive SHORTCUT (resolved to /content/drive/.shortcut-targets-by-id/...)
            if not os.path.normpath(os.path.abspath(str(folder))).startswith(DRIVE + "/MyDrive/"): raise PermissionError(f"Drive output folder must be inside MyDrive: {folder}")
            if not os.path.realpath(str(folder)).startswith(os.path.realpath(DRIVE) + "/"): raise PermissionError(f"{folder} resolves outside Google Drive")
        self.dir.mkdir(parents=True, exist_ok=True)
        self.api = self.fid = None; self.api_note = "not used (stand-in folder)" if fake else "disabled (CFG['drive_api'] = False)"
        if api_from is not None: self.api, self.api_note = api_from.api, api_from.api_note       # share one sign-in between sub-folders
        elif use_api and not fake: self._init_api()
        self.cv = threading.Condition(); self.pending = {}; self.busy = None; self.errors = []; self.done = {}; self._ids = itertools.count(); self.confirmers = []
        threading.Thread(target=self._worker, daemon=True).start()

    # ---------------------------------------------------------------- guards / API
    def _ok(self, name):
        base = name[:-4] if name.endswith(".tmp") else name
        if ".." in Path(base).parts or base.startswith("/"): return False
        return base in self.allowed or any(base.startswith(p) and len(base) > len(p) for p in self.prefixes)
    def _p(self, name):
        if not self._ok(name): raise PermissionError(f"'{name}' is not allowed in {self.dir}: names {self.allowed}, sub-folders {self.prefixes}")
        f = self.dir / name; f.parent.mkdir(parents=True, exist_ok=True); return f
    def _init_api(self):
        try:
            from google.colab import auth; auth.authenticate_user()
            from googleapiclient.discovery import build
            self.api = build("drive", "v3", cache_discovery=False)
            self.api_note = "ON: uploads confirmed by md5; trashed copies of this folder's checkpoints are deleted permanently"
        except Exception as e:
            self.api = None; self.api_note = (f"UNAVAILABLE ({type(e).__name__}: {e}): uploads are not confirmed and replaced checkpoints stay in the Drive "
                                              f"trash (empty it to recover the space)")
    def _folder_id(self):
        if self.fid or self.api is None: return self.fid
        rel = os.path.relpath(os.path.normpath(os.path.abspath(str(self.dir))), DRIVE + "/MyDrive").split(os.sep); parent = "root"
        try:
            for part in rel:                                  # walk the path as written; follow Drive shortcuts to their target folder
                q = (f"name = '{part}' and '{parent}' in parents and trashed = false and "
                     f"(mimeType = 'application/vnd.google-apps.folder' or mimeType = 'application/vnd.google-apps.shortcut')")
                r = self.api.files().list(q=q, fields="files(id,mimeType,shortcutDetails)", pageSize=10, supportsAllDrives=True,
                                          includeItemsFromAllDrives=True).execute().get("files", [])
                if not r: return None                         # folder not synced to Drive yet: retried later
                x = r[0]; parent = x["shortcutDetails"]["targetId"] if x["mimeType"].endswith("shortcut") else x["id"]
            self.fid = parent
        except Exception: return None
        return self.fid

    # ---------------------------------------------------------------- storage information
    def quota(self):
        if self.api:
            try:
                q = self.api.about().get(fields="storageQuota").execute()["storageQuota"]; lim = int(q.get("limit", 0)); use = int(q.get("usage", 0))
                return dict(limit=lim or None, usage=use, trash=int(q.get("usageInDriveTrash", 0)), free=(lim - use) if lim else None, source="Drive API")
            except Exception: pass
        du = shutil.disk_usage(self.dir if self.fake else DRIVE + "/MyDrive")
        return dict(limit=du.total, usage=du.used, trash=None, free=du.free, source="mounted file system (approximate)")
    def files(self): return {f.name: f.stat().st_size for f in sorted(self.dir.iterdir()) if f.is_file()}
    def status(self, expected_bytes=None):
        q = self.quota(); fl = self.files(); g = lambda b: "?" if b is None else f"{b / 2**30:.2f} GiB"
        log(f"  Drive output folder : {self.dir}")
        log(f"  folder size now     : {g(sum(fl.values()))} | files: " + (", ".join(f"{k} ({v / 2**20:.1f} MiB)" for k, v in fl.items()) or "none"))
        log(f"  Drive storage       : free {g(q['free'])} of {g(q['limit'])} (used {g(q['usage'])}" + (f", in trash {g(q['trash'])}" if q["trash"] is not None else "")
            + f") [{q['source']}]")
        log(f"  Drive API           : {self.api_note}")
        if expected_bytes: log(f"  this run will keep  : about {g(expected_bytes)} in this folder")
        unexpected = [k for k in fl if not self._ok(k)]
        if unexpected: log(f"  NOTE: files not written by this code are in the folder (left untouched): {unexpected}")
        return dict(quota=q, files=fl)
    def check_space(self, need_bytes, what):
        """Refuses to start writing large files when the free Drive space is below what the next writes need."""
        q = self.quota()
        if q["free"] is not None and q["free"] < need_bytes:
            raise RuntimeError(f"Google Drive has {q['free'] / 2**30:.2f} GiB free; {what} needs about {need_bytes / 2**30:.2f} GiB. Nothing large was written. "
                               f"Free at least {(need_bytes - q['free']) / 2**30:.2f} GiB more on Drive (deleted files only free space once the Drive trash is "
                               f"emptied) and re-run the cell.")
        free = "?" if q["free"] is None else f"{q['free'] / 2**30:.2f}"
        log(f"  Drive space check OK for {what}: need ~{need_bytes / 2**30:.2f} GiB, free {free} GiB")

    # ---------------------------------------------------------------- writes
    def write_text(self, name, text):
        tmp, dst = self._p(name + ".tmp"), self._p(name)
        with open(tmp, "w") as fh: fh.write(text); fh.flush(); os.fsync(fh.fileno())
        os.replace(tmp, dst)
    def write_json(self, name, obj): self.write_text(name, json.dumps(obj, indent=1, default=str))
    def push(self, local_file, name):
        """Queues an upload of a local checkpoint file. The file is snapshotted NOW (hard link), so later local saves cannot change what is
        uploaded; a newer request for the same name replaces a queued (not yet started) one."""
        self._p(name); snap = f"{local_file}.upload{next(self._ids)}"
        try: os.link(local_file, snap)
        except OSError: shutil.copyfile(local_file, snap)
        with self.cv:
            if self.errors: os.remove(snap); raise RuntimeError(f"a Google Drive upload failed: {self.errors[-1]} (training stopped; local state is saved)")
            old = self.pending.get(name); self.pending[name] = snap; self.cv.notify_all()
        if old and os.path.exists(old): os.remove(old)
    def _worker(self):
        while True:
            with self.cv:
                while not self.pending: self.cv.wait()
                name = next(iter(self.pending)); src = self.pending.pop(name); self.busy = name
            try: self._upload(src, name)
            except Exception as e: self.errors.append(f"{name}: {type(e).__name__}: {e}"); log(f"  GOOGLE DRIVE UPLOAD FAILED ({name}): {type(e).__name__}: {e}")
            finally:
                try: os.remove(src)
                except OSError: pass
                with self.cv: self.busy = None; self.cv.notify_all()
    def _upload(self, src, name):
        t0 = time.time(); tmp, dst = self._p(name + ".tmp"), self._p(name); h = hashlib.md5(); n = 0
        with open(src, "rb") as fi, open(tmp, "wb") as fo:          # the open handle keeps a consistent snapshot even if the local file is replaced meanwhile
            for b in iter(lambda: fi.read(1 << 24), b""): fo.write(b); h.update(b); n += len(b)
            fo.flush(); os.fsync(fo.fileno())
        os.replace(tmp, dst); md5 = h.hexdigest()
        self.done[name] = dict(bytes=n, md5=md5, written=time.strftime("%Y-%m-%d %H:%M:%S"), confirmed_by_drive_api=None)
        log(f"  Drive: {name} written ({n / 2**20:.0f} MiB in {time.time() - t0:.0f} s)" + ("; waiting for Drive to confirm the upload" if self.api else ""))
        if self.api:                                     # confirmation runs on its own thread: the next upload (and training) never waits for it
            def cf():
                self.done[name]["confirmed_by_drive_api"] = self._confirm(name, md5); self.purge_trash()
            t = threading.Thread(target=cf, daemon=True); t.start(); self.confirmers.append(t)
    def _confirm(self, name, md5, timeout=900):
        t0 = time.time()
        while time.time() - t0 < timeout:
            with self.cv:
                if name in self.pending: return "superseded by a newer upload"
            fid = self._folder_id()
            if fid:
                try:
                    r = self.api.files().list(q=f"name = '{name}' and '{fid}' in parents and trashed = false", fields="files(id,md5Checksum)").execute().get("files", [])
                    if any(x.get("md5Checksum") == md5 for x in r): log(f"  Drive: {name} upload CONFIRMED (md5 {md5[:10]} matches)"); return True
                except Exception: pass
            time.sleep(15)
        log(f"  Drive: {name} upload not confirmed within {timeout // 60} min (it may still be syncing)"); return False
    def purge_trash(self):
        """Deletes permanently the TRASHED copies of this folder's own files (replaced checkpoints); nothing else is touched."""
        fid = self._folder_id()
        if not fid: return 0
        freed = 0
        try:
            for x in self.api.files().list(q=f"'{fid}' in parents and trashed = true", fields="files(id,name,size)", pageSize=200).execute().get("files", []):
                base = x["name"][:-4] if x["name"].endswith(".tmp") else x["name"]
                if self._ok(base): self.api.files().delete(fileId=x["id"]).execute(); freed += int(x.get("size", 0))
        except Exception as e: log(f"  (could not clean this folder's trashed copies: {type(e).__name__}: {e})")
        if freed: log(f"  Drive: {freed / 2**20:.0f} MiB of replaced copies removed from the trash")
        return freed
    def wait(self, timeout=None):
        """Blocks until every queued upload is written (and confirmed, with the API)."""
        t0 = time.time()
        with self.cv:
            while self.pending or self.busy:
                if timeout and time.time() - t0 > timeout: return False
                self.cv.wait(5)
        for t in self.confirmers: t.join(timeout=max(1.0, 180 - (time.time() - t0)))      # at most ~3 min for Drive's confirmation
        if any(t.is_alive() for t in self.confirmers): log("  (Drive is still syncing; the files are written and will finish uploading in the background)")
        self.confirmers = [t for t in self.confirmers if t.is_alive()]
        if self.api: self.purge_trash()
        if self.errors: raise RuntimeError(f"Google Drive upload failed: {self.errors[-1]}")
        return True

    # ---------------------------------------------------------------- reading back
    def candidates(self, name): return [f for f in (self._p(name), self._p(name + ".tmp")) if f.exists()]
    def promote_tmp(self, name):
        """A complete '<name>.tmp' whose rename was interrupted becomes '<name>'."""
        os.replace(self._p(name + ".tmp"), self._p(name)); return self._p(name)
    def clean_stale_tmp(self):
        for f in self.dir.glob("*.tmp"):
            if self._ok(f.name[:-4]): f.unlink(); log(f"  removed an incomplete upload left by an interrupted run: {f.name}")
    def best_info(self, name="checkpoint_best.pt"): return best_meta(self._p(name)) if self._p(name).exists() else None
    def write_file(self, local_file, name):
        """Synchronous atomic copy of a small local file (e.g. a teacher-cache shard)."""
        tmp, dst = self._p(name + ".tmp"), self._p(name)
        with open(local_file, "rb") as fi, open(tmp, "wb") as fo: shutil.copyfileobj(fi, fo, 1 << 22); fo.flush(); os.fsync(fo.fileno())
        os.replace(tmp, dst)
    def write_csv(self, name, df): self.write_text(name, df.to_csv(index=False))
    def exists(self, name): return self._ok(name) and (self.dir / name).exists()
    def path(self, name): return self._p(name) if self._ok(name) else None


def best_meta(f):
    import torch
    try: ck = torch.load(f, map_location="cpu", weights_only=False, mmap=True)
    except Exception:
        try: ck = torch.load(f, map_location="cpu", weights_only=False)
        except Exception as e: return dict(error=f"{type(e).__name__}: {e}")
    r = dict(val_loss=float(ck["val"]["val_loss"]), epoch=int(ck["epoch"]), update=int(ck["update"]), sig=ck.get("sig")); del ck
    return r

# ####################################################################################################
# ---- VoxCeleb1: inspection of the real folder, header-level inventory, OFFICIAL partitions, manifests, trial lists, local copies
# ####################################################################################################
UID_RE = re.compile(r"(id\d{5})/([^/]+)/([^/]+\.wav)$")
TAR_RE = re.compile(r"(\.tar\.gz|\.tgz|\.tar)(\.part-?\d+|\.\d{3})?$")
TRIAL_NAMES = ("veri_test2.txt", "veri_test.txt", "list_test_all2.txt", "list_test_all.txt", "list_test_hard2.txt", "list_test_hard.txt",
               "iden_split.txt", "vox1_meta.csv")
OFFICIAL_TRIALS = ("veri_test2.txt", "veri_test.txt")       # VoxCeleb1-O (cleaned, original): test speakers only. E/H lists use DEV speakers -> not usable here

def uid_of(path):
    m = UID_RE.search(str(path).replace("\\", "/"))
    return f"{m.group(1)}/{m.group(2)}/{m.group(3)}" if m else None

def parse_wav_header(b, file_size=None):
    """RIFF/WAVE header -> dict(sr, channels, bits, fmt, n_frames). Walks the chunks (LIST etc. are skipped). Raises ValueError if invalid."""
    if len(b) < 12 or b[:4] not in (b"RIFF", b"RF64") or b[8:12] != b"WAVE": raise ValueError("not a RIFF/WAVE file")
    pos, fmt, data_size, data_off = 12, None, None, None
    while pos + 8 <= len(b):
        cid, size = b[pos:pos + 4], struct.unpack("<I", b[pos + 4:pos + 8])[0]
        if cid == b"fmt ":
            if pos + 8 + 16 > len(b): raise ValueError("truncated fmt chunk")
            af, ch, sr, _, ba, bits = struct.unpack("<HHIIHH", b[pos + 8:pos + 24]); fmt = dict(fmt=af, channels=ch, sr=sr, block_align=ba, bits=bits)
        elif cid == b"data":
            data_size, data_off = size, pos + 8; break
        pos += 8 + size + (size & 1)
    if fmt is None: raise ValueError("no fmt chunk")
    if data_size is None: raise ValueError("no data chunk in the first bytes")
    if file_size is not None and (data_size in (0, 0xFFFFFFFF) or data_off + data_size > file_size): data_size = max(0, file_size - data_off)
    if fmt["block_align"] <= 0: raise ValueError("block_align 0")
    return dict(**fmt, n_frames=data_size // fmt["block_align"])

def inspect_vox_root():
    """Real directory structure: audio folders (idXXXXX/<video>/<n>.wav), zip archives, split archives, metadata / trial files."""
    root = Path(CFG["vox_root"])
    if not root.is_dir(): raise FileNotFoundError(f"VoxCeleb1 folder not found: {root}")
    dirs, zips, parts, tars, meta, tree = [], [], [], [], {}, []
    for dp, dn, fn in os.walk(root):
        rel = Path(dp).relative_to(root); depth = len(rel.parts)
        ids = [d for d in dn if re.fullmatch(r"id\d{5}", d)]
        if ids: dirs.append(dict(base=dp, speakers=len(ids))); dn[:] = [d for d in dn if not re.fullmatch(r"id\d{5}", d)]
        if depth <= 2: tree.append(f"{'  ' * depth}{rel if depth else root.name}/  ({len(dn)} folders, {len(fn)} files" +
                                   (f", {len(ids)} speaker folders" if ids else "") + (f"; e.g. {sorted(fn)[:4]}" if fn else "") + ")")
        for f in fn:
            if f.lower().endswith(".zip"): zips.append(str(Path(dp) / f))
            elif re.search(r"vox1_(dev|test)_wav_parta[a-z]$", f): parts.append(str(Path(dp) / f))
            elif TAR_RE.search(f): tars.append(str(Path(dp) / f))
            if f in TRIAL_NAMES: meta.setdefault(f, str(Path(dp) / f))
        if depth >= 4: dn[:] = []
        dn[:] = [d for d in dn if not d.startswith(".")]
    log("  VoxCeleb1 folder (top levels):\n" + "\n".join("    " + t for t in tree[:40]))
    log(f"  audio folders: {[(Path(d['base']).relative_to(root).as_posix() or '.', d['speakers']) for d in dirs]} | zip archives: {[Path(z).name for z in zips]} | "
        f"split archives: {[Path(p).name for p in parts]} | tar archives: {len(tars)} file(s) {[Path(t).name for t in tars][:3]}{' ...' if len(tars) > 3 else ''} | "
        f"metadata: {sorted(meta)}")
    return dict(root=str(root), dirs=dirs, zips=zips, parts=parts, tars=tars, meta=meta)

def _id_dirs(base):
    out = []
    for dp, dn, fn in os.walk(base):
        ids = [d for d in dn if re.fullmatch(r"id\d{5}", d)]
        if ids: out.append(dict(base=dp, speakers=len(ids))); dn[:] = [d for d in dn if not re.fullmatch(r"id\d{5}", d)]
    return out

def extract_tars(tars):
    """tar / tar.gz archives, also split into pieces (<name>.tar.gz.part-000, -001, ...): the pieces are concatenated in order and streamed into
    ONE `tar` process that extracts only *.wav and *.txt files to LOCAL disk (never to Drive). A completion marker makes this a one-time step."""
    groups = {}
    for t in tars: groups.setdefault(TAR_RE.sub(lambda m: m.group(1), Path(t).name), []).append(t)
    dst = Path(assert_local(CFG["extract_dir"])) / "full"; dst.mkdir(parents=True, exist_ok=True)
    for g, ps in sorted(groups.items()):
        ps = sorted(ps, key=lambda p: [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", Path(p).name)]); mark = dst / f".extracted_{g}.ok"
        size = sum(os.path.getsize(p) for p in ps) / 2**30
        if mark.exists() and mark.read_text() == json.dumps([Path(p).name for p in ps]): log(f"  {g}: already extracted to {dst}"); continue
        require_space(dst, size * 1.15, f"extracting {g}", 5.0)
        gz = g.endswith((".tar.gz", ".tgz")); dec = (["-I", "pigz"] if shutil.which("pigz") else ["-z"]) if gz else []
        log(f"  extracting {g} ({len(ps)} piece(s), {size:.1f} GiB, read from Drive once) -> {dst} (local disk; *.wav and *.txt only) ...")
        t0 = time.time(); stop = threading.Event(); stalled = []
        cat = subprocess.Popen(["cat"] + ps, stdout=subprocess.PIPE)
        tarp = subprocess.Popen(["tar", "-x", *dec, "-f", "-", "-C", str(dst), "--wildcards", "*.wav", "*.txt"], stdin=cat.stdout, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, text=True)
        cat.stdout.close()
        def mon():                                       # progress every minute; no growth for CFG['stall_minutes'] -> stop instead of hanging
            last, t_last = -1.0, time.time()
            while not stop.wait(60):
                try: got = int(subprocess.run(["du", "-sb", str(dst)], capture_output=True, text=True).stdout.split()[0]) / 2**30
                except Exception: got = last
                log(f"    ... {got:.1f} GiB extracted | {(time.time() - t0) / 60:.0f} min")
                if got > last + 1e-3: last, t_last = got, time.time()
                elif time.time() - t_last > 60 * CFG["stall_minutes"]:
                    stalled.append(True); log(f"    no progress for {CFG['stall_minutes']} min (Drive read stalled?): stopping"); tarp.kill(); cat.kill(); return
        th = threading.Thread(target=mon, daemon=True); th.start()
        _, err = tarp.communicate(); cat.wait(); stop.set()
        if stalled: raise RuntimeError(f"extracting {g} stalled (Google Drive stopped delivering data). Re-run the cell: extraction restarts.")
        r = subprocess.CompletedProcess(tarp.args, tarp.returncode, "", err)
        errs = [l for l in (r.stderr or "").splitlines() if l.strip() and "*.txt: Not found in archive" not in l and "Exiting with failure status due to previous errors" not in l]
        if "*.wav: Not found in archive" in (r.stderr or ""): raise RuntimeError(f"{g} contains no .wav files")
        if (r.returncode and errs) or cat.returncode:
            raise RuntimeError(f"extracting {g} failed (tar exit {r.returncode}, cat exit {cat.returncode}): {(r.stderr or '')[-500:]} -- if a piece is missing or damaged, "
                               f"the archive cannot be read past it; re-run to retry")
        mark.write_text(json.dumps([Path(p).name for p in ps])); log(f"  {g}: extracted in {(time.time() - t0) / 60:.1f} min")
    return dst

SUBSET_TAR_SCRIPT = r"""
# streams a (concatenated) tar.gz from stdin and keeps ONLY the requested utterances; stops reading as soon as they are all extracted
import sys, os, re, json, time, tarfile
dst, want_file, quota, prog = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
want = set(json.load(open(want_file))); test_spk = {u.split("/")[0] for u in want}
UID = re.compile(r"(id\d{5})/([^/]+)/([^/]+\.wav)$"); TR = ("veri_test2.txt", "veri_test.txt")
class Count:
    def __init__(s, f): s.f, s.n = f, 0
    def read(s, k=-1):
        b = s.f.read(k); s.n += len(b); return b
src = Count(sys.stdin.buffer); seen = kept_t = kept_d = 0; t_last = 0; stopped = "end of archive"
def report(final=False):
    json.dump(dict(read_bytes=src.n, members=seen, test_kept=kept_t, test_wanted=len(want), dev_kept=kept_d, dev_quota=quota, final=final, stopped=stopped),
              open(prog + ".tmp", "w")); os.replace(prog + ".tmp", prog)
with tarfile.open(fileobj=src, mode="r|gz") as tf:
    for m in tf:
        seen += 1
        if time.time() - t_last > 10: report(); t_last = time.time()
        if not m.isfile(): continue
        base = os.path.basename(m.name); mt = UID.search(m.name); keep = False
        if mt:
            u = f"{mt.group(1)}/{mt.group(2)}/{mt.group(3)}"
            if u in want: keep = True; kept_t += 1
            elif mt.group(1) not in test_spk and kept_d < quota: keep = True; kept_d += 1
        elif base in TR: keep = True; u = base
        if keep:
            p = os.path.join(dst, u); os.makedirs(os.path.dirname(p), exist_ok=True); f = tf.extractfile(m)
            with open(p + ".part", "wb") as o: o.write(f.read())
            os.replace(p + ".part", p)
        if kept_t >= len(want) and kept_d >= quota: stopped = "all requested utterances extracted"; break
report(final=True)
"""

def extract_subset(tars, want, quota):
    """SANITY extraction: streams the archive and keeps only the official test utterances (from the trial list) plus the first `quota`
    dev utterances met; stops reading Drive as soon as both are complete. A .tar.gz must be read in order, so how much of it is read depends on
    where these files sit in the archive (printed)."""
    groups = {}
    for t in tars: groups.setdefault(TAR_RE.sub(lambda m: m.group(1), Path(t).name), []).append(t)
    if len(groups) != 1: raise RuntimeError(f"subset extraction expects one archive, found {sorted(groups)}")
    (g, ps), = groups.items(); ps = sorted(ps, key=lambda p: [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", Path(p).name)])
    dst = Path(assert_local(CFG["extract_dir"])) / "subset"; dst.mkdir(parents=True, exist_ok=True)
    rule = dict(archive=[Path(p).name for p in ps], test_utts=len(want), dev_quota=quota); mark = dst / ".subset.ok"
    if mark.exists() and json.load(open(mark)).get("rule") == rule: log(f"  subset already extracted to {dst}"); return dst, json.load(open(mark))
    size = sum(os.path.getsize(p) for p in ps) / 2**30; wf = dst / "want.json"; json.dump(sorted(want), open(wf, "w")); prog = dst / "progress.json"
    script = Path(assert_local(CFG["write_root"])) / "subset_tar.py"; script.write_text(SUBSET_TAR_SCRIPT)
    log(f"  SUBSET extraction from {g} ({len(ps)} pieces, {size:.1f} GiB in total): keeping the {len(want):,} official test utterances + the first "
        f"{quota:,} dev utterances, then stopping (local disk {dst})")
    t0 = time.time(); cat = subprocess.Popen(["cat"] + ps, stdout=subprocess.PIPE)
    ch = subprocess.Popen([sys.executable, "-I", str(script), str(dst), str(wf), str(quota), str(prog)], stdin=cat.stdout, stderr=subprocess.PIPE, text=True)
    cat.stdout.close(); last, t_last, stalled = -1, time.time(), False
    while ch.poll() is None:
        time.sleep(15)
        try: p = json.load(open(prog))
        except Exception: p = dict(read_bytes=0, members=0, test_kept=0, dev_kept=0)
        if p["read_bytes"] > last: last, t_last = p["read_bytes"], time.time()
        elif time.time() - t_last > 60 * CFG["stall_minutes"]: stalled = True; ch.kill(); cat.kill(); break
        if int(time.time() - t0) % 60 < 15:
            log(f"    ... read {p['read_bytes'] / 2**30:.1f} of {size:.1f} GiB | test {p['test_kept']:,}/{len(want):,} | dev {p['dev_kept']:,}/{quota:,} | {(time.time() - t0) / 60:.0f} min")
    err = ch.stderr.read() if ch.stderr else ""; cat.kill(); cat.wait()
    if stalled: raise RuntimeError("subset extraction stalled (Google Drive stopped delivering data). Re-run the cell to restart it.")
    if ch.returncode: raise RuntimeError(f"subset extraction failed (exit {ch.returncode}): {err[-500:]}")
    p = json.load(open(prog)); p.update(rule=rule, minutes=round((time.time() - t0) / 60, 1))
    log(f"  subset done in {p['minutes']} min: read {p['read_bytes'] / 2**30:.1f} of {size:.1f} GiB ({p['stopped']}) | test {p['test_kept']:,}/{len(want):,} | dev {p['dev_kept']:,}")
    if p["test_kept"] < len(want): log(f"  WARNING: only {p['test_kept']} of {len(want)} trial utterances are in the archive: the official EER will be reported as NOT official")
    atomic_json(p, mark); return dst, p

def _trial_utts(I):
    u = set()
    for n in OFFICIAL_TRIALS:
        if n in I["meta"]: t = read_trials(I["meta"][n]); u |= set(t.enrol) | set(t.test)
    return u

def effective_layout(mode="full"):
    """The audio sources actually used.
    full  : tar archives are the complete copy -> extracted locally and used INSTEAD of a (possibly partial) extracted folder on Drive.
    sanity: as little as possible is read from Drive: (1) an existing audio folder on Drive is used directly if it already holds every official
            trial utterance and >= CFG['sanity_dev_utts'] dev utterances; (2) otherwise only a SUBSET of the archive is extracted (the trial
            utterances + CFG['sanity_dev_utts'] dev utterances), and reading stops as soon as it is complete."""
    f = paths()["meta"] / f"vox_layout_{mode}.json"
    if f.exists(): return json.load(open(f))
    I = inspect_vox_root(); I["mode"] = mode; want = _trial_utts(I); quota = CFG["sanity_dev_utts"]
    if mode == "sanity" and I["dirs"]:
        have_t = sum(any(os.path.exists(Path(d["base"]) / u) for d in I["dirs"]) for u in sorted(want)) if want else 0
        n_dev = 0; tspk = {u.split("/")[0] for u in want}
        for d in I["dirs"]:
            for sp in sorted(os.listdir(d["base"])):
                if re.fullmatch(r"id\d{5}", sp) and sp not in tspk:
                    for dp, _, fn in os.walk(Path(d["base"]) / sp): n_dev += sum(x.endswith(".wav") for x in fn)
                if n_dev >= quota: break
        log(f"  Drive audio folder: {have_t}/{len(want)} trial utterances present, >= {n_dev:,} dev utterances")
        if (not want or have_t == len(want)) and n_dev >= quota:
            I["source_rule"] = "existing Drive audio folder (no archive read)"; atomic_json(I, f); return I
    if I["tars"]:
        if mode == "sanity":
            dst, p = extract_subset(I["tars"], want, quota); I["subset"] = p
            I["source_rule"] = f"SUBSET of the archive: {p['test_kept']} test + {p['dev_kept']} dev utterances (sanity run only; NOT the full VoxCeleb1-dev)"
        else:
            dst = extract_tars(I["tars"]); I["source_rule"] = "complete archive, extracted locally"
        loc = _id_dirs(dst)
        if I["dirs"]: log(f"  NOTE: the Drive folder(s) {[d['base'] for d in I['dirs']]} are NOT used ({I['source_rule']})")
        I["drive_dirs_ignored"] = I["dirs"]; I["dirs"] = loc
        for dp, _, fn in os.walk(dst):
            for x in fn:
                if x in TRIAL_NAMES: I["meta"].setdefault(x, str(Path(dp) / x))
    else: I["source_rule"] = "audio folders / zip archives as found"
    atomic_json(I, f); return I

def _assemble_parts(parts):
    """vox1_*_wav_partaa.. (official download pieces) -> one local zip (local disk, not Drive)."""
    out = []
    for group in sorted({re.sub(r"_parta[a-z]$", "", Path(p).name) for p in parts}):
        ps = sorted(p for p in parts if Path(p).name.startswith(group + "_parta")); dst = Path(assert_local(CFG["write_root"])) / "archives" / f"{group}.zip"
        if not dst.exists():
            need = sum(os.path.getsize(p) for p in ps) / 2**30; require_space(dst.parent if dst.parent.exists() else "/content", need, f"assembling {group}.zip", 5.0)
            dst.parent.mkdir(parents=True, exist_ok=True); tmp = dst.with_suffix(".zip.part"); t0 = time.time()
            log(f"  assembling {group}.zip from {len(ps)} pieces ({need:.1f} GiB) on local disk ...")
            with open(tmp, "wb") as fo:
                for p in ps:
                    with open(p, "rb") as fi: shutil.copyfileobj(fi, fo, 1 << 24)
            if not zipfile.is_zipfile(tmp): tmp.unlink(); raise RuntimeError(f"{group}: the assembled pieces are not a valid zip")
            os.replace(tmp, dst); log(f"  {dst} ready ({time.time() - t0:.0f} s)")
        out.append(str(dst))
    return out

def inventory(mode="full", rebuild=False):
    """One row per WAV file: uid, speaker, source, header fields, duration, validity + reason. Header-level only (one small read per file)."""
    P = paths(); ensure_dirs(P); f_inv, f_bad = P["meta"] / f"inventory_{mode}.csv", P["meta"] / f"excluded_inventory_{mode}.csv"
    if f_inv.exists() and not rebuild:
        inv = pd.read_csv(f_inv, dtype={"uid": str, "speaker": str, "source": str, "member": str}, keep_default_na=False)
        log(f"  inventory (cached): {len(inv):,} files, {int(inv.valid.sum()):,} valid"); return inv
    I = effective_layout(mode); zips = list(I["zips"]) + (_assemble_parts(I["parts"]) if I["parts"] and mode == "full" else [])
    tasks = []
    for d in I["dirs"]:
        for dp, _, fn in os.walk(d["base"]):
            for f in fn:
                if f.lower().endswith(".wav"): tasks.append(("dir", str(Path(dp) / f), ""))
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                for zi in zf.infolist():
                    if zi.filename.lower().endswith(".wav") and uid_of(zi.filename): tasks.append(("zip", z, zi.filename))
        except zipfile.BadZipFile as e: log(f"  WARNING: unreadable zip {z}: {e}")
    if not tasks: raise RuntimeError(f"no VoxCeleb1 WAV files found under {CFG['vox_root']} (looked for idXXXXX/<video>/<n>.wav in folders and zip archives)")
    log(f"  reading {len(tasks):,} WAV headers (one small read per file) ...")
    local = threading.local()
    def zf_of(z):
        if not hasattr(local, "z"): local.z = {}
        if z not in local.z: local.z[z] = zipfile.ZipFile(z)
        return local.z[z]
    def head(t):
        kind, src, mem = t; r = dict(uid=uid_of(mem or src), source=src, member=mem, kind=kind)
        r["speaker"] = r["uid"].split("/")[0] if r["uid"] else ""
        try:
            if kind == "dir":
                size = os.path.getsize(src)
                with open(src, "rb") as f: b = f.read(65536)
            else:
                zi = zf_of(src).getinfo(mem); size = zi.file_size
                with zf_of(src).open(mem) as f: b = f.read(65536)
            if size == 0: raise ValueError("empty file")
            h = parse_wav_header(b, size); r.update(sr=h["sr"], channels=h["channels"], bits=h["bits"], fmt=h["fmt"], n_frames=h["n_frames"], bytes=size)
            r["duration_s"] = h["n_frames"] / h["sr"] if h["sr"] else 0.0
            reason = ("not PCM/float WAV" if h["fmt"] not in (1, 3, 0xFFFE) else "no audio frames" if h["n_frames"] <= 0 else
                      "duration < 0.5 s" if r["duration_s"] < 0.5 else "")
        except Exception as e: reason = f"unreadable header: {type(e).__name__}: {e}"
        r["valid"] = not reason; r["reason"] = reason; return r
    t0 = time.time(); rows = []
    with ThreadPoolExecutor(32) as ex:
        for i, r in enumerate(ex.map(head, tasks, chunksize=64)):
            rows.append(r)
            if (i + 1) % 20000 == 0: log(f"    {i + 1:,}/{len(tasks):,} headers | {time.time() - t0:.0f} s")
    inv = pd.DataFrame(rows)
    for c in ["sr", "channels", "bits", "fmt", "n_frames", "bytes"]: inv[c] = inv.get(c, pd.Series(dtype=float)).fillna(0).astype(int)
    inv["duration_s"] = inv.get("duration_s", pd.Series(dtype=float)).fillna(0.0)
    dup = inv.uid.duplicated(keep="first") & inv.uid.ne("")
    if dup.any(): inv.loc[dup, ["valid", "reason"]] = [False, "duplicate uid (same file in another folder/archive; first copy kept)"]
    inv.loc[inv.uid.eq("") | inv.uid.isna(), ["valid", "reason"]] = [False, "path does not match idXXXXX/<video>/<n>.wav"]
    inv = inv.sort_values("uid", kind="stable").reset_index(drop=True); atomic_csv(inv, f_inv); atomic_csv(inv[~inv.valid], f_bad)
    log(f"  inventory: {len(inv):,} files, {int(inv.valid.sum()):,} valid, {int((~inv.valid).sum())} excluded (reasons recorded) | {time.time() - t0:.0f} s | "
        f"sample rates {inv[inv.valid].sr.value_counts().to_dict()} | channels {inv[inv.valid].channels.value_counts().to_dict()}")
    return inv

def read_trials(f):
    rows = []
    for ln in open(f):
        p = ln.split()
        if len(p) == 3 and p[0] in ("0", "1"): rows.append((int(p[0]), p[1], p[2]))
    return pd.DataFrame(rows, columns=["label", "enrol", "test"])

def prepare_dataset(mode="full"):
    """OFFICIAL partition (dev / test), the NON-OFFICIAL held-out validation speakers, manifests, trial lists, dataset report. Deterministic."""
    P = paths(); ensure_dirs(P); f_rep = P["meta"] / "dataset_validation_report.json"
    inv = inventory(mode); I = effective_layout(mode); ok = inv[inv.valid].copy()
    trials, tsrc = {}, None
    for name in OFFICIAL_TRIALS:
        if name in I["meta"]: trials[name] = read_trials(I["meta"][name])
    unusable = [n for n in I["meta"] if n.startswith("list_test")]
    if trials:
        tsrc = f"speakers of the official trial list(s) {sorted(trials)}"
        test_spk = sorted({u.split("/")[0] for t in trials.values() for u in pd.concat([t.enrol, t.test])})
    else:
        tests = ok[ok.source.str.contains("test", case=False) | ok.member.str.contains("test", case=False)]
        if len(tests) and tests.speaker.nunique() < ok.speaker.nunique():
            test_spk = sorted(tests.speaker.unique()); tsrc = "speakers stored in the 'test' folder/archive"
        else: test_spk = [s for s in TEST_SPEAKERS_DOC if s in set(ok.speaker)]; tsrc = "documented VoxCeleb1 test speaker ids id10270-id10309 (no trial list found)"
    test_set = set(test_spk); ok["partition"] = np.where(ok.speaker.isin(test_set), "test", "dev")
    dev_spk = sorted(ok[ok.partition == "dev"].speaker.unique()); rng = np.random.default_rng(CFG["seed"])
    n_val = CFG["n_val_speakers"] if I.get("subset") is None else min(CFG["n_val_speakers_subset"], max(1, len(dev_spk) // 5))
    val_spk = sorted(np.array(dev_spk)[rng.permutation(len(dev_spk))[:n_val]].tolist())
    ok.loc[ok.speaker.isin(set(val_spk)), "partition"] = "val"
    tr, va, te = (ok[ok.partition == p].sort_values("uid").reset_index(drop=True) for p in ("dev", "val", "test"))
    assert not set(tr.speaker) & set(te.speaker) and not set(va.speaker) & set(te.speaker) and not set(tr.speaker) & set(va.speaker)
    rng2 = np.random.default_rng(CFG["seed"] + 1); sub = tr.iloc[np.sort(rng2.permutation(len(tr))[:CFG["sanity_train_utts"]])].reset_index(drop=True)
    cols = ["uid", "speaker", "kind", "source", "member", "sr", "channels", "bits", "n_frames", "duration_s"]
    for name, df in [("train_manifest", tr), ("sanity_train_manifest", sub), ("validation_manifest", va), ("test_manifest", te)]: atomic_csv(df[cols], P["meta"] / f"{name}.csv")
    for name, t in trials.items(): atomic_csv(t, P["meta"] / f"trials_{name.replace('.txt', '')}.csv")
    have = set(te.uid); tri = {}
    for name, t in trials.items():
        miss = sorted((set(t.enrol) | set(t.test)) - have)
        tri[name] = dict(trials=len(t), target=int(t.label.sum()), utterances=len(set(t.enrol) | set(t.test)), missing_utterances=len(miss),
                         official_count=OFFICIAL.get(name.replace(".txt", "")), complete=not miss, first_missing=miss[:5])
    excl = inv[~inv.valid][["uid", "source", "member", "reason"]]
    rep = dict(code_version=CODE_VERSION, vox_root=CFG["vox_root"], mode=mode, source_rule=I.get("source_rule"), subset=I.get("subset"), layout=dict(audio_folders=I["dirs"], zip_archives=I["zips"], split_archives=I["parts"], tar_archives=I.get("tars", []),
                                                                       drive_folders_ignored=I.get("drive_dirs_ignored", []), metadata=I["meta"]),
               files=len(inv), valid=int(inv.valid.sum()), excluded=len(excl), excluded_reasons=excl.reason.str.split(":").str[0].value_counts().to_dict(),
               test_partition_source=tsrc,
               official=dict(dev=dict(speakers=int(tr.speaker.nunique() + va.speaker.nunique()), utterances=len(tr) + len(va), official_speakers=OFFICIAL["dev_speakers"],
                                      official_utterances=OFFICIAL["dev_utts"]),
                             test=dict(speakers=int(te.speaker.nunique()), utterances=len(te), official_speakers=OFFICIAL["test_speakers"], official_utterances=OFFICIAL["test_utts"])),
               train=dict(speakers=int(tr.speaker.nunique()), utterances=len(tr), hours=round(float(tr.duration_s.sum() / 3600), 2)),
               validation=dict(speakers=len(val_spk), utterances=len(va), hours=round(float(va.duration_s.sum() / 3600), 2), speaker_ids=val_spk,
                               note=f"NON-OFFICIAL: {len(val_spk)} VoxCeleb1-dev speakers held out with numpy default_rng({CFG['seed']}); VoxCeleb1 has no official SV validation split"),
               sanity_train=dict(utterances=len(sub), speakers=int(sub.speaker.nunique()), hours=round(float(sub.duration_s.sum() / 3600), 2),
                                 rule=f"numpy default_rng({CFG['seed'] + 1}) permutation of the training manifest, first {CFG['sanity_train_utts']}"),
               trials=tri, unusable_trial_lists={n: "uses VoxCeleb1-dev speakers, which are training data here" for n in unusable},
               durations=dict(min=round(float(ok.duration_s.min()), 2), median=round(float(ok.duration_s.median()), 2), mean=round(float(ok.duration_s.mean()), 2),
                              max=round(float(ok.duration_s.max()), 2), longer_than_crop=int((ok.duration_s > CFG["crop_seconds"]).sum())),
               sample_rates=ok.sr.value_counts().to_dict(), channels=ok.channels.value_counts().to_dict(), bits=ok.bits.value_counts().to_dict())
    atomic_csv(excl, P["meta"] / "excluded_files.csv"); atomic_json(rep, f_rep)
    log(f"  audio source: {I.get('source_rule')}")
    log(f"  OFFICIAL partition from {tsrc}:")
    log(f"    dev  : {rep['official']['dev']['speakers']} speakers / {rep['official']['dev']['utterances']:,} utterances (official {OFFICIAL['dev_speakers']} / {OFFICIAL['dev_utts']:,})")
    log(f"    test : {te.speaker.nunique()} speakers / {len(te):,} utterances (official {OFFICIAL['test_speakers']} / {OFFICIAL['test_utts']:,}) -> evaluation ONLY")
    log(f"  training   : {tr.speaker.nunique()} speakers / {len(tr):,} utterances / {rep['train']['hours']} h")
    log(f"  validation : {len(val_spk)} held-out dev speakers / {len(va):,} utterances (NON-OFFICIAL, speaker-disjoint from training and test)")
    log(f"  sanity     : {len(sub):,} training utterances ({sub.speaker.nunique()} speakers, {rep['sanity_train']['hours']} h)")
    for n, t in tri.items(): log(f"  trials {n}: {t['trials']:,} ({t['target']:,} target) | utterances {t['utterances']:,} | missing {t['missing_utterances']} -> "
                                 + ("COMPLETE (official EER possible)" if t["complete"] else "INCOMPLETE (no official EER from this list)"))
    if not tri: log("  NO official verification trial list found (veri_test2.txt / veri_test.txt): official EER cannot be computed; KD validation metrics only")
    for n in unusable: log(f"  {n}: not used (VoxCeleb1-E/H trials contain dev speakers = training data)")
    for k, (a, b) in dict(dev_speakers=(rep['official']['dev']['speakers'], OFFICIAL['dev_speakers']), test_utts=(len(te), OFFICIAL['test_utts'])).items():
        if a != b: log(f"  WARNING: {k} = {a}, official {b} (incomplete copy or extra files?)")
    return rep

def load_manifest(name): return pd.read_csv(paths()["meta"] / f"{name}.csv", dtype={"uid": str, "speaker": str, "source": str, "member": str}, keep_default_na=False)
def local_wav(uid): return Path(CFG["local_audio"]) / uid

def materialize(df, what):
    """Copies the manifest's WAV files to LOCAL disk (thread pool; resumable: complete files are skipped)."""
    assert_local(CFG["local_audio"]); need = [r for r in df.itertuples() if not local_wav(r.uid).exists()]      # copies are atomic (.part then rename)
    if not need: log(f"  [{what}] local audio ready ({len(df):,} files)"); return
    gib = sum(int(r.n_frames) * int(r.channels) * int(r.bits) // 8 for r in need if str(r.source).startswith(DRIVE) or r.kind != "dir") / 2**30
    require_space(CFG["local_audio"], gib, f"{what} audio", 3.0)
    log(f"  [{what}] copying {len(need):,} WAV files ({gib:.1f} GiB) to local disk ..."); t0 = time.time(); local = threading.local()
    def cp(r):
        dst = local_wav(r.uid); dst.parent.mkdir(parents=True, exist_ok=True); tmp = dst.with_name(dst.name + ".part")
        if r.kind == "dir":
            if not str(r.source).startswith(DRIVE):
                try: os.link(r.source, dst); return 1                       # locally extracted archive: hard link, no second copy
                except OSError: pass
            shutil.copyfile(r.source, tmp)
        else:
            if not hasattr(local, "z"): local.z = {}
            if r.source not in local.z: local.z[r.source] = zipfile.ZipFile(r.source)
            with local.z[r.source].open(r.member) as fi, open(tmp, "wb") as fo: shutil.copyfileobj(fi, fo, 1 << 20)
        os.replace(tmp, dst); return 1
    done = 0
    with ThreadPoolExecutor(16) as ex:
        for _ in ex.map(cp, need, chunksize=32):
            done += 1
            if done % 5000 == 0: log(f"    {done:,}/{len(need):,} | {time.time() - t0:.0f} s")
    log(f"  [{what}] {len(need):,} files copied in {time.time() - t0:.0f} s")

def decode_check(df, what):
    """Full decode of each file ONCE (finite samples, length = header). Failures are recorded and excluded with their reason."""
    import soundfile as sf
    P = paths(); fv, ff = P["meta"] / "decode_verified.txt", P["meta"] / "decode_failures.csv"
    ver = set(open(fv).read().split()) if fv.exists() else set(); bad = pd.read_csv(ff, dtype=str).set_index("uid").reason.to_dict() if ff.exists() else {}
    todo = [r for r in df.itertuples() if r.uid not in ver and r.uid not in bad]
    def chk(r):
        try:
            x, sr = sf.read(str(local_wav(r.uid)), dtype="float32", always_2d=True)
            if sr != int(r.sr) or len(x) != int(r.n_frames): return r.uid, f"decoded {len(x)} frames @ {sr} Hz, header {r.n_frames} @ {r.sr}"
            if not np.isfinite(x).all(): return r.uid, "non-finite samples"
            if float(np.abs(x).max()) == 0.0: return r.uid, "all-zero audio"
            return r.uid, ""
        except Exception as e: return r.uid, f"decode error: {type(e).__name__}: {e}"
    t0 = time.time(); new_ok = []
    with ThreadPoolExecutor(16) as ex:
        for i, (u, why) in enumerate(ex.map(chk, todo, chunksize=32)):
            if why: bad[u] = why
            else: new_ok.append(u)
            if (i + 1) % 20000 == 0: log(f"    {i + 1:,}/{len(todo):,} decoded | {time.time() - t0:.0f} s")
    if new_ok:
        with open(fv, "a") as f: f.write("\n".join(new_ok) + "\n")
    atomic_csv(pd.DataFrame(sorted(bad.items()), columns=["uid", "reason"]), ff)
    fails = df.uid[df.uid.isin(set(bad))].tolist()
    log(f"  [{what}] full decode check: {len(todo):,} decoded now ({len(df) - len(todo):,} earlier) in {time.time() - t0:.0f} s | failures {len(fails)}"
        + (f" -> excluded with reason, e.g. {fails[:2]}" if fails else ""))
    return df[~df.uid.isin(set(bad))].reset_index(drop=True)

def load_wave(uid, crop_seconds=None):
    """16 kHz mono float32. crop_seconds: centred crop (the SAME segment for teacher and student)."""
    import soundfile as sf
    x, sr = sf.read(str(local_wav(uid)), dtype="float32", always_2d=True); x = x.mean(axis=1) if x.shape[1] > 1 else x[:, 0]
    if sr != SR:
        import torch, torchaudio; x = torchaudio.functional.resample(torch.from_numpy(x), sr, SR).numpy()
    if crop_seconds:
        L = int(round(crop_seconds * SR))
        if len(x) > L: s = (len(x) - L) // 2; x = x[s:s + L]
    return np.ascontiguousarray(x, dtype=np.float32)

def crop_len(n_frames, sr, crop_seconds):
    n = int(round(int(n_frames) * SR / int(sr))) if int(sr) != SR else int(n_frames)
    return min(n, int(round(crop_seconds * SR))) if crop_seconds else n
# ####################################################################################################
# ---- scoring: cosine trials and EER (pure numpy)
# ####################################################################################################
def compute_eer(scores, labels):
    """EER (fraction) and threshold. FRR(t) = targets with score <= t, FAR(t) = non-targets with score > t; linear interpolation at the crossing."""
    s = np.asarray(scores, dtype=np.float64); y = np.asarray(labels).astype(bool); nt, nn_ = int(y.sum()), int((~y).sum())
    if nt == 0 or nn_ == 0: raise ValueError("EER needs target and non-target trials")
    o = np.argsort(s, kind="mergesort"); s, y = s[o], y[o]
    frr = np.concatenate([[0.0], np.cumsum(y) / nt]); far = np.concatenate([[1.0], 1.0 - np.cumsum(~y) / nn_]); d = frr - far
    k = int(np.argmax(d >= 0))
    if k == 0: return float(frr[0]), float(s[0])
    a = d[k - 1] / (d[k - 1] - d[k]); eer = frr[k - 1] + a * (frr[k] - frr[k - 1])
    return float(eer), float(s[k - 1])

def l2n(x, eps=1e-12): x = np.asarray(x, dtype=np.float64); return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def score_trials(emb, row_of, trials):
    """Cosine scores for the trials whose two utterances both have an embedding; returns (scores, labels, n_missing)."""
    keep = trials.enrol.map(lambda u: u in row_of) & trials.test.map(lambda u: u in row_of); t = trials[keep]
    e = l2n(emb); a = e[t.enrol.map(row_of).values]; b = e[t.test.map(row_of).values]
    return (a * b).sum(-1), t.label.values, int((~keep).sum())

def make_selection_trials(df, n_trials, seed):
    """NON-OFFICIAL trials among held-out DEV speakers (for choosing the target normalisation and for monitoring only): half target, half
    non-target pairs, sampled deterministically, no duplicates, no self-pairs."""
    rng = np.random.default_rng(seed); uids = df.uid.values; spk = df.speaker.values; by = {}
    for i, s in enumerate(spk): by.setdefault(s, []).append(i)
    S = [s for s in sorted(by) if len(by[s]) >= 2]; seen = set(); rows = []
    tries = 0
    while len(rows) < n_trials and tries < 50 * n_trials:
        tries += 1; tgt = len(rows) % 2 == 0
        if tgt: s = S[rng.integers(len(S))]; i, j = rng.choice(by[s], 2, replace=False)
        else:
            i, j = rng.integers(len(uids), size=2)
            if spk[i] == spk[j]: continue
        key = (min(i, j), max(i, j))
        if key in seen or i == j: continue
        seen.add(key); rows.append((int(tgt), uids[i], uids[j]))
    return pd.DataFrame(rows, columns=["label", "enrol", "test"])

def selection_sets(val_df):
    rng = np.random.default_rng(CFG["seed"] + 2); k = CFG["selection_val_utts_per_speaker"]
    parts = [g.iloc[np.sort(rng.permutation(len(g))[:k])] for _, g in val_df.groupby("speaker", sort=True)]
    sel = pd.concat(parts).sort_values("uid").reset_index(drop=True)
    return sel, make_selection_trials(sel, CFG["selection_trials"], CFG["seed"] + 3)

# ####################################################################################################
# ---- ReDimNet2-B6 teacher (frozen, pinned code commit + sha256-checked weights)
# ####################################################################################################
NORMS = ("none", "centre", "standardise")

def spec_name(spec):
    return f"{CFG['redimnet_label']} speaker embedding, " + dict(none="raw (no normalisation)", centre="mean-centred (training-set mean)",
                                                             standardise="standardised (training-set mean / std per dimension)")[spec["norm"]]

def norm_stats(E_tr, norm):
    """Label-free target normalisation from TRAINING-partition embeddings: 'none' -> (0, 1), 'centre' -> (mean, 1), 'standardise' -> (mean, std)."""
    D = E_tr.shape[1]
    if norm == "none": return np.zeros(D, np.float32), np.ones(D, np.float32)
    mu = E_tr.mean(0).astype(np.float32)
    if norm == "centre": return mu, np.ones(D, np.float32)
    sd = E_tr.std(0); return mu, np.maximum(sd, 1e-5 * max(float(sd.mean()), 1e-12)).astype(np.float32)

def _git(*a, cwd=None): return subprocess.run(["git", *a], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()

class ReDimNetTeacher:
    """Frozen ReDimNet2 (PalabraAI/redimnet2, MIT licence): source code pinned to CFG['redimnet_commit'], weights from the repository's release,
    checked against CFG['redimnet_weights_sha256']. Input: 16 kHz mono waveform (B, samples); output: (B, embed_dim) speaker embedding.
    The model has no padding mask (its signal normalisation, mel mean normalisation and pooling run over the whole input), so a batch only
    ever contains utterances of IDENTICAL length (checked)."""
    use_mask = False

    def __init__(self):
        import torch
        repo = Path(assert_local(CFG["redimnet_dir"])); want = CFG["redimnet_commit"]
        ok = (repo / ".git").exists()
        if ok:
            try: ok = _git("rev-parse", "HEAD", cwd=repo) == want and not _git("status", "--porcelain", cwd=repo)
            except Exception: ok = False
        if not ok:
            shutil.rmtree(repo, ignore_errors=True); log(f"  cloning {CFG['redimnet_repo']} @ {want[:12]} -> {repo}")
            _git("clone", "--quiet", CFG["redimnet_repo"], str(repo)); _git("checkout", "--quiet", want, cwd=repo)
        head = _git("rev-parse", "HEAD", cwd=repo)
        if head != want: raise RuntimeError(f"ReDimNet2 source is at {head}, expected the pinned commit {want}")
        if str(repo) not in sys.path: sys.path.insert(0, str(repo))
        for m in [m for m in sys.modules if m == "redimnet2" or m.startswith("redimnet2.")]: del sys.modules[m]
        import redimnet2 as R2
        if not Path(R2.__file__).resolve().is_relative_to(repo.resolve()): raise RuntimeError(f"imported redimnet2 from {R2.__file__}, not from the pinned clone {repo}")
        wname = f"{CFG['redimnet_model']}-{CFG['redimnet_dataset']}-{CFG['redimnet_train_type']}.pt"; url = R2.URL_TEMPLATE.format(weight_name=wname)
        wdir = Path(assert_local(CFG["torch_home"])) / "redimnet2"; wdir.mkdir(parents=True, exist_ok=True); wf = wdir / wname
        exp = CFG.get("redimnet_weights_sha256")
        if wf.exists() and exp and file_sha256(wf) != exp: log(f"  {wf.name}: sha256 mismatch (damaged download?) -> downloaded again"); wf.unlink()
        if not wf.exists():
            log(f"  downloading {url}"); tmp = wf.with_name(wf.name + ".part"); torch.hub.download_url_to_file(url, str(tmp), progress=False); os.replace(tmp, wf)
        sha = file_sha256(wf)
        if exp and sha != exp: raise RuntimeError(f"{wname}: sha256 {sha} != expected {exp} (the released file changed?). Training blocked.")
        try: ck = torch.load(wf, map_location="cpu", weights_only=True)
        except Exception: ck = torch.load(wf, map_location="cpu", weights_only=False)
        self.model = R2.ReDimNet2Wrap(**ck["model_config"]); res = self.model.load_state_dict(ck["state_dict"], strict=True)
        if res.missing_keys or res.unexpected_keys: raise RuntimeError(f"{wname}: missing {res.missing_keys[:3]}, unexpected {res.unexpected_keys[:3]}")
        self.model.eval().to("cuda")
        for p in self.model.parameters(): p.requires_grad_(False)
        self.amp = None; self._cpu = None; D = int(self.model.embed_dim)
        self.info = dict(model_id=f"PalabraAI/redimnet2:{wname[:-3]}", name=CFG["redimnet_label"], repo=CFG["redimnet_repo"], commit=head, model_class=type(self.model).__name__,
                         model_config=json.loads(json.dumps(ck["model_config"], default=str)), embed_dim=D, hidden_size=D,
                         params=int(sum(p.numel() for p in self.model.parameters())), sampling_rate=SR, attention_mask_used=False,
                         weight_files={wname: dict(bytes=os.path.getsize(wf), sha256=sha, url=url)},
                         training_data=dict(vox2="VoxCeleb2-dev (no speaker overlap with VoxCeleb1)").get(CFG["redimnet_dataset"], CFG["redimnet_dataset"]),
                         train_type=CFG["redimnet_train_type"], licence="MIT",
                         input_format="16 kHz mono float waveform (B, samples), identical lengths within a batch -> the model's own front end "
                                      "(signal normalisation, pre-emphasis, 72 log-mel bands, mean normalisation) -> embedding")
        del ck
        log(f"  {self.info['name']} ({wname}) code @ {head[:12]} | {self.info['params'] / 1e6:.2f}M params | {D}-D embedding | weights sha256 {sha[:16]}... | "
            f"trained on {self.info['training_data']}")

    def embed(self, waves, amp="auto"):
        """-> (B, D) fp32 embeddings. All waves must have the same length."""
        import torch
        if len({len(w) for w in waves}) != 1: raise RuntimeError("ReDimNet2 has no padding mask: one batch = utterances of identical length")
        amp = self.amp if amp == "auto" else amp; x = torch.from_numpy(np.stack(waves).astype(np.float32)).to("cuda")
        with torch.no_grad(), torch.autocast("cuda", dtype=amp or torch.float32, enabled=amp is not None): e = self.model(x)
        return e.float()

    def embed_cpu(self, waves):
        """Fallback for a single utterance that does not fit in GPU memory: same weights, fp32 on the CPU."""
        import torch, copy
        if self._cpu is None: self._cpu = copy.deepcopy(self.model).cpu().float().eval()
        with torch.no_grad(): return self._cpu(torch.from_numpy(np.stack(waves).astype(np.float32))).float()

    def smoke_test(self):
        import torch
        rng = np.random.default_rng(0); w = (0.1 * rng.standard_normal(4 * SR)).astype(np.float32); e = self.embed([w, w], amp=None)
        assert e.shape == (2, self.info["embed_dim"]) and bool(torch.isfinite(e).all()) and float((e[0] - e[1]).abs().max()) < 1e-4, tuple(e.shape)
        assert not self.model.training and sum(p.requires_grad for p in self.model.parameters()) == 0
        log(f"  smoke test on 4 s of noise: output {tuple(e.shape)}, finite, batch rows identical, eval mode, frozen -> OK")

def equal_length_batches(lens, max_sec, one=False):
    """Batches of utterances with IDENTICAL sample counts (long utterances cropped to crop_seconds all share one length); <= max_sec of audio."""
    by = {}
    for i, l in enumerate(lens): by.setdefault(int(l), []).append(int(i))
    out = []
    for l in sorted(by):
        per = 1 if one else max(1, int(max_sec * SR // max(l, 1))); idx = by[l]
        out += [idx[s:s + per] for s in range(0, len(idx), per)]
    return out

def run_teacher(T, df, crop, desc, max_sec=None):
    """Teacher embeddings for every utterance of df (float32 array in df order). Audio of the next batch is loaded while the GPU works;
    GPU out-of-memory -> smaller batch, and a single utterance that still does not fit runs on the CPU (same weights, fp32)."""
    import torch
    max_sec = max_sec or CFG["teacher_batch_seconds"]; lens = np.array([crop_len(r.n_frames, r.sr, crop) for r in df.itertuples()]); uids = df.uid.tolist()
    out = [None] * len(df); todo = equal_length_batches(lens, max_sec); t0 = t_log = time.time(); done = 0; n_cpu = 0
    ex = ThreadPoolExecutor(8); load = lambda b: ex.submit(lambda: [load_wave(uids[i], crop) for i in b])
    k = 0; fut = load(todo[0]) if todo else None
    while k < len(todo):
        b = todo[k]; ws = fut.result(); fut = load(todo[k + 1]) if k + 1 < len(todo) else None
        if len({len(w) for w in ws}) > 1:                # decoded length != header (cannot happen after decode_check): one by one
            todo[k:k + 1] = [[i] for i in b]; fut = load(todo[k]); continue
        try: y = T.embed(ws).cpu().numpy()
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(b) > 1:
                h = len(b) // 2; todo[k:k + 1] = [b[:h], b[h:]]; fut = load(todo[k]); continue
            y = T.embed_cpu(ws).numpy(); n_cpu += 1; log(f"    [{desc}] {uids[b[0]]} ({lens[b[0]] / SR:.0f} s) computed on the CPU (GPU memory)")
        for j, i in enumerate(b): out[i] = y[j]
        done += len(b); k += 1
        if time.time() - t_log > 60 or k == len(todo):
            el = time.time() - t0; t_log = time.time()
            log(f"    [{desc}] {done:,}/{len(df):,} utterances | {el / 60:.1f} min | ETA {el / max(done, 1) * (len(df) - done) / 60:.1f} min" + (f" | {n_cpu} on CPU" if n_cpu else ""))
    ex.shutdown(wait=False)
    return np.stack(out).astype(np.float32) if out else np.zeros((0, T.info["embed_dim"]), np.float32)

def verify_teacher(T, df, prefer_amp=None):
    """Teacher checks on REAL audio: output shape, finite values, repeatability, distinct utterances, identical-length batch == single,
    speaker separation (same-speaker vs different-speaker cosine), mixed precision vs fp32, frozen state."""
    import torch
    R = {}; D = T.info["embed_dim"]; crop = CFG["crop_seconds"]
    rows = df.sort_values("duration_s").iloc[np.linspace(0, len(df) - 1, 6).astype(int)]; ws = [load_wave(u, crop) for u in rows.uid]
    a = torch.stack([T.embed([w], amp=None)[0] for w in ws]); b = torch.stack([T.embed([w], amp=None)[0] for w in ws])
    R["output_shape"] = [1, int(a.shape[1])]; assert a.shape[1] == D
    R["repeat_max_abs_diff"] = float((a - b).abs().max()); R["finite"] = bool(torch.isfinite(a).all())
    En = torch.nn.functional.normalize(a, dim=-1); Cm = En @ En.T; off = Cm[~torch.eye(len(ws), dtype=torch.bool, device=Cm.device)]
    R["pairwise_cos_raw"] = dict(mean=float(off.mean()), max=float(off.max())); R["distinct_utterances"] = bool(off.max() < 0.99999)
    R["norm_raw"] = dict(mean=float(a.norm(dim=-1).mean()), min=float(a.norm(dim=-1).min()), max=float(a.norm(dim=-1).max()))
    w0 = ws[0]; w1 = ws[0][::-1].copy(); eb = T.embed([w0, w1], amp=None); e1 = T.embed([w1], amp=None)[0]
    R["equal_length_batch_vs_single_min_cos"] = float(min(torch.nn.functional.cosine_similarity(eb[0], a[0], dim=0), torch.nn.functional.cosine_similarity(eb[1], e1, dim=0)))
    # speaker separation on real training utterances (a trained speaker model must separate speakers clearly)
    g = df.groupby("speaker"); spk = [s for s, x in sorted(g, key=lambda t: t[0]) if len(x) >= 3][:8]
    su = pd.concat([g.get_group(s).sort_values("uid").iloc[:3] for s in spk]); se = torch.stack([T.embed([load_wave(u, crop)], amp=None)[0] for u in su.uid])
    sn = torch.nn.functional.normalize(se, dim=-1); sc = (sn @ sn.T).cpu().numpy(); lab = su.speaker.values
    iu = np.triu_indices(len(su), 1); same = lab[iu[0]] == lab[iu[1]]
    R["speaker_check"] = dict(speakers=len(spk), utterances=len(su), same_speaker_cos=float(sc[iu][same].mean()), different_speaker_cos=float(sc[iu][~same].mean()))
    R["speaker_check"]["gap"] = R["speaker_check"]["same_speaker_cos"] - R["speaker_check"]["different_speaker_cos"]
    # mixed precision: float16 (every GPU) unless a previous run of this teacher recorded float32
    cands = [prefer_amp] if prefer_amp else [CFG["teacher_amp_dtype"]]; T.amp = None
    for nm in cands:
        if nm in (None, "float32"): break
        dt = getattr(torch, nm); c = [float(torch.nn.functional.cosine_similarity(T.embed([w], amp=dt)[0], a[i], dim=0)) for i, w in enumerate(ws)]
        R["amp_vs_fp32_min_cos"] = min(c); R["amp_tested"] = nm
        if min(c) >= 0.999: T.amp = dt
    R["amp_used"] = str(T.amp).replace("torch.", "") if T.amp else "float32"
    R["frozen"] = dict(eval_mode=not T.model.training, trainable_params=sum(p.numel() for p in T.model.parameters() if p.requires_grad))
    ok = (R["repeat_max_abs_diff"] < 1e-3 and R["finite"] and R["distinct_utterances"] and R["frozen"]["trainable_params"] == 0 and R["frozen"]["eval_mode"]
          and R["equal_length_batch_vs_single_min_cos"] > 0.9999 and R["speaker_check"]["gap"] >= CFG["teacher_min_speaker_gap"])
    R["passed"] = bool(ok)
    log(f"  output (1, {D}) | repeat max|diff| {R['repeat_max_abs_diff']:.1e} | finite {R['finite']} | raw pairwise cos mean {R['pairwise_cos_raw']['mean']:.3f} "
        f"(distinct {R['distinct_utterances']}) | equal-length batch vs single min cos {R['equal_length_batch_vs_single_min_cos']:.6f}\n"
        f"  speaker check ({len(spk)} training speakers x 3 utterances): same-speaker cos {R['speaker_check']['same_speaker_cos']:.3f} vs different-speaker "
        f"{R['speaker_check']['different_speaker_cos']:.3f} (gap {R['speaker_check']['gap']:.3f}, needs >= {CFG['teacher_min_speaker_gap']})\n"
        f"  mixed precision vs fp32 min cos {R.get('amp_vs_fp32_min_cos', 'n/a')} -> using {R['amp_used']} | frozen {R['frozen']} -> {'PASS' if ok else 'FAIL'}")
    return R

def select_normalisation(E_tr, E_dv, dv):
    """Target normalisation (none / centre / standardise) chosen by EER on NON-OFFICIAL trials among held-out DEV speakers; statistics from the
    TRAINING partition only. The official test set is never used here. Ties (within 0.01 % EER) -> the simpler option."""
    sel, trials = selection_sets(dv); atomic_csv(trials, paths()["meta"] / "selection_trials_NONOFFICIAL.csv")
    rd = {u: i for i, u in enumerate(dv.uid)}; Es = E_dv[[rd[u] for u in sel.uid]]; row = {u: i for i, u in enumerate(sel.uid)}; res = []
    for n in NORMS:
        mu, sd = norm_stats(E_tr, n); s, y, miss = score_trials((Es - mu) / sd, row, trials); assert miss == 0
        res.append(dict(norm=n, eer=compute_eer(s, y)[0]))
    best = sorted(res, key=lambda r: (round(r["eer"], 4), NORMS.index(r["norm"])))[0]; spec = dict(kind="embedding", norm=best["norm"])
    log(f"  selection data: {len(E_tr):,} training utterances (statistics) | {len(sel):,} utterances of {sel.speaker.nunique()} held-out dev speakers | "
        f"{len(trials):,} NON-OFFICIAL trials ({int(trials.label.sum()):,} target)\n  EER per normalisation: " + " | ".join(f"{r['norm']} {r['eer'] * 100:.3f}%" for r in res)
        + f"\n  CHOSEN: {spec_name(spec)} (selection EER {best['eer'] * 100:.3f}%)")
    return dict(teacher=CFG["redimnet_label"], commit=None, chosen=dict(spec=spec, name=spec_name(spec), eer=best["eer"], dim=int(E_tr.shape[1])), candidates=res,
                selection_utterances=len(sel), selection_speakers=int(sel.speaker.nunique()), selection_trials=len(trials), training_utterances_for_statistics=len(E_tr),
                rule="lowest EER (cosine) on non-official trials among held-out dev speakers; ties -> simpler normalisation")

# ------------------------------------------------------------------ utterance-level teacher embedding cache (small: one vector per utterance)
CACHE_VERSION = 2
SHARD = 512

def teacher_identity(T):
    """Everything that determines the teacher's outputs (saved as teacher_info.json; caches are keyed by it)."""
    return json.loads(json.dumps(dict(T.info, amp=str(T.amp).replace("torch.", "") if T.amp else "float32"), default=str))

def cache_identity(tid, spec, crop):
    """The cache stores RAW embeddings, so it does not depend on the normalisation (spec['norm'] is applied when loading)."""
    d = dict(cache_version=CACHE_VERSION, model_id=tid["model_id"], commit=tid["commit"], weights={k: v["sha256"] for k, v in tid["weight_files"].items()},
             model_config=sha_of([json.dumps(tid["model_config"], sort_keys=True)]), crop_seconds=crop, crop_rule="centred crop" if crop else "full utterance",
             sampling_rate=SR, amp=tid["amp"], dim=tid["embed_dim"], dtype="float16", stored="RAW teacher embeddings (normalisation is applied when loading)")
    return sha_of([json.dumps(d, sort_keys=True)]), d

class EmbCache:
    """teacher_embedding_cache/<id>/ : meta.json + shard_XXXXX.npz (uids, emb fp16) + shard_XXXXX.json (sha1, count) written LAST = commit.
    Local mirror in /content; every committed shard is also copied to Drive (small files), so a recycled runtime does not recompute them."""
    def __init__(self, tid, spec, crop, drive_top=None):
        self.tid, self.spec, self.crop, self.drive = tid, spec, crop, drive_top; self.cid, self.meta = cache_identity(tid, spec, crop)
        self.local = paths()["teacher"] / "cache" / self.cid; self.local.mkdir(parents=True, exist_ok=True); self.rel = f"teacher_embedding_cache/{self.cid}"
        mf = self.local / "meta.json"
        if self.drive and self.drive.exists(f"{self.rel}/meta.json"):
            dm = json.load(open(self.drive.path(f"{self.rel}/meta.json")))
            if dm != json.loads(json.dumps(self.meta)): raise RuntimeError(f"Drive cache {self.rel}/meta.json does not match this configuration (stale or damaged): not reused")
        if mf.exists() and json.load(open(mf)) != json.loads(json.dumps(self.meta)): raise RuntimeError(f"local cache {mf} does not match this configuration")
        atomic_json(self.meta, mf)
        if self.drive and not self.drive.exists(f"{self.rel}/meta.json"): self.drive.write_json(f"{self.rel}/meta.json", self.meta)
        self._sync_from_drive()

    def _sync_from_drive(self):
        if not self.drive: return
        d = self.drive.dir / self.rel
        if not d.is_dir(): return
        n = 0
        for j in sorted(d.glob("shard_*.json")):
            if not (self.local / j.name).exists():
                npz = d / j.name.replace(".json", ".npz")
                if npz.exists() and file_sha1(npz) == json.load(open(j))["sha1"]:
                    shutil.copyfile(npz, self.local / npz.name); shutil.copyfile(j, self.local / j.name); n += 1
        if n: log(f"  teacher cache {self.cid}: {n} committed shards restored from Drive")

    def shards(self):
        out = []
        for j in sorted(self.local.glob("shard_*.json")):
            m = json.load(open(j)); npz = self.local / j.name.replace(".json", ".npz")
            if not npz.exists() or file_sha1(npz) != m["sha1"]: log(f"  damaged shard {npz.name}: moved aside"); (self.local / "corrupt").mkdir(exist_ok=True); os.replace(j, self.local / "corrupt" / j.name); continue
            out.append((npz, m))
        return out

    def done_uids(self):
        s = set()
        for npz, m in self.shards(): s.update(np.load(npz, allow_pickle=False)["uids"].tolist())
        return s

    def build(self, df, desc, T):
        if teacher_identity(T) != self.tid: raise RuntimeError("the loaded teacher differs from the identity this cache was opened with")
        have = self.done_uids(); todo = df[~df.uid.isin(have)].reset_index(drop=True)
        log(f"  [{desc}] teacher cache {self.cid} ({'centred crop <= ' + str(self.crop) + ' s' if self.crop else 'full utterances'}): "
            f"{len(df) - len(todo):,} done, {len(todo):,} to compute (committed every {SHARD * 8:,} utterances; an interruption loses at most that block)")
        k0 = len(list(self.local.glob("shard_*.json"))) + len(list((self.local / "corrupt").glob("*.json")) if (self.local / "corrupt").exists() else [])
        for s in range(0, len(todo), SHARD * 8):
            part = todo.iloc[s:s + SHARD * 8].reset_index(drop=True)
            E = run_teacher(T, part, self.crop, f"{desc}, block {s // (SHARD * 8) + 1}/{-(-len(todo) // (SHARD * 8))}")
            if not np.isfinite(E).all() or (np.linalg.norm(E, axis=1) == 0).any(): raise FloatingPointError(f"{desc}: invalid teacher embeddings")
            for q in range(0, len(part), SHARD):
                name = f"shard_{k0:05d}"; k0 += 1; npz = self.local / f"{name}.npz"; tmp = self.local / f"{name}.part.npz"
                np.savez(tmp, uids=np.array(part.uid[q:q + SHARD].tolist()), emb=E[q:q + SHARD].astype(np.float16)); os.replace(tmp, npz)
                m = dict(sha1=file_sha1(npz), count=int(min(SHARD, len(part) - q))); atomic_json(m, self.local / f"{name}.json")
                if self.drive: self.drive.write_file(npz, f"{self.rel}/{name}.npz"); self.drive.write_json(f"{self.rel}/{name}.json", m)
        return self.load(df)

    def complete(self, df): return set(df.uid) <= self.done_uids()
    def load(self, df):
        """-> float32 matrix aligned to df (exactly one valid embedding per utterance is required)."""
        rows, seen = {}, {}
        for npz, m in self.shards():
            z = np.load(npz, allow_pickle=False); u = z["uids"].tolist(); e = z["emb"]
            assert len(u) == m["count"] == len(e), f"{npz.name}: count mismatch"
            for i, x in enumerate(u):
                if x in seen: raise RuntimeError(f"teacher cache: {x} appears twice ({seen[x]}, {npz.name})")
                seen[x] = npz.name; rows[x] = e[i]
        miss = [u for u in df.uid if u not in rows]
        if miss: raise RuntimeError(f"teacher cache {self.cid}: {len(miss)} utterances missing (e.g. {miss[:3]}): run the cache step")
        E = np.stack([rows[u] for u in df.uid]).astype(np.float32) if len(df) else np.zeros((0, self.tid["embed_dim"]), np.float32)
        if not np.isfinite(E).all() or (np.linalg.norm(E, axis=1) == 0).any(): raise FloatingPointError("teacher cache contains invalid embeddings")
        return E
# ####################################################################################################
# ---- student: Zipformer-M source (icefall, unchanged) + pure-PyTorch Swoosh port (k2 replacement) + SV branch
# ####################################################################################################
# ------------------------------------------------------------------ Swoosh: pure-PyTorch port of k2's implementation (k2/python/csrc/torch/v2/autograd/swoosh.cu)
K2_SWOOSH_SRC = r"""# Pure-PyTorch port of k2's Swoosh functions (k2/python/csrc/torch/v2/autograd/swoosh.cu), written by the MT2KD ASR Stage 1 pipeline.
# Same formulas, same fp32 compute, same 8-bit stochastically-rounded derivative in the autograd path. Differences from k2:
# the random numbers for the stochastic rounding come from torch's RNG (k2: its own torch::rand call) -> statistically identical, not bitwise.
import torch
_P = {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}

def _forward(x, shift, coeff, offset):                    # k2 SwooshForward
    x = x.to(torch.float32); xo = x - shift
    l = torch.log1p(torch.exp(xo)); l = torch.where(torch.isinf(l), xo, l)
    return l - coeff * x - offset

def _forward_and_deriv(x, shift, coeff, offset):          # k2 SwooshForwardAndDeriv
    x = x.to(torch.float32); xo = x - shift
    denom = 1.0 + torch.exp(xo); d = (1.0 - coeff) - 1.0 / denom
    ld = torch.log(denom); ld = torch.where(torch.isinf(ld), xo, ld)
    return ld - coeff * x - offset, d

class _SwooshFunction(torch.autograd.Function):           # k2 SwooshFunction (autograd path)
    @staticmethod
    def forward(ctx, x, shift, coeff, offset, dropout_prob):
        x = x.to(torch.float32).contiguous(); xo = x - shift; e = torch.exp(xo)
        g = 1.0 - 1.0 / (1.0 + e)
        l = torch.log1p(e); l = torch.where(torch.isinf(l), xo, l); y = l - coeff * x - offset
        if dropout_prob != 0.0:
            drop = torch.rand_like(x) < dropout_prob
            y = torch.where(drop, torch.zeros_like(y), y * (1.0 / (1.0 - dropout_prob))); g = torch.where(drop, torch.full_like(g, coeff), g)
        g_int = (g * (255.0 / 1.005) + torch.rand_like(x)).to(torch.uint8)
        ctx.save_for_backward(g_int); ctx.coeff = coeff; ctx.dropout_prob = dropout_prob
        return y
    @staticmethod
    def backward(ctx, y_grad):
        (g,) = ctx.saved_tensors
        f = (g.to(torch.float32) * (1.005 / 255.0) - ctx.coeff) / (1.0 - ctx.dropout_prob)
        return y_grad.to(torch.float32) * f, None, None, None, None

def _swoosh(x, which, dropout_prob):
    shift, coeff, offset = _P[which]
    if not x.requires_grad:                               # k2: logaddexp path when no gradient is needed
        x = x.to(torch.float32); return torch.logaddexp(torch.zeros(1, device=x.device), x - shift) - coeff * x - offset
    return _SwooshFunction.apply(x, shift, coeff, offset, float(dropout_prob))

def swoosh_l(x, dropout_prob=0.0): return _swoosh(x, "l", dropout_prob)
def swoosh_r(x, dropout_prob=0.0): return _swoosh(x, "r", dropout_prob)
def swoosh_l_forward(x): return _forward(x, *_P["l"])
def swoosh_r_forward(x): return _forward(x, *_P["r"])
def swoosh_l_forward_and_deriv(x): return _forward_and_deriv(x, *_P["l"])
def swoosh_r_forward_and_deriv(x): return _forward_and_deriv(x, *_P["r"])
"""
STUB_FUNCS = {"swoosh_l", "swoosh_r", "swoosh_l_forward", "swoosh_r_forward", "swoosh_l_forward_and_deriv", "swoosh_r_forward_and_deriv"}

def setup_zipformer():
    """Private copy of icefall's Zipformer source (unchanged except `import k2` -> the Swoosh port). Returns source information."""
    ZDIR = CFG["zipformer_dir"]; SRC_Z = "/content/icefall/egs/librispeech/ASR/zipformer"
    if not os.path.isdir(SRC_Z):
        subprocess.run("rm -rf /content/icefall; git clone --depth 1 https://github.com/k2-fsa/icefall.git /content/icefall", shell=True, check=True)
    os.makedirs(ZDIR + "/icefall", exist_ok=True)
    for f in ["zipformer.py", "scaling.py", "subsampling.py", "encoder_interface.py", "optim.py"]:
        if not os.path.exists(f"{ZDIR}/{f}"): shutil.copy(f"{SRC_Z}/{f}", f"{ZDIR}/{f}")
    o_ = open(f"{ZDIR}/optim.py").read()                 # lhotse is only used by optim.py's self-tests: replaced by an equivalent seed function
    if "from lhotse.utils import fix_random_seed" in o_:
        o_ = o_.replace("from lhotse.utils import fix_random_seed",
                        "def fix_random_seed(s):\n    import random as _r, numpy as _n\n    _r.seed(s); _n.random.seed(s); torch.manual_seed(s)", 1)
        open(f"{ZDIR}/optim.py", "w").write(o_)
    open(ZDIR + "/icefall/__init__.py", "a").close()
    open(ZDIR + "/icefall/utils.py", "w").write(
        'import torch\n\ndef torch_autocast(device_type="cuda", **kwargs):\n    try:\n        return torch.amp.autocast(device_type, **kwargs)\n'
        '    except (AttributeError, TypeError):\n        return torch.cuda.amp.autocast(**kwargs)\n')
    if not os.path.exists(f"{ZDIR}/k2_swoosh.py") or open(f"{ZDIR}/k2_swoosh.py").read() != K2_SWOOSH_SRC: open(f"{ZDIR}/k2_swoosh.py", "w").write(K2_SWOOSH_SRC)
    t_ = open(f"{ZDIR}/scaling.py").read()
    if "\nimport k2\n" in t_: t_ = t_.replace("\nimport k2\n", "\nimport k2_swoosh as k2\n", 1); open(f"{ZDIR}/scaling.py", "w").write(t_)
    used = set(re.findall(r"\bk2\.(\w+)", t_ + open(f"{ZDIR}/zipformer.py").read() + open(f"{ZDIR}/subsampling.py").read()))
    if used - STUB_FUNCS: raise RuntimeError(f"the Zipformer source uses k2 functions the port does not provide: {used - STUB_FUNCS}")
    sys.path = [p for p in sys.path if p != ZDIR]; sys.path.insert(0, ZDIR); importlib.invalidate_caches()
    for m_ in [k for k in list(sys.modules) if k in ("zipformer", "scaling", "subsampling", "encoder_interface", "k2_swoosh", "optim") or k == "icefall" or k.startswith("icefall.")]:
        if not (getattr(sys.modules[m_], "__file__", "") or "").startswith(ZDIR): del sys.modules[m_]          # e.g. imported earlier from another folder
    try: commit = subprocess.run(["git", "-C", "/content/icefall", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: commit = "unknown"
    return dict(dir=ZDIR, icefall_commit=commit, k2_functions_used=sorted(used),
                files={f: file_sha1(f"{ZDIR}/{f}")[:12] for f in ["zipformer.py", "scaling.py", "subsampling.py", "k2_swoosh.py", "optim.py"]})

def verify_swoosh(device="cuda"):
    """Swoosh port vs reference: forward exactness (fp32/fp16 inputs, extremes), derivative, 8-bit stochastic backward (bounded, unbiased),
    the icefall modules that call it, and the real k2 if it happens to be installed."""
    import torch, torch.nn.functional as F
    setup_zipformer(); import k2_swoosh as P, scaling as S
    dev = torch.device(device if torch.cuda.is_available() else "cpu"); R = dict(source="k2/python/csrc/torch/v2/autograd/swoosh.cu (ported)")
    xs = torch.cat([torch.linspace(-60, 60, 200001), torch.tensor([-65504., -1e4, -100., 100., 1e4, 65504.])]).to(dev)
    for w, (sh, co, of) in {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}.items():
        x64 = xs.double(); ref = torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x64 - sh) - co * x64 - of; dref = torch.sigmoid(x64 - sh) - co
        tol = 2e-6 * (1 + ref.abs())
        y = getattr(P, f"swoosh_{w}_forward")(xs); yd, d = getattr(P, f"swoosh_{w}_forward_and_deriv")(xs)
        e_fwd = float(((y.double() - ref).abs() / tol).max()); e_fd = float(((yd.double() - ref).abs() / tol).max()); e_d = float((d.double() - dref).abs().max())
        y16 = getattr(P, f"swoosh_{w}_forward")(xs.half()); y16_ok = bool(torch.isfinite(y16).all()) and y16.dtype == torch.float32
        xg = xs.clone().requires_grad_(True); yg = getattr(P, f"swoosh_{w}")(xg); yg.backward(torch.ones_like(yg)); g = xg.grad.double()
        q = 1.005 / 255.0; e_q = float((g - dref).abs().max()); bias = float((g - dref).mean())
        xr = torch.randn(2_000_000, device=dev) * 6; xr.requires_grad_(True); yr = getattr(P, f"swoosh_{w}")(xr); yr.backward(torch.ones_like(yr))
        bias_r = float((xr.grad.double() - (torch.sigmoid(xr.detach().double() - sh) - co)).mean())
        e_nog = float(((getattr(P, f"swoosh_{w}")(xs).double() - ref).abs() / tol).max())
        mod = (S.SwooshL if w == "l" else S.SwooshR)(); e_mod = float(((mod(xs).double() - ref).abs() / tol).max())
        own = (S.SwooshLForward if w == "l" else S.SwooshRForward)(xs); e_own = float(((own.double() - ref).abs() / tol)[torch.isfinite(own)].max())
        R[w] = dict(forward_err_rel=e_fwd, forward_and_deriv_err_rel=e_fd, deriv_maxerr=e_d, fp16_input_finite_fp32_output=y16_ok, no_grad_path_err_rel=e_nog,
                    backward_maxerr=e_q, backward_quant_step=q, backward_bias_grid=bias, backward_bias_random=bias_r, icefall_module_err_rel=e_mod,
                    icefall_reference_SwooshForward_err_rel=e_own)
        ok = e_fwd <= 1 and e_fd <= 1 and e_d < 1e-6 and y16_ok and e_nog <= 1 and e_q <= q + 1e-6 and abs(bias_r) < 2e-4 and e_mod <= 1 and e_own <= 1
        R[w]["passed"] = ok
        log(f"  swoosh_{w}: forward rel.err {e_fwd:.2f} (<=1 = within 2e-6 relative) | deriv {e_d:.1e} | fp16 inputs finite {y16_ok} | backward |err| {e_q:.4f} "
            f"<= 8-bit step {q:.4f}, mean bias {bias_r:+.1e} | icefall SwooshL/R module {e_mod:.2f}, icefall reference {e_own:.2f} -> {'OK' if ok else 'FAIL'}")
    # the fused ActivationDropoutAndLinear path (uses forward_and_deriv in backward) vs autograd through the reference, dropout 0
    torch.manual_seed(0); lin = S.ActivationDropoutAndLinear(64, 32, bias=True, activation="SwooshL", dropout_p=0.0).to(dev)
    x = (torch.randn(50, 64, device=dev) * 4).requires_grad_(True); y = lin(x); y.square().sum().backward(); gx = x.grad.clone(); gw = lin.weight.grad.clone()
    x2 = x.detach().double().requires_grad_(True); W = lin.weight.detach().double().requires_grad_(True); b = lin.bias.detach().double()
    y2 = F.linear(torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x2 - 4) - 0.08 * x2 - 0.035, W, b); y2.square().sum().backward()
    R["fused_linear"] = dict(out_rel=float((y.double() - y2).abs().max() / y2.abs().max()), dx_rel=float((gx.double() - x2.grad).abs().max() / x2.grad.abs().max()),
                             dw_rel=float((gw.double() - W.grad).abs().max() / W.grad.abs().max()))
    R["fused_linear"]["passed"] = max(R["fused_linear"].values()) < 1e-4
    log(f"  ActivationDropoutAndLinear(SwooshL) vs float64 autograd: output {R['fused_linear']['out_rel']:.1e}, dx {R['fused_linear']['dx_rel']:.1e}, "
        f"dW {R['fused_linear']['dw_rel']:.1e} (relative)")
    try:
        import k2 as real
        if not hasattr(real, "swoosh_l_forward"): raise ImportError
        xr = torch.randn(100000, device=dev) * 8
        R["real_k2"] = dict(forward_l_maxdiff=float((real.swoosh_l_forward(xr) - P.swoosh_l_forward(xr)).abs().max()),
                            forward_r_maxdiff=float((real.swoosh_r_forward(xr) - P.swoosh_r_forward(xr)).abs().max()),
                            deriv_l_maxdiff=float((real.swoosh_l_forward_and_deriv(xr)[1] - P.swoosh_l_forward_and_deriv(xr)[1]).abs().max()))
        log(f"  compared with the installed k2: {R['real_k2']}")
    except Exception: R["real_k2"] = "k2 not installed for this PyTorch (not downgraded); compared against k2's published formulas and algorithm instead"
    R["passed"] = R["l"]["passed"] and R["r"]["passed"] and R["fused_linear"]["passed"]
    R["claim"] = ("forward and derivative: same formulas as k2 in fp32 (verified to <= 2e-6 relative against float64); autograd backward: same 8-bit "
                  "stochastic quantisation of the derivative as k2 (bounded by one step, unbiased); random numbers differ, so results are not bit-identical to a k2 run")
    return R

def lengths_to_mask(lens, T):
    import torch
    return torch.arange(T, device=lens.device)[None, :] < lens[:, None]

def set_batch_count(model, bc):
    for name, mod in model.named_modules():
        if hasattr(mod, "batch_count"): mod.batch_count = bc
        if hasattr(mod, "name"): mod.name = name

def fbank(wav):
    """Student frontend = your v1 ASR fbank (80 bins, 25 ms / 10 ms, povey window, snip_edges=False)."""
    import torch, torchaudio.compliance.kaldi as kaldi
    return kaldi.fbank(torch.from_numpy(wav)[None], num_mel_bins=80, frame_length=25.0, frame_shift=10.0, dither=0.0, energy_floor=1e-10,
                       sample_frequency=float(SR), snip_edges=False, low_freq=20.0, high_freq=-400.0, remove_dc_offset=True,
                       preemphasis_coefficient=0.97, window_type="povey", use_energy=False)

def make_batches(lens, epoch, seed, shuffle, max_sec, max_utts, idx=None):
    """Duration batches over `idx` (default: all): padded seconds <= max_sec and <= max_utts items; sorted inside shuffled pools of 1000;
    deterministic for (seed, epoch, idx)."""
    lens = np.asarray(lens); idx = np.arange(len(lens)) if idx is None else np.asarray(sorted(idx))
    rng = np.random.default_rng(seed + 1000 * epoch + (0 if len(idx) == len(lens) else 7919 * len(idx)))
    order = idx[rng.permutation(len(idx))] if shuffle else idx; out = []
    for s in range(0, len(order), 1000):
        ids = order[s:s + 1000]; ids = ids[np.argsort(lens[ids], kind="stable")]; cur, mx = [], 0
        for i in ids:
            nm = max(mx, lens[i])
            if cur and (nm * (len(cur) + 1) / SR > max_sec or len(cur) >= max_utts): out.append(cur); cur, nm = [], lens[i]
            cur.append(int(i)); mx = nm
        if cur: out.append(cur)
    if shuffle: out = [out[j] for j in rng.permutation(len(out))]
    return out

# ------------------------------------------------------------------ Zipformer-M student (unchanged architecture) + the MT2KD SV branch
ZIP_ARGS = dict(output_downsampling_factor=2, downsampling_factor=(1, 2, 4, 8, 4, 2), num_encoder_layers=(2, 2, 3, 4, 3, 2),
                encoder_dim=(192, 256, 384, 512, 384, 256), encoder_unmasked_dim=(192, 192, 256, 256, 256, 192), query_head_dim=(32,) * 6,
                pos_head_dim=(4,) * 6, value_head_dim=(12,) * 6, num_heads=(4, 4, 4, 8, 4, 4), feedforward_dim=(512, 768, 1024, 1536, 1024, 768),
                cnn_module_kernel=(31, 31, 15, 15, 15, 31), pos_dim=48, causal=False)        # identical to the project's ASR student
UNUSED_ICEFALL = re.compile(r"(^|\.)layers\.\d+\.bypass_scale$")     # Zipformer2EncoderLayer.bypass_scale: defined, never used by icefall ('# TODO: remove it')
def fbank_frames(n): return (n + 80) // 160
def embed_frames(n): return (fbank_frames(n) - 7) // 2              # Conv2dSubsampling output (~50 Hz) = the rate at which every encoder stack returns its output

def build_sv_student(teacher_dim):
    """-> (student, msv) on CPU. student = the project's Zipformer-M (same parameter names: encoder_embed.*, encoder.*);
    student.sv_features() returns the output of encoder stack `sv_block` (1-based), msv = M_SV."""
    import torch, torch.nn as nn
    info = setup_zipformer()
    from subsampling import Conv2dSubsampling
    from zipformer import Zipformer2
    import zipformer as zmod
    assert zmod.__file__.startswith(CFG["zipformer_dir"]), "Zipformer imported from the wrong folder"
    class _StopAfterBlock(Exception): pass
    class ZipformerStudent(nn.Module):
        def __init__(self):
            super().__init__(); self.encoder_embed = Conv2dSubsampling(in_channels=80, out_channels=192); self.encoder = Zipformer2(**ZIP_ARGS)
            self.sv_index = CFG["sv_block"] - 1; self.sv_dim = int(ZIP_ARGS["encoder_dim"][self.sv_index])
        def forward(self, feats, feat_lens):                       # full encoder (as in the ASR project): (B, T/4, 512) at 25 Hz
            x, xl = self.encoder_embed(feats, feat_lens)
            enc, el = self.encoder(x.permute(1, 0, 2), xl, ~lengths_to_mask(xl, x.size(1)))
            return enc.permute(1, 0, 2), el
        def sv_features(self, feats, feat_lens, early_exit=True):
            """Output of encoder.encoders[sv_index] (the THIRD stack for sv_block=3): (B, T50, 384), valid lengths T50 = embed frames.
            Taken with a forward hook inside icefall's own Zipformer2.forward (no re-implementation); with early_exit the later stacks are not run."""
            x, xl = self.encoder_embed(feats, feat_lens); cap = {}
            def hook(m, i, o):
                cap["y"] = o
                if early_exit: raise _StopAfterBlock()
            h = self.encoder.encoders[self.sv_index].register_forward_hook(hook)
            try: self.encoder(x.permute(1, 0, 2), xl, ~lengths_to_mask(xl, x.size(1)))
            except _StopAfterBlock: pass
            finally: h.remove()
            y = cap["y"].permute(1, 0, 2)
            assert y.size(1) == x.size(1) and y.size(2) == self.sv_dim, (tuple(y.shape), tuple(x.shape))
            return y, xl
        def sv_params(self):
            """Parameters that the SV branch actually uses: the frontend and stacks 1..sv_block."""
            return list(self.encoder_embed.named_parameters(prefix="encoder_embed")) + \
                   [(f"encoder.encoders.{i}.{n}", p) for i in range(self.sv_index + 1) for n, p in self.encoder.encoders[i].named_parameters()]
    class MSV(nn.Module):
        """MT2KD M_SV: 'an attentive pooling module consisting of two 1D convolution layers' + 'a linear layer projecting the speaker embeddings'.
        Here: Conv1d(C->A,1) -> tanh -> Conv1d(A->C,1) -> softmax over VALID frames (channel-wise weights) -> attentive mean and std (2C)
        -> Linear(2C -> teacher dim). Computed in fp32."""
        def __init__(self, C, A, D):
            super().__init__(); self.att = nn.Sequential(nn.Conv1d(C, A, 1), nn.Tanh(), nn.Conv1d(A, C, 1)); self.proj = nn.Linear(2 * C, D)
            self.C, self.A, self.D = C, A, D
        def forward(self, y, lens):
            with torch.autocast("cuda", enabled=False):
                x = y.float().transpose(1, 2); m = lengths_to_mask(lens, x.size(2))[:, None, :]
                e = self.att(x).masked_fill(~m, float("-inf")); w = torch.softmax(e, dim=2)
                mu = (w * x).sum(2); var = (w * x * x).sum(2) - mu * mu
                return self.proj(torch.cat([mu, var.clamp(min=1e-5).sqrt()], 1))
    s = ZipformerStudent(); s.source_info = info
    return s, MSV(s.sv_dim, CFG["msv_attention_channels"], teacher_dim)

def inspect_student(student, msv):
    enc = student.encoder; stacks = []
    for i, m in enumerate(enc.encoders):
        inner = getattr(m, "encoder", m); stacks.append(dict(index=i + 1, type=type(m).__name__, downsample=int(enc.downsampling_factor[i]), layers=len(inner.layers),
                                                             dim=int(enc.encoder_dim[i]), params=sum(p.numel() for p in m.parameters())))
    assert [s["layers"] for s in stacks] == list(ZIP_ARGS["num_encoder_layers"]) and [s["dim"] for s in stacks] == list(ZIP_ARGS["encoder_dim"])
    k = student.sv_index; sv_enc = sum(p.numel() for _, p in student.sv_params()); msv_p = sum(p.numel() for p in msv.parameters())
    R = dict(source=student.source_info, stacks=stacks, total_params=sum(p.numel() for p in student.parameters()), sv_block=k + 1, sv_block_dim=stacks[k]["dim"],
             sv_block_internal_rate_hz=50.0 / stacks[k]["downsample"], sv_block_output_rate_hz=50.0, sv_encoder_params=sv_enc, msv_params=msv_p,
             paper_table_vii_sv_params_block3=18.9e6,
             note=(f"SV branch = Conv2dSubsampling + stacks 1..{k + 1} ({sv_enc / 1e6:.2f}M) + M_SV ({msv_p / 1e6:.2f}M); MT2KD Table VII lists 18.9M SV parameters "
                   f"for the third block with a 192-D teacher (its M_SV projection is smaller)"))
    log(f"  Zipformer-M (icefall {student.source_info['icefall_commit'][:10]}): stacks " + ", ".join(f"#{s['index']} {s['layers']}x{s['dim']}/{s['downsample']}" for s in stacks)
        + f" | {R['total_params'] / 1e6:.2f}M params")
    log(f"  SV branch: stack #{k + 1} = encoder.encoders[{k}] ({stacks[k]['layers']} layers, {stacks[k]['dim']}-D, runs at {R['sv_block_internal_rate_hz']} Hz, "
        f"returned at 50 Hz) | {R['note']}")
    return R

class SVData:
    def __init__(self, df, targets, crop): self.u = df.uid.tolist(); self.t = targets; self.crop = crop
    def __len__(self): return len(self.u)
    def __getitem__(self, i):
        import torch
        w = load_wave(self.u[i], self.crop)
        return dict(i=i, n=len(w), fb=fbank(w), t=None if self.t is None else torch.from_numpy(self.t[i]))

def sv_collate(b):
    import torch
    from torch.nn.utils.rnn import pad_sequence
    return dict(i=torch.tensor([x["i"] for x in b]), n=torch.tensor([x["n"] for x in b]),
                fb=pad_sequence([x["fb"] for x in b], batch_first=True, padding_value=math.log(1e-10)), fb_lens=torch.tensor([x["fb"].size(0) for x in b]),
                t=None if b[0]["t"] is None else torch.stack([x["t"] for x in b]))

def sv_loader(df, targets, batches, crop, num_workers, seed=0):
    import torch
    def wi(_): torch.set_num_threads(1)
    return torch.utils.data.DataLoader(SVData(df, targets, crop), batch_sampler=batches, num_workers=num_workers, collate_fn=sv_collate, pin_memory=True,
                                       worker_init_fn=wi, prefetch_factor=4 if num_workers else None, generator=torch.Generator().manual_seed(int(seed)),
                                       timeout=600 if num_workers else 0)            # a stuck/dead worker raises after 10 min instead of hanging
# ####################################################################################################
# ---- KD objective, evaluation, optimiser, resumable training
# ####################################################################################################
RESUME_KEYS = {"sig", "student", "msv", "opt", "scaler", "scheduler", "state", "rng_cpu", "rng_cuda", "rng_np", "rng_py"}
class SimulatedDisconnect(Exception):
    """Raised by the resume test to imitate a runtime that dies: nothing is saved at that moment."""

def kd_loss(v, t, eps):
    """MT2KD Eq. 8: 1 - CosineSim(teacher, student) between UN-normalised vectors; fp32; eps guards the denominator."""
    v = v.float(); t = t.float(); cos = (v * t).sum(-1) / (v.norm(dim=-1) * t.norm(dim=-1)).clamp(min=eps)
    return (1.0 - cos).mean(), cos

def sv_forward(student, msv, b, dev, amp):
    import torch
    fb, fl = b["fb"].to(dev, non_blocking=True), b["fb_lens"].to(dev)
    with torch.autocast("cuda", dtype=amp): y, yl = student.sv_features(fb, fl)
    return msv(y, yl)

def student_embeddings(student, msv, df, crop, dev, amp, max_sec=600.0):
    """Student SV embeddings (eval mode, no gradient) in df order."""
    import torch
    lens = np.array([crop_len(r.n_frames, r.sr, crop) for r in df.itertuples()]); out = np.zeros((len(df), msv.D), np.float32)
    was = student.training; student.eval(); msv.eval()
    with torch.no_grad():
        for b in sv_loader(df, None, make_batches(lens, 0, 0, False, max_sec, CFG["max_batch_utts"]), crop, CFG["num_workers"]):
            out[b["i"].numpy()] = sv_forward(student, msv, b, dev, amp).float().cpu().numpy()
    student.train(was); msv.train(was); return out

def sv_evaluate(student, msv, C, dev, amp, df=None, targets=None, trials=None):
    """KD metrics on a manifest (no parameter updates) + EER on NON-OFFICIAL validation trials (monitoring only)."""
    import torch
    df = C["dv"] if df is None else df; targets = C["T_dv"] if targets is None else targets; trials = C.get("sel_trials") if trials is None else trials
    E = student_embeddings(student, msv, df, CFG["crop_seconds"], dev, amp); inv = int((~np.isfinite(E).all(1)).sum())
    loss, cos = kd_loss(torch.from_numpy(np.nan_to_num(E)), torch.from_numpy(targets), CFG["cos_eps"])
    tn = l2n(targets); base = float(np.linalg.norm(tn.mean(0)))           # best possible cosine of ONE constant vector = |mean of unit targets|
    r = dict(val_loss=float(loss), val_cos=float(cos.mean()), val_cos_constant_baseline=base, teacher_norm=float(np.linalg.norm(targets, axis=1).mean()),
             student_norm=float(np.linalg.norm(np.nan_to_num(E), axis=1).mean()), invalid_embeddings=inv, val_utts=len(df))
    if trials is not None and len(trials):
        s, y, miss = score_trials(np.nan_to_num(E), {u: i for i, u in enumerate(df.uid)}, trials)
        if len(s) and y.any() and (~y.astype(bool)).any(): r["val_eer_nonofficial"] = compute_eer(s, y)[0]
    return r, E

def make_optimizer(student, msv, cfg):
    """scaled_adam: icefall ScaledAdam (two named-parameter groups: Zipformer frontend+stacks 1..3, M_SV) + Eden; adamw: AdamW (scheduler None)."""
    import torch
    enc = [(n, p) for n, p in student.sv_params() if p.requires_grad]; head = [("msv." + n, p) for n, p in msv.named_parameters()]
    if cfg["optimizer"] == "scaled_adam":
        from optim import ScaledAdam, Eden
        opt = ScaledAdam([dict(named_params=enc, lr=cfg["base_lr"]), dict(named_params=head, lr=cfg["base_lr"] * cfg["msv_lr_scale"])],
                         lr=cfg["base_lr"], clipping_scale=cfg["clipping_scale"])
        return opt, Eden(opt, cfg["lr_batches"], cfg["lr_epochs"], warmup_batches=cfg["eden_warmup_batches"], warmup_start=cfg["eden_warmup_start"])
    if cfg["optimizer"] != "adamw": raise ValueError(cfg["optimizer"])
    ps = [p for _, p in enc]; hp = [p for _, p in head]
    return torch.optim.AdamW([dict(params=[p for p in ps if p.ndim >= 2], weight_decay=cfg["weight_decay"]), dict(params=[p for p in ps if p.ndim < 2], weight_decay=0.0),
                              dict(params=hp, weight_decay=0.0, lr=cfg["adamw_lr"] * cfg["msv_lr_scale"])], lr=cfg["adamw_lr"]), None

def freeze_unused(student):
    """Only the frontend and stacks 1..sv_block are on the SV path; the later stacks keep their initial values and get no optimiser state."""
    used = {n for n, _ in student.sv_params()}
    for n, p in student.named_parameters(): p.requires_grad_(n in used)
    return sum(p.numel() for p in student.parameters() if p.requires_grad)

def new_models(C, dev):
    set_seed(CFG["seed"]); s, m = build_sv_student(C["dim"]); freeze_unused(s); return s.to(dev), m.to(dev)

def _load_resumable(f):
    import torch
    try: ck = torch.load(f, map_location="cpu", weights_only=False)
    except Exception as e: return None, f"unreadable ({type(e).__name__}: {str(e)[:120]})"
    if not isinstance(ck, dict) or not RESUME_KEYS <= set(ck): return None, "not a complete resumable checkpoint"
    return ck, ""

def _position(S): return (S["update"], S["epoch"], len(S["seen"]))

def memory_probe(student, msv, C, dev, amp, max_sec):
    import torch
    L = int(round(CFG["crop_seconds"] * SR)); total = torch.cuda.get_device_properties(0).total_memory
    while max_sec >= 20:
        nb = max(1, min(CFG["max_batch_utts"], int(max_sec * SR // L))); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); peak = float("nan")
        try:
            T_ = fbank_frames(L); b = dict(fb=torch.randn(nb, T_, 80), fb_lens=torch.full((nb,), T_)); v = sv_forward(student, msv, b, dev, amp)
            loss, _ = kd_loss(v, torch.randn(nb, msv.D, device=dev), 1e-8); loss.backward(); peak = torch.cuda.max_memory_allocated(); ok = peak < CFG["mem_fraction"] * total
        except torch.OutOfMemoryError: ok = False
        for p in list(student.parameters()) + list(msv.parameters()): p.grad = None
        torch.cuda.empty_cache()
        log(f"  memory probe: {nb} x {L / SR:.0f} s = {nb * L / SR:.0f} s padded -> peak {peak / 2**30:.2f} GiB of {total / 2**30:.1f} GiB {'OK' if ok else '-> reducing'}")
        if ok: return max_sec
        max_sec *= 0.8
    raise RuntimeError("even a 20 s batch does not fit in GPU memory")

def batch_count_of(update, MBS, cfg, progress):
    ic = update * MBS * cfg["accum_steps"] / cfg["ref_duration"]          # icefall: batches of the reference duration seen so far
    return max(ic, 20000.0 * progress) if cfg["batch_count_mode"] == "progress" else ic

def sv_train(mode, C, P=None, drive=None, overrides=None, interrupt_after_update=None, probe=True):
    """Resumes automatically from the most advanced VALID checkpoint among: local latest, Drive checkpoint_latest.pt (+ an upload whose rename was interrupted)."""
    import torch
    P = P or paths(mode); ensure_dirs(P); dev = torch.device("cuda"); amp = amp_dtype(); cfg = dict(CFG, **MODES[mode]); cfg.update(overrides or {})
    LATEST, BEST = P["latest"] / "checkpoint_latest.pt", P["best"] / "checkpoint_best.pt"; N = len(C["tr"])
    lens = np.array([crop_len(r.n_frames, r.sr, CFG["crop_seconds"]) for r in C["tr"].itertuples()])
    student, msv = new_models(C, dev); params = [p for _, p in student.sv_params()] + list(msv.parameters()); opt, sch = make_optimizer(student, msv, cfg)
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 12, enabled=amp == torch.float16)
    sig_keys = ["seed", "optimizer", "accum_steps", "max_epochs", "max_batch_utts", "crop_seconds", "sv_block", "msv_attention_channels", "cos_eps", "batch_count_mode", "msv_lr_scale"] + \
               (["base_lr", "lr_batches", "lr_epochs", "eden_warmup_batches", "eden_warmup_start", "clipping_scale"] if cfg["optimizer"] == "scaled_adam"
                else ["adamw_lr", "weight_decay", "warmup_updates", "min_lr_frac", "grad_clip"])
    SIG = sha_of([json.dumps({k: cfg[k] for k in sig_keys}, sort_keys=True, default=str), C["manifest_sha"], C["dev_sha"], C["cache_ids"], C["norm_sha"], CODE_VERSION])
    S = dict(epoch=0, batch=0, update=0, best=None, es_counter=0, train_hist=[], val_hist=[], elapsed=0.0, skips=0, finished=None, seen=[], plan_remaining=None,
             ep_sum=0.0, ep_n=0, max_batch_seconds=None, sessions=0, resumes=[], peak_gpu_gib=0.0)
    cands = [("local disk", LATEST)] + ([("Google Drive", f) for f in drive.candidates("checkpoint_latest.pt")] if drive else []); chosen = None
    for src, f in cands:
        if not Path(f).exists(): continue
        ck, why = _load_resumable(f)
        if ck is None: log(f"  ignored {src} file {f}: {why} (e.g. an interrupted save or upload)"); continue
        if ck["sig"] != SIG and src == "local disk":
            dst = P["ckpt"].parent / f"{P['ckpt'].name}_other_config_{time.strftime('%Y%m%d_%H%M%S')}"; del ck; os.replace(P["ckpt"], dst); ensure_dirs(P)
            log(f"  local checkpoints of a DIFFERENT configuration moved aside: {dst}"); continue
        if ck["sig"] != SIG:
            raise RuntimeError(f"{f} ({src}) is a checkpoint of a DIFFERENT configuration / data / teacher cache (sig {ck['sig']}, this run {SIG}). Left untouched: "
                               f"move it out of {drive.dir} to start a new run, or restore the settings it was made with.")
        k = _position(ck["state"]); log(f"  valid checkpoint on {src}: {Path(f).name}, epoch {ck['state']['epoch'] + 1}, batch {ck['state']['batch']}, global step {k[0]}")
        if chosen is None or k > chosen[2]: chosen = (src, f, k, ck)
        else: del ck
    if chosen:
        src, f, k, ck = chosen
        if src == "Google Drive" and str(f).endswith(".tmp"): f = drive.promote_tmp("checkpoint_latest.pt")
        student.load_state_dict(ck["student"], strict=True); msv.load_state_dict(ck["msv"], strict=True); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sch is not None and ck["scheduler"].get("eden"): sch.load_state_dict(ck["scheduler"]["eden"])
        S = ck["state"]; torch.set_rng_state(ck["rng_cpu"].cpu()); torch.cuda.set_rng_state(ck["rng_cuda"].cpu()); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
        log("\n" + "#" * 70 + "\n#  RESUMING FROM CHECKPOINT\n" + "#" * 70)
        log(f"  source {src}: {f}\n  epoch {S['epoch'] + 1}/{cfg['max_epochs']} | batch position {S['batch']} ({len(S['seen'])}/{N} utterances of this epoch done) | "
            f"global optimizer step {S['update']} | best val KD loss {S['best']['val_loss'] if S['best'] else None} | early-stop counter {S['es_counter']}/{cfg['patience']}")
        S.setdefault("resumes", []).append(dict(source=src, update=S["update"], epoch=S["epoch"], batch=S["batch"], seen=len(S["seen"]), time=time.strftime("%Y-%m-%d %H:%M:%S")))
        if src == "Google Drive": shutil.copyfile(f, str(LATEST) + ".part"); os.replace(str(LATEST) + ".part", LATEST)
        del ck
        if cfg["max_batch_seconds"] < S["max_batch_seconds"]:
            log(f"  max_batch_seconds lowered {S['max_batch_seconds']:.0f} -> {cfg['max_batch_seconds']:.0f}: the remaining utterances of this epoch are re-batched")
            S["max_batch_seconds"] = cfg["max_batch_seconds"]; S["plan_remaining"] = sorted(set(range(N)) - set(S["seen"])); S["batch"] = 0
    else: log("\n" + "#" * 70 + "\n#  STARTING NEW TRAINING RUN\n" + "#" * 70)
    if drive: drive.clean_stale_tmp()
    # ---- the best checkpoint is never replaced by a worse model (a newer best.pt from work lost in an interruption is a floor)
    lb = best_meta(BEST) if BEST.exists() else None
    if lb and lb.get("sig") != SIG: dst = BEST.with_name(f"best_other_config_{time.strftime('%Y%m%d_%H%M%S')}.pt"); os.replace(BEST, dst); lb = None
    db = drive.best_info() if drive else None
    if db and "error" in db: log(f"  Drive checkpoint_best.pt unreadable ({db['error']}): will be replaced by the next best model"); db = None
    if db and db.get("sig") != SIG: raise RuntimeError(f"{drive.dir / 'checkpoint_best.pt'} belongs to a different configuration; left untouched. Move it away to start a new run.")
    floor = None
    for src_, m in (("Google Drive", db), ("local disk", lb)):
        if m and (S["best"] is None or m["val_loss"] < S["best"]["val_loss"]) and (floor is None or m["val_loss"] < floor["val_loss"]): floor = dict(m, source=src_)
    if floor: log(f"  existing best checkpoint on {floor['source']} (val KD loss {floor['val_loss']:.5f}) is only replaced by a better model")
    if drive:
        if db and (lb is None or lb["val_loss"] > db["val_loss"]): shutil.copyfile(drive.dir / "checkpoint_best.pt", str(BEST) + ".part"); os.replace(str(BEST) + ".part", BEST)
        elif lb and (db is None or lb["val_loss"] < db["val_loss"]): drive.push(BEST, "checkpoint_best.pt")
    S["sessions"] += 1
    if drive and not drive.fake:
        n_all = sum(p.numel() for p in student.parameters()) + sum(p.numel() for p in msv.parameters()); n_tr = sum(p.numel() for p in params)
        drive.check_space(2 * (4.05 * n_all + 8.2 * n_tr + 4.05 * n_all) + 4.05 * n_all + 50e6, "checkpoints (latest + best, each via a temporary copy, + exported weights)")
    if S["max_batch_seconds"] is None: S["max_batch_seconds"] = memory_probe(student, msv, C, dev, amp, cfg["max_batch_seconds"]) if probe else cfg["max_batch_seconds"]
    MBS = S["max_batch_seconds"]
    def lr_at(update, prog):
        w = cfg["adamw_lr"] * min(1.0, (update + 1) / cfg["warmup_updates"]); cosv = cfg["min_lr_frac"] + (1 - cfg["min_lr_frac"]) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
        return min(w, cfg["adamw_lr"] * cosv) if update < cfg["warmup_updates"] else cfg["adamw_lr"] * cosv
    progress = lambda: (S["epoch"] + len(S["seen"]) / N) / cfg["max_epochs"]
    def cur_lr():
        if sch is None:
            lr = lr_at(S["update"], progress()); opt.param_groups[0]["lr"] = opt.param_groups[1]["lr"] = lr; opt.param_groups[2]["lr"] = lr * cfg["msv_lr_scale"]; return lr
        sch.step_epoch(S["epoch"]); sch.step_batch(S["update"] + 1); return sch.get_last_lr()[0]
    run_cfg = dict(code_version=CODE_VERSION, sig=SIG, mode=mode, cfg={k: cfg[k] for k in sorted(cfg) if not callable(cfg[k])}, max_batch_seconds_effective=MBS,
                   teacher=dict(model_id=C["tid"]["model_id"], commit=C["tid"]["commit"], representation=spec_name(C["spec"]), spec=C["spec"], dim=C["dim"],
                                normalisation=f"{C['spec']['norm']} ({mode} training-set statistics, label-free), sha {C['norm_sha']}"),
                   caches=C["cache_ids"], data=dict(train_utts=N, train_speakers=int(C["tr"].speaker.nunique()), train_hours=round(float(C["tr"].duration_s.sum() / 3600), 3),
                                                     train_manifest_sha=C["manifest_sha"], val_utts=len(C["dv"]), val_manifest_sha=C["dev_sha"]),
                   preprocessing=dict(student="80-D kaldi fbank (25 ms / 10 ms, povey, snip_edges=False) of the same centred <= crop_seconds segment the teacher saw",
                                      teacher="ReDimNet2 front end (signal normalisation, pre-emphasis, 72 log-mel, mean normalisation); 16 kHz mono"),
                   student=dict(model="Zipformer-M (icefall)", zip_args=ZIP_ARGS, sv_block=CFG["sv_block"], sv_block_dim=student.sv_dim,
                                msv="attentive pooling (Conv1d-tanh-Conv1d, masked softmax) -> attentive mean+std -> Linear", trainable_params=int(sum(p.numel() for p in params))),
                   loss="1 - CosineSim(v_teacher, M_SV(R_student^(3))) (MT2KD Eq. 8); no speaker labels")
    atomic_json(run_cfg, P["logs"] / "run_config.json")
    if drive:
        try:
            import yaml; drive.write_text("config.yaml", yaml.safe_dump(json.loads(json.dumps(run_cfg, default=str)), sort_keys=False))
        except Exception as e: log(f"  (config.yaml not written: {e})")
    log(f"  train {N:,} utts ({C['tr'].speaker.nunique()} speakers) | val {len(C['dv']):,} | max {cfg['max_epochs']} epochs | "
        + (f"ScaledAdam + Eden (base lr {cfg['base_lr']}, lr_batches {cfg['lr_batches']}, lr_epochs {cfg['lr_epochs']}, warm-up {cfg['eden_warmup_batches']})" if sch else f"AdamW {cfg['adamw_lr']}")
        + f" | {str(amp).split('.')[-1]} | batches <= {MBS:.0f} s | teacher {C['dim']}-D | batch_count {cfg['batch_count_mode']}")
    t_ref = [time.time()]; last_local = [time.time()]; last_drive = [time.time()]; t_session = time.time(); budget = cfg.get("max_session_hours")
    def sched_state(): return dict(update=S["update"], progress=progress(), lr=float(opt.param_groups[0]["lr"]), eden=sch.state_dict() if sch is not None else None)
    def save_latest(to_drive=False):
        S["elapsed"] += time.time() - t_ref[0]; t_ref[0] = time.time()
        atomic_torch_save(dict(sig=SIG, student=student.state_dict(), msv=msv.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), scheduler=sched_state(), state=S,
                               rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state(), rng_np=np.random.get_state(), rng_py=random.getstate(), cfg=run_cfg["cfg"],
                               teacher=run_cfg["teacher"], caches=C["cache_ids"], manifests=dict(train=C["manifest_sha"], val=C["dev_sha"]), code_version=CODE_VERSION), LATEST)
        atomic_csv(pd.DataFrame(S["train_hist"]), P["logs"] / "training_log.csv"); last_local[0] = time.time()
        if to_drive and drive: drive.push(LATEST, "checkpoint_latest.pt"); drive.write_csv("training_log.csv", pd.DataFrame(S["train_hist"])); last_drive[0] = time.time()
    def write_val():
        V = pd.DataFrame(S["val_hist"]); atomic_csv(V, P["logs"] / "validation_log.csv")
        if drive: drive.write_csv("validation_log.csv", V)
    student.train(); msv.train(); win = []; pending = []; stop = None
    try:
        while S["epoch"] < cfg["max_epochs"] and not S["finished"]:
            ep = S["epoch"]; batches = make_batches(lens, ep, cfg["seed"], True, MBS, cfg["max_batch_utts"], idx=S["plan_remaining"])
            k0 = S["batch"]; t_ep = time.time(); torch.cuda.reset_peak_memory_stats(); nb = [len(x) for x in batches]
            log(f"epoch {ep + 1}: {len(batches)} batches, {min(nb)}-{max(nb)} utterances per batch" + (f" | continuing at batch {k0}" if k0 or S["plan_remaining"] is not None else ""))
            tb = time.time()
            for k, b in enumerate(sv_loader(C["tr"], C["T_tr"], batches[k0:], CFG["crop_seconds"], cfg["num_workers"], seed=cfg["seed"] + ep), start=k0):
                set_batch_count(student, batch_count_of(S["update"], MBS, cfg, progress()))
                v = sv_forward(student, msv, b, dev, amp); loss, cos = kd_loss(v, b["t"].to(dev), cfg["cos_eps"]); n_inv = int((~torch.isfinite(v).all(-1)).sum())
                if not torch.isfinite(loss) or n_inv:
                    S["skips"] += 1; log(f"  non-finite loss / {n_inv} invalid student embeddings at step {S['update']} (batch skipped, {S['skips']} in a row)")
                    opt.zero_grad(set_to_none=True)
                    for kk in pending: S["seen"] += batches[kk]
                    pending = []; S["seen"] += batches[k]; S["batch"] = k + 1
                    if S["skips"] > cfg["max_consecutive_skips"]: save_latest(True); raise FloatingPointError("too many non-finite losses; state saved")
                    continue
                scaler.scale(loss / cfg["accum_steps"]).backward(); pending.append(k)
                win.append((float(loss), float(cos.mean()), float(v.detach().norm(dim=-1).mean()), float(b["t"].norm(dim=-1).mean()), len(b["i"]), int(b["n"].sum())))
                if (k + 1) % cfg["accum_steps"] and k + 1 < len(batches): continue
                lr = cur_lr(); scaler.unscale_(opt); gn = float(torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"] if sch is None else float("inf")))
                if math.isfinite(gn): scaler.step(opt); S["skips"] = 0
                else:
                    S["skips"] += 1; log(f"  WARNING: non-finite gradient norm at step {S['update']} (update skipped, {S['skips']} in a row)")
                    if scaler.is_enabled(): scaler.step(opt)
                    if S["skips"] > cfg["max_consecutive_skips"]: save_latest(True); raise FloatingPointError("too many non-finite gradients; state saved")
                if math.isfinite(gn) and gn > 1e3: log(f"  WARNING: large gradient norm {gn:.1f} at step {S['update']}")
                scaler.update(); opt.zero_grad(set_to_none=True); S["update"] += 1
                for kk in pending: S["seen"] += batches[kk]
                for w_ in win[-len(pending):]: S["ep_sum"] += w_[0] * w_[4]; S["ep_n"] += w_[4]
                pending = []; S["batch"] = k + 1; u = S["update"]
                if u % cfg["log_every"] == 0 or u == 1:
                    w = np.array(win); win = []; dt = time.time() - tb; tb = time.time(); pk = torch.cuda.max_memory_allocated() / 2**30; S["peak_gpu_gib"] = max(S["peak_gpu_gib"], pk)
                    rec = dict(update=u, epoch=ep + 1, batch=k + 1, loss=float(np.average(w[:, 0], weights=w[:, 4])), cos=float(np.average(w[:, 1], weights=w[:, 4])),
                               student_norm=float(w[:, 2].mean()), teacher_norm=float(w[:, 3].mean()), lr=lr, grad_norm=gn, batch_count=batch_count_of(u, MBS, cfg, progress()),
                               gpu_peak_gib=round(pk, 3), sec_per_batch=dt / len(w), utts_per_sec=float(w[:, 4].sum() / max(dt, 1e-9)), audio_sec_per_sec=float(w[:, 5].sum() / SR / max(dt, 1e-9)))
                    S["train_hist"].append(rec); el = time.time() - t_ep
                    log(f"[ep {ep + 1}/{cfg['max_epochs']} | b {k + 1}/{len(batches)} | step {u}] KD loss {rec['loss']:.4f} | cos {rec['cos']:.4f} | |v_s| {rec['student_norm']:.2f} "
                        f"|v_t| {rec['teacher_norm']:.2f} | |g| {gn:.2f} | lr {lr:.2e} | {rec['utts_per_sec']:.0f} utt/s | GPU {pk:.1f} GiB | epoch ETA "
                        f"{el / max(k + 1 - k0, 1) * (len(batches) - k - 1) / 60:.1f} min")
                if drive and (u % cfg["drive_ckpt_every_updates"] == 0 or time.time() - last_drive[0] > 60 * cfg["drive_ckpt_every_minutes"]):
                    save_latest(True); log(f"  checkpoint saved at step {u} (epoch {ep + 1}, batch {k + 1}/{len(batches)}): local + Google Drive")
                elif u % cfg["local_ckpt_every_updates"] == 0 or time.time() - last_local[0] > 60 * cfg["ckpt_every_minutes"]: save_latest(); log(f"  checkpoint saved at step {u} (local)")
                if interrupt_after_update is not None and u == interrupt_after_update: raise SimulatedDisconnect(f"simulated runtime loss after step {u}")
                if budget and time.time() - t_session > 3600 * budget: save_latest(True); write_val(); stop = f"session time budget of {budget} h reached"; break
            if stop: break
            assert len(set(S["seen"])) == N == len(S["seen"]), f"epoch {ep + 1} saw {len(set(S['seen']))} distinct of {N} utterances ({len(S['seen'])} total)"
            v_, _ = sv_evaluate(student, msv, C, dev, amp); tr_loss = S["ep_sum"] / max(S["ep_n"], 1)
            v_.update(epoch=ep + 1, update=S["update"], train_loss=tr_loss, lr=float(opt.param_groups[0]["lr"]), epoch_minutes=(time.time() - t_ep) / 60)
            improved = S["best"] is None or v_["val_loss"] < S["best"]["val_loss"] - cfg["min_delta"]
            if improved and floor and v_["val_loss"] >= floor["val_loss"]:
                S["best"] = {k: floor[k] for k in ("val_loss", "epoch", "update")}; S["es_counter"] = 0; improved = False; floor = None
            elif improved:
                S["best"] = dict(val_loss=v_["val_loss"], val_cos=v_["val_cos"], epoch=ep + 1, update=S["update"], val_eer_nonofficial=v_.get("val_eer_nonofficial")); S["es_counter"] = 0; floor = None
                atomic_torch_save(dict(student=student.state_dict(), msv=msv.state_dict(), val=v_, epoch=ep + 1, update=S["update"], sig=SIG, cfg=run_cfg["cfg"], teacher=run_cfg["teacher"],
                                       caches=C["cache_ids"], zip_args=ZIP_ARGS, sv_block=CFG["sv_block"], msv_dims=dict(C=msv.C, A=msv.A, D=msv.D), code_version=CODE_VERSION), BEST)
                if drive: drive.push(BEST, "checkpoint_best.pt")
            else: S["es_counter"] += 1
            v_["best"] = improved; v_["es_counter"] = S["es_counter"]; S["val_hist"].append(v_)
            log(f"=== epoch {ep + 1}: train KD loss {tr_loss:.4f} | VAL KD loss {v_['val_loss']:.4f} | val cos {v_['val_cos']:.4f} (one constant vector: {v_['val_cos_constant_baseline']:.4f}) | "
                f"val EER (non-official trials) {v_.get('val_eer_nonofficial', float('nan')) * 100:.2f}% | invalid {v_['invalid_embeddings']} | step {S['update']} | "
                f"{v_['epoch_minutes']:.1f} min" + (" | NEW BEST" if improved else f" | no improvement ({S['es_counter']}/{cfg['patience']})") + " ===")
            S["epoch"] = ep + 1; S["batch"] = 0; S["seen"] = []; S["plan_remaining"] = None; S["ep_sum"] = 0.0; S["ep_n"] = 0
            if S["es_counter"] >= cfg["patience"]: S["finished"] = f"early stopping after epoch {ep + 1} (best epoch {S['best']['epoch']})"
            elif S["epoch"] >= cfg["max_epochs"]: S["finished"] = f"max_epochs {cfg['max_epochs']} reached"
            save_latest(True); write_val(); plots(P, drive)
    except torch.OutOfMemoryError:
        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); save_latest(True); write_val()
        if drive: drive.wait()
        raise RuntimeError(f"CUDA out of memory. State saved at step {S['update']}. Set CFG['max_batch_seconds'] lower (effective {MBS:.0f}) and re-run.")
    except KeyboardInterrupt:
        save_latest(True); write_val(); log("interrupted: state saved; waiting for the Google Drive upload ...")
        if drive: drive.wait()
        raise
    if stop: log(f"{stop}: state saved at step {S['update']}. Re-run the cell to continue.")
    del student, msv, opt, params; torch.cuda.empty_cache()
    if drive: drive.wait()
    return dict(finished=S["finished"], stop=stop, state=S, P=P)

# ####################################################################################################
# ---- resume test and learning check (short runs on tiny subsets; a LOCAL stand-in folder replaces Drive)
# ####################################################################################################
DRIVE_MODE_FILES = ("checkpoint_latest.pt", "checkpoint_best.pt", "student_encoder.pt", "sv_projection.pt", "training_log.csv", "validation_log.csv",
                    "final_report.json", "training_curves.png", "config.yaml", "train_manifest.csv")

def tiny_context(C, n_train, n_dev, dev_from_train=False):
    trs = C["tr"].sort_values(["duration_s", "uid"]); it = np.linspace(0, len(trs) // 2, n_train).astype(int)
    tr = trs.iloc[it].drop_duplicates("uid"); Ttr = C["T_tr"][[C["row_tr"][u] for u in tr.uid]]
    if dev_from_train: dv, Tdv = tr, Ttr
    else:
        dvs = C["dv"].sort_values(["duration_s", "uid"]); dv = dvs.iloc[np.linspace(0, len(dvs) // 2, n_dev).astype(int)].drop_duplicates("uid"); Tdv = C["T_dv"][[C["row_dv"][u] for u in dv.uid]]
    tr, dv = tr.reset_index(drop=True), dv.reset_index(drop=True)
    return dict(C, tr=tr, dv=dv, T_tr=Ttr, T_dv=Tdv, sel_trials=None, manifest_sha=sha_of(sorted(tr.uid)), dev_sha=sha_of(sorted(dv.uid)))

def _tinyP(root, name):
    P = dict(ckpt=root / name / "ckpt", logs=root / name / "logs", results=root / name / "results"); P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest"); return P

def resume_test(C):
    """Deliberately interrupted short runs (24 / 8 utterances): run A killed without saving; local checkpoints deleted (= runtime recycle) and an
    incomplete upload planted; run B must resume from the Drive copy; killed again; run C must resume from the more advanced local checkpoint and finish.
    Checks: resume source / epoch / batch / global step; each step done once with the reference's (epoch, batch); losses = reference (3 % / 10 %)."""
    import torch, gc
    root = Path(CFG["write_root"]) / "resume_test"
    if root.exists(): shutil.rmtree(root)
    Ct = tiny_context(C, 24, 8); LE, DE = 3, 7
    ov = dict(max_epochs=4, eden_warmup_batches=10, warmup_updates=4, local_ckpt_every_updates=LE, drive_ckpt_every_updates=DE, ckpt_every_minutes=1e9, drive_ckpt_every_minutes=1e9, log_every=1,
              max_batch_seconds=40.0, max_batch_utts=3, patience=99, max_session_hours=None)    # <= 3 utterances per batch -> 8 batches/epoch, 32 steps
    def go(P, ddir, interrupt=None):
        drv = DriveStore(ddir, DRIVE_MODE_FILES, fake=True); out = "finished"
        try: sv_train("sanity", Ct, P=P, drive=drv, overrides=ov, interrupt_after_update=interrupt, probe=False)
        except SimulatedDisconnect as e: out = str(e)
        drv.wait(); gc.collect(); torch.cuda.empty_cache(); return out
    def state(P): ck = torch.load(P["latest"] / "checkpoint_latest.pt", map_location="cpu", weights_only=False); s = ck["state"]; del ck; return s
    log("---- reference run (uninterrupted)"); Pr = _tinyP(root, "reference"); go(Pr, root / "drive_reference"); R = state(Pr)
    ref = {h["update"]: h for h in R["train_hist"]}; total = R["update"]; ends = {v["update"] for v in R["val_hist"]}
    lpts = {u for u in ref if u % LE == 0} | ends; dpts = {u for u in ref if u % DE == 0} | ends; lastle = lambda pts, u: max([p for p in pts if p <= u], default=0)
    c1 = next((u for u in range(DE + 2, total) if u not in lpts | dpts), None)
    c2 = next((u for u in range(c1 + 2, total) if u not in lpts | dpts and lastle(lpts, u) > lastle(dpts, u) > lastle(dpts, c1)), None) if c1 else None
    if c2 is None: raise RuntimeError(f"resume test: the reference run has only {total} steps, too few for two interruptions")
    expect = lambda u: (ref[u]["epoch"], 0) if u in ends else (ref[u]["epoch"] - 1, ref[u]["batch"])
    Pb, Db = _tinyP(root, "interrupted"), root / "drive_interrupted"; checks = []
    log(f"---- run A: killed after step {c1}"); go(Pb, Db, c1)
    shutil.rmtree(Pb["ckpt"]); (Db / "checkpoint_latest.pt.tmp").write_bytes(os.urandom(1 << 20))
    log(f"---- runtime recycle simulated -> run B must resume from Drive at step {lastle(dpts, c1)}; killed after step {c2}"); go(Pb, Db, c2)
    log(f"---- run C: must resume from the local checkpoint at step {lastle(lpts, c2)} and finish"); go(Pb, Db)
    F = state(Pb); res = F["resumes"]; exp = [("Google Drive", lastle(dpts, c1)), ("local disk", lastle(lpts, c2))]
    checks.append(("resume sources and global steps", [(r["source"], r["update"]) for r in res] == exp, f"{[(r['source'], r['update']) for r in res]} expected {exp}"))
    pos = [(r["epoch"], r["batch"]) for r in res]; epos = [expect(u) for _, u in exp]
    checks.append(("resumed epoch and batch position", pos == epos, f"{pos} expected {epos}"))
    upd = [h["update"] for h in F["train_hist"]]
    checks.append(("every optimizer step done exactly once", upd == list(range(1, total + 1)), f"{len(upd)} steps, reference {total}"))
    checks.append(("each step used the reference's epoch/batch", all((h["epoch"], h["batch"]) == (ref[h["update"]]["epoch"], ref[h["update"]]["batch"]) for h in F["train_hist"]), ""))
    rel = {h["update"]: abs(h["loss"] - ref[h["update"]]["loss"]) / max(ref[h["update"]]["loss"], 1e-6) for h in F["train_hist"]}; first = [rel[u + 1] for _, u in exp]
    checks.append(("first loss after each resume == reference", max(first) < 3e-2, f"relative differences {[f'{x:.1e}' for x in first]}"))
    checks.append(("all losses == reference", max(rel.values()) < 0.1, f"max relative difference {max(rel.values()):.1e}"))
    checks.append(("final state == reference", F["epoch"] == R["epoch"] and F["update"] == R["update"] and bool(F["finished"])
                   and abs(F["val_hist"][-1]["val_loss"] - R["val_hist"][-1]["val_loss"]) <= 0.1 * max(R["val_hist"][-1]["val_loss"], 1e-3),
                   f"epochs {F['epoch']}/{R['epoch']}, steps {F['update']}/{R['update']}, final val loss {F['val_hist'][-1]['val_loss']:.4f}/{R['val_hist'][-1]['val_loss']:.4f}"))
    checks.append(("incomplete upload ignored and removed", not (Db / "checkpoint_latest.pt.tmp").exists(), ""))
    bm = best_meta(Db / "checkpoint_best.pt"); vbest = min(v["val_loss"] for v in F["val_hist"])
    checks.append(("best checkpoint holds the lowest validation loss", bm["val_loss"] <= vbest + CFG["min_delta"] + 1e-9, f"{bm['val_loss']:.5f} vs {vbest:.5f}"))
    for n, okk, info in checks: log(f"  [{'PASS' if okk else 'FAIL'}] {n}: {info}")
    rep = dict(passed=all(c[1] for c in checks), code_version=CODE_VERSION, time=time.strftime("%Y-%m-%d %H:%M:%S"), interrupts=[c1, c2], total_steps=total,
               checks=[dict(check=n, passed=bool(o), info=i) for n, o, i in checks])
    atomic_json(rep, paths()["logs"] / "resume_test_report.json")
    if rep["passed"]: shutil.rmtree(root); log("RESUME TEST PASSED")
    else: log(f"RESUME TEST FAILED: training is blocked (test folders kept in {root})")
    return rep

def learning_check(C, n=64, epochs=24):
    """~300 optimizer steps on 64 training utterances (validation = the same utterances, eval mode): the student must reach
    cos >= CFG['learning_check_min_cos'] with the teacher, i.e. clearly more than one constant vector can (printed)."""
    import torch, gc
    root = Path(CFG["write_root"]) / "learning_check"
    if root.exists(): shutil.rmtree(root)
    Ct = tiny_context(C, n, 0, dev_from_train=True); P = _tinyP(root, "run")
    ov = dict(max_epochs=epochs, eden_warmup_batches=50, warmup_updates=50, local_ckpt_every_updates=10**9, drive_ckpt_every_updates=10**9, ckpt_every_minutes=1e9, drive_ckpt_every_minutes=1e9,
              log_every=50, max_batch_seconds=40.0, patience=10**9, max_session_hours=None)
    log(f"---- learning check: {len(Ct['tr'])} training utterances, {epochs} epochs, optimizer {CFG['optimizer']}")
    r = sv_train("sanity", Ct, P=P, drive=None, overrides=ov, probe=False); gc.collect(); torch.cuda.empty_cache(); S = r["state"]
    curve = [round(v["val_cos"], 3) for v in S["val_hist"]]; base = S["val_hist"][-1]["val_cos_constant_baseline"]; ok = curve[-1] >= CFG["learning_check_min_cos"]
    log(f"  cosine with the teacher per epoch (same 64 utterances, eval mode): {curve}\n  final {curve[-1]:.3f} (needs >= {CFG['learning_check_min_cos']}; one constant vector: {base:.3f})")
    rep = dict(passed=bool(ok), code_version=CODE_VERSION, optimizer=CFG["optimizer"], cos_curve=curve, final_cos=curve[-1], constant_baseline=base, steps=S["update"],
               time=time.strftime("%Y-%m-%d %H:%M:%S"))
    atomic_json(rep, paths()["logs"] / "learning_check.json"); shutil.rmtree(root)
    log("LEARNING CHECK PASSED" if ok else "LEARNING CHECK FAILED: the student cannot fit even 64 teacher embeddings -> training blocked")
    return rep
# ####################################################################################################
# ---- context, official EER, sanity checks, curves, final report and exports
# ####################################################################################################
TOP_FILES = ("teacher_info.json", "teacher_selection_report.json", "dataset_validation_report.json", "train_manifest.csv", "validation_manifest.csv",
             "test_manifest.csv", "excluded_files.csv", "resume_test_report.json", "selection_trials_NONOFFICIAL.csv")

def usable(df):
    """Drops files that failed the full decode (their reason is in excluded_files.csv)."""
    f = paths()["meta"] / "decode_failures.csv"
    if not f.exists(): return df
    bad = set(pd.read_csv(f, dtype=str).uid); return df[~df.uid.isin(bad)].reset_index(drop=True)

def mode_manifests(mode):
    tr = usable(load_manifest("sanity_train_manifest" if mode == "sanity" else "train_manifest")); dv = usable(load_manifest("validation_manifest"))
    te = usable(load_manifest("test_manifest")); return tr, dv, te

def complete_trials():
    out = {}
    for n in OFFICIAL_TRIALS:
        f = paths()["meta"] / f"trials_{n.replace('.txt', '')}.csv"
        if f.exists(): out[n] = pd.read_csv(f, dtype={"enrol": str, "test": str})
    return out

def build_context(mode, tid, spec, drive_top):
    tr, dv, te = mode_manifests(mode); cache = EmbCache(tid, spec, CFG["crop_seconds"], drive_top)
    E_tr, E_dv = cache.load(tr), cache.load(dv); mu, sd = norm_stats(E_tr, spec["norm"]); norm_sha = hashlib.sha1(mu.tobytes() + sd.tobytes()).hexdigest()[:12]
    f = paths()["teacher"] / f"normalisation_{mode}_{spec['norm']}_{norm_sha}.npz"
    if not f.exists():
        np.savez(str(f) + ".part.npz", mu=mu, sd=sd, train_uids_sha=sha_of(sorted(tr.uid))); os.replace(str(f) + ".part.npz", f)
        if drive_top: drive_top.write_file(f, f"{cache.rel}/normalisation_{mode}_{spec['norm']}.npz")
    _, sel_trials = selection_sets(dv)
    return dict(mode=mode, tr=tr, dv=dv, te=te, T_tr=((E_tr - mu) / sd).astype(np.float32), T_dv=((E_dv - mu) / sd).astype(np.float32), mu=mu, sd=sd, norm_sha=norm_sha,
                spec=spec, dim=int(E_tr.shape[1]), tid=tid, cache_ids=[cache.cid, cache_identity(tid, spec, None)[0]], manifest_sha=sha_of(sorted(tr.uid)),
                dev_sha=sha_of(sorted(dv.uid)), sel_trials=sel_trials, row_tr={u: i for i, u in enumerate(tr.uid)}, row_dv={u: i for i, u in enumerate(dv.uid)})

def official_eer(E, df, label):
    """EER on every COMPLETE official VoxCeleb1-O trial list (cosine, no score normalisation)."""
    row = {u: i for i, u in enumerate(df.uid)}; out = {}
    for n, t in complete_trials().items():
        s, y, miss = score_trials(E, row, t)
        if miss: out[n] = dict(status=f"NOT OFFICIAL: {miss} of {len(t)} trials lack an utterance (missing or excluded files)", trials_scored=len(s),
                               eer_on_available=compute_eer(s, y)[0] if len(s) else None)
        else: out[n] = dict(status="official", trials=len(t), target=int(t.label.sum()), eer=compute_eer(s, y)[0], scoring="cosine, no score normalisation")
        e = out[n].get("eer", out[n].get("eer_on_available")); log(f"  {label} | {n}: " + (f"EER {e * 100:.3f}% on {len(s):,} trials ({out[n]['status']})" if e is not None else out[n]["status"]))
    if not out: log(f"  {label}: no official trial list available -> official EER not computed")
    return out

def teacher_official_eer(C, tid, spec, drive_top):
    if not complete_trials(): return {}
    E = EmbCache(tid, spec, None, drive_top).load(C["te"])
    out = dict(kd_target=official_eer((E - C["mu"]) / C["sd"], C["te"], f"TEACHER ({C['spec']['norm']} = KD target)"))
    if C["spec"]["norm"] != "none": out["raw"] = official_eer(E, C["te"], "TEACHER (raw embedding)")
    return out

def sanity_checks(C, tv, tsel, tid):
    """Numbered PASS/FAIL checks (task section 17). Critical failures block training."""
    import torch, gc
    dev = torch.device("cuda"); amp = amp_dtype(); R = []; st = {}
    def chk(no, name, fn, critical=True):
        t0 = time.time()
        try: info = fn(); R.append(dict(no=no, check=name, critical=critical, status="PASS", info=info)); log(f"[{no:2d}] PASS  {name}: {info}")
        except Exception as e: R.append(dict(no=no, check=name, critical=critical, status="FAIL", info=f"{type(e).__name__}: {e}")); log(f"[{no:2d}] {'FAIL' if critical else 'WARN'}  {name}: {type(e).__name__}: {e}")
    def c1():
        tr, dv, te = C["tr"], C["dv"], C["te"]; a, b, c = set(tr.speaker), set(dv.speaker), set(te.speaker)
        assert not a & b and not a & c and not b & c, "speaker overlap between train / validation / test"
        assert not set(tr.uid) & set(te.uid) and len(tr) and len(dv), "empty manifest or test utterances in training"
        ver = set(open(paths()["meta"] / "decode_verified.txt").read().split()); assert set(tr.uid) | set(dv.uid) <= ver, "manifest files not fully decoded"
        return f"train {len(tr):,} utts / {len(a)} spk | val {len(dv):,} / {len(b)} spk | test {len(te):,} / {len(c)} spk | speaker-disjoint, all decoded"
    def c2():
        w = list(tid["weight_files"].values())[0]; assert tid["commit"] == CFG["redimnet_commit"] and w["sha256"] == CFG["redimnet_weights_sha256"] and tid["params"] > 1e6
        return f"{tid['model_id']} | code @ {tid['commit'][:12]} (pinned) | weights sha256 {w['sha256'][:16]}... (expected) | {tid['params'] / 1e6:.2f}M params | {tid['model_class']}"
    def c3(): assert tv["output_shape"] == [1, tid["embed_dim"]] and tv["finite"]; return f"forward pass on real audio: output {tuple(tv['output_shape'])}, finite, repeat max|diff| {tv['repeat_max_abs_diff']:.1e}"
    def c4():
        sc = tv["speaker_check"]; assert sc["gap"] >= CFG["teacher_min_speaker_gap"]
        return f"speaker embedding separates speakers: same-speaker cos {sc['same_speaker_cos']:.3f} vs different {sc['different_speaker_cos']:.3f} | target: {spec_name(C['spec'])} (selection EER {tsel['chosen']['eer'] * 100:.3f}%)"
    def c5():
        assert tv["distinct_utterances"] and tv["equal_length_batch_vs_single_min_cos"] > 0.9999
        return f"distinct utterances (raw pairwise cos {tv['pairwise_cos_raw']['mean']:.3f}); equal-length batch == single (min cos {tv['equal_length_batch_vs_single_min_cos']:.6f}); no padding ever used"
    def c6():
        T = C["T_tr"]; n = np.linalg.norm(T, axis=1); assert np.isfinite(T).all() and (n > 0).all()
        u = l2n(T[:500]); c = u @ u.T; off = c[~np.eye(len(u), dtype=bool)]
        return f"{T.shape[1]}-D targets ({C['spec']['norm']}) | norm mean {n.mean():.1f} | pairwise cos mean {off.mean():.3f} (one constant vector reaches {np.linalg.norm(l2n(T).mean(0)):.3f})"
    def c7():
        st["S"], st["M"] = new_models(C, dev); st["info"] = inspect_student(st["S"], st["M"])
        return f"{st['info']['total_params'] / 1e6:.2f}M params, stacks {[s['layers'] for s in st['info']['stacks']]}, trainable (SV path + M_SV) {sum(p.numel() for p in st['S'].parameters() if p.requires_grad) / 1e6 + st['info']['msv_params'] / 1e6:.2f}M"
    def c8():
        rows = C["tr"].sort_values("duration_s").iloc[np.linspace(0, len(C["tr"]) - 1, 4).astype(int)].reset_index(drop=True); st["rows"] = rows
        idx = [C["row_tr"][u] for u in rows.uid]; b = sv_collate([SVData(C["tr"], C["T_tr"], CFG["crop_seconds"])[i] for i in idx]); st["b"] = b
        S = st["S"]; S.eval()
        with torch.no_grad():
            y, yl = S.sv_features(b["fb"].to(dev), b["fb_lens"].to(dev)); y2, _ = S.sv_features(b["fb"].to(dev), b["fb_lens"].to(dev), early_exit=False)
        S.train(); assert y.shape[-1] == 384 and yl.tolist() == [embed_frames(int(n)) for n in b["n"]], (tuple(y.shape), yl.tolist())
        d = float((y - y2).abs().max()); assert d < 1e-5, d
        return f"encoder.encoders[{S.sv_index}] output {tuple(y.shape)} (B, T50, 384), lengths {yl.tolist()} = embed frames; early exit == full forward (max diff {d:.1e})"
    def c9():
        S, M = st["S"], st["M"]; b = st["b"]; S.eval(); M.eval(); diffs = []
        with torch.no_grad():
            v = sv_forward(S, M, b, dev, amp)
            for k in range(len(b["n"])):
                L = int(b["fb_lens"][k]); vk = sv_forward(S, M, dict(fb=b["fb"][k:k + 1, :L], fb_lens=b["fb_lens"][k:k + 1]), dev, amp)
                diffs.append(float(torch.nn.functional.cosine_similarity(v[k:k + 1].float(), vk.float()).item()))
        S.train(); M.train(); assert v.shape == (len(b["n"]), C["dim"]) and min(diffs) > 0.999, diffs
        return f"M_SV output {tuple(v.shape)} | padded batch vs each utterance alone: min cosine {min(diffs):.6f}"
    def c10(): assert st["M"].D == C["dim"] == C["T_tr"].shape[1]; return f"student embedding {st['M'].D}-D == teacher {C['dim']}-D ({spec_name(C['spec'])})"
    def c11():
        S, M = st["S"], st["M"]; b = st["b"]; set_batch_count(S, 20000); v = sv_forward(S, M, b, dev, amp); loss, cos = kd_loss(v, b["t"].to(dev), CFG["cos_eps"])
        vn, tn = v.detach().double().cpu().numpy(), b["t"].double().numpy(); man = float(np.mean(1 - (vn * tn).sum(1) / (np.linalg.norm(vn, axis=1) * np.linalg.norm(tn, axis=1))))
        assert torch.isfinite(loss) and abs(float(loss) - man) < 1e-4, (float(loss), man); st["loss"] = loss
        return f"KD loss {float(loss):.5f} == numpy {man:.5f} (1 - cosine of un-normalised vectors) | cos {[round(float(c), 3) for c in cos]}"
    def c12():
        # fp16 (T4): backward through a SCALED loss, as training does with GradScaler; otherwise small gradients underflow to exactly 0
        k = 2.0 ** 12 if amp == torch.float16 else 1.0; (st["loss"] * k).backward(); S, M = st["S"], st["M"]; out = []
        for p in list(S.parameters()) + list(M.parameters()):
            if p.grad is not None: p.grad.div_(k)
        zero = [n for n, p in S.sv_params() if p.grad is not None and not bool(p.grad.any())]
        for g, mods in {"encoder_embed": [S.encoder_embed], **{f"stack{i + 1}": [S.encoder.encoders[i]] for i in range(S.sv_index + 1)}, "M_SV": [M]}.items():
            gs = [p.grad for m in mods for n, p in m.named_parameters() if p.grad is not None and not UNUSED_ICEFALL.search(n)]; assert gs, f"{g}: no gradient"
            nrm = float(torch.sqrt(sum((x.float() ** 2).sum() for x in gs))); assert all(torch.isfinite(x).all() for x in gs) and nrm > 0, g; out.append(f"{g} {nrm:.1e}")
        later = [n for n, p in S.named_parameters() if p.grad is not None and not any(n.startswith(f"encoder.encoders.{i}.") for i in range(S.sv_index + 1)) and not n.startswith("encoder_embed.")]
        assert not later, f"gradients outside the SV path: {later[:4]}"
        assert not zero, f"{len(zero)} tensors on the SV path have an all-zero gradient: {zero[:5]}"
        return " | ".join(out) + f" | every SV-path tensor has a non-zero gradient{' (loss scaled by 4096 for fp16, as GradScaler does)' if k > 1 else ''} | stacks {S.sv_index + 2}-6 receive no gradient (not on the SV path)"
    def c13():
        S, M = st["S"], st["M"]; opt, sch = make_optimizer(S, M, dict(CFG, **MODES["sanity"])); named = dict(S.sv_params() + [("msv." + n, p) for n, p in M.named_parameters()])
        before = {k: v.detach().clone() for k, v in named.items()}; frozen_before = {n: p.detach().clone() for n, p in S.named_parameters() if not p.requires_grad}
        unused = {k for k, v in named.items() if v.grad is None}; assert all(UNUSED_ICEFALL.search(k) for k in unused), sorted(unused)[:5]
        opt.step(); opt.zero_grad(set_to_none=True)
        changed = [k for k, v in named.items() if not torch.equal(v.detach(), before[k]) and k not in unused]; same = [k for k in named if k not in unused and k not in changed]
        assert len(same) <= 0.01 * len(named), f"{len(same)} trainable tensors not updated: {same[:5]}"
        fz = [n for n, p in S.named_parameters() if n in frozen_before and not torch.equal(p.detach(), frozen_before[n])]; assert not fz, fz[:3]
        return f"{type(opt).__name__}: {len(changed)}/{len(named) - len(unused)} trainable tensors updated; {len(frozen_before)} tensors of stacks {S.sv_index + 2}-6 unchanged"
    def c14():
        assert tv["frozen"]["trainable_params"] == 0 and tv["frozen"]["eval_mode"] and not st["b"]["t"].requires_grad
        return f"{tid['name']}: eval mode, 0 trainable parameters, run under no_grad; training uses CACHED teacher vectors (no teacher in the graph)"
    def c15(): assert torch.isfinite(st["loss"]); return "loss, embeddings and gradients finite (checks 11-12)"
    def c16():
        lens = np.array([crop_len(r.n_frames, r.sr, CFG["crop_seconds"]) for r in C["tr"].itertuples()]); B = make_batches(lens, 0, CFG["seed"], True, CFG["max_batch_seconds"], CFG["max_batch_utts"])
        flat = [i for b in B for i in b]; assert sorted(flat) == list(range(len(lens))), "batches do not cover every utterance exactly once"
        assert all(len(b) <= CFG["max_batch_utts"] and len(b) * max(lens[b]) / SR <= CFG["max_batch_seconds"] + 1e-6 for b in B)
        assert st["b"]["fb_lens"].tolist() == [fbank_frames(int(n)) for n in st["b"]["n"]]
        return f"{len(B)} length-sorted batches, every utterance once, <= {CFG['max_batch_seconds']:.0f} s / {CFG['max_batch_utts']} utts, fbank lengths = formula"
    def c17():
        S = st["S"]; b = st["b"]; S.eval(); ds = []
        with torch.no_grad():
            y, yl = S.sv_features(b["fb"].to(dev), b["fb_lens"].to(dev))
            for k in range(len(b["n"])):
                L = int(b["fb_lens"][k]); yk, ykl = S.sv_features(b["fb"][k:k + 1, :L].to(dev), b["fb_lens"][k:k + 1].to(dev)); n_ = int(ykl[0]) - 8
                ds.append(float((y[k, :n_] - yk[0, :n_]).abs().max() / yk[0, :n_].abs().max()))
        S.train(); assert max(ds) < 5e-3, ds
        return f"third-stack features of each utterance: padded batch == alone (max rel. diff {max(ds):.1e}; last 8 frames excluded: stack downsampling pairs the last frame with padding)"
    def c18():
        Ttr = EmbCache(tid, C["spec"], CFG["crop_seconds"]).load(C["tr"]); assert Ttr.shape[0] == len(C["tr"])
        return f"cache {C['cache_ids'][0]}: one valid embedding per utterance (train {len(C['tr']):,}, val {len(C['dv']):,}), shards sha1-verified, metadata = this teacher configuration"
    def c19():
        S, M = st["S"], st["M"]; f = paths()["logs"] / "sanity_ckpt.pt"; atomic_torch_save(dict(student=S.state_dict(), msv=M.state_dict()), f)
        ck = torch.load(f, map_location=dev, weights_only=False); S2, M2 = new_models(C, dev); S2.load_state_dict(ck["student"], strict=True); M2.load_state_dict(ck["msv"], strict=True)
        outs = []
        for a, m in [(S, M), (S2, M2)]:
            a.eval(); m.eval()
            with torch.no_grad(): outs.append(sv_forward(a, m, st["b"], dev, amp))
            a.train(); m.train()
        d = float((outs[0] - outs[1]).abs().max()); os.remove(f); assert d < 1e-5, d; return f"atomic save + strict reload, identical outputs (max diff {d:.1e})"
    def c21():
        r, _ = sv_evaluate(st["S"], st["M"], C, dev, amp, df=C["dv"].iloc[:64], targets=C["T_dv"][:64], trials=pd.DataFrame(columns=["label", "enrol", "test"]))
        assert np.isfinite(r["val_loss"]) and r["invalid_embeddings"] == 0; return f"validation pass on 64 utterances: KD loss {r['val_loss']:.4f}, cos {r['val_cos']:.4f}, no parameter update"
    def cM():
        mb = memory_probe(st["S"], st["M"], C, dev, amp, CFG["max_batch_seconds"]); return f"worst-case batch fits with max_batch_seconds {mb:.0f}"
    plan = [(1, "dataset manifest validation", c1), (2, "ReDimNet2 checkpoint loading", c2), (3, "ReDimNet2 forward pass", c3), (4, "teacher speaker embedding", c4),
            (5, "teacher batching without padding", c5), (6, "teacher embedding validation", c6), (7, "Zipformer-M initialisation", c7), (8, "third-block representation extraction", c8),
            (9, "student pooling / projection (M_SV)", c9), (10, "teacher / student dimensional compatibility", c10), (11, "KD loss calculation", c11),
            (12, "backpropagation", c12), (13, "student parameter update", c13), (14, "teacher remains frozen", c14), (15, "no NaN / Inf", c15),
            (16, "variable-length batching", c16), (17, "padding-mask correctness", c17), (18, "teacher cache integrity", c18), (19, "checkpoint save / reload", c19),
            (21, "validation pass", c21), (23, "GPU memory of the worst-case batch", cM)]
    for no, name, fn in plan:
        chk(no, name, fn)
        if no == 7 and R[-1]["status"] == "FAIL": break
    st.clear(); gc.collect(); torch.cuda.empty_cache()
    crit = [r["no"] for r in R if r["critical"] and r["status"] == "FAIL"]; missing = [n for n, _, _ in plan if n not in {r["no"] for r in R}]
    rep = dict(code_version=CODE_VERSION, checks=R, critical_failed=crit + missing, all_critical_passed=not crit and not missing, cache=C["cache_ids"], time=time.strftime("%Y-%m-%d %H:%M:%S"))
    atomic_json(rep, paths()["logs"] / "sanity_checks.json")
    log(f"\nSANITY CHECKS: {sum(r['status'] == 'PASS' for r in R)}/{len(plan)} passed | critical failures: {rep['critical_failed'] or 'none'} (20 = resume test and 22 = official teacher EER are separate steps)")
    return rep

def plots(P, drive=None):
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        ft, fv = P["logs"] / "training_log.csv", P["logs"] / "validation_log.csv"
        if not fv.exists() or fv.stat().st_size < 40: return
        V = pd.read_csv(fv); T = pd.read_csv(ft) if ft.exists() and ft.stat().st_size > 10 else None
        fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))
        if T is not None and len(T): ax[0].plot(T["update"], T["loss"], alpha=0.5, label="train (logged steps)")
        ax[0].plot(V["update"], V["train_loss"], "o-", label="train (epoch)"); ax[0].plot(V["update"], V["val_loss"], "o-", label="validation"); ax[0].set(title="KD loss = 1 - cos", xlabel="global step"); ax[0].legend()
        ax[1].plot(V["update"], V["val_cos"], "o-", label="validation cos"); ax[1].plot(V["update"], V["val_cos_constant_baseline"], "--", c="gray", label="one constant vector"); ax[1].set(title="cosine with the teacher", xlabel="global step"); ax[1].legend()
        if "val_eer_nonofficial" in V: ax[2].plot(V["update"], V["val_eer_nonofficial"] * 100, "o-"); ax[2].set(title="EER % on NON-OFFICIAL validation trials (monitor)", xlabel="global step")
        if T is not None and len(T): ax[3].semilogy(T["update"], T["grad_norm"].clip(lower=1e-6), label="grad norm"); a3 = ax[3].twinx(); a3.plot(T["update"], T["lr"], c="tab:red"); ax[3].set(title="gradient norm / lr (red)", xlabel="global step")
        fig.tight_layout(); f = P["results"] / "training_curves.png"; fig.savefig(f, dpi=110); plt.close(fig)
        if drive: drive.write_file(f, "training_curves.png")
    except Exception as e: log(f"  (plot not updated: {type(e).__name__}: {e})")

def final_report(mode, C, P, drive, tsel, tid, teer, dsrep, sanity, rt, lc, compare=None, tv=None):
    tv_speaker = (tv or {}).get("speaker_check")
    import torch
    LATEST, BEST = P["latest"] / "checkpoint_latest.pt", P["best"] / "checkpoint_best.pt"
    ck = torch.load(LATEST, map_location="cpu", weights_only=False); S = ck["state"]; del ck
    rep = dict(RESULT_TYPE=("SANITY RUN (~2,000 utterances): a pipeline check, NOT a Stage 1 research result" if mode == "sanity" else "FULL Stage 1"),
               status=("COMPLETED: " + S["finished"]) if S["finished"] else f"NOT FINISHED (epoch {S['epoch'] + 1}, step {S['update']}): re-run the cell to resume",
               teacher=dict(model=tid["name"], model_id=tid["model_id"], repo=tid["repo"], commit=tid["commit"], params=tid["params"], training_data=tid["training_data"],
                            train_type=tid["train_type"], embedding_dim=C["dim"], representation=spec_name(C["spec"]), normalisation=C["spec"]["norm"],
                            normalisation_selection=dict(rule=tsel["rule"], candidates=tsel["candidates"], selection_eer_nonofficial=tsel["chosen"]["eer"]),
                            precision=tid["amp"], weights=tid["weight_files"], speaker_check=tv_speaker),
               student=dict(model="Zipformer-M (icefall, unchanged)", sv_branch=f"Conv2dSubsampling + encoder stacks 1-{CFG['sv_block']} (third stack: 3 layers, 384-D)",
                            msv="attentive pooling (two 1-D convolutions) -> attentive mean+std (768) -> Linear", embedding_dim=C["dim"]),
               objective="L_SV_KD = 1 - CosineSim(v_teacher, M_SV(R_student^(3))) (MT2KD Eq. 8); no speaker labels used in training",
               data=dict(dataset="VoxCeleb1", partition=dsrep["test_partition_source"], official=dsrep["official"], train_utterances=len(C["tr"]), train_speakers=int(C["tr"].speaker.nunique()),
                         validation_utterances=len(C["dv"]), validation_speakers=int(C["dv"].speaker.nunique()), validation_note=dsrep["validation"]["note"],
                         excluded_files=int(len(_ex := pd.read_csv(paths()["meta"] / "excluded_files.csv", dtype=str))),
                         excluded_reasons=_ex.reason.str.split(":").str[0].value_counts().to_dict(), excluded_list="excluded_files.csv"),
               training=dict(epochs=S["epoch"], optimizer_steps=S["update"], best=S["best"], early_stopping=dict(counter=S["es_counter"]), hours=round(S["elapsed"] / 3600, 3),
                             sessions=S["sessions"], resumes=S.get("resumes", []), peak_gpu_gib=S.get("peak_gpu_gib")),
               cache=dict(ids=C["cache_ids"], location=f"{CFG['drive_out']}/teacher_embedding_cache/", dim=C["dim"], dtype="float16"),
               sanity_checks=dict(passed=sanity["all_critical_passed"], failed=sanity["critical_failed"]), resume_test=dict(passed=rt.get("passed")), learning_check=lc,
               teacher_eer_official=teer, references_nonofficial=compare,
               limitations=["the teacher is ReDimNet2-B6 trained on VoxCeleb2-dev, not the paper's ECAPA-TDNN: results are not directly comparable with MT2KD Table VII",
                            "ReDimNet2 is recent (2026) with few independent reproductions; its official EER is measured here on the same trials as the student",
                            "the validation split is NON-OFFICIAL (held-out dev speakers); it was used to choose the target normalisation and for early stopping",
                            "KD targets and student inputs use a centred <= crop_seconds segment of each utterance; official EER uses full utterances",
                            "Zipformer step-dependent regularisation follows training progress (batch_count_mode='progress'), not icefall's absolute schedule",
                            "the paper's M_SV is described only as attentive pooling with two 1-D convolutions + a linear layer; the attentive mean+std form is an interpretation",
                            "VoxCeleb1-E/H trial lists are not usable (their speakers are training data here)",
                            "teacher embeddings are computed on the same centred <= crop_seconds segment as the student (training / validation); test uses full utterances"])
    if BEST.exists():
        dev = torch.device("cuda"); s, m = new_models(C, dev); b = torch.load(BEST, map_location=dev, weights_only=False); s.load_state_dict(b["student"]); m.load_state_dict(b["msv"])
        v, _ = sv_evaluate(s, m, C, dev, amp_dtype()); rep["best_checkpoint_validation"] = dict(epoch=b["epoch"], step=b["update"], val_kd_loss=v["val_loss"], val_cos=v["val_cos"],
                                                                                          val_eer_nonofficial=v.get("val_eer_nonofficial"), reload_matches=abs(v["val_loss"] - b["val"]["val_loss"]) < 2e-3)
        if S["finished"] and complete_trials():
            E = student_embeddings(s, m, C["te"], None, dev, amp_dtype(), max_sec=300.0); rep["student_eer_official"] = official_eer(E, C["te"], "STUDENT (best checkpoint)")
        if S["finished"]:
            le, lp = P["results"] / "student_encoder.pt", P["results"] / "sv_projection.pt"
            atomic_torch_save(dict(student=b["student"], zip_args=ZIP_ARGS, sv_block=CFG["sv_block"], epoch=b["epoch"], note="full Zipformer-M state; stacks after sv_block are untrained in SV-only Stage 1"), le)
            atomic_torch_save(dict(msv=b["msv"], dims=b["msv_dims"], teacher=b["teacher"], normalisation=dict(kind=C["spec"]["norm"], mu=C["mu"], sd=C["sd"], apply="(embedding - mu) / sd"), sv_block=CFG["sv_block"]), lp)
            if drive: drive.push(le, "student_encoder.pt"); drive.push(lp, "sv_projection.pt")
        del s, m; torch.cuda.empty_cache()
    atomic_json(rep, P["results"] / "final_report.json"); plots(P, drive)
    if drive: drive.write_json("final_report.json", rep); drive.wait()
    log("\n" + "=" * 100 + f"\nSV STAGE 1 FINAL REPORT  [{rep['RESULT_TYPE']}]\n" + "=" * 100)
    for k in ["status", "teacher", "student", "data", "training", "best_checkpoint_validation", "teacher_eer_official", "student_eer_official", "resume_test", "learning_check"]:
        if k in rep: log(f"{k:28s}: {rep[k]}")
    return rep
'''

os.makedirs(_MOD_DIR, exist_ok=True)
for _name, _src in _MODULES.items():
    _f = os.path.join(_MOD_DIR, _name + ".py")
    if not os.path.exists(_f) or open(_f).read() != _src: open(_f, "w").write(_src)
if _MOD_DIR not in sys.path: sys.path.insert(0, _MOD_DIR)
importlib.invalidate_caches()
for _name in _MODULES:
    if _name in sys.modules: importlib.reload(sys.modules[_name])
MA = importlib.import_module("mt2kd_mod_asr")      # WavLM ASR cell (LibriSpeech, WavLM teacher, alignment, Drive store, WER code)
MB = importlib.import_module("mt2kd_mod_at")       # AT teacher-test cell (FSD50K, SSLAM teacher, AudioSet-ID mapping, mAP)
MC = importlib.import_module("mt2kd_mod_sv")       # ReDimNet2 SV cell (VoxCeleb1, ReDimNet2 teacher, Zipformer-M + stack-3 SV branch, EER)

# ---- the web app's own backend files (copied unchanged from the laptop: backend/mt2kd_infer.py, backend/ctc_lm.py)
APP_SOURCES = {}
APP_SOURCES['mt2kd_infer'] = r'''"""MT2KD inference: loads the deploy folder written by MT2KD_EXPORT_CELL.py and runs ASR, audio tagging and speaker embeddings.

The model code below is copied unchanged from the training cells (Cell 1: MC.build_sv_student / MSV, MA.fbank, MA.greedy_decode)
and was verified in the export cell (strict weight loading, dev-clean WER equal to training, reload test).
"""
from __future__ import annotations

import contextlib
import json
import sys
import threading
import time
import zipfile
from pathlib import Path

import numpy as np

SR = 16000
REQUIRED = ("model.pt", "labels.json", "zipformer_src.zip")


class DeployFolderError(RuntimeError):
    pass


def _check_deploy(d: Path):
    missing = [f for f in REQUIRED if not (d / f).exists()]
    if missing:
        raise DeployFolderError(
            f"deploy folder {d} is missing {missing}. Download MT2KD/deploy/<run>/ from Google Drive into this folder.")


def _prepare_zipformer(d: Path) -> Path:
    """Unpack the exact Zipformer source saved by the export cell (icefall files + Swoosh port)."""
    z = d / "zipformer_src.zip"
    out = d / "_zipformer"
    stamp = out / ".source_stamp"
    sig = f"{z.stat().st_size}-{int(z.stat().st_mtime)}"
    if not (stamp.exists() and stamp.read_text() == sig):
        import shutil
        shutil.rmtree(out, ignore_errors=True)
        out.mkdir(parents=True)
        with zipfile.ZipFile(z) as f:
            f.extractall(out)
        stamp.write_text(sig)
    return out


class MT2KD:
    """One Zipformer encoder (M or S, sizes read from model.pt) + Stage-2 heads. Thread-safe (one inference at a time)."""

    def __init__(self, deploy_dir: str | Path, threads: int | None = None):
        import torch
        import torch.nn as nn

        self.torch = torch
        d = Path(deploy_dir).resolve()
        _check_deploy(d)
        zdir = _prepare_zipformer(d)
        if str(zdir) not in sys.path:
            sys.path.insert(0, str(zdir))
        for m in [k for k in list(sys.modules) if k in ("zipformer", "scaling", "subsampling", "encoder_interface", "k2_swoosh")
                  or k == "icefall" or k.startswith("icefall.")]:
            del sys.modules[m]
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            from subsampling import Conv2dSubsampling
            from zipformer import Zipformer2
        if threads:
            torch.set_num_threads(int(threads))

        def lengths_to_mask(lens, T):
            return torch.arange(T, device=lens.device)[None, :] < lens[:, None]

        class _StopAfterBlock(Exception):
            pass

        class ZipformerStudent(nn.Module):
            def __init__(self, zip_args, sv_block=3):
                super().__init__()
                self.encoder_embed = Conv2dSubsampling(in_channels=80, out_channels=192)
                self.encoder = Zipformer2(**zip_args)
                self.sv_index = sv_block - 1

            def forward(self, feats, feat_lens):
                x, xl = self.encoder_embed(feats, feat_lens)
                enc, el = self.encoder(x.permute(1, 0, 2), xl, ~lengths_to_mask(xl, x.size(1)))
                return enc.permute(1, 0, 2), el

            def sv_features(self, feats, feat_lens):
                x, xl = self.encoder_embed(feats, feat_lens)
                cap = {}

                def hook(m, i, o):
                    cap["y"] = o
                    raise _StopAfterBlock()

                h = self.encoder.encoders[self.sv_index].register_forward_hook(hook)
                try:
                    self.encoder(x.permute(1, 0, 2), xl, ~lengths_to_mask(xl, x.size(1)))
                except _StopAfterBlock:
                    pass
                finally:
                    h.remove()
                return cap["y"].permute(1, 0, 2), xl

        class MSV(nn.Module):
            def __init__(self, C, A, D):
                super().__init__()
                self.att = nn.Sequential(nn.Conv1d(C, A, 1), nn.Tanh(), nn.Conv1d(A, C, 1))
                self.proj = nn.Linear(2 * C, D)

            def forward(self, y, lens):
                x = y.float().transpose(1, 2)
                m = lengths_to_mask(lens, x.size(2))[:, None, :]
                e = self.att(x).masked_fill(~m, float("-inf"))
                w = torch.softmax(e, dim=2)
                mu = (w * x).sum(2)
                var = (w * x * x).sum(2) - mu * mu
                return self.proj(torch.cat([mu, var.clamp(min=1e-5).sqrt()], 1))

        t0 = time.time()
        ck = torch.load(d / "model.pt", map_location="cpu", weights_only=False)
        if ck.get("format") != "mt2kd-infer-v1":
            raise DeployFolderError(f"model.pt has format {ck.get('format')!r}; expected 'mt2kd-infer-v1' (re-run the export cell)")
        self.vocab = list(ck["vocab"])
        self.student = ZipformerStudent(ck["zip_args"], ck.get("sv_block", 3))
        self.student.load_state_dict(ck["student"], strict=True)
        D_enc = int(ck["heads"]["ctc.weight"].shape[1])           # encoder output size: 512 (Zipformer-M) or 256 (Zipformer-S)
        self.heads = nn.ModuleDict(dict(ctc=nn.Linear(D_enc, len(self.vocab)), at=nn.Linear(D_enc, ck["at_classes"]), msv=MSV(**ck["msv"])))
        self.model_name = ck.get("model_name", "Zipformer-M")
        self.heads.load_state_dict(ck["heads"], strict=True)
        for name, mod in self.student.named_modules():
            if hasattr(mod, "batch_count"):
                mod.batch_count = ck["batch_count"]
            if hasattr(mod, "name"):
                mod.name = name
        self.student.eval()
        self.heads.eval()
        self._mask = lengths_to_mask
        self.lock = threading.Lock()

        self.labels = json.loads((d / "labels.json").read_text(encoding="utf-8"))
        self.report = json.loads((d / "export_report.json").read_text(encoding="utf-8")) if (d / "export_report.json").exists() else {}
        self.source = ck.get("source", {})
        self.threshold = dict(ck.get("sv_threshold") or {})
        cal = d / "sv_threshold.json"                    # written later by the calibration step (VoxCeleb1 validation trials)
        if cal.exists():
            self.threshold = json.loads(cal.read_text(encoding="utf-8"))
        self.final_report = None
        fr = d / "final_report.json"                     # optional: Cell 2's final_report.json (official test-set numbers)
        if fr.exists():
            self.final_report = json.loads(fr.read_text(encoding="utf-8"))
        self.params = sum(p.numel() for p in self.student.parameters()) + sum(p.numel() for p in self.heads.parameters())
        self.deploy_dir = d
        self.load_seconds = time.time() - t0
        self.lm_decoder, self.lm_config, self.lm_error = None, None, None
        self._load_lm(d / "lm")

    def _load_lm(self, lmdir: Path):
        """Optional word LM for CTC beam search (written by MT2KD_LM_DECODING_CELL.py into deploy/lm/)."""
        cfg = lmdir / "lm_config.json"
        if not cfg.exists():
            return
        try:
            from . import ctc_lm
        except ImportError:
            import ctc_lm
        try:
            c = json.loads(cfg.read_text(encoding="utf-8"))
            t0 = time.time()
            lm = ctc_lm.NgramLM.load_cached(lmdir / c["lm"], log=lambda *a: print("[MT2KD]", *a, flush=True))
            hw = lmdir / "hotwords.txt"                  # optional: one extra word per line (e.g. YOUTUBE)
            if hw.exists():
                words = [l.strip() for l in hw.read_text(encoding="utf-8").splitlines() if l.strip() and not l.strip().startswith("#")]
                lm.add_hotwords(words)
                print(f"[MT2KD] hotwords added: {len(words)}", flush=True)
            self.lm_decoder = ctc_lm.Decoder(self.vocab, lm, alpha=c["alpha"], beta=c["beta"], beam=c.get("beam", 64))
            self.lm_config = c
            print(f"[MT2KD] language model ready in {time.time() - t0:.0f} s (alpha {c['alpha']}, beta {c['beta']}, beam {c.get('beam', 64)})", flush=True)
        except Exception as e:
            self.lm_error = f"{type(e).__name__}: {e}"
            print(f"[MT2KD] language model NOT loaded ({self.lm_error}); greedy decoding only", flush=True)

    # ------------------------------------------------------------------ front end (identical to training)
    def _fbank(self, wav: np.ndarray):
        import torchaudio.compliance.kaldi as kaldi
        x = self.torch.from_numpy(np.ascontiguousarray(wav, dtype=np.float32))[None]
        return kaldi.fbank(x, num_mel_bins=80, frame_length=25.0, frame_shift=10.0, dither=0.0, energy_floor=1e-10,
                           sample_frequency=float(SR), snip_edges=False, low_freq=20.0, high_freq=-400.0, remove_dc_offset=True,
                           preemphasis_coefficient=0.97, window_type="povey", use_energy=False)

    def _greedy(self, ids):
        out, prev = [], -1
        for k in ids:
            if k != prev and k != 0:
                out.append(self.vocab[k])
            prev = k
        return " ".join("".join(out).split())

    def _encode(self, wav):
        fb = self._fbank(wav)[None]
        fl = self.torch.tensor([fb.size(1)])
        enc, el = self.student(fb, fl)
        return enc, el, fb, fl

    # ------------------------------------------------------------------ tasks
    def transcribe(self, wav: np.ndarray, max_chunk_s: float = 30.0) -> dict:
        """Greedy CTC and (if available) CTC beam search + word LM. Audio longer than max_chunk_s is cut at quiet points
        (training utterances are <= ~35 s)."""
        segs = split_at_quiet_points(wav, max_chunk_s)
        greedy, logps = [], []
        with self.lock, self.torch.no_grad():
            for a, b in segs:
                enc, el, _, _ = self._encode(wav[a:b])
                lg = self.heads["ctc"](enc.repeat_interleave(2, dim=1))[0, :int(el[0]) * 2]
                greedy.append(self._greedy(lg.argmax(-1).tolist()))
                logps.append(lg.float().log_softmax(-1).numpy())
        out = dict(text_greedy=" ".join(t for t in greedy if t), segments=len(segs), text_lm=None, lm_seconds=None)
        if self.lm_decoder is not None:
            t0 = time.time()
            out["text_lm"] = " ".join(t for t in (self.lm_decoder.decode(lp) for lp in logps) if t)
            out["lm_seconds"] = time.time() - t0
        return out

    def tag(self, wav: np.ndarray, top_k: int = 5, window_s: float = 10.0) -> dict:
        """Clip-level sigmoid scores (200 FSD50K classes). Longer audio: 10 s windows (training crop), max over windows."""
        W = int(window_s * SR)
        if len(wav) <= W:
            starts = [0]
        else:
            hop = W // 2
            starts = list(range(0, len(wav) - W + 1, hop))
            if starts[-1] != len(wav) - W:
                starts.append(len(wav) - W)
        best = None
        with self.lock, self.torch.no_grad():
            for s in starts:
                enc, el, _, _ = self._encode(wav[s:s + W])
                m = self._mask(el, enc.size(1)).unsqueeze(-1).to(enc.dtype)
                pooled = (enc * m).sum(1) / m.sum(1).clamp(min=1)
                p = self.torch.sigmoid(self.heads["at"](pooled))[0].numpy()
                best = p if best is None else np.maximum(best, p)
        order = np.argsort(-best)[:max(1, int(top_k))]
        return dict(labels=[dict(index=int(i), name=self.labels[i]["name"], mid=self.labels[i]["mid"], score=float(best[i])) for i in order],
                    windows=len(starts))

    def embed(self, wav: np.ndarray, max_s: float = 90.0) -> np.ndarray:
        """192-D speaker embedding from encoder stack 3 + M_SV (L2-normalised). Very long audio: centre max_s seconds."""
        L = int(max_s * SR)
        if len(wav) > L:
            s = (len(wav) - L) // 2
            wav = wav[s:s + L]
        with self.lock, self.torch.no_grad():
            fb = self._fbank(wav)[None]
            fl = self.torch.tensor([fb.size(1)])
            y, yl = self.student.sv_features(fb, fl)
            e = self.heads["msv"](y, yl)[0].numpy().astype(np.float64)
        return e / max(np.linalg.norm(e), 1e-12)


# ---------------------------------------------------------------------- audio helpers
def split_at_quiet_points(wav: np.ndarray, max_s: float, search_s: float = 8.0):
    """[(start, end)] segments of <= max_s seconds, each cut at the quietest 25 ms frame in the last search_s seconds."""
    n, M = len(wav), int(max_s * SR)
    if n <= M:
        return [(0, n)]
    out, a, win = [], 0, int(0.025 * SR)
    while n - a > M:
        lo, hi = a + M - int(search_s * SR), a + M
        frames = (hi - lo) // win
        e = np.square(wav[lo:lo + frames * win]).reshape(frames, win).mean(1)
        cut = lo + int(np.argmin(e)) * win + win // 2
        out.append((a, cut))
        a = cut
    out.append((a, n))
    return out


def decode_audio(data: bytes, filename: str = "") -> tuple[np.ndarray, dict]:
    """Bytes of an audio file -> 16 kHz mono float32 + info. WAV/FLAC/OGG/MP3 via soundfile; others via torchaudio if it can."""
    import io
    info = {}
    try:
        import soundfile as sf
        x, sr = sf.read(io.BytesIO(data), dtype="float32", always_2d=True)
    except Exception as e1:
        try:
            import torchaudio
            t, sr = torchaudio.load(io.BytesIO(data))
            x = t.numpy().T
        except Exception:
            raise ValueError(f"could not read {filename or 'the file'} as audio ({type(e1).__name__}). Use WAV, FLAC, OGG or MP3.")
    info["sample_rate_in"] = int(sr)
    info["channels"] = int(x.shape[1])
    w = x.mean(axis=1) if x.shape[1] > 1 else x[:, 0]
    if sr != SR:
        import torch
        import torchaudio
        w = torchaudio.functional.resample(torch.from_numpy(np.ascontiguousarray(w)), int(sr), SR).numpy()
    w = np.ascontiguousarray(w, dtype=np.float32)
    if not np.isfinite(w).all():
        raise ValueError("the audio contains invalid samples (NaN or infinity)")
    info["duration_s"] = len(w) / SR
    return w, info


def peaks(wav: np.ndarray, n: int = 160) -> list[float]:
    """Peak amplitude per block, scaled to 0..1, for drawing the waveform."""
    if len(wav) == 0:
        return []
    k = max(1, len(wav) // n)
    p = np.abs(wav[: k * n].reshape(-1, k)).max(axis=1) if len(wav) >= n else np.abs(wav)
    m = float(p.max()) or 1.0
    return [round(float(v) / m, 4) for v in p]
'''
APP_SOURCES['ctc_lm'] = r'''"""Pure-Python CTC prefix beam search with a word n-gram language model (ARPA). No compiled dependencies.

Character CTC (blank = index 0, space = ' ') + word LM, as in Hannun et al. 2014 ("first-pass LVCSR with CTC + LM"):
  score(prefix) = log P_ctc(prefix | audio) + alpha * ln P_lm(words) + beta * (number of words)
A word gets its LM score when it is completed (space or end of audio). A partial word that is not the beginning of any
LM word gets a penalty (alpha * UNK_LOG10 * ln 10), so the search prefers real words.
"""
from __future__ import annotations

import gzip
import math
import pickle
from pathlib import Path

import numpy as np

LN10 = math.log(10.0)
NEG = -1e30
UNK_LOG10 = -10.0                      # log10 score for a word the LM does not know (as pyctcdecode's default offset)


def _lse(a, b):
    if a < b:
        a, b = b, a
    if b <= NEG:
        return a
    return a + math.log1p(math.exp(b - a))


class NgramLM:
    """Back-off n-gram LM from an ARPA file. Words are upper case (LibriSpeech LMs are)."""

    def __init__(self, prob: dict, bow: dict, order: int, prefixes: set):
        self.prob, self.bow, self.order, self.prefixes = prob, bow, order, prefixes
        self.extra = {}                                  # hotwords: word -> log10 unigram score

    # ---------------------------------------------------------------- loading
    @classmethod
    def from_arpa(cls, path: str | Path, log=print) -> "NgramLM":
        path = Path(path)
        op = gzip.open if path.suffix == ".gz" else open
        prob, bow, order, sec = {}, {}, 0, 0
        with op(path, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                if line.startswith("\\"):
                    if line.endswith("-grams:"):
                        sec = int(line[1:line.index("-")]); order = max(order, sec)
                    else:
                        sec = 0
                    continue
                if sec == 0:
                    continue
                parts = line.split("\t")
                if len(parts) < 2:
                    parts = line.split()
                    lp, words, bo = float(parts[0]), " ".join(parts[1:1 + sec]), (float(parts[1 + sec]) if len(parts) > 1 + sec else None)
                else:
                    lp, words, bo = float(parts[0]), parts[1], (float(parts[2]) if len(parts) > 2 else None)
                prob[words] = lp
                if bo is not None and bo != 0.0:
                    bow[words] = bo
        lm = cls(prob, bow, order, cls._prefixes(prob))
        log(f"  ARPA loaded: order {order}, {len(prob):,} n-grams, {sum(1 for w in prob if ' ' not in w):,} words")
        return lm

    @staticmethod
    def _prefixes(prob):
        out = set()
        for w in prob:
            if " " not in w and not w.startswith("<"):
                for k in range(1, len(w) + 1):
                    out.add(w[:k])
        return out

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(dict(prob=self.prob, bow=self.bow, order=self.order), f, protocol=pickle.HIGHEST_PROTOCOL)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        return cls(d["prob"], d["bow"], d["order"], cls._prefixes(d["prob"]))

    @classmethod
    def load_cached(cls, arpa_path, log=print):
        """Parse the ARPA file once, then reuse a pickle cache next to it (much faster start-up)."""
        arpa_path = Path(arpa_path)
        cache = arpa_path.with_name(arpa_path.name + ".cache.pkl")
        if cache.exists() and cache.stat().st_mtime >= arpa_path.stat().st_mtime:
            try:
                return cls.load(cache)
            except Exception:
                pass
        lm = cls.from_arpa(arpa_path, log=log)
        try:
            lm.save(cache)
        except Exception:
            pass
        return lm

    def add_hotwords(self, words, log10_score: float = -4.0):
        for w in words:
            w = w.upper().strip()
            if w and " " not in w:
                self.extra[w] = log10_score
                for k in range(1, len(w) + 1):
                    self.prefixes.add(w[:k])

    # ---------------------------------------------------------------- scoring
    def known(self, w):
        return w in self.prob or w in self.extra

    def log10p(self, ctx: tuple, w: str) -> float:
        """log10 P(w | ctx) with Katz-style back-off. ctx = up to order-1 previous words."""
        if w not in self.prob:
            return self.extra.get(w, UNK_LOG10)
        ctx = ctx[-(self.order - 1):] if self.order > 1 else ()
        back = 0.0
        for k in range(len(ctx), -1, -1):
            hist = ctx[len(ctx) - k:]
            key = " ".join(hist + (w,))
            p = self.prob.get(key)
            if p is not None:
                return p + back
            if k > 0:
                back += self.bow.get(" ".join(hist), 0.0)
        return UNK_LOG10


class Decoder:
    def __init__(self, vocab: list, lm: NgramLM | None, alpha: float = 0.5, beta: float = 1.0, beam: int = 32,
                 token_min_logp: float = math.log(1e-3), blank_skip: float = 0.999):
        self.vocab = list(vocab)
        assert self.vocab[1] == " ", "vocab[1] must be the space symbol"
        self.lm, self.alpha, self.beta, self.beam = lm, alpha, beta, beam
        self.token_min_logp, self.blank_skip_logp = token_min_logp, math.log(blank_skip)
        self.space = 1

    # LM part of a prefix's score, cached per prefix string (the LM only depends on the text)
    def _lm(self, pre: str, cache: dict):
        r = cache.get(pre)
        if r is not None:
            return r
        if not pre:
            r = (0.0, ("<s>",), 0.0)                    # committed LM score, LM context, partial-word penalty
        elif pre.endswith(" "):                          # "... W " -> the word W was just completed
            prev = pre[:-1]
            head = prev[:prev.rfind(" ") + 1]            # text before W ("" or ending with a space)
            w = prev[len(head):]
            base, ctx, _ = self._lm(head, cache)
            s = self.alpha * self.lm.log10p(ctx, w) * LN10 + self.beta
            r = (base + s, (ctx + (w,))[-max(1, self.lm.order - 1):], 0.0)
        else:                                            # "... par" -> partial word "par"
            head = pre[:pre.rfind(" ") + 1]
            part = pre[len(head):]
            base, ctx, _ = self._lm(head, cache)
            pen = 0.0 if part in self.lm.prefixes else self.alpha * UNK_LOG10 * LN10
            r = (base, ctx, pen)
        cache[pre] = r
        return r

    def _final_lm(self, pre: str, cache: dict) -> float:
        """LM score at the end of the audio: complete the last word, then </s>."""
        if self.lm is None:
            return 0.0
        p = pre.rstrip(" ")
        if not p:
            return 0.0
        full, ctx, _ = self._lm(p + " ", cache)
        return full + self.alpha * self.lm.log10p(ctx, "</s>") * LN10

    def decode(self, logp: np.ndarray) -> str:
        """logp: (T, V) natural-log CTC posteriors (log_softmax output)."""
        if self.lm is None:
            ids = logp.argmax(-1); out, prev = [], -1
            for k in ids:
                if k != prev and k != 0:
                    out.append(self.vocab[k])
                prev = k
            return " ".join("".join(out).split())
        V = logp.shape[1]
        cache = {}
        beams = {"": [0.0, NEG]}                         # prefix -> [log p ending in blank, log p ending in non-blank]
        cid = {c: i for i, c in enumerate(self.vocab)}
        for t in range(logp.shape[0]):
            row = logp[t]
            lpb = float(row[0])
            nxt = {}
            if lpb > self.blank_skip_logp:               # blank-dominated frame: no new prefixes
                for pre, (pb, pnb) in beams.items():
                    nb = _lse(pb, pnb) + lpb
                    nn_ = pnb + float(row[cid[pre[-1]]]) if pre else NEG
                    nxt[pre] = [nb, nn_]
            else:
                cand = [c for c in range(1, V) if row[c] > self.token_min_logp]
                for pre, (pb, pnb) in beams.items():
                    tot = _lse(pb, pnb)
                    e = nxt.setdefault(pre, [NEG, NEG])
                    e[0] = _lse(e[0], tot + lpb)
                    last = cid[pre[-1]] if pre else -1
                    if last >= 0:
                        e[1] = _lse(e[1], pnb + float(row[last]))
                    for c in cand:
                        lc = float(row[c])
                        if c == self.space and (not pre or pre.endswith(" ")):
                            e[0] = _lse(e[0], tot + lc)      # leading / double space: no new prefix
                            continue
                        new = pre + self.vocab[c]
                        ne = nxt.setdefault(new, [NEG, NEG])
                        ne[1] = _lse(ne[1], (pb if c == last else tot) + lc)
            # prune
            scored = []
            for pre, (pb, pnb) in nxt.items():
                base, _, pen = self._lm(pre, cache)
                scored.append((_lse(pb, pnb) + base + pen, pre))
            scored.sort(reverse=True)
            beams = {pre: nxt[pre] for _, pre in scored[:self.beam]}
        best, best_s = "", NEG
        for pre, (pb, pnb) in beams.items():
            s = _lse(pb, pnb) + self._final_lm(pre, cache)
            if s > best_s:
                best, best_s = pre, s
        return " ".join(best.split())
'''


# ####################################################################################################################
# ====  PART 2: official LibriSpeech test-clean WER of the DEPLOYED app (Zipformer-S + app LM decoder, alpha/beta from lm_config.json)  ====
# ####################################################################################################################
import gc, multiprocessing as mp
import torch
ROOT = "/content/drive/MyDrive/MT2KD"
DEPLOY_DRIVE = f"{ROOT}/deploy/mt2kd_S20_main"                       # the exported app folder (read; only lm/lm_config.json gets the result added)
OUT = f"{ROOT}/results/LM_app_decoder_test_clean"                    # NEW folder: report + per-utterance CSV
LOCAL = Path("/content/app_deploy_S20"); APPDIR = Path("/content/mt2kd_app_backend")
NSTEP = 5; NPROC = max(1, os.cpu_count() or 1); T0 = time.time()
def log(*a): print(*a, flush=True)
def STEP(n, text): log("\n" + "=" * 112 + f"\n▶ STEP {n}/{NSTEP}  {text}\n" + "=" * 112)
def DONE(n, text): log(f"✅ STEP {n} COMPLETED: {text}")

STEP(1, "Google Drive + the deployed app folder -> local disk")
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive as _gd; _gd.mount("/content/drive")
MA.CFG["drive_api"] = False
need = ["model.pt", "labels.json", "zipformer_src.zip", "lm/3-gram.pruned.1e-7.arpa.gz", "lm/lm_config.json"]
for f in need: log(f"  {'found  ' if Path(DEPLOY_DRIVE, f).exists() else 'MISSING'} {DEPLOY_DRIVE}/{f}")
if not all(Path(DEPLOY_DRIVE, f).exists() for f in need): raise SystemExit("run MT2KD_EXPORT_ZIPFORMER_S_CELL.py first")
shutil.rmtree(LOCAL, ignore_errors=True); shutil.copytree(DEPLOY_DRIVE, LOCAL)
Path(OUT).mkdir(parents=True, exist_ok=True)
DONE(1, f"copied | {NPROC} CPU cores")

STEP(2, "load the app exactly as the web server does (app's own backend code, model + LM)")
APPDIR.mkdir(parents=True, exist_ok=True)
for _n, _s in APP_SOURCES.items(): (APPDIR / f"{_n}.py").write_text(_s)
if str(APPDIR) not in sys.path: sys.path.insert(0, str(APPDIR))
import mt2kd_infer, ctc_lm
APP = mt2kd_infer.MT2KD(LOCAL, threads=NPROC)
if APP.lm_decoder is None: raise SystemExit(f"LM not loaded: {APP.lm_error}")
C = APP.lm_config; A_, B_, BEAM = C["alpha"], C["beta"], C.get("beam", 64)
log(f"  {APP.model_name}, {APP.params / 1e6:.2f}M parameters | LM alpha {A_}, beta {B_}, beam {BEAM} (hotwords: none on Drive -> LibriSpeech LM only)")
DONE(2, "app loaded")

STEP(3, "LibriSpeech test-clean (official test set) -> CTC posteriors, same segmentation as the app (> 30 s cut at quiet points)")
TEST = MA.inventory("test-clean").reset_index(drop=True); MA.materialize(TEST, "test-clean")
LP, GREEDY, NSEG = {}, [], 0; t0 = time.time()
with torch.no_grad():
    for i, r in enumerate(TEST.itertuples()):
        w = MA.load_audio(r.relpath, int(r.n_samples)); segs = mt2kd_infer.split_at_quiet_points(w, 30.0); NSEG += len(segs) > 1; g = []
        for k, (a, b) in enumerate(segs):
            enc, el, _, _ = APP._encode(w[a:b]); lg = APP.heads["ctc"](enc.repeat_interleave(2, dim=1))[0, :int(el[0]) * 2]
            g.append(APP._greedy(lg.argmax(-1).tolist())); LP[(i, k)] = lg.float().log_softmax(-1).numpy()
        GREEDY.append(" ".join(t for t in g if t))
        if (i + 1) % 500 == 0: log(f"    {i + 1:,}/{len(TEST):,} utterances | {time.time() - t0:.0f} s")
RG = MA.error_rates(list(TEST.text), GREEDY)
log(f"  greedy: WER {RG['wer'] * 100:.2f}% CER {RG['cer'] * 100:.2f}% (training report 43.36%, LM run 43.19%) | {NSEG} utterances split | {time.time() - t0:.0f} s")
DONE(3, f"{len(TEST):,} utterances, {len(LP):,} segments")

STEP(4, f"LM decoding with the app decoder (ctc_lm.py), {NPROC} processes")
_G = dict(LM=APP.lm_decoder.lm, LP=LP, V=APP.vocab)
def _dec(key): return key, ctc_lm.Decoder(_G["V"], _G["LM"], alpha=A_, beta=B_, beam=BEAM).decode(_G["LP"][key])
t0 = time.time(); RES = {}
with mp.get_context("fork").Pool(NPROC) as pool:
    for j, (key, h) in enumerate(pool.imap_unordered(_dec, list(LP), chunksize=8)):
        RES[key] = h
        if (j + 1) % 500 == 0: log(f"    {j + 1:,}/{len(LP):,} segments | {time.time() - t0:.0f} s")
LMH = [" ".join(t for t in (RES[(i, k)] for k in range(sum(1 for kk in LP if kk[0] == i))) if t) for i in range(len(TEST))]
RL = MA.error_rates(list(TEST.text), LMH)
log(f"  WITH LM: WER {RL['wer'] * 100:.2f}% CER {RL['cer'] * 100:.2f}% | greedy {RG['wer'] * 100:.2f}% -> relative reduction {(1 - RL['wer'] / RG['wer']) * 100:.1f}% | "
    f"{time.time() - t0:.0f} s ({(time.time() - t0) * NPROC / len(TEST):.2f} s per utterance on 1 core)")
DONE(4, f"test-clean {RG['wer'] * 100:.2f}% -> {RL['wer'] * 100:.2f}% with the LM")

STEP(5, "save: report + per-utterance CSV; add the test-clean result to the deployed lm_config.json")
pd.DataFrame(dict(uid=TEST.uid, ref=TEST.text, greedy=GREEDY, with_lm=LMH)).to_csv(f"{OUT}/test_clean_per_utterance.csv", index=False)
REP = dict(model=APP.model_name, parameters=APP.params, decoder="ctc_lm (web app)", alpha=A_, beta=B_, beam=BEAM, hotwords="none",
           test_clean=dict(utterances=len(TEST), greedy_wer=RG["wer"], greedy_cer=RG["cer"], lm_wer=RL["wer"], lm_cer=RL["cer"]),
           tuned_on=C.get("tuned_on"), created=time.strftime("%Y-%m-%d %H:%M:%S"))
json.dump(REP, open(f"{OUT}/test_clean_report.json", "w"), indent=1)
cfg = json.load(open(f"{DEPLOY_DRIVE}/lm/lm_config.json")); cfg["test_clean"] = REP["test_clean"]
json.dump(cfg, open(f"{DEPLOY_DRIVE}/lm/lm_config.json.part", "w"), indent=1); os.replace(f"{DEPLOY_DRIVE}/lm/lm_config.json.part", f"{DEPLOY_DRIVE}/lm/lm_config.json")
log(f"  {OUT}/test_clean_report.json\n  {OUT}/test_clean_per_utterance.csv\n  {DEPLOY_DRIVE}/lm/lm_config.json (+ test_clean)")
DONE(5, f"ALL DONE in {(time.time() - T0) / 60:.0f} min | test-clean: greedy {RG['wer'] * 100:.2f}% -> LM {RL['wer'] * 100:.2f}%")
